<img src="https://raw.githubusercontent.com/senanurcetin/enveda-casmi26/main/site/assets/nexus-midnight.svg" width="360" alt="NEXUS">

# CASMI26: Purged Family External Validation

The original-snapshot novelty study did not find the required independent population. This is a separately registered experiment: choose external generic families before predictions, remove their spectra and molecular identities from the entire training/reference corpus, rebuild every artifact, fit every supervised weight from scratch, then seal both candidate budgets and the development choice before test features. Historical test families remain protected. Generalization is relative to this purged training snapshot. No original-snapshot novelty, independent laboratory replication, paid compute, service deployment or competition submission is claimed.


In [ ]:
import hashlib,importlib.metadata,json,os,re,subprocess,sys,time
from pathlib import Path
sources=json.loads('{"src/casmi/__init__.py": "", "src/casmi/channels/__init__.py": "", "src/casmi/channels/analog.py": "\\"\\"\\"D2 \\u2014 mass-shifted analog propagation.\\n\\nA Class-2 molecule has no reference spectrum, but its structural relatives usually do. Relatives\\nthat differ by one substituent fragment into largely the same ions, either at the same m/z (the\\nsubstituent was lost) or shifted by the precursor difference (it was retained). We find such\\nanalog spectra in two stages and propagate their structures onto the candidates:\\n\\n  1. prefilter: every library spectrum is a sparse vector over fragment-m/z bins and\\n     neutral-loss bins; one sparse mat-vec scores a query spectrum against the whole library;\\n  2. refine: the top hits are rescored with mass-shifted entropy similarity;\\n  3. propagate: candidate score = max over analogs a of  sim(a)^p \\u00b7 Tanimoto(fp_c, fp_a).\\n\\"\\"\\"\\nfrom dataclasses import asdict, dataclass\\nfrom pathlib import Path\\nimport hashlib\\nimport json\\nimport zipfile\\n\\nimport numpy as np\\nimport pandas as pd\\nimport scipy.sparse as sp\\n\\nfrom casmi.spectra.clean import CleanConfig, clean_peaks_fast\\nfrom casmi.channels.prefilter import eligible_top_k\\nfrom casmi.channels.reference_selection import select_reference_rows, SELECTION_VERSION\\nfrom casmi.spectra.similarity import entropy_similarity\\n\\nBIN_DA = 0.05\\nMAX_MZ = 1300.0\\nN_BINS = int(MAX_MZ / BIN_DA) + 1\\n\\n\\n@dataclass(frozen=True)\\nclass AnalogConfig:\\n    max_shift_da: float = 200.0\\n    prefilter_top: int = 200\\n    keep_structures: int = 100\\n    per_structure_adduct: int = 3\\n    reference_policy: str = \\"legacy\\"  # opt-in: source_balanced (requires new index cache)\\n    tol_da: float = 0.02\\n    powers: tuple = (1.0, 2.0, 4.0)\\n    clean: CleanConfig = CleanConfig(min_rel_intensity=0.01, top_k_per_window=6, window_da=50, max_peaks=64)\\n\\n\\ndef _vector(mz, it, precursor_mz):\\n    \\"\\"\\"Sparse (indices, values) over [fragment bins | neutral-loss bins], L2-normalised.\\"\\"\\"\\n    w = np.sqrt(it.astype(np.float64))\\n    frag = np.minimum((mz / BIN_DA).astype(np.int64), N_BINS - 1)\\n    loss_mz = precursor_mz - mz\\n    ok = loss_mz > 0.5\\n    loss = N_BINS + np.minimum((loss_mz[ok] / BIN_DA).astype(np.int64), N_BINS - 1)\\n    idx = np.concatenate([frag, loss])\\n    val = np.concatenate([w, w[ok]])\\n    # duplicate bins (two peaks in one bin) are summed by the CSR constructor\\n    norm = np.sqrt((val ** 2).sum())\\n    return idx, val / norm if norm > 0 else val\\n\\n\\nclass AnalogIndex:\\n    def __init__(self, store, key_of_row, cfg: AnalogConfig = AnalogConfig(), cache: Path | None = None):\\n        self.store, self.cfg = store, cfg\\n        self._clean_cache = {}\\n        meta = store.meta\\n        expected = None\\n        stamp = None\\n        if cfg.reference_policy != \\"legacy\\":\\n            # A policy-specific cache never overwrites the frozen leaderboard\\n            # baseline\'s analog_index.npz. Nonlegacy loads are fail-closed.\\n            expected = self._select(meta, key_of_row)\\n            stamp = self._cache_stamp(store, key_of_row, expected)\\n            if cache is not None:\\n                path = Path(cache)\\n                cache = path.with_name(\\n                    f\\"{path.stem}_{cfg.reference_policy}_v1{path.suffix}\\"\\n                )\\n        loaded = False\\n        if cache is not None and Path(cache).exists():\\n            try:\\n                with np.load(cache, allow_pickle=False) as z:\\n                    if expected is not None and (\\n                        str(z[\\"selection_version\\"].item()) != SELECTION_VERSION\\n                        or str(z[\\"selection_digest\\"].item()) != stamp\\n                        or not np.array_equal(z[\\"rows\\"], expected)\\n                    ):\\n                        raise ValueError(\\"Cached D2 selection provenance mismatch\\")\\n                    cached_rows = z[\\"rows\\"]\\n                    cached_matrix = sp.csr_matrix(\\n                        (z[\\"data\\"], z[\\"indices\\"], z[\\"indptr\\"]),\\n                        shape=tuple(z[\\"shape\\"]),\\n                    )\\n                    if cached_matrix.shape != (len(cached_rows), 2 * N_BINS):\\n                        raise ValueError(\\"Cached D2 matrix shape is invalid\\")\\n                    if len(cached_rows) and (cached_rows.min() < 0 or cached_rows.max() >= len(meta)):\\n                        raise ValueError(\\"Cached D2 reference row index out of range\\")\\n                    self.rows, self.matrix = cached_rows, cached_matrix\\n                    loaded = True\\n            except (KeyError, OSError, ValueError, zipfile.BadZipFile):\\n                if expected is None:\\n                    raise  # fail closed on malformed frozen legacy caches\\n        if not loaded:\\n            self.rows = expected if expected is not None else self._select(meta, key_of_row)\\n            self.matrix = self._build(self.rows)\\n            if cache is not None:\\n                output = dict(rows=self.rows, data=self.matrix.data,\\n                              indices=self.matrix.indices, indptr=self.matrix.indptr,\\n                              shape=np.array(self.matrix.shape))\\n                if expected is not None:\\n                    output.update(selection_version=np.array(SELECTION_VERSION),\\n                                  selection_digest=np.array(stamp))\\n                np.savez(cache, **output)\\n        # Legacy and selection-validated caches are useful for exploratory\\n        # runs but cannot independently attest a freshly rebuilt D2 matrix.\\n        # Only cache=None sets the scientifically stronger fresh-origin bit.\\n        self.cache_origin = (\\n            \\"recomputed_without_cache\\" if cache is None\\n            else \\"loaded_cache\\" if loaded else \\"built_with_cache\\"\\n        )\\n        self.key = key_of_row[self.rows]\\n        self.prec = meta.precursor_mz.to_numpy()[self.rows]\\n        self.adduct = meta.adduct.astype(str).to_numpy()[self.rows]\\n        self.nmass = meta.neutral_mass.to_numpy()[self.rows]\\n\\n    def fresh_index_provenance(self):\\n        \\"\\"\\"Content digest for a D2 matrix rebuilt from the current store.\\n\\n        Only callable on a cache-free index. The digest binds selected rows,\\n        sparse matrix bytes and its configuration to the run record.\\n        It is a reproducibility checksum, NOT an external dataset signature.\\n        \\"\\"\\"\\n        if self.cache_origin != \\"recomputed_without_cache\\":\\n            raise ValueError(\\n                \\"D2 index was not freshly rebuilt; cached indices cannot \\"\\n                \\"support a decision-grade provenance record\\"\\n            )\\n        digest = hashlib.sha256(b\\"casmi-d2-fresh-index-v1\\\\\\\\n\\")\\n        cfg = json.dumps(asdict(self.cfg), sort_keys=True, separators=(\\",\\", \\":\\"))\\n        digest.update(cfg.encode(\\"utf-8\\"))\\n        for field in (\\n            self.rows, np.asarray(self.matrix.shape, dtype=np.int64),\\n            self.matrix.indptr, self.matrix.indices, self.matrix.data,\\n        ):\\n            array = np.ascontiguousarray(field)\\n            digest.update(str(array.dtype).encode(\\"ascii\\"))\\n            digest.update(np.asarray(array.shape, dtype=np.int64).tobytes())\\n            blob = memoryview(array).cast(\\"B\\")\\n            for start in range(0, len(blob), 1024 * 1024):\\n                digest.update(blob[start:start + 1024 * 1024])\\n        return {\\n            \\"schema\\": \\"casmi-d2-fresh-index-v1\\",\\n            \\"status\\": \\"REBUILT_WITHOUT_CACHE\\",\\n            \\"index_sha256\\": digest.hexdigest(),\\n            \\"n_references\\": int(self.matrix.shape[0]),\\n            \\"n_nonzero\\": int(self.matrix.nnz),\\n            \\"reference_policy\\": self.cfg.reference_policy,\\n            \\"cache_used\\": False,\\n        }\\n\\n    def _select(self, meta, key_of_row):\\n        return select_reference_rows(\\n            meta, key_of_row,\\n            per_structure_adduct=self.cfg.per_structure_adduct,\\n            policy=self.cfg.reference_policy,\\n        )\\n\\n    def _cache_stamp(self, store, keys, rows):\\n        \\"\\"\\"Hash selected metadata, source-array file stamps and index config.\\n\\n        The selected rows are recomputed on every nonlegacy cache load, so\\n        updated eligibility or changes in available libraries invalidate the\\n        cached index even if the number of spectra is unchanged.\\n        \\"\\"\\"\\n        digest = hashlib.sha256()\\n        digest.update(SELECTION_VERSION.encode())\\n        digest.update(json.dumps(asdict(self.cfg), sort_keys=True).encode())\\n        digest.update(np.asarray(rows, dtype=np.int64).tobytes())\\n        selected = store.meta.iloc[rows][[\\n            \\"ingest_lib\\", \\"adduct\\", \\"precursor_mz\\", \\"neutral_mass\\",\\n            \\"ppm\\", \\"n_peaks\\", \\"is_timstof\\", \\"test_adduct\\",\\n        ]]\\n        digest.update(pd.util.hash_pandas_object(selected, index=False).values.tobytes())\\n        digest.update(pd.util.hash_pandas_object(\\n            pd.Series(np.asarray(keys)[rows], dtype=\\"string\\"), index=False,\\n        ).values.tobytes())\\n        for field in (\\"mz\\", \\"it\\", \\"offsets\\"):\\n            ref = getattr(store, field, None)\\n            filename = getattr(ref, \\"filename\\", None)\\n            if filename is None:\\n                continue\\n            stat = Path(filename).stat()\\n            digest.update(\\n                f\\"{field}:{stat.st_size}:{stat.st_mtime_ns}\\".encode()\\n            )\\n        return digest.hexdigest()\\n\\n    def _clean(self, row):\\n        hit = self._clean_cache.get(row)\\n        if hit is None:\\n            mz, it = self.store.peaks(row)\\n            hit = clean_peaks_fast(mz, it, float(self.store.meta.precursor_mz.iat[row]), self.cfg.clean)\\n            if len(self._clean_cache) < 300_000:\\n                self._clean_cache[row] = hit\\n        return hit\\n\\n    def _build(self, rows):\\n        if len(rows) == 0:\\n            return sp.csr_matrix((0, 2 * N_BINS), dtype=np.float32)\\n        indptr, indices, data = [0], [], []\\n        prec = self.store.meta.precursor_mz.to_numpy()\\n        for r in rows:\\n            mz, it = self._clean(r)\\n            i, v = _vector(mz, it, prec[r]) if mz.size else (np.empty(0, np.int64), np.empty(0))\\n            indices.append(i)\\n            data.append(v.astype(np.float32))\\n            indptr.append(indptr[-1] + i.size)\\n        self._clean_cache.clear()\\n        m = sp.csr_matrix((np.concatenate(data), np.concatenate(indices), np.array(indptr)),\\n                          shape=(len(rows), 2 * N_BINS))\\n        m.sum_duplicates()\\n        return m\\n\\n    def search(self, query_spectra, query_mass, excluded_rows=None):\\n        \\"\\"\\"Best refined similarity per analog structure: dict key -> (sim, analog neutral mass).\\"\\"\\"\\n        best = {}\\n        # A structure-level knockout must be applied BEFORE selecting the top\\n        # analog hits. Previously self-references displaced allowed hits in the\\n        # shortlist and were discarded only afterwards.\\n        excluded = (np.asarray(excluded_rows, dtype=np.int64)\\n                    if excluded_rows is not None else np.empty(0, np.int64))\\n        not_excluded = (~np.isin(self.rows, excluded)\\n                        if excluded.size else np.ones(len(self.rows), dtype=bool))\\n        mass_ok = np.isfinite(self.nmass) & np.isfinite(query_mass) & (\\n            np.abs(self.nmass - query_mass) <= self.cfg.max_shift_da\\n        )\\n        for q in query_spectra:\\n            if q[\\"mz_c\\"].size == 0:\\n                continue\\n            i, v = _vector(q[\\"mz_c\\"], q[\\"it_c\\"], q[\\"precursor_mz\\"])\\n            # Dense CSR mat-vec avoids a large sparse x sparse temporary.\\n            dq = np.zeros(2 * N_BINS, np.float32)\\n            np.add.at(dq, i, v.astype(np.float32))\\n            scores = self.matrix @ dq\\n            allowed = not_excluded & mass_ok & (self.adduct == q[\\"adduct\\"])\\n            top = eligible_top_k(scores, allowed, self.cfg.prefilter_top)\\n            for t in top:\\n                rm, ri = self._clean(self.rows[t])\\n                sim = entropy_similarity(q[\\"mz_c\\"], q[\\"it_c\\"], rm, ri, self.cfg.tol_da,\\n                                         float(q[\\"precursor_mz\\"] - self.prec[t]))\\n                k = self.key[t]\\n                if sim > best.get(k, (0.0, 0.0))[0]:\\n                    best[k] = (sim, float(self.nmass[t]))\\n        items = sorted(best.items(), key=lambda kv: -kv[1][0])[: self.cfg.keep_structures]\\n        return dict(items)\\n\\n\\ndef tanimoto_matrix(fa, fb):\\n    \\"\\"\\"Tanimoto between packed-bit fingerprint rows: fa [n, 256] vs fb [m, 256] -> [n, m].\\"\\"\\"\\n    a = np.unpackbits(fa, axis=1).astype(np.float32)\\n    b = np.unpackbits(fb, axis=1).astype(np.float32)\\n    inter = a @ b.T\\n    return inter / (a.sum(1)[:, None] + b.sum(1)[None, :] - inter + 1e-9)\\n\\n\\ndef propagate(analogs: dict, cand_rows, cand_mass, fps, row_of, powers):\\n    \\"\\"\\"Per-candidate analog features. `analogs`: key -> (sim, analog neutral mass).\\"\\"\\"\\n    n = len(cand_rows)\\n    feats = {f\\"an_p{int(p)}\\": np.zeros(n) for p in powers}\\n    feats.update(an_sim=np.zeros(n), an_tani=np.zeros(n), an_dmass=np.full(n, 999.0),\\n                 an_n_close=np.zeros(n), an_self=np.zeros(n))\\n    keys = [k for k in analogs if k in row_of]\\n    if not keys or n == 0:\\n        return feats\\n    sims = np.array([analogs[k][0] for k in keys])\\n    amass = np.array([analogs[k][1] for k in keys])\\n    T = tanimoto_matrix(fps[cand_rows], fps[[row_of[k] for k in keys]])\\n    for p in powers:\\n        feats[f\\"an_p{int(p)}\\"] = (T * sims[None, :] ** p).max(axis=1)\\n    best = (T * sims[None, :] ** 4).argmax(axis=1)\\n    feats[\\"an_sim\\"] = sims[best]\\n    feats[\\"an_tani\\"] = T[np.arange(n), best]\\n    feats[\\"an_dmass\\"] = cand_mass - amass[best]\\n    feats[\\"an_n_close\\"] = ((T >= 0.7) & (sims[None, :] >= 0.5)).sum(axis=1)\\n    feats[\\"an_self\\"] = np.where(T >= 0.999, sims[None, :], 0).max(axis=1)\\n    return feats\\n", "src/casmi/channels/bondmodel.py": "\\"\\"\\"D5 \\u2014 learned bond-cleavage likelihood.\\n\\nD4 counts how much of the spectrum a candidate *can* explain, which favours candidates that\\nfragment into many pieces. D5 asks the reverse question as well: of the cleavages this candidate\\n*should* show, which do we see? Each cleavage event (one acyclic bond, or two bonds of the same\\nring) has a type; its probability of producing an observed fragment ion is learned from library\\nspectra of training structures. A candidate\'s score is the Bernoulli log-likelihood of the\\nobserved/unobserved pattern of its events, averaged over events and spectra.\\n\\"\\"\\"\\nimport numpy as np\\nfrom rdkit import Chem\\n\\nfrom casmi.chem.masses import ELECTRON, PROTON\\n\\nH = 1.00782503207\\nNA = 22.9897692820\\nTOL_DA, TOL_PPM = 0.01, 10.0\\nMIN_REL = 0.01\\n\\n\\ndef _atom_masses(mol):\\n    pt = Chem.GetPeriodicTable()\\n    return np.array([pt.GetMassForIsotope(a.GetSymbol(), pt.GetMostCommonIsotope(a.GetSymbol()))\\n                     + a.GetTotalNumHs() * H for a in mol.GetAtoms()])\\n\\n\\ndef _bond_key(b):\\n    a1, a2 = b.GetBeginAtom(), b.GetEndAtom()\\n    ends = sorted([(a1.GetSymbol(), a1.GetDegree(), a1.GetIsAromatic()), (a2.GetSymbol(), a2.GetDegree(), a2.GetIsAromatic())])\\n    return f\\"{ends[0][0]}{ends[0][1]}{\'a\' if ends[0][2] else \'\'}-{b.GetBondTypeAsDouble():g}-{ends[1][0]}{ends[1][1]}{\'a\' if ends[1][2] else \'\'}\\"\\n\\n\\ndef _split(mol, bonds, masses):\\n    \\"\\"\\"Masses of the two pieces after removing `bonds`, or None if the molecule stays connected.\\"\\"\\"\\n    em = Chem.RWMol(mol)\\n    for b in bonds:\\n        bd = mol.GetBondWithIdx(b)\\n        em.RemoveBond(bd.GetBeginAtomIdx(), bd.GetEndAtomIdx())\\n    frags = Chem.GetMolFrags(em, asMols=False, sanitizeFrags=False)\\n    if len(frags) != 2:\\n        return None\\n    return tuple(float(masses[list(f)].sum()) for f in frags)\\n\\n\\ndef cleavage_events(smiles):\\n    \\"\\"\\"[(event key, (piece mass 1, piece mass 2))]: acyclic single bonds + bond pairs within a ring.\\"\\"\\"\\n    mol = Chem.MolFromSmiles(smiles)\\n    if mol is None or mol.GetNumAtoms() > 120:\\n        return []\\n    masses = _atom_masses(mol)\\n    out = []\\n    ring_bonds = set()\\n    for ring in mol.GetRingInfo().BondRings():\\n        ring_bonds.update(ring)\\n    for b in mol.GetBonds():\\n        if b.GetIdx() in ring_bonds or b.GetBondTypeAsDouble() > 1.5:\\n            continue\\n        pieces = _split(mol, [b.GetIdx()], masses)\\n        if pieces:\\n            out.append((\\"A:\\" + _bond_key(b), pieces))\\n    for ring in mol.GetRingInfo().BondRings():\\n        if len(ring) > 7 or any(mol.GetBondWithIdx(i).GetIsAromatic() for i in ring):\\n            continue\\n        for i in range(len(ring)):\\n            for j in range(i + 2, len(ring)):\\n                if i == 0 and j == len(ring) - 1:\\n                    continue\\n                pieces = _split(mol, [ring[i], ring[j]], masses)\\n                if pieces:\\n                    k = \\"|\\".join(sorted([_bond_key(mol.GetBondWithIdx(ring[i])), _bond_key(mol.GetBondWithIdx(ring[j]))]))\\n                    out.append((\\"R:\\" + k, pieces))\\n    return out\\n\\n\\ndef _carriers(mode, adduct):\\n    if mode.lower().startswith(\\"n\\") or adduct.endswith(\\"-\\"):\\n        return [-PROTON]\\n    return [PROTON] + ([NA - ELECTRON] if \\"Na\\" in adduct else [])\\n\\n\\ndef observed(events, mz, rel, mode, adduct, precursor_mz):\\n    \\"\\"\\"Per event: 1 if one of its pieces (\\u00b12 H, precursor carrier) matches a peak >= MIN_REL.\\"\\"\\"\\n    keep = (rel >= MIN_REL) & (mz < precursor_mz - 1.5)\\n    mz = np.sort(mz[keep])\\n    obs = np.zeros(len(events), np.int8)\\n    if len(mz) == 0:\\n        return obs\\n    shifts = np.array([c + k * H for c in _carriers(mode, adduct) for k in (-2, -1, 0, 1, 2)])\\n    for e, (_, pieces) in enumerate(events):\\n        ions = (np.array(pieces)[:, None] + shifts[None, :]).ravel()\\n        pos = np.searchsorted(mz, ions)\\n        lo = mz[np.clip(pos - 1, 0, len(mz) - 1)]\\n        hi = mz[np.clip(pos, 0, len(mz) - 1)]\\n        d = np.minimum(np.abs(ions - lo), np.abs(ions - hi))\\n        obs[e] = int((d <= np.maximum(TOL_DA, ions * TOL_PPM * 1e-6)).any())\\n    return obs\\n\\n\\nclass BondModel:\\n    \\"\\"\\"Event-type propensities: P(observed | event key, ion mode), Beta-smoothed toward the global rate.\\"\\"\\"\\n\\n    def __init__(self, counts=None, prior=4.0):\\n        self.counts = counts or {}          # (key, mode) -> [n, observed]\\n        self.prior = prior\\n        self._p = None\\n\\n    def add(self, events, obs, mode):\\n        for (k, _), o in zip(events, obs):\\n            c = self.counts.setdefault((k, mode), [0, 0])\\n            c[0] += 1\\n            c[1] += int(o)\\n\\n    def finalize(self):\\n        n = sum(c[0] for c in self.counts.values())\\n        o = sum(c[1] for c in self.counts.values())\\n        base = o / max(n, 1)\\n        self.base = base\\n        self._p = {k: (c[1] + self.prior * base) / (c[0] + self.prior) for k, c in self.counts.items()}\\n        return self\\n\\n    def p(self, key, mode):\\n        return self._p.get((key, mode), self.base)\\n\\n    def score(self, events, spectra):\\n        \\"\\"\\"(mean log-likelihood per event, observed share weighted by propensity) averaged over spectra.\\"\\"\\"\\n        if not events or not spectra:\\n            return 0.0, 0.0\\n        lls, ws = [], []\\n        for s in spectra:\\n            mode = \\"neg\\" if (str(s[\\"mode\\"]).lower().startswith(\\"n\\") or str(s[\\"adduct\\"]).endswith(\\"-\\")) else \\"pos\\"\\n            it = np.asarray(s[\\"it\\"], float)\\n            rel = it / it.max() if len(it) and it.max() > 0 else it\\n            obs = observed(events, np.asarray(s[\\"mz\\"], float), rel, mode, str(s[\\"adduct\\"]), float(s[\\"precursor_mz\\"]))\\n            p = np.clip(np.array([self.p(k, mode) for k, _ in events]), 1e-3, 1 - 1e-3)\\n            lls.append(float(np.mean(np.where(obs == 1, np.log(p), np.log(1 - p)))))\\n            ws.append(float((p * obs).sum() / p.sum()))\\n        return float(np.mean(lls)), float(np.mean(ws))\\n", "src/casmi/channels/fpinfer.py": "\\"\\"\\"D3 inference shared by validation (scripts/add_d3_features.py) and the submission.\\"\\"\\"\\nimport numpy as np\\nimport torch\\n\\nfrom casmi.chem.masses import TEST_ADDUCTS\\nfrom casmi.spectra.clean import CleanConfig, clean_peaks_fast\\n\\nMAX_PEAKS = 64\\nFP_CLEAN = CleanConfig(min_rel_intensity=0.005, top_k_per_window=8, window_da=50, max_peaks=MAX_PEAKS)\\nADDUCT_ID = {a: i for i, a in enumerate(TEST_ADDUCTS)}\\n\\n\\n@torch.no_grad()\\ndef molecule_logp(models, spectra, device):\\n    \\"\\"\\"Mean predicted bit probability over a molecule\'s spectra and over models.\\n    Returns (log p(bit=1), log p(bit=0)) as NumPy arrays of length n_bits.\\"\\"\\"\\n    n = len(spectra)\\n    mz, it, npk = np.zeros((n, MAX_PEAKS), np.float32), np.zeros((n, MAX_PEAKS), np.float32), np.zeros(n, np.int64)\\n    for j, s in enumerate(spectra):\\n        cm, ci = clean_peaks_fast(s[\\"mz\\"], s[\\"it\\"], s[\\"precursor_mz\\"], FP_CLEAN)\\n        mz[j, :cm.size], it[j, :cm.size], npk[j] = cm, ci, cm.size\\n    t = lambda a, dt=torch.float32: torch.as_tensor(np.asarray(a), dtype=dt, device=device)\\n    x = dict(mz=t(mz), it=t(it), npk=t(npk, torch.long),\\n             prec=t([s[\\"precursor_mz\\"] for s in spectra]),\\n             adduct=t([ADDUCT_ID.get(s[\\"adduct\\"], 0) for s in spectra], torch.long),\\n             ce=t([s[\\"ce_ev\\"] for s in spectra]), positive=t([float(s[\\"mode\\"] == \\"positive\\") for s in spectra]))\\n    p = torch.stack([torch.sigmoid(m(**x)) for m in models]).mean(dim=(0, 1)).clamp(1e-4, 1 - 1e-4)\\n    return torch.log(p).cpu().numpy(), torch.log1p(-p).cpu().numpy()\\n", "src/casmi/channels/fpmodel.py": "\\"\\"\\"D3 \\u2014 spectrum \\u2192 fingerprint model and candidate scoring.\\n\\nA small set transformer reads \\u2264 64 peaks (no positional encoding: peaks are a set) plus one\\nglobal token (precursor mass, adduct, collision energy, polarity) and predicts the 2048-bit\\nMorgan fingerprint used by the pool, so a prediction can be scored directly against candidates.\\n\\nm/z values need ~0.01 Da resolution over a 1,000 Da range, which a single linear projection cannot\\nrepresent; they are encoded with sinusoids at log-spaced wavelengths (as in DreaMS / MIST).\\nEach peak is described both by its m/z and by its neutral loss from the precursor.\\n\\"\\"\\"\\nfrom dataclasses import asdict, dataclass\\n\\nimport numpy as np\\nimport torch\\nimport torch.nn as nn\\n\\nN_ADDUCTS = 10\\n\\n\\n@dataclass\\nclass FPModelConfig:\\n    n_bits: int = 2048\\n    d_model: int = 256\\n    n_layers: int = 4\\n    n_heads: int = 8\\n    d_ff: int = 1024\\n    dropout: float = 0.1\\n    n_freq: int = 32\\n    min_wavelength: float = 0.01\\n    max_wavelength: float = 1000.0\\n\\n\\nclass FourierMass(nn.Module):\\n    def __init__(self, n_freq, lo, hi):\\n        super().__init__()\\n        wl = torch.logspace(np.log10(lo), np.log10(hi), n_freq)\\n        self.register_buffer(\\"w\\", 2 * np.pi / wl)\\n\\n    def forward(self, x):                      # x [..., ] -> [..., 2 * n_freq]\\n        a = x.unsqueeze(-1) * self.w\\n        return torch.cat([torch.sin(a), torch.cos(a)], dim=-1)\\n\\n\\nclass FPModel(nn.Module):\\n    def __init__(self, cfg: FPModelConfig = FPModelConfig()):\\n        super().__init__()\\n        self.cfg = cfg\\n        f = 2 * cfg.n_freq\\n        self.enc = FourierMass(cfg.n_freq, cfg.min_wavelength, cfg.max_wavelength)\\n        self.peak_proj = nn.Sequential(nn.Linear(2 * f + 2, cfg.d_model), nn.GELU(), nn.Linear(cfg.d_model, cfg.d_model))\\n        self.adduct_emb = nn.Embedding(N_ADDUCTS, 32)\\n        self.glob_proj = nn.Sequential(nn.Linear(f + 32 + 2, cfg.d_model), nn.GELU(), nn.Linear(cfg.d_model, cfg.d_model))\\n        layer = nn.TransformerEncoderLayer(cfg.d_model, cfg.n_heads, cfg.d_ff, cfg.dropout,\\n                                           activation=\\"gelu\\", batch_first=True, norm_first=True)\\n        self.encoder = nn.TransformerEncoder(layer, cfg.n_layers, enable_nested_tensor=False)\\n        self.head = nn.Sequential(nn.LayerNorm(cfg.d_model), nn.Linear(cfg.d_model, cfg.d_model),\\n                                  nn.GELU(), nn.Linear(cfg.d_model, cfg.n_bits))\\n\\n    def forward(self, mz, it, npk, prec, adduct, ce, positive):\\n        \\"\\"\\"mz, it [B, P]; npk [B]; prec, ce, positive [B]; adduct [B] (long). Returns logits [B, n_bits].\\"\\"\\"\\n        B, P = mz.shape\\n        pad = torch.arange(P, device=mz.device)[None, :] >= npk[:, None]\\n        loss = (prec[:, None] - mz).clamp(min=0)\\n        peak = torch.cat([self.enc(mz), self.enc(loss), it.sqrt().unsqueeze(-1), torch.log1p(it * 100).unsqueeze(-1)], -1)\\n        tok = self.peak_proj(peak)\\n        ce_n = torch.nan_to_num(ce, nan=30.0) / 100.0\\n        glob = torch.cat([self.enc(prec), self.adduct_emb(adduct), ce_n[:, None], positive[:, None]], -1)\\n        x = torch.cat([self.glob_proj(glob)[:, None, :], tok], dim=1)\\n        mask = torch.cat([torch.zeros(B, 1, dtype=torch.bool, device=mz.device), pad], dim=1)\\n        h = self.encoder(x, src_key_padding_mask=mask)\\n        return self.head(h[:, 0])\\n\\n\\ndef save(model, path, extra=None):\\n    torch.save({\\"cfg\\": asdict(model.cfg), \\"state\\": model.state_dict(), **(extra or {})}, path)\\n\\n\\ndef load(path, device=\\"cpu\\"):\\n    ck = torch.load(path, map_location=device, weights_only=False)\\n    m = FPModel(FPModelConfig(**ck[\\"cfg\\"])).to(device)\\n    m.load_state_dict(ck[\\"state\\"])\\n    return m.eval()\\n\\n\\ndef candidate_scores(logp_on, logp_off, cand_bits):\\n    \\"\\"\\"Bernoulli log-likelihood of each candidate fingerprint under the predicted bit probabilities.\\n\\n    logp_on / logp_off: [n_bits] molecule-level log p(bit=1) / log p(bit=0)\\n    cand_bits: [n_cand, n_bits] {0,1}\\n    Returns (mean log-likelihood per bit, cosine between probabilities and candidate bits).\\n    \\"\\"\\"\\n    ll = (cand_bits * logp_on + (1 - cand_bits) * logp_off).mean(axis=1)\\n    p = np.exp(logp_on)\\n    cos = (cand_bits @ p) / (np.linalg.norm(p) * np.sqrt(cand_bits.sum(axis=1)) + 1e-9)\\n    return ll, cos\\n", "src/casmi/channels/fragments.py": "\\"\\"\\"D4 \\u2014 fragment explanation: how much of the query spectrum a candidate structure can explain.\\n\\nCandidates in one mass window are mostly isomers, so formula-level evidence cannot separate them;\\nbond topology can. Every candidate is broken at one or two bonds (combinatorial fragmentation in\\nthe spirit of MetFrag, depth 2), every resulting connected piece becomes a neutral fragment mass,\\nand each fragment is ionised with the precursor\'s charge carrier and \\u00b12 hydrogen rearrangements.\\nA query peak is explained when some fragment ion lies within tolerance.\\n\\nFeatures per candidate (averaged and maxed over the molecule\'s spectra):\\n  d4_int   share of fragment-peak intensity explained\\n  d4_pk    share of fragment peaks explained\\nPeaks within 1.5 Da of the precursor are ignored (every candidate explains the precursor).\\n\\"\\"\\"\\nimport numpy as np\\nfrom numba import njit\\n\\nfrom casmi.chem.masses import ELECTRON, PROTON\\nfrom casmi.spectra.clean import CleanConfig, clean_peaks_fast\\n\\nH = 1.00782503207\\nNA = 22.9897692820\\nFRAG_CLEAN = CleanConfig(max_above_precursor=-1.5, min_rel_intensity=0.01, max_peaks=60)\\nTOL_DA, TOL_PPM = 0.01, 10.0\\nMAX_ATOMS = 160\\n\\n\\n@njit(cache=True)\\ndef _find(parent, a):\\n    while parent[a] != a:\\n        parent[a] = parent[parent[a]]\\n        a = parent[a]\\n    return a\\n\\n\\n@njit(cache=True)\\ndef _pieces(n, atom_mass, bu, bv, skip1, skip2, out, n_out, total):\\n    parent = np.arange(n)\\n    for b in range(len(bu)):\\n        if b == skip1 or b == skip2:\\n            continue\\n        ra, rb = _find(parent, bu[b]), _find(parent, bv[b])\\n        if ra != rb:\\n            parent[ra] = rb\\n    acc = np.zeros(n)\\n    for a in range(n):\\n        acc[_find(parent, a)] += atom_mass[a]\\n    for a in range(n):\\n        m = acc[a]\\n        if m > 0.0 and m < total - 1e-6 and n_out < len(out):\\n            out[n_out] = m\\n            n_out += 1\\n    return n_out\\n\\n\\n@njit(cache=True)\\ndef _fragments(atom_mass, bu, bv):\\n    n, m = len(atom_mass), len(bu)\\n    total = atom_mass.sum()\\n    out = np.empty(3 * (m + m * (m - 1) // 2) + 1)\\n    k = 0\\n    for i in range(m):\\n        k = _pieces(n, atom_mass, bu, bv, i, -1, out, k, total)\\n        for j in range(i + 1, m):\\n            k = _pieces(n, atom_mass, bu, bv, i, j, out, k, total)\\n    return out[:k]\\n\\n\\n@njit(cache=True)\\ndef _fragments3(atom_mass, bu, bv):\\n    \\"\\"\\"Depth 3: every bond triple as well (breaks rings in two places plus one more cut).\\"\\"\\"\\n    n, m = len(atom_mass), len(bu)\\n    total = atom_mass.sum()\\n    base = _fragments(atom_mass, bu, bv)\\n    out = np.empty(len(base) + 4 * (m * (m - 1) * (m - 2) // 6) + 1)\\n    out[:len(base)] = base\\n    k = len(base)\\n    for i in range(m):\\n        for j in range(i + 1, m):\\n            for l in range(j + 1, m):\\n                parent = np.arange(n)\\n                for b in range(m):\\n                    if b == i or b == j or b == l:\\n                        continue\\n                    ra, rb = _find(parent, bu[b]), _find(parent, bv[b])\\n                    if ra != rb:\\n                        parent[ra] = rb\\n                acc = np.zeros(n)\\n                for a in range(n):\\n                    acc[_find(parent, a)] += atom_mass[a]\\n                for a in range(n):\\n                    if acc[a] > 0.0 and acc[a] < total - 1e-6 and k < len(out):\\n                        out[k] = acc[a]\\n                        k += 1\\n    return out[:k]\\n\\n\\nMAX_BONDS_DEPTH3 = 70\\n\\n\\ndef fragment_masses(mol, depth=2):\\n    \\"\\"\\"Sorted unique neutral fragment masses of an RDKit molecule (heavy atoms carry their Hs).\\"\\"\\"\\n    n = mol.GetNumAtoms()\\n    if n < 2 or n > MAX_ATOMS:\\n        return np.empty(0)\\n    atom_mass = np.array([_mono(a) for a in mol.GetAtoms()])\\n    bu = np.array([b.GetBeginAtomIdx() for b in mol.GetBonds()], np.int64)\\n    bv = np.array([b.GetEndAtomIdx() for b in mol.GetBonds()], np.int64)\\n    if len(bu) == 0:\\n        return np.empty(0)\\n    f = _fragments3(atom_mass, bu, bv) if depth >= 3 and len(bu) <= MAX_BONDS_DEPTH3 else _fragments(atom_mass, bu, bv)\\n    return np.unique(np.round(f, 4))\\n\\n\\n_MONO = {}\\n\\n\\ndef _mono(atom):\\n    from rdkit.Chem import GetPeriodicTable\\n    s = atom.GetSymbol()\\n    if s not in _MONO:\\n        pt = GetPeriodicTable()\\n        _MONO[s] = pt.GetMassForIsotope(s, pt.GetMostCommonIsotope(s))\\n    return _MONO[s] + atom.GetTotalNumHs() * H\\n\\n\\ndef ion_masses(frags, mode, adduct, h_max=2):\\n    \\"\\"\\"Fragment ions for the precursor\'s charge carrier, with -h_max..+h_max hydrogen rearrangements.\\"\\"\\"\\n    if mode.lower().startswith(\\"n\\") or adduct.endswith(\\"-\\"):\\n        carriers = [-PROTON]\\n    else:\\n        carriers = [PROTON] + ([NA - ELECTRON] if \\"Na\\" in adduct else [])\\n    shifts = np.array([c + k * H for c in carriers for k in range(-h_max, h_max + 1)])\\n    return np.sort((frags[:, None] + shifts[None, :]).ravel())\\n\\n\\ndef explained(peaks_mz, peaks_it, ions, tol_da=TOL_DA, tol_ppm=TOL_PPM):\\n    \\"\\"\\"(intensity share, peak share) of the peaks within tolerance of some ion.\\"\\"\\"\\n    if len(peaks_mz) == 0 or len(ions) == 0:\\n        return 0.0, 0.0\\n    pos = np.searchsorted(ions, peaks_mz)\\n    lo = ions[np.clip(pos - 1, 0, len(ions) - 1)]\\n    hi = ions[np.clip(pos, 0, len(ions) - 1)]\\n    d = np.minimum(np.abs(peaks_mz - lo), np.abs(peaks_mz - hi))\\n    hit = d <= np.maximum(tol_da, peaks_mz * tol_ppm * 1e-6)\\n    return float(peaks_it[hit].sum() / peaks_it.sum()), float(hit.mean())\\n\\n\\nclass FragmentScorer:\\n    \\"\\"\\"Caches fragment masses per structure; scores (query spectra \\u00d7 candidates).\\"\\"\\"\\n\\n    def __init__(self, smiles_of, tol_da=TOL_DA, tol_ppm=TOL_PPM, h_max=2, weight=\\"sqrt\\", depth=2):\\n        \\"\\"\\"Defaults reproduce the D4 the current rankers were trained with.\\"\\"\\"\\n        self.smiles_of = smiles_of\\n        self.tol_da, self.tol_ppm, self.h_max, self.weight, self.depth = tol_da, tol_ppm, h_max, weight, depth\\n        self.cache = {}\\n\\n    def frags(self, key):\\n        f = self.cache.get(key)\\n        if f is None:\\n            from rdkit import Chem\\n            mol = Chem.MolFromSmiles(self.smiles_of(key))\\n            f = fragment_masses(mol, self.depth) if mol is not None else np.empty(0)\\n            self.cache[key] = f\\n        return f\\n\\n    def prepare(self, spectra):\\n        \\"\\"\\"Cleaned fragment peaks per spectrum (sqrt intensity, precursor region removed).\\"\\"\\"\\n        out = []\\n        for s in spectra:\\n            mz, it = clean_peaks_fast(s[\\"mz\\"], s[\\"it\\"], s[\\"precursor_mz\\"], FRAG_CLEAN)\\n            mz, it = np.asarray(mz, np.float64), np.asarray(it, np.float64)\\n            it = {\\"sqrt\\": np.sqrt(it), \\"raw\\": it, \\"log\\": np.log1p(100 * it), \\"count\\": np.ones_like(it)}[self.weight]\\n            if len(mz):\\n                out.append((mz, it, str(s.get(\\"mode\\", \\"\\")), str(s.get(\\"adduct\\", \\"\\"))))\\n        return out\\n\\n    def score(self, spectra, keys):\\n        peaks = self.prepare(spectra)\\n        d4 = np.zeros((len(keys), 4))\\n        if not peaks:\\n            return d4\\n        for i, k in enumerate(keys):\\n            f = self.frags(k)\\n            v = np.array([explained(mz, it, ion_masses(f, mode, adduct, self.h_max), self.tol_da, self.tol_ppm)\\n                          for mz, it, mode, adduct in peaks])\\n            d4[i] = (v[:, 0].mean(), v[:, 0].max(), v[:, 1].mean(), v[:, 1].max())\\n        return d4\\n\\n\\nD4_COLUMNS = [\\"d4_int\\", \\"d4_int_max\\", \\"d4_pk\\", \\"d4_pk_max\\"]\\nD4X = dict(depth=3)              # second D4 block (\\"d4x_\\"): depth-3 cleavage (probe_d4.py: best alone)\\n\\n\\ndef add_d4(f, scorer, spectra, prefix=\\"d4\\"):\\n    \\"\\"\\"D4 columns plus within-list rank / gap / z for the molecule\'s candidate table `f`.\\"\\"\\"\\n    import pandas as pd\\n    d4 = scorer.score(spectra, f.pool_key.to_numpy())\\n    names = [c.replace(\\"d4\\", prefix, 1) for c in D4_COLUMNS]\\n    f = f.assign(**{c: d4[:, j] for j, c in enumerate(names)})\\n    for c in (f\\"{prefix}_int\\", f\\"{prefix}_pk\\"):\\n        v = f[c].to_numpy()\\n        sd = v.std()\\n        f[f\\"{c}_rank\\"] = pd.Series(v).rank(ascending=False, method=\\"min\\").to_numpy()\\n        f[f\\"{c}_gap\\"] = v.max() - v\\n        f[f\\"{c}_z\\"] = (v - v.mean()) / sd if sd > 0 else 0.0\\n    return f\\n\\n\\ndef add_d4_table(tbl, spectra_of_qid, smiles_of, cache_path=None, prefix=\\"d4\\", **scorer_kw):\\n    \\"\\"\\"D4 for every (qid, cls) group of a ranker table; fragment masses cached on disk.\\"\\"\\"\\n    import pickle\\n    import pandas as pd\\n    scorer = FragmentScorer(smiles_of, **scorer_kw)\\n    if cache_path is not None and cache_path.exists():\\n        scorer.cache.update(pickle.loads(cache_path.read_bytes()))\\n    parts = [add_d4(g.reset_index(drop=True), scorer, spectra_of_qid[qid], prefix)\\n             for (qid, _), g in tbl.groupby([\\"qid\\", \\"cls\\"], sort=False)]\\n    if cache_path is not None:\\n        cache_path.write_bytes(pickle.dumps(scorer.cache, protocol=pickle.HIGHEST_PROTOCOL))\\n    return pd.concat(parts, ignore_index=True)\\n", "src/casmi/channels/library.py": "\\"\\"\\"D1 \\u2014 direct library match: compare the query\'s spectra with each candidate\'s own reference spectra.\\n\\nFor every candidate structure we look up its library spectra (by metric key), keep the most\\ncomparable ones (same ionisation mode, test adduct, plausible mass), and score the best\\nquery \\u00d7 reference entropy similarity. Knocked-out spectra are never looked at.\\n\\"\\"\\"\\nfrom dataclasses import dataclass\\n\\nimport numpy as np\\n\\nfrom casmi.spectra.clean import CleanConfig, clean_peaks_fast\\nfrom casmi.spectra.similarity import entropy_similarity\\n\\n\\n@dataclass(frozen=True)\\nclass LibraryConfig:\\n    tol_da: float = 0.02\\n    max_ppm: float = 20.0              # reference spectra with worse precursor error are not trusted\\n    max_refs_per_candidate: int = 40   # cap work per candidate; timsTOF + same adduct first\\n    clean: CleanConfig = CleanConfig(min_rel_intensity=0.01, top_k_per_window=6, window_da=50, max_peaks=64)\\n\\n\\nclass LibraryIndex:\\n    \\"\\"\\"Store rows grouped by metric key, pre-filtered for trustworthiness.\\"\\"\\"\\n\\n    def __init__(self, store, key_of_row, cfg: LibraryConfig = LibraryConfig()):\\n        self.store, self.cfg = store, cfg\\n        meta = store.meta\\n        ok = (np.abs(meta.ppm.to_numpy()) <= cfg.max_ppm) & (meta.n_peaks.to_numpy() >= 3)\\n        rows = np.flatnonzero(ok)\\n        keys = key_of_row[rows]\\n        order = np.argsort(keys.astype(str), kind=\\"stable\\")\\n        rows, keys = rows[order], keys[order].astype(str)\\n        starts = np.r_[0, np.flatnonzero(keys[1:] != keys[:-1]) + 1]\\n        ends = np.r_[starts[1:], len(keys)]\\n        self.rows_of = {keys[s]: rows[s:e] for s, e in zip(starts, ends)}\\n        # priority for picking references: timsTOF first, then same-adduct (applied per query)\\n        self.is_timstof = meta.is_timstof.to_numpy()\\n        self.adduct = meta.adduct.astype(str).to_numpy()\\n        self.mode = meta.ionization_mode.astype(str).to_numpy()\\n        self._cache = {}\\n\\n    def _clean(self, row):\\n        hit = self._cache.get(row)\\n        if hit is None:\\n            mz, it = self.store.peaks(row)\\n            hit = clean_peaks_fast(mz, it, float(self.store.meta.precursor_mz.iat[row]), self.cfg.clean)\\n            if len(self._cache) < 400_000:\\n                self._cache[row] = hit\\n        return hit\\n\\n    def references(self, key, mode, adducts, excluded):\\n        rows = self.rows_of.get(key)\\n        if rows is None:\\n            return rows\\n        if excluded is not None and len(excluded):\\n            rows = rows[~np.isin(rows, excluded)]\\n        rows = rows[self.mode[rows] == mode]\\n        if len(rows) > self.cfg.max_refs_per_candidate:\\n            prio = self.is_timstof[rows].astype(int) * 2 + np.isin(self.adduct[rows], list(adducts)).astype(int)\\n            rows = rows[np.argsort(-prio, kind=\\"stable\\")[: self.cfg.max_refs_per_candidate]]\\n        return rows\\n\\n    def score(self, query_spectra, key, excluded):\\n        \\"\\"\\"Best and mean per-spectrum match using ONLY same-mode references.\\n\\n        A molecule can have spectra collected in different ionization modes.\\n        Each query spectrum must be compared exclusively with references from\\n        its own mode; using query_spectra[0][\\"mode\\"] for every spectrum\\n        incorrectly compares positive and negative ionizations. If a mode has\\n        no valid references, its query spectra contribute zero to the mean.\\n        \\"\\"\\"\\n        if not query_spectra:\\n            return 0.0, 0.0, 0\\n        by_mode = {}\\n        for index, spectrum in enumerate(query_spectra):\\n            by_mode.setdefault(spectrum[\\"mode\\"], []).append(index)\\n        per_spec = np.zeros(len(query_spectra), dtype=np.float64)\\n        used_rows = set()\\n        for mode, indices in by_mode.items():\\n            rows = self.references(\\n                key, mode,\\n                {query_spectra[i][\\"adduct\\"] for i in indices},\\n                excluded,\\n            )\\n            if rows is None or len(rows) == 0:\\n                continue\\n            references = [self._clean(int(row)) for row in rows]\\n            used_rows.update(int(row) for row in rows)\\n            for i in indices:\\n                query = query_spectra[i]\\n                per_spec[i] = max(\\n                    entropy_similarity(\\n                        query[\\"mz_c\\"], query[\\"it_c\\"], rm, ri,\\n                        self.cfg.tol_da, 0.0,\\n                    )\\n                    for rm, ri in references\\n                )\\n        if not used_rows:\\n            return 0.0, 0.0, 0\\n        return float(per_spec.max()), float(per_spec.mean()), len(used_rows)\\n", "src/casmi/channels/prefilter.py": "\\"\\"\\"Numerically safe, deterministic top-k prefilter for MS/MS analog retrieval.\\n\\nAll eligibility restrictions, especially structure-level knockout rows, must\\nbe enforced *before* top-k selection. Otherwise withheld/self-references can\\nconsume the entire shortlist and silently suppress genuine analogs.\\n\\nThis module requires only NumPy, so offline numerical regression tests need\\nneither the full spectrum store nor GPU/chemistry dependencies.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport numpy as np\\n\\n\\ndef eligible_top_k(scores, allowed, k: int) -> np.ndarray:\\n    \\"\\"\\"Indices of up to k highest-scoring eligible, positive, finite entries.\\n\\n    Ties are resolved by ascending original index (stable, deterministic);\\n    inputs are never mutated. The order is descending score, then index.\\n    \\"\\"\\"\\n    scores = np.asarray(scores)\\n    allowed = np.asarray(allowed)\\n    if scores.ndim != 1 or allowed.ndim != 1 or scores.shape != allowed.shape:\\n        raise ValueError(\\"scores and allowed must be aligned 1D vectors\\")\\n    if not (np.issubdtype(scores.dtype, np.floating)\\n            or np.issubdtype(scores.dtype, np.integer)):\\n        raise ValueError(\\"scores must be real numeric values\\")\\n    if allowed.dtype != np.dtype(bool):\\n        raise ValueError(\\"allowed must be a boolean vector\\")\\n    if type(k) is not int or k < 0:\\n        raise ValueError(\\"k must be a nonnegative integer\\")\\n    if k == 0 or scores.size == 0:\\n        return np.empty(0, dtype=np.int64)\\n    eligible = np.flatnonzero(allowed & np.isfinite(scores) & (scores > 0))\\n    if eligible.size <= k:\\n        return eligible[np.lexsort((eligible, -scores[eligible]))]\\n    # Use the score of the kth-best candidate as a cutoff, then break ties\\n    # deterministically by index without sorting the whole reference library.\\n    local_values = scores[eligible]\\n    cutoff = np.partition(local_values, local_values.size - k)[local_values.size - k]\\n    better = eligible[local_values > cutoff]\\n    tied = eligible[local_values == cutoff]\\n    chosen = np.concatenate((better, tied[: k - better.size]))\\n    return chosen[np.lexsort((chosen, -scores[chosen]))]\\n", "src/casmi/channels/reference_selection.py": "\\"\\"\\"Reference sampling for the CASMI D2 analog index.\\n\\nThe historical policy selects up to N spectra per (standard key, adduct),\\npreferring timsTOF rows. This can eliminate every independent reference\\nacquired in another library before a cross-library C1 knockout is applied.\\n\\nThe opt-in source_balanced policy retains the best row, reserves one\\nnon-timsTOF source (if the best is timsTOF and N >= 2), then chooses distinct\\nlibraries before filling unused slots by quality. It controls reference\\ncoverage; it does NOT guarantee improved MRR or metric-key equivalence.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nPOLICIES = (\\"legacy\\", \\"source_balanced\\")\\nSELECTION_VERSION = \\"casmi-d2-reference-selection-v1\\"\\n\\n\\ndef select_reference_rows(meta: pd.DataFrame, key_of_row, *,\\n                          per_structure_adduct: int = 3,\\n                          policy: str = \\"legacy\\") -> np.ndarray:\\n    \\"\\"\\"Return sorted, unique spectrum row IDs for building the D2 index.\\n\\n    The legacy branch intentionally reproduces AnalogIndex._select,\\n    including its global timsTOF and peak-count priority and stable ties.\\n    Source balancing does not raise the existing per-(key, adduct) cap.\\n    \\"\\"\\"\\n    if policy not in POLICIES:\\n        raise ValueError(f\\"Unknown selection policy {policy!r}\\")\\n    if type(per_structure_adduct) is not int or per_structure_adduct < 1:\\n        raise ValueError(\\"per_structure_adduct must be a positive integer\\")\\n    keys = np.asarray(key_of_row)\\n    if len(meta) != len(keys):\\n        raise ValueError(\\"Reference key and metadata lengths differ\\")\\n    needed = {\\"test_adduct\\", \\"ppm\\", \\"n_peaks\\", \\"is_timstof\\", \\"ingest_lib\\", \\"adduct\\"}\\n    if not needed.issubset(meta.columns):\\n        raise ValueError(f\\"Missing index metadata: {sorted(needed - set(meta.columns))}\\")\\n    ok = (meta.test_adduct.to_numpy(dtype=bool) &\\n          (np.abs(meta.ppm.to_numpy(dtype=float)) <= 20) &\\n          (meta.n_peaks.to_numpy(dtype=float) >= 5))\\n    rows = np.flatnonzero(ok)\\n    if len(rows) == 0:\\n        return np.empty(0, dtype=np.int64)\\n    t = pd.DataFrame({\\"row\\": rows})\\n    t[\\"key\\"] = keys[rows]\\n    t[\\"adduct\\"] = meta.adduct.to_numpy()[rows]\\n    t[\\"prio\\"] = (\\n        meta.is_timstof.to_numpy(dtype=bool)[rows].astype(np.int64) * 1000\\n        + np.clip(meta.n_peaks.to_numpy(dtype=float)[rows], 0, 999)\\n    )\\n    t = t.sort_values(\\"prio\\", ascending=False, kind=\\"stable\\")\\n    if policy == \\"legacy\\":\\n        # Exact historical selection path.\\n        t = t.groupby([\\"key\\", \\"adduct\\"], observed=True, sort=False).head(per_structure_adduct)\\n        return np.sort(t.row.to_numpy(dtype=np.int64))\\n\\n    t[\\"lib\\"] = meta.ingest_lib.astype(str).to_numpy()[t.row.to_numpy()]\\n    t[\\"timstof\\"] = meta.is_timstof.to_numpy(dtype=bool)[t.row.to_numpy()]\\n    selected = []\\n    for _, group in t.groupby([\\"key\\", \\"adduct\\"], observed=True, sort=False):\\n        group_rows = group.row.to_numpy(dtype=np.int64)\\n        if len(group_rows) <= per_structure_adduct:\\n            selected.extend(group_rows)\\n            continue\\n        libs = group.lib.to_numpy()\\n        timstof = group.timstof.to_numpy(dtype=bool)\\n        chosen = [0]  # Never displace the highest-quality reference.\\n        used_libs = {libs[0]}\\n        if per_structure_adduct > 1 and timstof[0]:\\n            for i in range(1, len(group_rows)):\\n                if not timstof[i] and libs[i] not in used_libs:\\n                    chosen.append(i)\\n                    used_libs.add(libs[i])\\n                    break\\n        for i in range(1, len(group_rows)):\\n            if len(chosen) == per_structure_adduct:\\n                break\\n            if libs[i] not in used_libs:\\n                chosen.append(i)\\n                used_libs.add(libs[i])\\n        if len(chosen) < per_structure_adduct:\\n            known = set(chosen)\\n            for i in range(1, len(group_rows)):\\n                if len(chosen) == per_structure_adduct:\\n                    break\\n                if i not in known:\\n                    chosen.append(i)\\n        selected.extend(group_rows[chosen])\\n    return np.sort(np.asarray(selected, dtype=np.int64))\\n", "src/casmi/chem/__init__.py": "", "src/casmi/chem/canon.py": "\\"\\"\\"Structure identity exactly as the official metric computes it.\\n\\nThe metric canonicalises every SMILES with RDKit 2026.03.3\'s TautomerEnumerator and compares the\\nfirst InChIKey block. Our pool keys, deduplication and validation labels all go through the same\\nfunction, so \\"same key here\\" means \\"same key on the leaderboard\\".\\n\\"\\"\\"\\nimport warnings\\nfrom functools import lru_cache\\n\\nimport rdkit\\nfrom rdkit import Chem, RDLogger\\nfrom rdkit.Chem.MolStandardize import rdMolStandardize\\n\\nEXPECTED_RDKIT = \\"2026.03.3\\"\\nif rdkit.__version__ != EXPECTED_RDKIT:\\n    warnings.warn(f\\"RDKit {rdkit.__version__} != metric\'s {EXPECTED_RDKIT}; InChIKey14 keys may not match the leaderboard.\\")\\n\\nRDLogger.DisableLog(\\"rdApp.*\\")\\n_ENUMERATOR = rdMolStandardize.TautomerEnumerator()\\n\\n\\ndef ik14_from_mol(mol):\\n    try:\\n        return Chem.MolToInchiKey(_ENUMERATOR.Canonicalize(mol))[:14]\\n    except (ValueError, RuntimeError, TypeError):\\n        return None\\n\\n\\n@lru_cache(maxsize=500_000)\\ndef canonical_ik14(smiles):\\n    \\"\\"\\"Tautomer-canonical InChIKey14 (metric-equivalent), or None if RDKit cannot parse it.\\"\\"\\"\\n    if not isinstance(smiles, str):\\n        return None\\n    try:\\n        mol = Chem.MolFromSmiles(smiles)\\n    except (ValueError, RuntimeError, TypeError):\\n        return None\\n    return None if mol is None else ik14_from_mol(mol)\\n\\n\\ndef heavy_atom_key(smiles):\\n    \\"\\"\\"Heavy-atom composition, the metric\'s cheap pre-filter before tautomer canonicalisation.\\"\\"\\"\\n    mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None\\n    if mol is None:\\n        return None\\n    counts = {}\\n    for atom in mol.GetAtoms():\\n        if atom.GetSymbol() != \\"H\\":\\n            counts[atom.GetSymbol()] = counts.get(atom.GetSymbol(), 0) + 1\\n    return tuple(sorted(counts.items()))\\n\\n\\n@lru_cache(maxsize=1_000_000)\\ndef skeleton_key(smiles):\\n    \\"\\"\\"Heavy-atom connectivity only: bonds single, no aromaticity, charges, H or stereo.\\n    Tautomers (and protomers) share it, so two structures can only have the same metric key if\\n    they share this key. It costs ~0.2 ms against 10\\u2013300 ms for tautomer canonicalisation, so it\\n    serves as an exact pre-filter before the expensive comparison.\\"\\"\\"\\n    mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None\\n    if mol is None:\\n        return None\\n    # The official metric compares only InChIKey\'s first block, which excludes\\n    # isotopic layers. Clear isotopes before removing explicit isotope hydrogens.\\n    for atom in mol.GetAtoms():\\n        atom.SetIsotope(0)\\n    rw = Chem.RWMol(Chem.RemoveHs(mol, sanitize=False))\\n    for atom in rw.GetAtoms():\\n        atom.SetFormalCharge(0)\\n        atom.SetIsAromatic(False)\\n        atom.SetNoImplicit(True)\\n        atom.SetNumExplicitHs(0)\\n        atom.SetChiralTag(Chem.ChiralType.CHI_UNSPECIFIED)\\n    for bond in rw.GetBonds():\\n        bond.SetBondType(Chem.BondType.SINGLE)\\n        bond.SetIsAromatic(False)\\n        bond.SetStereo(Chem.BondStereo.STEREONONE)\\n    return Chem.MolToSmiles(rw, canonical=True)\\n", "src/casmi/chem/keys.py": "\\"\\"\\"Lazy, persistent map from pool key (standard InChIKey14) to metric key (tautomer-canonical).\\n\\nMeasured: the two keys agree for only 88.5% of COCONUT structures (natural products carry many\\ntautomerisable groups), so scoring on the standard key would be wrong ~1 time in 9. Computing the\\nmetric key costs 10\\u2013300 ms per structure, so we compute it only where it matters \\u2014 query truths,\\nthe head of each ranked list, and output deduplication \\u2014 and cache it on disk across runs.\\n\\"\\"\\"\\nfrom pathlib import Path\\n\\nimport pandas as pd\\n\\nfrom casmi.chem.canon import canonical_ik14\\n\\n\\nclass MetricKeys:\\n    def __init__(self, cache_path: Path):\\n        self.path = Path(cache_path)\\n        self.map = {}\\n        if self.path.exists():\\n            t = pd.read_parquet(self.path)\\n            self.map = dict(zip(t.pool_key, t.metric_key))\\n        self._dirty = 0\\n\\n    def get(self, pool_key: str, smiles: str):\\n        key = self.map.get(pool_key)\\n        if key is None and pool_key not in self.map:\\n            key = canonical_ik14(smiles)\\n            self.map[pool_key] = key\\n            self._dirty += 1\\n        return key\\n\\n    def save(self):\\n        if self._dirty:\\n            pd.DataFrame({\\"pool_key\\": list(self.map), \\"metric_key\\": list(self.map.values())}).to_parquet(self.path, index=False)\\n            self._dirty = 0\\n\\n    def dedupe_ranked(self, pool_keys, smiles_of, top_n=25):\\n        \\"\\"\\"Map a ranked list of pool keys to metric keys, drop unparseable and duplicate structures,\\n        keep the first `top_n`. Returns [(pool_key, metric_key)] in rank order.\\"\\"\\"\\n        out, seen = [], set()\\n        for pk in pool_keys:\\n            mk = self.get(pk, smiles_of(pk))\\n            if mk is None or mk in seen:\\n                continue\\n            seen.add(mk)\\n            out.append((pk, mk))\\n            if len(out) == top_n:\\n                break\\n        return out\\n", "src/casmi/chem/masses.py": "\\"\\"\\"Monoisotopic masses, formula parsing and adduct arithmetic.\\n\\nEverything here is plain Python/NumPy (no RDKit) so it can run over millions of rows.\\nThe adduct parser handles every string in train (`[2M+Na]+`, `[M-2H2O+H]+`, `[M+CH2O2-H]-`, ...),\\nnot only the ten test adducts, because library spectra with other adducts are still evidence.\\n\\"\\"\\"\\nimport re\\nfrom functools import lru_cache\\n\\nimport numpy as np\\n\\nELECTRON = 0.000548579909\\nPROTON = 1.007276466621\\n\\nMONOISOTOPIC = {\\n    \\"H\\": 1.00782503207, \\"D\\": 2.0141017778, \\"C\\": 12.0, \\"N\\": 14.0030740048, \\"O\\": 15.99491461956,\\n    \\"P\\": 30.97376163, \\"S\\": 31.97207100, \\"F\\": 18.99840322, \\"Cl\\": 34.96885268, \\"Br\\": 78.9183371,\\n    \\"I\\": 126.904473, \\"Si\\": 27.9769265325, \\"B\\": 11.0093054, \\"Se\\": 79.9165213, \\"Na\\": 22.9897692809,\\n    \\"K\\": 38.96370668, \\"Li\\": 7.01600455, \\"Mg\\": 23.9850417, \\"Ca\\": 39.96259098, \\"Fe\\": 55.9349375,\\n    \\"Cu\\": 62.9295975, \\"Zn\\": 63.9291422, \\"As\\": 74.9215965, \\"Al\\": 26.98153863, \\"Co\\": 58.933195,\\n    \\"Ni\\": 57.9353429, \\"Mn\\": 54.9380451, \\"Cr\\": 51.9405075, \\"Sn\\": 119.9021947, \\"Hg\\": 201.970643,\\n    \\"Pt\\": 194.9647911, \\"Ag\\": 106.905097, \\"Au\\": 196.9665687, \\"Gd\\": 157.9241039, \\"Ge\\": 73.9211778,\\n    \\"Ti\\": 47.9479463, \\"V\\": 50.9439595, \\"Mo\\": 97.9054082, \\"W\\": 183.9509312, \\"Te\\": 129.9062244,\\n    \\"Ru\\": 101.9043493, \\"Rh\\": 102.905504, \\"Pd\\": 105.903486, \\"Sb\\": 120.9038157, \\"Bi\\": 208.9803987,\\n    \\"Ba\\": 137.9052472, \\"Sr\\": 87.9056121, \\"Tl\\": 204.9744275, \\"Pb\\": 207.9766521, \\"Cs\\": 132.905451933,\\n    \\"Rb\\": 84.911789738,\\n}\\n\\n_ELEMENT_RE = re.compile(r\\"([A-Z][a-z]?)(\\\\d*)\\")\\n\\n\\n@lru_cache(maxsize=200_000)\\ndef formula_counts(formula: str) -> dict:\\n    \\"\\"\\"\'C15H10O5\' -> {\'C\': 15, \'H\': 10, \'O\': 5}. Ignores charge suffixes such as \'+\' or \'-\'.\\"\\"\\"\\n    counts = {}\\n    for element, n in _ELEMENT_RE.findall(formula.rstrip(\\"+-\\")):\\n        counts[element] = counts.get(element, 0) + (int(n) if n else 1)\\n    return counts\\n\\n\\n@lru_cache(maxsize=200_000)\\ndef formula_mass(formula: str) -> float:\\n    \\"\\"\\"Monoisotopic neutral mass of a molecular formula; NaN if it contains an element we do not\\n    tabulate (organometallics such as Ir or Os, far outside this competition\'s chemistry).\\"\\"\\"\\n    counts = formula_counts(formula)\\n    if any(e not in MONOISOTOPIC for e in counts):\\n        return float(\\"nan\\")\\n    return sum(MONOISOTOPIC[e] * n for e, n in counts.items())\\n\\n\\n# ----------------------------------------------------------------------------- adducts\\n_ADDUCT_RE = re.compile(r\\"^\\\\[(\\\\d*)M((?:[+-]\\\\d*[A-Za-z0-9]+)*)\\\\](\\\\d*)([+-])$\\")\\n_PART_RE = re.compile(r\\"([+-])(\\\\d*)([A-Za-z][A-Za-z0-9]*)\\")\\n# Common adduct shorthands that are not literal formulas\\n_ALIASES = {\\"ACN\\": \\"C2H3N\\", \\"FA\\": \\"CH2O2\\", \\"Hac\\": \\"C2H4O2\\", \\"HAc\\": \\"C2H4O2\\", \\"TFA\\": \\"C2HF3O2\\", \\"IsoProp\\": \\"C3H8O\\"}\\n\\nTEST_ADDUCTS = (\\"[M+H]+\\", \\"[M+NH4]+\\", \\"[M-H2O+H]+\\", \\"[M-2H2O+H]+\\", \\"[M+Na]+\\", \\"[M+K]+\\",\\n                \\"[M-H]-\\", \\"[M-H2O-H]-\\", \\"[M+CH2O2-H]-\\", \\"[M+Cl]-\\")\\n\\n\\n@lru_cache(maxsize=4096)\\ndef parse_adduct(adduct: str):\\n    \\"\\"\\"Return (n_molecules, mass_shift, charge) with ion_mz = (n*M + mass_shift) / |charge|.\\n\\n    `mass_shift` already includes the electron mass for the charge state. Returns None when the\\n    string is not a recognisable adduct (the caller decides whether to drop the spectrum).\\n    \\"\\"\\"\\n    if not isinstance(adduct, str):\\n        return None\\n    m = _ADDUCT_RE.match(adduct.strip())\\n    if not m:\\n        return None\\n    n_mol = int(m.group(1)) if m.group(1) else 1\\n    z = int(m.group(3)) if m.group(3) else 1\\n    charge = z if m.group(4) == \\"+\\" else -z\\n    shift = 0.0\\n    for sign, mult, part in _PART_RE.findall(m.group(2)):\\n        mass = formula_mass(_ALIASES.get(part, part))\\n        if np.isnan(mass):\\n            return None\\n        shift += (1 if sign == \\"+\\" else -1) * (int(mult) if mult else 1) * mass\\n    shift -= charge * ELECTRON          # a cation has lost electrons, an anion gained them\\n    return n_mol, shift, charge\\n\\n\\ndef ion_mz(neutral_mass: float, adduct: str) -> float:\\n    \\"\\"\\"Ion m/z for a neutral mass and adduct; NaN when the adduct string is not recognised.\\"\\"\\"\\n    parsed = parse_adduct(adduct)\\n    if parsed is None:\\n        return np.nan\\n    n, shift, charge = parsed\\n    return (n * neutral_mass + shift) / abs(charge)\\n\\n\\ndef neutral_mass(precursor_mz, adduct: str):\\n    \\"\\"\\"Invert `ion_mz`. Works on scalars or NumPy arrays of precursor m/z for one adduct.\\"\\"\\"\\n    parsed = parse_adduct(adduct)\\n    if parsed is None:\\n        return np.nan if np.ndim(precursor_mz) == 0 else np.full(np.shape(precursor_mz), np.nan)\\n    n, shift, charge = parsed\\n    return (np.asarray(precursor_mz, dtype=float) * abs(charge) - shift) / n\\n\\n\\ndef ppm(measured, theoretical):\\n    return (np.asarray(measured, dtype=float) - theoretical) / theoretical * 1e6\\n", "src/casmi/chem/npscore.py": "\\"\\"\\"RDKit natural-product likeness (Ertl et al. 2008, rdkit.Contrib.NP_Score), cached per key.\\n\\nComputed from the structure for every candidate alike: unlike COCONUT\'s own np_likeness column it\\ncarries no information about which database a candidate came from.\\n\\"\\"\\"\\n_MODEL = None\\n_CACHE = {}\\n\\n\\ndef np_likeness(key, smiles):\\n    global _MODEL\\n    v = _CACHE.get(key)\\n    if v is None:\\n        from rdkit import Chem\\n        from rdkit.Contrib.NP_Score import npscorer\\n        if _MODEL is None:\\n            _MODEL = npscorer.readNPModel()\\n        mol = Chem.MolFromSmiles(smiles)\\n        v = float(npscorer.scoreMol(mol, _MODEL)) if mol is not None else 0.0\\n        _CACHE[key] = v\\n    return v\\n", "src/casmi/config.py": "\\"\\"\\"Paths and runtime environment. Resolves the same code for local runs and Kaggle notebooks.\\"\\"\\"\\nimport os\\nfrom pathlib import Path\\n\\nON_KAGGLE = Path(\\"/kaggle/input\\").exists()\\nPROJECT_ROOT = Path(__file__).resolve().parents[2]\\n\\n\\ndef _find_competition_data() -> Path:\\n    # Explicit research snapshots must take precedence over mounted competition\\n    # discovery, especially after creating a family-purged training corpus.\\n    if os.environ.get(\\"CASMI_DATA\\"):\\n        return Path(os.environ[\\"CASMI_DATA\\"])\\n    if ON_KAGGLE:\\n        hits = sorted(Path(\\"/kaggle/input\\").rglob(\\"test.parquet\\"))\\n        if hits:\\n            return hits[0].parent\\n    return Path(os.environ.get(\\"CASMI_DATA\\", \\"C:/Users/MONSTER/kaggle-data/casmi26\\"))\\n\\n\\nDATA_DIR = _find_competition_data()\\n# Built artifacts (spectrum store, pool, models). Large, never committed.\\n# /tmp on Kaggle: scratch disk, not saved as notebook output (which would slow every commit)\\nARTIFACT_DIR = Path(os.environ.get(\\"CASMI_ARTIFACTS\\",\\n                                   \\"/tmp/casmi_artifacts\\" if ON_KAGGLE else str(DATA_DIR / \\"artifacts\\")))\\nOUTPUT_DIR = Path(\\"/kaggle/working\\") if ON_KAGGLE else PROJECT_ROOT / \\"runs\\"\\nSEED = 42\\n\\n# Candidate-pool variant: \\"\\" = train \\u222a COCONUT (default), \\"_pcl\\" = + PubChemLite. All pool-derived\\n# artifacts carry the tag, so variants can live side by side and be compared on the same queries.\\nPOOL_TAG = os.environ.get(\\"CASMI_POOL_TAG\\", \\"\\")\\nPOOL_FILE = f\\"pool{POOL_TAG}.parquet\\"\\nPOOL_FP_FILE = f\\"pool_fp2048{POOL_TAG}.npy\\"\\nPOOL_FP_KEYS_FILE = f\\"pool_fp_keys{POOL_TAG}.parquet\\"\\n", "src/casmi/engine.py": "\\"\\"\\"Per-molecule engine: neutral mass -> candidates -> channel features -> ranking.\\n\\nThe baseline ranks by D1 (direct library similarity), breaking ties by mass error. Later stages\\nreplace `rank` with the learned ranker; the candidate/feature interface stays the same, so every\\nchannel is validated through exactly the code path the submission uses.\\n\\"\\"\\"\\nfrom dataclasses import dataclass, field\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom casmi.channels.analog import AnalogConfig, propagate\\nfrom casmi.channels.library import LibraryConfig, LibraryIndex\\nfrom casmi.chem.masses import neutral_mass\\nfrom casmi.spectra.clean import clean_peaks_fast\\n\\n\\n@dataclass(frozen=True)\\nclass EngineConfig:\\n    tol_ppm: float = 10.0\\n    mass_offset_ppm: float = 0.0      # calibrated instrument offset (measured +1.5 ppm on np-examples)\\n    top_n: int = 25\\n    library: LibraryConfig = field(default_factory=LibraryConfig)\\n    analog: AnalogConfig = field(default_factory=AnalogConfig)\\n    generate: bool = False            # Class-3 hedge: add biotransformed analogs as candidates\\n    gen_max_analogs: int = 15\\n    gen_cap: int = 300\\n    gen_consensus: bool = False       # v17 features: seed / route consensus and transform type\\n    gen_expand: bool = False          # research opt-in, legacy-preserving ordered-pair expansion\\n    np_score: bool = False            # RDKit NP-likeness for every candidate (chemistry, not membership)\\n\\n\\ndef molecule_mass(spectra, offset_ppm=0.0):\\n    masses = [float(neutral_mass(s[\\"precursor_mz\\"], s[\\"adduct\\"])) for s in spectra]\\n    masses = [m for m in masses if np.isfinite(m)]\\n    m = float(np.median(masses)) if masses else np.nan\\n    return m / (1 + offset_ppm * 1e-6)\\n\\n\\nclass BaselineEngine:\\n    def __init__(self, pool, library: LibraryIndex, cfg: EngineConfig = EngineConfig(), analog=None, fps=None):\\n        \\"\\"\\"`analog` (AnalogIndex) and `fps` (packed pool fingerprints) enable the D2 channel.\\"\\"\\"\\n        self.pool, self.library, self.cfg = pool, library, cfg\\n        self.analog, self.fps = analog, fps\\n\\n    def features(self, spectra, excluded_rows=None, excluded_keys=None) -> pd.DataFrame:\\n        for s in spectra:   # clean once per query, with the library\'s cleaning settings\\n            s[\\"mz_c\\"], s[\\"it_c\\"] = clean_peaks_fast(s[\\"mz\\"], s[\\"it\\"], s[\\"precursor_mz\\"], self.library.cfg.clean)\\n        mass = molecule_mass(spectra, self.cfg.mass_offset_ppm)\\n        win = self.pool.window(mass, self.cfg.tol_ppm, excluded_keys)\\n        keys = self.pool.key[win.idx]\\n        d1 = np.array([self.library.score(spectra, k, excluded_rows) for k in keys]).reshape(-1, 3)\\n        f = pd.DataFrame({\\"pool_key\\": keys, \\"ppm\\": win.ppm, \\"d1_max\\": d1[:, 0], \\"d1_mean\\": d1[:, 1],\\n                          \\"d1_nref\\": d1[:, 2].astype(int), \\"n_cand\\": len(keys)})\\n        analogs = None\\n        if self.analog is not None and len(keys):\\n            analogs = self.analog.search(spectra, mass, excluded_rows)\\n            for name, v in propagate(analogs, win.idx, self.pool.mass[win.idx], self.fps,\\n                                     self.pool.row_of, self.cfg.analog.powers).items():\\n                f[name] = v\\n        if self.cfg.generate:\\n            if analogs is None and self.analog is not None:\\n                analogs = self.analog.search(spectra, mass, excluded_rows)\\n            f = f.assign(gen=0, gen_sim=0.0, gen_depth=0)\\n            if self.cfg.gen_expand:\\n                f = f.assign(audit_legacy=1)\\n            if self.cfg.gen_consensus:\\n                f = f.assign(gen_sim_sum=0.0, gen_n_seeds=0, gen_n_paths=0, gen_t1=-1, gen_t2=-1)\\n            g = self._generated(analogs or {}, mass, excluded_keys or set())\\n            if len(g):\\n                f = pd.concat([f, g], ignore_index=True)\\n                f[\\"n_cand\\"] = len(f)\\n        if self.cfg.np_score and len(f):\\n            from casmi.chem.npscore import np_likeness\\n            f[\\"np_score\\"] = [np_likeness(k, self.pool.smiles_of(k)) for k in f.pool_key]\\n        return f\\n\\n    def _generated(self, analogs, mass, excluded_keys):\\n        \\"\\"\\"Biotransformed analogs at the query mass that are not already pool candidates.\\"\\"\\"\\n        from rdkit import Chem\\n        from rdkit.Chem.Descriptors import ExactMolWt\\n        from casmi.generate.transforms import propose, propose_agg\\n        from casmi.pool.pool import generated_candidate_identity\\n        order = sorted(analogs.items(), key=lambda kv: -kv[1][0])[: self.cfg.gen_max_analogs]\\n        seeds = [(self.pool.smiles_of(k), sim, am) for k, (sim, am) in order if k in self.pool.row_of]\\n        rows, seen = [], {}\\n        if self.cfg.gen_consensus:\\n            if self.cfg.gen_expand:\\n                from casmi.generate.expanded import expanded_proposals\\n                legacy_seeds = [(self.pool.smiles_of(k), sim, am) for k, (sim, am) in order[:15] if k in self.pool.row_of]\\n                props = expanded_proposals(seeds, mass, self.cfg.tol_ppm, self.cfg.gen_max_analogs, self.cfg.gen_cap,\\n                                           legacy_analogs=legacy_seeds)\\n                items = sorted(props.items(), key=lambda kv: (not kv[1][\\"audit_legacy\\"], -kv[1][\\"sim_max\\"]))\\n            else:\\n                props = propose_agg(seeds, mass, self.cfg.tol_ppm, self.cfg.gen_max_analogs, self.cfg.gen_cap)\\n                items = sorted(props.items(), key=lambda kv: -kv[1][\\"sim_max\\"])\\n        else:\\n            if self.cfg.gen_expand:\\n                raise ValueError(\\"Expanded generation requires consensus features\\")\\n            items = [(smi, dict(sim_max=sim, depth=path.count(\\"+\\") + 1))\\n                     for smi, sim, path in propose(seeds, mass, self.cfg.tol_ppm, self.cfg.gen_max_analogs, self.cfg.gen_cap)]\\n        for smi, r in items:\\n            mol = Chem.MolFromSmiles(smi)\\n            ik = Chem.MolToInchiKey(mol)[:14] if mol is not None else \\"\\"\\n            if not ik or (ik in self.pool.row_of and ik not in excluded_keys):\\n                continue\\n            if ik in seen:                                     # same structure reached as another SMILES\\n                if self.cfg.gen_consensus:\\n                    row = rows[seen[ik]]\\n                    if self.cfg.gen_expand and row[-1] != r[\\"audit_legacy\\"]:\\n                        continue  # Do not alter legacy candidate features with new routes.\\n                    row[5] += r[\\"sim_sum\\"]\\n                    row[6] += r[\\"n_seeds\\"]\\n                    row[7] += r[\\"n_paths\\"]\\n                continue\\n            seen[ik] = len(rows)\\n            generated_key, smi = generated_candidate_identity(mol)\\n            row = [generated_key, smi, (mass - ExactMolWt(mol)) / ExactMolWt(mol) * 1e6, r[\\"sim_max\\"], r[\\"depth\\"]]\\n            if self.cfg.gen_consensus:\\n                row += [r[\\"sim_sum\\"], r[\\"n_seeds\\"], r[\\"n_paths\\"], r[\\"t1\\"], r[\\"t2\\"]]\\n            if self.cfg.gen_expand:\\n                row += [r[\\"audit_legacy\\"]]\\n            rows.append(row)\\n        if not rows:\\n            return pd.DataFrame()\\n        cols = [\\"pool_key\\", \\"gen_smiles\\", \\"ppm\\", \\"gen_sim\\", \\"gen_depth\\"]\\n        if self.cfg.gen_consensus:\\n            cols += [\\"gen_sim_sum\\", \\"gen_n_seeds\\", \\"gen_n_paths\\", \\"gen_t1\\", \\"gen_t2\\"]\\n        if self.cfg.gen_expand:\\n            cols += [\\"audit_legacy\\"]\\n        g = pd.DataFrame(rows, columns=cols)\\n        self.pool.register_generated(g.pool_key, g.gen_smiles)\\n        g = g.drop(columns=\\"gen_smiles\\").assign(d1_max=0.0, d1_mean=0.0, d1_nref=0, gen=1)\\n        if analogs:\\n            keys = [k for k in analogs if k in self.pool.row_of]\\n            local = np.vstack([self.pool.packed_fps(g.pool_key.to_numpy(), self.fps),\\n                               self.fps[[self.pool.row_of[k] for k in keys]]])\\n            row_of = {k: len(g) + i for i, k in enumerate(keys)}\\n            gm = mass / (1 + g.ppm.to_numpy() * 1e-6)\\n            for name, v in propagate({k: analogs[k] for k in keys}, np.arange(len(g)), gm, local, row_of,\\n                                     self.cfg.analog.powers).items():\\n                g[name] = v\\n        return g\\n\\n    def rank(self, feats: pd.DataFrame, depth=60):\\n        \\"\\"\\"Ranked pool keys (deeper than 25, so output deduplication can refill the list).\\"\\"\\"\\n        if feats.empty:\\n            return []\\n        primary = feats.d1_max.to_numpy()\\n        secondary = feats.d1_mean.to_numpy()\\n        if \\"an_p4\\" in feats:   # interim rule until the learned ranker: best of direct and propagated evidence\\n            primary = np.maximum(primary, feats.an_p4.to_numpy())\\n            secondary = feats.an_p2.to_numpy()\\n        order = np.lexsort((np.abs(feats.ppm.to_numpy()), -secondary, -primary))\\n        return feats.pool_key.to_numpy()[order][:depth].tolist()\\n", "src/casmi/generate/__init__.py": "", "src/casmi/generate/expanded.py": "\\"\\"\\"Explicitly budgeted proposal expansion preserving historical candidates/features.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom .transforms import propose_agg\\n\\n\\ndef expanded_proposals(analogs, query_mass, tol_ppm=10.0, max_analogs=30, cap=900, *, legacy_analogs=None):\\n    if type(max_analogs) is not int or max_analogs < 15 or type(cap) is not int or cap < 300:\\n        raise ValueError(\\"Expanded policy must preserve the 15-seed / 300-candidate legacy budget\\")\\n    legacy = propose_agg(analogs[:15] if legacy_analogs is None else legacy_analogs, query_mass, tol_ppm, 15, 300)\\n    expanded = propose_agg(analogs, query_mass, tol_ppm, max_analogs, cap, ordered_pairs=True)\\n    out = {smiles: {**record, \\"audit_legacy\\": 1} for smiles, record in legacy.items()}\\n    for smiles, record in expanded.items():\\n        if smiles not in out and len(out) < cap:\\n            out[smiles] = {**record, \\"audit_legacy\\": 0}\\n    return out\\n", "src/casmi/generate/transforms.py": "\\"\\"\\"Class-3 hedge: propose unseen structures by applying one or two common natural-product\\nbiotransformations to spectral analogs of the query.\\n\\nA Class-3 molecule is in no database, so retrieval scores 0 on it by construction. Natural products\\ncome in families, though: the analog search (D2) often finds a library structure whose spectrum\\nmatches the query up to a precursor shift \\u0394. When \\u0394 equals a common biotransformation (methylation,\\nhydroxylation, glycosylation, acylation, ...) the query is plausibly that analog with the\\nmodification at one of its sites. Every site gives one candidate; the metric ignores stereo\\n(InChIKey block 1), so connectivity is all that has to be right.\\n\\n`propose(analogs, query_mass)` returns generated SMILES with the analog they came from; scoring\\nand placement in the ranked list happen elsewhere (they need D3/D4 and a confidence gate).\\n\\"\\"\\"\\nfrom dataclasses import dataclass\\n\\nimport numpy as np\\nfrom rdkit import Chem, RDLogger\\nfrom rdkit.Chem import AllChem\\nfrom rdkit.Chem.Descriptors import ExactMolWt\\n\\nRDLogger.DisableLog(\\"rdApp.*\\")\\n\\nHEX = \\"C1OC(CO)C(O)C(O)C1O\\"          # hexose (glucose/galactose connectivity)\\nDHEX = \\"C1OC(C)C(O)C(O)C1O\\"          # 6-deoxyhexose (rhamnose/fucose)\\nPENT = \\"C1OCC(O)C(O)C1O\\"             # pentose (pyranose)\\nGLCA = \\"C1OC(C(=O)O)C(O)C(O)C1O\\"     # glucuronic acid\\n\\n\\n@dataclass(frozen=True)\\nclass Transform:\\n    name: str\\n    delta: float\\n    smarts: tuple\\n\\n\\ndef _t(name, formula_delta, *smarts):\\n    return Transform(name, formula_delta, tuple(smarts))\\n\\n\\n# forward transforms; each is also applied in reverse where a removal SMARTS is given\\nTRANSFORMS = [\\n    _t(\\"methyl\\", 14.01565, \\"[OX2H1:1]>>[O:1]C\\", \\"[NX3;H1,H2;!$(N-C=O):1]>>[N:1]C\\", \\"[cH1:1]>>[c:1]C\\"),\\n    _t(\\"demethyl\\", -14.01565, \\"[O:1]-[CH3]>>[O:1]\\", \\"[N:1]-[CH3]>>[N:1]\\"),\\n    _t(\\"hydroxyl\\", 15.99491, \\"[cH1:1]>>[c:1]O\\", \\"[CX4;H1,H2:1]>>[C:1]O\\"),\\n    _t(\\"dehydroxyl\\", -15.99491, \\"[c:1]-[OH]>>[c:1]\\", \\"[CX4:1]-[OH]>>[C:1]\\"),\\n    _t(\\"methoxyl\\", 30.01056, \\"[cH1:1]>>[c:1]OC\\"),\\n    _t(\\"demethoxyl\\", -30.01056, \\"[c:1]-[OX2]-[CH3]>>[c:1]\\"),\\n    _t(\\"hydrogenation\\", 2.01565, \\"[C:1]=[C:2]>>[C:1]-[C:2]\\"),\\n    _t(\\"dehydrogenation\\", -2.01565, \\"[CX4;!H0:1]-[CX4;!H0:2]>>[C:1]=[C:2]\\", \\"[CX4;H1,H2:1]-[OH:2]>>[C:1]=[O:2]\\"),\\n    _t(\\"hydration\\", 18.01056, \\"[C:1]=[C:2]>>[C:1](O)-[C:2]\\"),\\n    _t(\\"dehydration\\", -18.01056, \\"[CX4;!H0:1]-[CX4:2]-[OH]>>[C:1]=[C:2]\\"),\\n    _t(\\"hexose\\", 162.05282, f\\"[OX2H1:1]>>[O:1]{HEX}\\"),\\n    _t(\\"dehexose\\", -162.05282, \\"[O:1]-[CH1;R1]1-[OX2;R1]-[CH1;R1](-[CH2]-[OH])-[CH1;R1](-[OH])-[CH1;R1](-[OH])-[CH1;R1]1-[OH]>>[O:1]\\"),\\n    _t(\\"deoxyhexose\\", 146.05791, f\\"[OX2H1:1]>>[O:1]{DHEX}\\"),\\n    _t(\\"dedeoxyhexose\\", -146.05791, \\"[O:1]-[CH1;R1]1-[OX2;R1]-[CH1;R1](-[CH3])-[CH1;R1](-[OH])-[CH1;R1](-[OH])-[CH1;R1]1-[OH]>>[O:1]\\"),\\n    _t(\\"pentose\\", 132.04226, f\\"[OX2H1:1]>>[O:1]{PENT}\\"),\\n    _t(\\"glucuronide\\", 176.03209, f\\"[OX2H1:1]>>[O:1]{GLCA}\\"),\\n    _t(\\"acetyl\\", 42.01057, \\"[OX2H1:1]>>[O:1]C(C)=O\\"),\\n    _t(\\"deacetyl\\", -42.01057, \\"[O:1]-C(=O)-[CH3]>>[O:1]\\"),\\n    _t(\\"malonyl\\", 86.00039, \\"[OX2H1:1]>>[O:1]C(=O)CC(=O)O\\"),\\n    _t(\\"sulfate\\", 79.95682, \\"[OX2H1:1]>>[O:1]S(=O)(=O)O\\"),\\n    _t(\\"prenyl\\", 68.06260, \\"[cH1:1]>>[c:1]CC=C(C)C\\", \\"[OX2H1:1]>>[O:1]CC=C(C)C\\"),\\n    _t(\\"coumaroyl\\", 146.03678, \\"[OX2H1:1]>>[O:1]C(=O)C=Cc1ccc(O)cc1\\"),\\n    _t(\\"caffeoyl\\", 162.03169, \\"[OX2H1:1]>>[O:1]C(=O)C=Cc1ccc(O)c(O)c1\\"),\\n    _t(\\"feruloyl\\", 176.04734, \\"[OX2H1:1]>>[O:1]C(=O)C=Cc1ccc(O)c(OC)c1\\"),\\n    _t(\\"galloyl\\", 152.01096, \\"[OX2H1:1]>>[O:1]C(=O)c1cc(O)c(O)c(O)c1\\"),\\n    # v2 (diagnose_c3.py: the formula changes the misses needed most often)\\n    _t(\\"homolog\\", 14.01565, \\"[CH2;!R:1]-[CH2;!R:2]>>[C:1]-C-[C:2]\\", \\"[CH3;!R:1]>>[C:1]C\\"),\\n    _t(\\"dehomolog\\", -14.01565, \\"[CH2;!R:1]-[CH2;!R]-[CH2;!R:2]>>[C:1]-[C:2]\\", \\"[CH2;!R:1]-[CH3]>>[C:1]\\"),\\n    _t(\\"carbonyl_reduction\\", 2.01565, \\"[#6X3;$([#6](-[#6])-[#6]):1]=[O:2]>>[C:1]-[O:2]\\"),\\n    _t(\\"carbonyl_to_ch2\\", -13.97926, \\"[#6X3;$([#6](-[#6])-[#6]):1]=[O]>>[C:1]\\"),\\n    _t(\\"decarboxyl\\", -43.98983, \\"[#6:1]-C(=O)-[OH]>>[#6:1]\\"),\\n    _t(\\"chloro\\", 33.96103, \\"[cH1:1]>>[c:1]Cl\\"),\\n    _t(\\"dechloro\\", -33.96103, \\"[#6:1]-Cl>>[#6:1]\\"),\\n    _t(\\"n_acetyl\\", 42.01057, \\"[NX3;H1,H2;!$(N-C=O):1]>>[N:1]C(C)=O\\"),\\n    _t(\\"c_hexose\\", 162.05282, f\\"[cH1:1]>>[c:1]{HEX}\\"),\\n    _t(\\"desulfate\\", -79.95682, \\"[O:1]-S(=O)(=O)-[OH]>>[O:1]\\"),\\n    _t(\\"demalonyl\\", -86.00039, \\"[O:1]-C(=O)-[CH2]-C(=O)-[OH]>>[O:1]\\"),\\n    _t(\\"depentose\\", -132.04226, \\"[O:1]-[CH1;R1]1-[OX2;R1]-[CH2;R1]-[CH1;R1](-[OH])-[CH1;R1](-[OH])-[CH1;R1]1-[OH]>>[O:1]\\"),\\n    _t(\\"deglucuronide\\", -176.03209, \\"[O:1]-[CH1;R1]1-[OX2;R1]-[CH1;R1](-C(=O)-[OH])-[CH1;R1](-[OH])-[CH1;R1](-[OH])-[CH1;R1]1-[OH]>>[O:1]\\"),\\n    _t(\\"degalloyl\\", -152.01096, \\"[O:1]-C(=O)-c1cc(O)c(O)c(O)c1>>[O:1]\\"),\\n    _t(\\"decoumaroyl\\", -146.03678, \\"[O:1]-C(=O)-C=C-c1ccc(O)cc1>>[O:1]\\"),\\n    _t(\\"decaffeoyl\\", -162.03169, \\"[O:1]-C(=O)-C=C-c1ccc(O)c(O)c1>>[O:1]\\"),\\n    _t(\\"deferuloyl\\", -176.04734, \\"[O:1]-C(=O)-C=C-c1ccc(O)c(OC)c1>>[O:1]\\"),\\n]\\n_RXN = {t.name: [AllChem.ReactionFromSmarts(s) for s in t.smarts] for t in TRANSFORMS}\\nDELTAS = np.array([t.delta for t in TRANSFORMS])\\n\\n\\ndef matching(delta, tol_da, *, ordered_pairs=False):\\n    \\"\\"\\"Single transforms and ordered pairs whose total mass change equals `delta`.\\"\\"\\"\\n    single = [(t,) for t in TRANSFORMS if abs(t.delta - delta) <= tol_da]\\n    pairs = [(a, b) for i, a in enumerate(TRANSFORMS) for b in TRANSFORMS[i:]\\n             if abs(a.delta + b.delta - delta) <= tol_da]\\n    if ordered_pairs:\\n        # Reaction order can change which sites exist in the intermediate.\\n        # Append reverse routes after historical routes to preserve their priority.\\n        return single + pairs + [(b, a) for a, b in pairs if a.name != b.name]\\n    return single + pairs\\n\\n\\ndef apply(mol, transform, cap=60):\\n    \\"\\"\\"Every single-site product of `transform` on `mol` (sanitised, unique by SMILES).\\"\\"\\"\\n    out = {}\\n    for rxn in _RXN[transform.name]:\\n        try:\\n            products = rxn.RunReactants((mol,), maxProducts=cap)\\n        except Exception:\\n            continue\\n        for (p,) in products:\\n            try:\\n                Chem.SanitizeMol(p)\\n                smi = Chem.MolToSmiles(p)\\n            except Exception:\\n                continue\\n            if smi and \\".\\" not in smi:\\n                out.setdefault(smi, p)\\n        if len(out) >= cap:\\n            break\\n    return out\\n\\n\\ndef propose(analogs, query_mass, tol_ppm=10.0, max_analogs=15, cap=300):\\n    \\"\\"\\"analogs: [(smiles, similarity, analog_mass)], best first.\\n    Returns [(smiles, similarity, transform names)] of structures at the query mass.\\"\\"\\"\\n    tol_da = query_mass * tol_ppm * 1e-6\\n    seen, out = set(), []\\n    for smi, sim, amass in analogs[:max_analogs]:\\n        paths = matching(query_mass - amass, tol_da * 2)\\n        if not paths:\\n            continue\\n        mol = Chem.MolFromSmiles(smi)\\n        if mol is None:\\n            continue\\n        for path in paths:\\n            level = {smi: mol}\\n            for t in path:\\n                nxt = {}\\n                for m in level.values():\\n                    nxt.update(apply(m, t))\\n                level = nxt\\n                if not level:\\n                    break\\n            for psmi, p in level.items():\\n                if psmi in seen:\\n                    continue\\n                seen.add(psmi)\\n                if abs(ExactMolWt(p) - query_mass) <= tol_da:\\n                    out.append((psmi, sim, \\"+\\".join(t.name for t in path)))\\n            if len(out) >= cap:\\n                return out\\n    return out\\n\\n\\nT_INDEX = {t.name: i for i, t in enumerate(TRANSFORMS)}\\n\\n\\ndef propose_agg(analogs, query_mass, tol_ppm=10.0, max_analogs=15, cap=300, *, ordered_pairs=False):\\n    \\"\\"\\"Like `propose`, but every product carries its consensus: how many distinct analog seeds and\\n    (seed, path) routes produce it. analogs: [(smiles, similarity, analog_mass)], best first.\\n    Returns {smiles: dict(sim_max, sim_sum, n_seeds, n_paths, depth, t1, t2)}.\\"\\"\\"\\n    tol_da = query_mass * tol_ppm * 1e-6\\n    out = {}\\n    for si, (smi, sim, amass) in enumerate(analogs[:max_analogs]):\\n        paths = matching(query_mass - amass, tol_da * 2, ordered_pairs=ordered_pairs)\\n        if not paths:\\n            continue\\n        mol = Chem.MolFromSmiles(smi)\\n        if mol is None:\\n            continue\\n        for path in paths:\\n            level = {smi: mol}\\n            for t in path:\\n                nxt = {}\\n                for m in level.values():\\n                    nxt.update(apply(m, t))\\n                level = nxt\\n                if not level:\\n                    break\\n            for psmi, prod in level.items():\\n                rec = out.get(psmi)\\n                if rec is None:\\n                    if len(out) >= cap or abs(ExactMolWt(prod) - query_mass) > tol_da:\\n                        continue\\n                    rec = out[psmi] = dict(sim_max=sim, seeds=set(), sims={}, n_paths=0, depth=len(path),\\n                                           t1=T_INDEX[path[0].name], t2=T_INDEX[path[1].name] if len(path) > 1 else -1)\\n                rec[\\"seeds\\"].add(si)\\n                rec[\\"sims\\"][si] = sim\\n                rec[\\"n_paths\\"] += 1\\n                rec[\\"sim_max\\"] = max(rec[\\"sim_max\\"], sim)\\n                if len(path) < rec[\\"depth\\"]:\\n                    rec[\\"depth\\"], rec[\\"t1\\"] = len(path), T_INDEX[path[0].name]\\n                    rec[\\"t2\\"] = T_INDEX[path[1].name] if len(path) > 1 else -1\\n    for rec in out.values():\\n        rec[\\"n_seeds\\"] = len(rec.pop(\\"seeds\\"))\\n        rec[\\"sim_sum\\"] = float(sum(rec.pop(\\"sims\\").values()))\\n    return out\\n", "src/casmi/pool/__init__.py": "", "src/casmi/pool/build_safety.py": "\\"\\"\\"Exclusive publication for a newly built immutable CASMI candidate pool.\\n\\nA preflight `exists()` check does not guard against a second writer racing\\nto publish its file. Stage the Parquet on the same filesystem and atomically\\ncreate a hard link at the destination; unlike os.replace, os.link refuses\\nan existing destination. Clean the staging file on all outcomes.\\n\\nFilesystems without hard-link support fail closed. Readers must use immutable\\nversioned pool tags rather than expecting an existing file to be replaced.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom contextlib import contextmanager\\nimport os\\nfrom pathlib import Path\\nimport tempfile\\n\\n\\n@contextmanager\\ndef fresh_pool_file(destination):\\n    \\"\\"\\"Yield a temporary Parquet path; publish it only on successful exit.\\"\\"\\"\\n    destination = Path(destination)\\n    if not destination.name or destination.name in (\\".\\", \\"..\\"):\\n        raise ValueError(\\"An explicit non-root pool output path is required\\")\\n    if os.path.lexists(destination):\\n        raise FileExistsError(\\"Refusing to overwrite frozen candidate pool\\")\\n    destination.parent.mkdir(parents=True, exist_ok=True)\\n    fd, name = tempfile.mkstemp(\\n        prefix=f\\".{destination.name}-building-\\", suffix=\\".parquet\\",\\n        dir=destination.parent,\\n    )\\n    os.close(fd)\\n    stage = Path(name)\\n    try:\\n        yield stage\\n        if not stage.is_file() or stage.stat().st_size == 0:\\n            raise ValueError(\\"Candidate pool build produced an empty output file\\")\\n        # Hard-link creation, unlike rename or os.replace, fails if a second\\n        # process has created the final path during the build.\\n        os.link(stage, destination, follow_symlinks=False)\\n    finally:\\n        stage.unlink(missing_ok=True)\\n", "src/casmi/pool/fingerprint_audit.py": "\\"\\"\\"Read-only quality summary for packed CASMI Morgan fingerprints.\\n\\nA SHA-256 inventory can consistently sign invalid zero-bit vectors. This\\naudit adds a numeric sanity check without changing or disclosing source rows.\\nIt does not recompute RDKit chemistry from SMILES and cannot certify model\\naccuracy or upstream dataset provenance.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport numpy as np\\n\\nBITS_BYTES = 256\\n\\n\\ndef fingerprint_quality_summary(pool_keys, fp_keys, fps, *, batch_rows: int = 65536) -> dict:\\n    \\"\\"\\"Verify row identity, vector format, and count zero vectors in chunks.\\"\\"\\"\\n    if type(batch_rows) is not int or not 1 <= batch_rows <= 1000000:\\n        raise ValueError(\\"batch_rows must be an integer from 1 to 1000000\\")\\n    if (not isinstance(fps, np.ndarray) or fps.dtype != np.uint8\\n            or fps.ndim != 2 or fps.shape[1] != BITS_BYTES):\\n        raise ValueError(\\"Fingerprint vectors must be packed uint8 Morgan 2048 bits\\")\\n    source = np.asarray(pool_keys)\\n    recorded = np.asarray(fp_keys)\\n    if source.ndim != 1 or recorded.ndim != 1:\\n        raise ValueError(\\"Fingerprint keys must be one-dimensional\\")\\n    if len(source) != len(recorded) or len(source) != len(fps):\\n        raise ValueError(\\"Pool keys and fingerprint vectors have different row counts\\")\\n    if not np.array_equal(source, recorded):\\n        raise ValueError(\\"Fingerprint key order differs from candidate pool\\")\\n    if len(fps) == 0:\\n        raise ValueError(\\"Empty candidate pools cannot pass fingerprint quality audit\\")\\n\\n    zero_rows = 0\\n    for start in range(0, len(fps), batch_rows):\\n        fragment = fps[start:start + batch_rows]\\n        zero_rows += int(np.count_nonzero(~fragment.any(axis=1)))\\n    return {\\n        \\"status\\": \\"PASS_NONZERO_FINGERPRINTS\\" if zero_rows == 0\\n                  else \\"INVALID_ZERO_FINGERPRINTS\\",\\n        \\"candidate_count\\": len(fps),\\n        \\"zero_fingerprint_rows\\": zero_rows,\\n        \\"zero_fraction\\": float(zero_rows / len(fps)) if len(fps) else 0.0,\\n        \\"method\\": \\"Read-only, row-aligned uint8 bits scanned in bounded batches\\",\\n        \\"warning\\": (\\n            \\"Does not authenticate Kaggle source, recalculate chemistry or \\"\\n            \\"prove candidate SMILES are structurally correct\\"\\n        ),\\n    }\\n", "src/casmi/pool/fingerprint_provenance.py": "\\"\\"\\"Content-bound provenance for CASMI candidate-pool Morgan fingerprints.\\n\\nFingerprint row order alone does not prove that the bit vectors were computed\\nfrom the CURRENT candidate SMILES: a corrected de-salted COCONUT parent may\\nshare a key with an old molecule but have different input structure bytes.\\nAn immutable research snapshot binds pool_key + smiles + mass AND fingerprint\\nbytes to the same recorded generation, without disclosing source rows.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\nimport math\\n\\nimport numpy as np\\n\\nSCHEMA = \\"casmi-morgan-fingerprint-provenance-v1\\"\\nFP_SIZE_BYTES = 256  # radius 2, Morgan 2048 bits\\n\\n\\ndef pool_row_digest(pool) -> str:\\n    \\"\\"\\"Hash ordered (key, SMILES, mass) without materializing a JSON dataset.\\"\\"\\"\\n    required = (\\"pool_key\\", \\"smiles\\", \\"mass\\")\\n    if any(col not in pool for col in required):\\n        raise ValueError(\\"Candidate pool lacks key, SMILES or mass\\")\\n    sha = hashlib.sha256()\\n    sha.update(b\\"casmi-pool-rows-v1\\\\n\\")\\n    for key, smiles, mass in pool.loc[:, list(required)].itertuples(index=False, name=None):\\n        if not isinstance(key, str) or not key:\\n            raise ValueError(\\"Invalid candidate pool key\\")\\n        if not isinstance(smiles, str) or not smiles:\\n            raise ValueError(\\"Missing candidate pool SMILES\\")\\n        try:\\n            mass_value = float(mass)\\n        except (ValueError, TypeError) as exc:\\n            raise ValueError(\\"Invalid candidate mass\\") from exc\\n        if not math.isfinite(mass_value) or mass_value <= 0:\\n            raise ValueError(\\"Invalid candidate mass\\")\\n        line = json.dumps([key, smiles, mass_value.hex()],\\n                          separators=(\\",\\", \\":\\"), ensure_ascii=True)\\n        sha.update(line.encode(\\"utf-8\\"))\\n        sha.update(b\\"\\\\n\\")\\n    return sha.hexdigest()\\n\\n\\ndef _array_digest(fps) -> str:\\n    if not isinstance(fps, np.ndarray) or fps.dtype != np.uint8:\\n        raise ValueError(\\"Packed fingerprints must be a NumPy uint8 array\\")\\n    if fps.ndim != 2 or fps.shape[1] != FP_SIZE_BYTES:\\n        raise ValueError(\\"Expected 2048-bit packed fingerprints with 256 columns\\")\\n    if not fps.flags.c_contiguous:\\n        raise ValueError(\\"Packed fingerprint array must be C-contiguous\\")\\n    payload = memoryview(fps).cast(\\"B\\")\\n    digest = hashlib.sha256()\\n    for start in range(0, len(payload), 1024 * 1024):\\n        digest.update(payload[start:start + 1024 * 1024])\\n    return digest.hexdigest()\\n\\n\\ndef make_fingerprint_provenance(pool, keys, fps) -> dict:\\n    \\"\\"\\"Reject mismatched row ordering and produce a content-bound descriptor.\\"\\"\\"\\n    names = np.asarray(keys)\\n    expected = pool[\\"pool_key\\"].to_numpy()\\n    if names.ndim != 1 or len(names) != len(expected):\\n        raise ValueError(\\"Fingerprint-key row count differs from candidate pool\\")\\n    if not np.array_equal(names, expected):\\n        raise ValueError(\\"Fingerprint keys do not match ordered candidate pool keys\\")\\n    if len(fps) != len(expected):\\n        raise ValueError(\\"Fingerprint-vector row count differs from candidate pool\\")\\n    return {\\n        \\"schema\\": SCHEMA,\\n        \\"n_structures\\": len(expected),\\n        \\"fingerprint\\": \\"Morgan radius=2, 2048 bits, packed uint8\\",\\n        \\"pool_rows_sha256\\": pool_row_digest(pool),\\n        \\"fingerprint_bytes_sha256\\": _array_digest(fps),\\n        \\"limitations\\": (\\n            \\"Local byte-alignment attestation only. Does not authenticate \\"\\n            \\"Kaggle provenance, reconstruct chemistry or prove model accuracy.\\"\\n        ),\\n    }\\n\\n\\ndef verify_fingerprint_provenance(pool, keys, fps, manifest) -> dict:\\n    \\"\\"\\"Fail closed for stale SMILES, reordered keys, mutated bits or old schema.\\"\\"\\"\\n    if not isinstance(manifest, dict) or manifest.get(\\"schema\\") != SCHEMA:\\n        raise ValueError(\\"Missing or unsupported fingerprint provenance schema\\")\\n    actual = make_fingerprint_provenance(pool, keys, fps)\\n    for field in (\\"n_structures\\", \\"fingerprint\\", \\"pool_rows_sha256\\",\\n                  \\"fingerprint_bytes_sha256\\"):\\n        if type(manifest.get(field)) is not type(actual[field]) or manifest[field] != actual[field]:\\n            raise ValueError(f\\"Fingerprint provenance mismatch: {field}\\")\\n    return {\\n        \\"status\\": \\"POOL_FINGERPRINTS_VERIFIED\\",\\n        \\"schema\\": SCHEMA,\\n        \\"n_structures\\": actual[\\"n_structures\\"],\\n        \\"pool_rows_sha256\\": actual[\\"pool_rows_sha256\\"],\\n        \\"fingerprint_bytes_sha256\\": actual[\\"fingerprint_bytes_sha256\\"],\\n    }\\n", "src/casmi/pool/pool.py": "\\"\\"\\"A3 \\u2014 candidate pool: unique structures (metric key) sorted by neutral monoisotopic mass.\\n\\nSources are tracked as a bitmask so the ranker can learn source priors (e.g. that structures known\\nonly from the drug-like enveda-180 library are unlikely answers for a natural-product test set)\\nand so validation can knock out a structure from the pool (Class-3 simulation).\\n\\"\\"\\"\\nfrom dataclasses import dataclass\\nimport hashlib\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nSOURCE_BITS = {\\"train\\": 1, \\"enveda180_only\\": 2, \\"coconut\\": 4, \\"chebi\\": 8, \\"lipidmaps\\": 16, \\"pubchem\\": 32, \\"pubchem_lit\\": 64}\\n\\n\\ndef generated_candidate_identity(mol):\\n    \\"\\"\\"Bind generated cache identity to full canonical representation, not InChI14 alone.\\n\\n    Standard InChI can merge tautomers whose representation-sensitive Morgan\\n    fingerprints and fragments differ. Content IDs keep those caches independent;\\n    official metric identity is still handled separately by MetricKeys.\\n    \\"\\"\\"\\n    from rdkit import Chem\\n    smiles = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)\\n    standard = Chem.MolToInchiKey(mol)[:14]\\n    return \\"G\\" + standard + \\":\\" + hashlib.sha256(smiles.encode()).hexdigest(), smiles\\n\\n\\n@dataclass\\nclass Window:\\n    idx: np.ndarray          # pool row indices\\n    ppm: np.ndarray          # signed ppm of each candidate relative to the query mass\\n\\n\\nclass CandidatePool:\\n    def __init__(self, table: pd.DataFrame):\\n        \\"\\"\\"table: pool_key, smiles, mass, sources (bitmask) [+ any extra per-structure columns].\\"\\"\\"\\n        needed = {\\"pool_key\\", \\"smiles\\", \\"mass\\"}\\n        if not needed.issubset(table.columns):\\n            raise ValueError(f\\"Candidate pool missing columns: {sorted(needed - set(table.columns))}\\")\\n        # Missing candidates can be dropped as before, but present impossible\\n        # masses must not silently produce NaN/inf ppm values or divide by 0.\\n        t = table.dropna(subset=[\\"pool_key\\", \\"mass\\"]).copy()\\n        try:\\n            mass = pd.to_numeric(t[\\"mass\\"], errors=\\"raise\\").to_numpy(dtype=np.float64)\\n        except (TypeError, ValueError) as exc:\\n            raise ValueError(\\"Candidate pool contains non-numeric masses\\") from exc\\n        if np.any(~np.isfinite(mass)) or np.any(mass <= 0.0):\\n            raise ValueError(\\"Candidate pool masses must be finite and strictly positive\\")\\n        t[\\"mass\\"] = mass\\n        t = t.sort_values(\\"mass\\", kind=\\"stable\\")\\n        t = t.drop_duplicates(\\"pool_key\\", keep=\\"first\\").reset_index(drop=True)\\n        self.table = t\\n        self.mass = t[\\"mass\\"].to_numpy(np.float64)\\n        self.key = t[\\"pool_key\\"].to_numpy()\\n        self.smiles = t[\\"smiles\\"].to_numpy()\\n        self.row_of = dict(zip(self.key, range(len(t))))\\n        # generated (Class-3 hedge) structures live outside the table: key -> SMILES / packed fingerprint\\n        self.extra_smiles, self.extra_fp = {}, {}\\n\\n    def smiles_of(self, key):\\n        r = self.row_of.get(key)\\n        return self.smiles[r] if r is not None else self.extra_smiles[key]\\n\\n    def register_generated(self, keys, smiles):\\n        \\"\\"\\"Atomically register chemically valid generated structures.\\n\\n        Active IDs bind standard InChIKey14 and canonical-SMILES SHA-256.\\n        Legacy \'G\'+InChIKey14 remains readable but ambiguous reuse is rejected.\\n        Invalid records cannot contribute\\n        all-zero fingerprints or leave a half-written candidate overlay.\\n        \\"\\"\\"\\n        from rdkit import Chem\\n        from rdkit.Chem import rdFingerprintGenerator\\n\\n        keys = list(keys)\\n        smiles = list(smiles)\\n        if len(keys) != len(smiles):\\n            raise ValueError(\\"Generated key and SMILES counts differ\\")\\n        gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)\\n        staged = {}\\n        for i, (key, smi) in enumerate(zip(keys, smiles)):\\n            if not isinstance(key, str) or not isinstance(smi, str):\\n                raise ValueError(f\\"Malformed generated candidate at row {i}\\")\\n            try:\\n                mol = Chem.MolFromSmiles(smi)\\n                if mol is None or mol.GetNumHeavyAtoms() == 0:\\n                    raise ValueError\\n                expected = \\"G\\" + Chem.MolToInchiKey(mol)[:14]\\n                content_key, canonical_smiles = generated_candidate_identity(mol)\\n                if key not in (expected, content_key):\\n                    raise ValueError\\n                previous = self.extra_smiles.get(key, staged.get(key, (None, None))[0])\\n                if previous is not None:\\n                    prior = Chem.MolToSmiles(Chem.MolFromSmiles(previous), canonical=True, isomericSmiles=True)\\n                    if prior != canonical_smiles:\\n                        raise ValueError\\n                    continue\\n                bits = gen.GetFingerprintAsNumPy(mol).astype(np.uint8)\\n                if bits.size != 2048 or not np.any(bits):\\n                    raise ValueError\\n            except (ValueError, TypeError, RuntimeError):\\n                # Never log the source SMILES or derived molecular identity.\\n                raise ValueError(f\\"Invalid generated candidate identity at row {i}\\") from None\\n            staged[key] = (canonical_smiles, np.packbits(bits))\\n        for key, (smi, fp) in staged.items():\\n            self.extra_smiles[key] = smi\\n            self.extra_fp[key] = fp\\n\\n    def packed_fps(self, keys, fps):\\n        \\"\\"\\"Aligned packed fingerprints for frozen pool and generated candidates.\\"\\"\\"\\n        if (not isinstance(fps, np.ndarray) or fps.dtype != np.uint8\\n                or fps.ndim != 2 or fps.shape != (len(self), 256)):\\n            raise ValueError(\\"Pool fingerprints must be row-aligned packed uint8 Morgan 2048 bits\\")\\n        names = list(keys)\\n        out = np.empty((len(names), 256), np.uint8)\\n        for i, key in enumerate(names):\\n            row = self.row_of.get(key)\\n            if row is not None:\\n                out[i] = fps[row]\\n            elif key in self.extra_fp:\\n                out[i] = self.extra_fp[key]\\n            else:\\n                raise KeyError(\\"Requested fingerprint key is absent from pool and generated overlay\\")\\n        return out\\n\\n    def __len__(self):\\n        return len(self.mass)\\n\\n    def window(self, query_mass, tol_ppm=10.0, exclude=None) -> Window:\\n        \\"\\"\\"All candidates with |ppm(candidate vs query)| <= tol_ppm, minus excluded keys.\\"\\"\\"\\n        try:\\n            if isinstance(query_mass, (bool, np.bool_)) or not np.isscalar(query_mass):\\n                raise ValueError\\n            mass_value = float(query_mass)\\n            if not np.isfinite(mass_value) or mass_value <= 0:\\n                raise ValueError\\n        except (TypeError, ValueError, OverflowError) as exc:\\n            raise ValueError(\\"Query mass must be a finite positive scalar\\") from exc\\n        try:\\n            if isinstance(tol_ppm, (bool, np.bool_)) or not np.isscalar(tol_ppm):\\n                raise ValueError\\n            tolerance = float(tol_ppm)\\n            if not np.isfinite(tolerance) or tolerance < 0:\\n                raise ValueError\\n        except (TypeError, ValueError, OverflowError) as exc:\\n            raise ValueError(\\"Mass tolerance ppm must be a finite nonnegative scalar\\") from exc\\n        d = mass_value * tolerance * 1e-6\\n        lo = np.searchsorted(self.mass, mass_value - d, side=\\"left\\")\\n        hi = np.searchsorted(self.mass, mass_value + d, side=\\"right\\")\\n        idx = np.arange(lo, hi)\\n        if exclude is not None:\\n            excluded = list(exclude)\\n            if excluded:\\n                idx = idx[~np.isin(self.key[idx], excluded)]\\n        return Window(idx, (mass_value - self.mass[idx]) / self.mass[idx] * 1e6)\\n", "src/casmi/pool/source_reconciliation.py": "\\"\\"\\"Fail-closed reconciliation before merging CASMI candidate source records.\\n\\nSame first-block InChIKey indicates matched connectivity but does not justify\\njoining arbitrarily different neutral masses. Some source rows lack complete\\nidentity fields; those may be excluded, but an impossible *present* mass and\\na material same-key mass contradiction must never be silently accepted.\\n\\nThis is a source-identity quality gate, not a proof that source providers\\nhave correct chemical annotations.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nREQUIRED = (\\"pool_key\\", \\"smiles\\", \\"mass\\")\\nDEFAULT_MAX_MASS_SPREAD_DA = 0.01\\nDEFAULT_MAX_MASS_SPREAD_PPM = 20.0\\n\\n\\ndef reconcile_source_rows(\\n    records: pd.DataFrame, *,\\n    max_spread_da: float = DEFAULT_MAX_MASS_SPREAD_DA,\\n    max_spread_ppm: float = DEFAULT_MAX_MASS_SPREAD_PPM,\\n    quarantine_conflicts: bool = False,\\n) -> tuple[pd.DataFrame, dict]:\\n    \\"\\"\\"Return whole, complete source rows after auditing duplicate key masses.\\n\\n    The intended source order (training library before external providers)\\n    is preserved. Crucially, the first source\'s SMILES and mass must both\\n    exist on the SAME row before groupby.first() is used by the caller.\\n    \\"\\"\\"\\n    if not isinstance(records, pd.DataFrame):\\n        raise TypeError(\\"Candidate source records must be a DataFrame\\")\\n    absent = sorted(set(REQUIRED) - set(records.columns))\\n    if absent:\\n        raise ValueError(\\"Candidate sources missing required fields: \\" + \\", \\".join(absent))\\n    if (not np.isfinite(max_spread_da) or max_spread_da < 0\\n            or not np.isfinite(max_spread_ppm) or max_spread_ppm < 0):\\n        raise ValueError(\\"Mass disagreement tolerances must be finite nonnegative values\\")\\n    complete = records.dropna(subset=list(REQUIRED)).copy()\\n    omitted = len(records) - len(complete)\\n    if complete.empty:\\n        raise ValueError(\\"No complete candidate source identity rows available\\")\\n    # A source key is the standard InChIKey connectivity block. Reject\\n    # malformed *present* keys instead of merging them as new structures.\\n    if (not complete[\\"pool_key\\"].map(lambda value:\\n                                    isinstance(value, str) and len(value) == 14\\n                                    and value.isascii() and value.isalpha()\\n                                    and value.isupper()).all()):\\n        raise ValueError(\\"Malformed candidate source InChIKey14\\")\\n    if not complete[\\"smiles\\"].map(lambda value:\\n                                   isinstance(value, str) and bool(value.strip())).all():\\n        raise ValueError(\\"Empty or malformed candidate source SMILES\\")\\n    try:\\n        mass = pd.to_numeric(complete[\\"mass\\"], errors=\\"raise\\").to_numpy(dtype=np.float64)\\n    except (TypeError, ValueError, OverflowError) as exc:\\n        raise ValueError(\\"Non-numeric candidate source mass\\") from exc\\n    if np.any(~np.isfinite(mass)) or np.any(mass <= 0):\\n        raise ValueError(\\"Candidate source mass must be finite and positive\\")\\n    complete[\\"mass\\"] = mass\\n\\n    ranges = complete.groupby(\\"pool_key\\", sort=False)[\\"mass\\"].agg([\\"min\\", \\"max\\"])\\n    discrepancies = (ranges[\\"max\\"] - ranges[\\"min\\"]).to_numpy(dtype=float)\\n    thresholds = np.maximum(\\n        float(max_spread_da),\\n        ranges[\\"min\\"].to_numpy(dtype=float) * float(max_spread_ppm) * 1e-6,\\n    )\\n    conflict_keys = ranges.index[discrepancies > thresholds]\\n    conflicts = len(conflict_keys)\\n    if conflicts and not quarantine_conflicts:\\n        # Aggregate count only: no licensed source identifiers, chemical\\n        # structures, or per-molecule annotations escape into log messages.\\n        raise ValueError(\\n            f\\"Conflicting neutral mass for {conflicts} shared InChIKey14 groups; \\"\\n            \\"quarantine or correct providers before building a new candidate pool\\"\\n        )\\n    quarantined = complete.pool_key.isin(conflict_keys)\\n    quarantined_rows = int(quarantined.sum())\\n    if conflicts:\\n        complete = complete.loc[~quarantined].copy()\\n        if complete.empty:\\n            raise ValueError(\\"No complete candidate rows remain after conflict quarantine\\")\\n    summary = {\\n        \\"status\\": \\"SOURCE_ROWS_RECONCILED\\",\\n        \\"input_rows\\": int(len(records)),\\n        \\"complete_rows\\": int(len(complete)),\\n        \\"incomplete_rows_excluded\\": int(omitted),\\n        \\"distinct_pool_keys\\": int(complete.pool_key.nunique()),\\n        \\"shared_pool_keys\\": int((complete.groupby(\\"pool_key\\").size() > 1).sum()),\\n        \\"mass_conflict_groups\\": 0,\\n        \\"mass_conflict_policy\\": \\"quarantine_entire_identity\\" if quarantine_conflicts else \\"reject\\",\\n        \\"quarantined_mass_conflict_groups\\": int(conflicts),\\n        \\"quarantined_source_rows\\": quarantined_rows,\\n        \\"max_source_mass_spread_da\\": float(discrepancies.max(initial=0.0)),\\n        \\"mass_threshold_da\\": float(max_spread_da),\\n        \\"mass_threshold_ppm\\": float(max_spread_ppm),\\n        \\"interpretation\\": \\"Pre-aggregation identity consistency only; not RDKit verification\\",\\n    }\\n    return complete, summary\\n", "src/casmi/pool/structure_identity.py": "\\"\\"\\"Chemically consistent neutral parent identities for multi-fragment inputs.\\n\\nCOCONUT supplies some multi-component SMILES strings. Taking the largest\\nfragment without recomputing its standard InChIKey and monoisotopic mass\\ncreates an inconsistent candidate identity. Candidate pool keys are standard\\n(non-tautomer-canonical) InChIKey first blocks, as in the train.parquet pool.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\n\\ndef neutral_parent_identity(smiles: str) -> tuple[str, str, float, str] | None:\\n    \\"\\"\\"Return (parent canonical SMILES, standard InChIKey14, exact mass, formula).\\n\\n    None signifies malformed SMILES, missing InChI support, or a charged\\n    parent. No placeholder molecule is ever substituted for invalid input.\\n    Largest *heavy-atom* fragment is selected, preserving the original\\n    source\'s parent-fragment rule. Caller must decide which rows to discard.\\n    \\"\\"\\"\\n    from rdkit import Chem\\n    from rdkit.Chem.Descriptors import ExactMolWt\\n    from rdkit.Chem import rdMolDescriptors\\n    import math\\n\\n    if not isinstance(smiles, str) or not smiles.strip():\\n        return None\\n    try:\\n        molecule = Chem.MolFromSmiles(smiles)\\n        if molecule is None:\\n            return None\\n        parts = Chem.GetMolFrags(molecule, asMols=True)\\n        if not parts:\\n            return None\\n        parent = max(parts, key=lambda mol: mol.GetNumHeavyAtoms())\\n        if parent.GetNumHeavyAtoms() == 0 or Chem.GetFormalCharge(parent) != 0:\\n            return None\\n        canonical = Chem.MolToSmiles(parent, canonical=True)\\n        standard_key = Chem.MolToInchiKey(parent)\\n        mass = float(ExactMolWt(parent))\\n        formula = rdMolDescriptors.CalcMolFormula(parent)\\n    except (ValueError, RuntimeError, TypeError):\\n        return None\\n    if (not isinstance(standard_key, str) or len(standard_key) < 14\\n            or not standard_key[:14].isalpha()\\n            or not math.isfinite(mass) or mass <= 0):\\n        return None\\n    return canonical, standard_key[:14], mass, formula\\n", "src/casmi/pool/structure_quality.py": "\\"\\"\\"Explicit whole-candidate quarantine before immutable pool publication.\\n\\nThis verifies parseability and nonzero Morgan features, not chemical validity,\\nprovider identity accuracy or biological plausibility. No row details are logged.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom concurrent.futures import ProcessPoolExecutor\\n\\nimport numpy as np\\n\\n\\ndef _invalid_reasons(smiles):\\n    from rdkit import Chem, RDLogger\\n    from rdkit.Chem import rdFingerprintGenerator\\n\\n    RDLogger.DisableLog(\\"rdApp.*\\")\\n    generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)\\n    reasons = []\\n    for value in smiles:\\n        try:\\n            mol = Chem.MolFromSmiles(value) if isinstance(value, str) else None\\n            if mol is None:\\n                reasons.append(\\"unparseable\\")\\n            elif mol.GetNumHeavyAtoms() == 0:\\n                reasons.append(\\"no_heavy_atoms\\")\\n            elif generator.GetFingerprint(mol).GetNumOnBits() == 0:\\n                reasons.append(\\"zero_morgan\\")\\n            else:\\n                reasons.append(\\"\\")\\n        except (ValueError, TypeError, RuntimeError):\\n            reasons.append(\\"fingerprint_error\\")\\n    return reasons\\n\\n\\ndef quarantine_invalid_candidates(pool, *, workers=1):\\n    \\"\\"\\"Return retained rows and aggregate accounting; never silently zero-fill.\\"\\"\\"\\n    if not pool.pool_key.is_unique:\\n        raise ValueError(\\"Structure quarantine requires one candidate per identity\\")\\n    values = pool.smiles.tolist()\\n    chunks = [values[start:start + 5000] for start in range(0, len(values), 5000)]\\n    if workers == 1:\\n        nested = [_invalid_reasons(chunk) for chunk in chunks]\\n    else:\\n        with ProcessPoolExecutor(max_workers=workers) as executor:\\n            nested = list(executor.map(_invalid_reasons, chunks))\\n    reasons = np.asarray([reason for chunk in nested for reason in chunk], dtype=object)\\n    kept = pool.loc[reasons == \\"\\"].copy().reset_index(drop=True)\\n    if kept.empty:\\n        raise ValueError(\\"No candidates remain after structure quarantine\\")\\n    return kept, {\\n        \\"policy\\": \\"quarantine_invalid_selected_candidate_identity\\",\\n        \\"input_candidates\\": len(pool), \\"retained_candidates\\": len(kept),\\n        \\"quarantined_candidates\\": int(np.count_nonzero(reasons != \\"\\")),\\n        \\"reasons\\": {name: int(np.count_nonzero(reasons == name))\\n                    for name in (\\"unparseable\\", \\"no_heavy_atoms\\", \\"zero_morgan\\", \\"fingerprint_error\\")},\\n        \\"scope\\": \\"Parseable, heavy-atom-containing, nonzero Morgan structures; not provider identity authentication\\",\\n    }\\n", "src/casmi/predict.py": "\\"\\"\\"End-to-end prediction for one molecule or a whole test file \\u2014 the code path the submission runs.\\n\\n    engine features (D1, D2, D3) -> priors + list context -> ranker score -> metric-key dedupe\\n    -> 25 SMILES, best first\\n\\nEvery molecule is wrapped in a fallback: if anything fails, it is answered with its mass-window\\ncandidates ordered by mass error, so a single bad molecule can never cost a row (a missing row\\nscores 0 and an exception would lose the whole submission).\\n\\"\\"\\"\\nimport time\\nimport traceback\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom casmi.rank.features import add_context, add_priors\\n\\n\\nclass Predictor:\\n    def __init__(self, engine, pool, keys, ranker=None, fp_models=None, device=\\"cpu\\"):\\n        self.engine, self.pool, self.keys = engine, pool, keys\\n        self.ranker, self.fp_models, self.device = ranker, fp_models or [], device\\n        self.timings = {\\"features\\": 0.0, \\"d3\\": 0.0, \\"d4\\": 0.0, \\"rank\\": 0.0, \\"dedupe\\": 0.0}\\n        self.frag = None\\n        self.frag_x = None\\n        self.failures = []\\n\\n    # ------------------------------------------------------------------ one molecule\\n    def _d3(self, f, spectra):\\n        from casmi.channels.fpmodel import candidate_scores\\n        from casmi.channels.fpinfer import molecule_logp\\n        t = time.perf_counter()\\n        lp_on, lp_off = molecule_logp(self.fp_models, spectra, self.device)\\n        bits = np.unpackbits(self.pool.packed_fps(f.pool_key.to_numpy(), self.engine.fps), axis=1).astype(np.float32)\\n        f[\\"d3_ll\\"], f[\\"d3_cos\\"] = candidate_scores(lp_on, lp_off, bits)\\n        for c in (\\"d3_ll\\", \\"d3_cos\\"):\\n            v = f[c].to_numpy()\\n            f[f\\"{c}_rank\\"] = pd.Series(v).rank(ascending=False, method=\\"min\\").to_numpy()\\n            f[f\\"{c}_gap\\"] = v.max() - v\\n            sd = v.std()\\n            f[f\\"{c}_z\\"] = (v - v.mean()) / sd if sd > 0 else 0.0\\n        self.timings[\\"d3\\"] += time.perf_counter() - t\\n        return f\\n\\n    def features(self, spectra, *, excluded_rows=None, excluded_keys=None,\\n                 knocked_keys=frozenset()):\\n        \\"\\"\\"Use the inference path with explicit research-only reference/pool knockouts.\\n\\n        Defaults preserve ordinary inference. Validation also clears reference-count\\n        priors for withheld structures rather than leaking their original membership.\\n        \\"\\"\\"\\n        started = time.perf_counter()\\n        f = self.engine.features([dict(s) for s in spectra], excluded_rows, excluded_keys)\\n        self.timings[\\"features\\"] += time.perf_counter() - started\\n        if f.empty:\\n            return f\\n        f = add_context(add_priors(f, self.pool, knocked_keys))\\n        if self.fp_models:\\n            f = self._d3(f, spectra)\\n        if self.ranker is not None and any(c.startswith(\\"d4_\\") for c in self.ranker.columns):\\n            from casmi.channels.fragments import FragmentScorer, add_d4\\n            t = time.perf_counter()\\n            if self.frag is None:\\n                self.frag = FragmentScorer(self.pool.smiles_of)\\n            f = add_d4(f, self.frag, spectra)\\n            if any(c.startswith(\\"d4x_\\") for c in self.ranker.columns):\\n                from casmi.channels.fragments import D4X\\n                if self.frag_x is None:\\n                    self.frag_x = FragmentScorer(self.pool.smiles_of, **D4X)\\n                f = add_d4(f, self.frag_x, spectra, \\"d4x\\")\\n            self.timings[\\"d4\\"] += time.perf_counter() - t\\n        return f\\n\\n    def rank(self, spectra, top_n=25, *, excluded_rows=None, excluded_keys=None,\\n             knocked_keys=frozenset()):\\n        f = self.features(spectra, excluded_rows=excluded_rows, excluded_keys=excluded_keys,\\n                          knocked_keys=knocked_keys)\\n        if f.empty:\\n            return []\\n        t = time.perf_counter()\\n        if self.ranker is not None:\\n            order = np.argsort(-self.ranker.predict(f), kind=\\"stable\\")\\n            ranked = f.pool_key.to_numpy()[order][:60].tolist()\\n        else:\\n            ranked = self.engine.rank(f)\\n        self.timings[\\"rank\\"] += time.perf_counter() - t\\n        t = time.perf_counter()\\n        out = self.keys.dedupe_ranked(ranked, self.pool.smiles_of, top_n)\\n        self.timings[\\"dedupe\\"] += time.perf_counter() - t\\n        return out                                   # [(pool_key, metric_key)] best first\\n\\n    def fallback_keys(self, spectra, top_n=25):\\n        \\"\\"\\"Mass-window candidates ordered by mass error: the answer when the model path fails.\\"\\"\\"\\n        from casmi.engine import molecule_mass\\n        win = self.pool.window(molecule_mass(spectra), 20.0)\\n        keys = self.pool.key[win.idx][np.argsort(np.abs(win.ppm))].tolist()\\n        return self.keys.dedupe_ranked(keys, self.pool.smiles_of, top_n)\\n\\n    # ------------------------------------------------------------------ whole test file\\n    def predict_file(self, test: pd.DataFrame, log_every=50) -> pd.DataFrame:\\n        rows, t0 = [], time.time()\\n        groups = test.groupby(\\"molecule_id\\", sort=False)\\n        for i, (mid, g) in enumerate(groups):\\n            spectra = [dict(mz=np.asarray(r.ms2_mzs, np.float64), it=np.asarray(r.ms2_normalized_intensities, np.float64),\\n                            precursor_mz=float(r.precursor_mz), adduct=str(r.adduct),\\n                            ce_ev=float(np.mean(r.collision_energy_ev)) if r.collision_energy_ev is not None and len(r.collision_energy_ev) else np.nan,\\n                            mode=str(r.ionization_mode)) for r in g.itertuples()]\\n            try:\\n                picked = self.rank(spectra)\\n            except Exception:\\n                self.failures.append((mid, traceback.format_exc(limit=2)))\\n                picked = []\\n            if len(picked) < 25:                     # refill by metric key, so no structure appears twice\\n                seen = {mk for _, mk in picked}\\n                picked += [(pk, mk) for pk, mk in self.fallback_keys(spectra) if mk not in seen][: 25 - len(picked)]\\n            smiles = [self.pool.smiles_of(pk) for pk, _ in picked]\\n            rows.append((mid, \\";\\".join(smiles) if smiles else \\"C\\"))   # never an empty field\\n            if log_every and (i + 1) % log_every == 0:\\n                print(f\\"  {i + 1}/{groups.ngroups} molecules, {time.time() - t0:.0f}s\\", flush=True)\\n        self.keys.save()\\n        return pd.DataFrame(rows, columns=[\\"molecule_id\\", \\"smiles\\"])\\n\\n\\ndef validate_submission(sub: pd.DataFrame, test: pd.DataFrame, max_rank=25):\\n    \\"\\"\\"The format rules the scorer enforces, checked before writing.\\"\\"\\"\\n    assert list(sub.columns) == [\\"molecule_id\\", \\"smiles\\"], sub.columns\\n    assert not sub.isna().any().any(), \\"nulls\\"\\n    assert sub.molecule_id.is_unique, \\"duplicate molecule_id\\"\\n    missing = set(test.molecule_id) - set(sub.molecule_id)\\n    assert not missing, f\\"{len(missing)} molecules missing\\"\\n    n = sub.smiles.str.split(\\";\\").map(len)\\n    assert (n >= 1).all() and (n <= max_rank).all(), \\"guess count out of range\\"\\n", "src/casmi/rank/__init__.py": "", "src/casmi/rank/features.py": "\\"\\"\\"Ranker features: channel outputs + knockout-consistent priors + within-molecule list context.\\n\\nTwo leakage rules are enforced here, because the ranker would otherwise learn them:\\n  * priors derived from training-set membership (has spectra, spectrum count, which libraries)\\n    are recomputed after the knockout \\u2014 a simulated Class-2 answer must look exactly like a real\\n    one, i.e. like a structure with no reference spectra;\\n  * list-context features are computed per molecule over the candidates the engine actually saw.\\n\\"\\"\\"\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom casmi.pool.pool import SOURCE_BITS\\n\\nCHANNEL_COLS = [\\"d1_max\\", \\"d1_mean\\", \\"an_p1\\", \\"an_p2\\", \\"an_p4\\", \\"an_sim\\", \\"an_tani\\", \\"an_self\\"]\\nCONTEXT_OF = [\\"d1_max\\", \\"an_p4\\", \\"an_p2\\", \\"an_tani\\"]\\n\\n\\ndef add_priors(feats: pd.DataFrame, pool, knocked_keys=frozenset()) -> pd.DataFrame:\\n    t = pool.table\\n    rows = np.fromiter((pool.row_of.get(k, -1) for k in feats.pool_key), np.int64, len(feats))\\n    gen = rows < 0                                     # generated structures: in no source\\n    knocked = feats.pool_key.isin(knocked_keys).to_numpy() | gen\\n    rows = np.where(gen, 0, rows)\\n    src = np.where(gen, 0, t.sources.to_numpy()[rows])\\n    in_train = ((src & SOURCE_BITS[\\"train\\"]) > 0) & ~knocked\\n    feats = feats.assign(\\n        prior_in_train=in_train.astype(np.int8),\\n        prior_in_coconut=((src & SOURCE_BITS[\\"coconut\\"]) > 0).astype(np.int8),\\n        prior_enveda180_only=(((src & SOURCE_BITS[\\"enveda180_only\\"]) > 0) & ~knocked).astype(np.int8),\\n        prior_log_spectra=np.where(knocked, 0.0, np.log1p(t.n_spectra.to_numpy()[rows])),\\n        prior_np_likeness=np.where(gen, np.nan, t.np_likeness.to_numpy(dtype=float)[rows]),\\n        prior_has_sugar=np.where(gen, np.nan, t.has_sugar.map({True: 1.0, False: 0.0}).to_numpy(dtype=float)[rows]),\\n        abs_ppm=np.abs(feats.ppm.to_numpy()),\\n    )\\n    return feats\\n\\n\\ndef add_context(feats: pd.DataFrame) -> pd.DataFrame:\\n    \\"\\"\\"Rank, gap-to-best and z-score of key channels within the molecule\'s candidate list.\\"\\"\\"\\n    out = {}\\n    for c in CONTEXT_OF:\\n        if c not in feats:\\n            continue\\n        v = feats[c].to_numpy(dtype=float)\\n        best = v.max() if len(v) else 0.0\\n        sd = v.std() if len(v) > 1 else 0.0\\n        out[f\\"{c}_rank\\"] = pd.Series(v).rank(ascending=False, method=\\"min\\").to_numpy()\\n        out[f\\"{c}_gap\\"] = best - v\\n        out[f\\"{c}_z\\"] = (v - v.mean()) / sd if sd > 0 else np.zeros_like(v)\\n    out[\\"log_n_cand\\"] = np.full(len(feats), np.log1p(len(feats)))\\n    return feats.assign(**out)\\n\\n\\ndef feature_columns(feats: pd.DataFrame):\\n    drop = {\\"pool_key\\", \\"ppm\\", \\"n_cand\\", \\"label\\", \\"qid\\", \\"cls\\", \\"gen_smiles\\", \\"stratum\\"}\\n    return [c for c in feats.columns if c not in drop and not c.startswith(\\"audit_\\")]\\n", "src/casmi/rank/ranker.py": "\\"\\"\\"E \\u2014 LightGBM LambdaRank over per-molecule candidate lists, seed-bagged.\\n\\nGrouping is by (query, simulated class): each group is one molecule\'s candidate list with exactly\\none relevant item (or none, for recall misses \\u2014 those groups carry no gradient and are dropped).\\n\\"\\"\\"\\nfrom dataclasses import dataclass, field\\n\\nimport lightgbm as lgb\\nimport numpy as np\\nimport pandas as pd\\n\\nMONOTONE_UP = {\\"d1_max\\", \\"d1_mean\\", \\"an_p1\\", \\"an_p2\\", \\"an_p4\\", \\"an_self\\"}\\n\\n\\n@dataclass\\nclass RankerConfig:\\n    seeds: tuple = (0, 1, 2, 3, 4)\\n    params: dict = field(default_factory=lambda: dict(\\n        objective=\\"lambdarank\\", metric=\\"map\\", eval_at=[25], lambdarank_truncation_level=25,\\n        learning_rate=0.05, num_leaves=31, min_data_in_leaf=50, feature_fraction=0.8,\\n        bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, verbose=-1))\\n    num_boost_round: int = 2000\\n    early_stopping: int = 100\\n\\n\\nclass Ranker:\\n    def __init__(self, cfg: RankerConfig = RankerConfig()):\\n        self.cfg, self.models, self.columns = cfg, [], None\\n\\n    @staticmethod\\n    def _groups(df):\\n        return df.groupby([\\"qid\\", \\"cls\\"], sort=False).size().to_numpy()\\n\\n    def fit(self, train: pd.DataFrame, valid: pd.DataFrame, columns, class_weight=None):\\n        self.columns = list(columns)\\n        keep = lambda d: d[d.groupby([\\"qid\\", \\"cls\\"]).label.transform(\\"max\\") > 0]\\n        train, valid = keep(train), keep(valid)\\n        mono = [1 if c in MONOTONE_UP else 0 for c in self.columns]\\n        w = train.cls.map(class_weight).fillna(1.0).to_numpy() if class_weight else None\\n        dtr = lgb.Dataset(train[self.columns], train.label, group=self._groups(train), weight=w, free_raw_data=False)\\n        dva = lgb.Dataset(valid[self.columns], valid.label, group=self._groups(valid), reference=dtr)\\n        self.models, self.best_iters = [], []\\n        for seed in self.cfg.seeds:\\n            params = {**self.cfg.params, \\"seed\\": seed, \\"monotone_constraints\\": mono}\\n            m = lgb.train(params, dtr, self.cfg.num_boost_round, valid_sets=[dva],\\n                          callbacks=[lgb.early_stopping(self.cfg.early_stopping, verbose=False)])\\n            self.models.append(m)\\n            self.best_iters.append(m.best_iteration)\\n        return self\\n\\n    def predict(self, feats: pd.DataFrame) -> np.ndarray:\\n        X = feats[self.columns]\\n        return np.mean([m.predict(X, num_iteration=m.best_iteration) for m in self.models], axis=0)\\n\\n    def importance(self) -> pd.Series:\\n        imp = np.mean([m.feature_importance(\\"gain\\") for m in self.models], axis=0)\\n        return pd.Series(imp, index=self.columns).sort_values(ascending=False)\\n", "src/casmi/spectra/__init__.py": "", "src/casmi/spectra/build_safety.py": "\\"\\"\\"Exclusive, staged publication of immutable CASMI spectrum-store artifacts.\\n\\nThe A1 builder must never open an existing spectrum store for writing.\\nWriting four related output files directly into the public experiment\'s\\nhistorical store can silently truncate valid metadata/peak snapshots if\\na build fails midway. This context manager stages a complete new directory\\non the same filesystem and publishes it only after the caller\'s verification.\\n\\nNo external dependencies or CI environment required.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom contextlib import contextmanager\\nimport os\\nfrom pathlib import Path\\nimport shutil\\nimport tempfile\\n\\n\\n@contextmanager\\ndef fresh_store_target(destination):\\n    \\"\\"\\"Yield a new staging directory; publish once only on successful exit.\\n\\n    Existing destinations, including empty directories and dangling symlinks,\\n    are refused. The staging directory is removed on errors. The caller must\\n    complete and verify every artifact before leaving the context.\\n    The same-file-system rename makes publication atomic on supported OSes;\\n    parallel writers should be managed externally with a project lock.\\n    \\"\\"\\"\\n    destination = Path(destination)\\n    if not destination.name or destination.name in (\\".\\", \\"..\\"):\\n        raise ValueError(\\"An explicit, non-root output directory is required\\")\\n    if os.path.lexists(destination):\\n        raise FileExistsError(\\n            f\\"Refusing to overwrite an existing spectrum store: {destination}\\"\\n        )\\n    destination.parent.mkdir(parents=True, exist_ok=True)\\n    stage = Path(tempfile.mkdtemp(\\n        prefix=f\\".{destination.name}-building-\\", dir=destination.parent,\\n    ))\\n    try:\\n        yield stage\\n        if os.path.lexists(destination):\\n            raise FileExistsError(\\n                f\\"Output appeared during build; refusing to overwrite: {destination}\\"\\n            )\\n        stage.rename(destination)\\n    except BaseException:\\n        shutil.rmtree(stage, ignore_errors=True)\\n        raise\\n", "src/casmi/spectra/clean.py": "\\"\\"\\"Peak cleaning. One function is used for library spectra (offline) and query spectra (online),\\nso the two can never drift apart.\\n\\n`clean_peaks` is the readable NumPy reference; `clean_peaks_fast` is the numba version used on\\nmillions of spectra. tests/test_spectra.py asserts they agree on random spectra.\\n\\"\\"\\"\\nfrom dataclasses import dataclass\\n\\nimport numba as nb\\nimport numpy as np\\n\\n\\n@dataclass(frozen=True)\\nclass CleanConfig:\\n    max_above_precursor: float = 2.0   # Da; a fragment cannot outweigh its singly charged precursor\\n    min_rel_intensity: float = 0.001   # relative to base peak\\n    window_da: float = 50.0            # windowed top-k keeps low-mass fragments a global top-N drops\\n    top_k_per_window: int = 0          # 0 disables windowed filtering\\n    max_peaks: int = 150               # global cap after the filters above\\n    merge_tol_da: float = 0.005        # peaks closer than this are merged (centroiding artefacts)\\n\\n\\ndef clean_peaks(mz, intensity, precursor_mz, cfg: CleanConfig = CleanConfig()):\\n    \\"\\"\\"Return (mz, intensity) sorted by m/z, float32, intensity normalised to base peak = 1.\\"\\"\\"\\n    mz = np.asarray(mz, dtype=np.float64)\\n    it = np.asarray(intensity, dtype=np.float64)\\n    # Fragment m/z must be strictly positive; negative/zero bins can corrupt\\n    # downstream sparse-vector indexing and spectral similarity assumptions.\\n    keep = np.isfinite(mz) & (mz > 0) & np.isfinite(it) & (it > 0)\\n    if precursor_mz is not None and np.isfinite(precursor_mz):\\n        keep &= mz <= precursor_mz + cfg.max_above_precursor\\n    mz, it = mz[keep], it[keep]\\n    if mz.size == 0:\\n        return np.empty(0, np.float32), np.empty(0, np.float32)\\n\\n    order = np.argsort(mz, kind=\\"stable\\")\\n    mz, it = mz[order], it[order]\\n    if cfg.merge_tol_da > 0 and mz.size > 1:\\n        # merge runs of peaks closer than merge_tol_da: intensity-weighted m/z, summed intensity\\n        group = np.concatenate([[0], np.cumsum(np.diff(mz) > cfg.merge_tol_da)])\\n        if group[-1] + 1 < mz.size:\\n            w = np.bincount(group, weights=it)\\n            mz = np.bincount(group, weights=mz * it) / w\\n            it = w\\n\\n    it = it / it.max()\\n    keep = it >= cfg.min_rel_intensity\\n    mz, it = mz[keep], it[keep]\\n\\n    if cfg.top_k_per_window and mz.size > cfg.top_k_per_window:\\n        win = np.floor(mz / cfg.window_da).astype(np.int64)\\n        # rank peaks within each window by intensity (descending)\\n        order = np.lexsort((-it, win))\\n        rank = np.empty(mz.size, np.int64)\\n        starts = np.r_[0, np.flatnonzero(np.diff(win[order])) + 1]\\n        run = np.arange(mz.size) - np.repeat(starts, np.diff(np.r_[starts, mz.size]))\\n        rank[order] = run\\n        keep = rank < cfg.top_k_per_window\\n        mz, it = mz[keep], it[keep]\\n\\n    if mz.size > cfg.max_peaks:\\n        top = np.argpartition(-it, cfg.max_peaks - 1)[:cfg.max_peaks]\\n        top.sort()\\n        mz, it = mz[top], it[top]\\n    return mz.astype(np.float32), it.astype(np.float32)\\n\\n\\n@nb.njit(cache=True)\\ndef _clean_nb(mz, it, precursor_mz, max_above, min_rel, window_da, top_k, max_peaks, merge_tol):\\n    n = mz.size\\n    keep = np.empty(n, np.bool_)\\n    lim = precursor_mz + max_above if np.isfinite(precursor_mz) else np.inf\\n    m = 0\\n    for i in range(n):\\n        ok = np.isfinite(mz[i]) and mz[i] > 0 and np.isfinite(it[i]) and it[i] > 0 and mz[i] <= lim\\n        keep[i] = ok\\n        m += ok\\n    mz, it = mz[keep], it[keep]\\n    if m == 0:\\n        return np.empty(0, np.float32), np.empty(0, np.float32)\\n    order = np.argsort(mz, kind=\\"mergesort\\")\\n    mz, it = mz[order], it[order]\\n\\n    # chain-merge consecutive peaks closer than merge_tol\\n    out_mz = np.empty(m, np.float64)\\n    out_it = np.empty(m, np.float64)\\n    g = 0\\n    wsum, isum = mz[0] * it[0], it[0]\\n    for i in range(1, m):\\n        if merge_tol > 0 and mz[i] - mz[i - 1] <= merge_tol:\\n            wsum += mz[i] * it[i]\\n            isum += it[i]\\n        else:\\n            out_mz[g], out_it[g] = wsum / isum, isum\\n            g += 1\\n            wsum, isum = mz[i] * it[i], it[i]\\n    out_mz[g], out_it[g] = wsum / isum, isum\\n    g += 1\\n    mz, it = out_mz[:g], out_it[:g]\\n\\n    it = it / it.max()\\n    sel = it >= min_rel\\n    mz, it = mz[sel], it[sel]\\n\\n    if top_k > 0 and mz.size > top_k:\\n        win = np.floor(mz / window_da)\\n        keepw = np.zeros(mz.size, np.bool_)\\n        s = 0\\n        while s < mz.size:\\n            e = s\\n            while e < mz.size and win[e] == win[s]:\\n                e += 1\\n            seg = np.argsort(-it[s:e], kind=\\"mergesort\\")\\n            for j in range(min(top_k, e - s)):\\n                keepw[s + seg[j]] = True\\n            s = e\\n        mz, it = mz[keepw], it[keepw]\\n\\n    if mz.size > max_peaks:\\n        top = np.sort(np.argsort(-it, kind=\\"mergesort\\")[:max_peaks])\\n        mz, it = mz[top], it[top]\\n    return mz.astype(np.float32), it.astype(np.float32)\\n\\n\\ndef clean_peaks_fast(mz, intensity, precursor_mz, cfg: CleanConfig = CleanConfig()):\\n    \\"\\"\\"numba equivalent of `clean_peaks` (~50x faster).\\"\\"\\"\\n    p = np.nan if precursor_mz is None else float(precursor_mz)\\n    return _clean_nb(np.asarray(mz, np.float64), np.asarray(intensity, np.float64), p,\\n                     cfg.max_above_precursor, cfg.min_rel_intensity, cfg.window_da,\\n                     cfg.top_k_per_window, cfg.max_peaks, cfg.merge_tol_da)\\n", "src/casmi/spectra/similarity.py": "\\"\\"\\"Spectral similarities, compiled with numba.\\n\\nEntropy similarity (Li et al., Nature Methods 2021) with intensity weighting:\\n    sim = sum over matched peak pairs [ (a+b)ln(a+b) - a ln a - b ln b ] / ln 4\\nwith each spectrum\'s intensities normalised to sum 1. This closed form follows from\\n1 - (2 S_AB - S_A - S_B) / ln 4 and needs only the matched pairs.\\n\\n`shift` enables the mass-shifted (\\"modified\\") variant used for analog search: a peak may match\\neither at the same m/z or at m/z + shift, where shift = precursor difference between the spectra.\\n\\"\\"\\"\\nimport numba as nb\\nimport numpy as np\\n\\nLN4 = np.log(4.0)\\n\\n\\n@nb.njit(cache=True)\\ndef _entropy(p):\\n    s = 0.0\\n    for x in p:\\n        if x > 0:\\n            s -= x * np.log(x)\\n    return s\\n\\n\\n@nb.njit(cache=True)\\ndef _valid_spectrum(mz, it):\\n    \\"\\"\\"Reject invalid numerical peak arrays before any logarithm or square root.\\"\\"\\"\\n    if mz.ndim != 1 or it.ndim != 1 or mz.size != it.size:\\n        raise ValueError(\\"Spectrum m/z and intensity must be aligned 1D arrays\\")\\n    for i in range(mz.size):\\n        if (not np.isfinite(mz[i]) or mz[i] < 0.0\\n                or not np.isfinite(it[i]) or it[i] < 0.0):\\n            return False\\n    return True\\n\\n\\n@nb.njit(cache=True)\\ndef weight_intensity(it):\\n    \\"\\"\\"Normalise positive intensity; return zeros for zero/invalid total.\\"\\"\\"\\n    total = it.sum()\\n    if not np.isfinite(total) or total <= 0.0:\\n        return np.zeros(it.size, dtype=np.float64)\\n    p = it / total\\n    s = _entropy(p)\\n    if s < 3.0:\\n        w = 0.25 + 0.25 * s\\n        p = p ** w\\n        p = p / p.sum()\\n    return p\\n\\n\\n@nb.njit(cache=True)\\ndef _match(mz_a, mz_b, tol_da, shift):\\n    \\"\\"\\"Greedy one-to-one matching of peaks within tol_da, allowing an optional mass shift.\\n    Returns index pairs sorted by m/z proximity (closest pairs claimed first).\\"\\"\\"\\n    na, nb_ = mz_a.size, mz_b.size\\n    cand_i, cand_j, cand_d = [], [], []\\n    for s_idx in range(2 if shift != 0.0 else 1):\\n        off = shift if s_idx == 1 else 0.0\\n        j0 = 0\\n        for i in range(na):\\n            lo = mz_a[i] + off - tol_da\\n            while j0 < nb_ and mz_b[j0] < lo:\\n                j0 += 1\\n            j = j0\\n            while j < nb_ and mz_b[j] <= mz_a[i] + off + tol_da:\\n                cand_i.append(i)\\n                cand_j.append(j)\\n                cand_d.append(abs(mz_b[j] - mz_a[i] - off))\\n                j += 1\\n    n = len(cand_i)\\n    order = np.argsort(np.array(cand_d)) if n else np.empty(0, np.int64)\\n    used_a = np.zeros(na, np.bool_)\\n    used_b = np.zeros(nb_, np.bool_)\\n    pi, pj = [], []\\n    for k in order:\\n        i, j = cand_i[k], cand_j[k]\\n        if not used_a[i] and not used_b[j]:\\n            used_a[i] = True\\n            used_b[j] = True\\n            pi.append(i)\\n            pj.append(j)\\n    return pi, pj\\n\\n\\n@nb.njit(cache=True)\\ndef entropy_similarity(mz_a, it_a, mz_b, it_b, tol_da=0.02, shift=0.0):\\n    if not _valid_spectrum(mz_a, it_a) or not _valid_spectrum(mz_b, it_b):\\n        return 0.0\\n    if mz_a.size == 0 or mz_b.size == 0:\\n        return 0.0\\n    pa = weight_intensity(it_a.astype(np.float64))\\n    pb = weight_intensity(it_b.astype(np.float64))\\n    if pa.sum() <= 0.0 or pb.sum() <= 0.0:\\n        return 0.0\\n    pi, pj = _match(mz_a.astype(np.float64), mz_b.astype(np.float64), tol_da, shift)\\n    s = 0.0\\n    for k in range(len(pi)):\\n        a, b = pa[pi[k]], pb[pj[k]]\\n        if a <= 0.0 or b <= 0.0:\\n            continue  # 0*log(0) is mathematically zero, not NaN.\\n        s += (a + b) * np.log(a + b) - a * np.log(a) - b * np.log(b)\\n    return s / LN4\\n\\n\\n@nb.njit(cache=True)\\ndef cosine_similarity(mz_a, it_a, mz_b, it_b, tol_da=0.02, shift=0.0):\\n    \\"\\"\\"Greedy (modified, if shift != 0) cosine on sqrt intensities.\\"\\"\\"\\n    if not _valid_spectrum(mz_a, it_a) or not _valid_spectrum(mz_b, it_b):\\n        return 0.0\\n    if mz_a.size == 0 or mz_b.size == 0:\\n        return 0.0\\n    a = np.sqrt(it_a.astype(np.float64))\\n    b = np.sqrt(it_b.astype(np.float64))\\n    pi, pj = _match(mz_a.astype(np.float64), mz_b.astype(np.float64), tol_da, shift)\\n    s = 0.0\\n    for k in range(len(pi)):\\n        s += a[pi[k]] * b[pj[k]]\\n    norm = np.sqrt((a * a).sum()) * np.sqrt((b * b).sum())\\n    return s / norm if norm > 0.0 and np.isfinite(norm) else 0.0\\n", "src/casmi/spectra/store.py": "\\"\\"\\"A1 \\u2014 on-disk spectrum store: cleaned peaks as flat float32 memmaps + a metadata table.\\n\\nLayout (artifacts/store/):\\n  mz.f32, it.f32   all peaks back to back\\n  offsets.i64      spectrum i owns peaks[offsets[i]:offsets[i+1]]\\n  meta.parquet     one row per spectrum (library, structure, adduct, masses, ppm, CE, ...)\\n\\"\\"\\"\\nfrom pathlib import Path\\n\\nimport operator\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom casmi.spectra.store_integrity import inspect_store_layout\\n\\n\\nclass SpectrumStore:\\n    def __init__(self, root: Path):\\n        root = Path(root)\\n        self.meta = pd.read_parquet(root / \\"meta.parquet\\")\\n        if \\"n_peaks\\" not in self.meta:\\n            raise ValueError(\\"Spectrum metadata lacks per-row n_peaks counts\\")\\n        layout = inspect_store_layout(\\n            root, expected_spectra=len(self.meta),\\n            metadata_n_peaks=self.meta[\\"n_peaks\\"].to_numpy(),\\n        )\\n        self.offsets = layout.offsets\\n        self.layout = layout\\n        if layout.peak_count:\\n            self.mz = np.memmap(root / \\"mz.f32\\", dtype=np.float32, mode=\\"r\\")\\n            self.it = np.memmap(root / \\"it.f32\\", dtype=np.float32, mode=\\"r\\")\\n        else:\\n            # A valid empty peak store cannot be memmapped by NumPy.\\n            self.mz = np.empty(0, dtype=np.float32)\\n            self.it = np.empty(0, dtype=np.float32)\\n\\n    def close(self):\\n        \\"\\"\\"Release memory-mapped peak files (important for Windows file locks).\\n\\n        Any arrays returned by peaks() must not be accessed after close().\\n        Calling close() repeatedly is safe. The source files are not changed.\\n        \\"\\"\\"\\n        for name in (\\"mz\\", \\"it\\"):\\n            array = getattr(self, name, None)\\n            mapped = getattr(array, \\"_mmap\\", None)\\n            if mapped is not None and not mapped.closed:\\n                mapped.close()\\n        self.mz = None\\n        self.it = None\\n\\n    def __enter__(self):\\n        return self\\n\\n    def __exit__(self, exc_type, exc_value, traceback):\\n        self.close()\\n        return False\\n\\n    def __len__(self):\\n        return len(self.meta)\\n\\n    def peaks(self, i):\\n        if self.mz is None or self.it is None:\\n            raise ValueError(\\"SpectrumStore is closed\\")\\n        if isinstance(i, (bool, np.bool_)):\\n            raise TypeError(\\"Spectrum index must be an integer row ID\\")\\n        row = operator.index(i)\\n        if row < 0 or row >= len(self.meta):\\n            raise IndexError(f\\"Spectrum row {row} is outside the store\\")\\n        a, b = int(self.offsets[row]), int(self.offsets[row + 1])\\n        return np.asarray(self.mz[a:b]), np.asarray(self.it[a:b])\\n", "src/casmi/spectra/store_integrity.py": "\\"\\"\\"Fast fail-closed structural checks for an existing CASMI MS/MS peak store.\\n\\nThe store format consists of contiguous float32 m/z/intensity arrays, int64\\noffsets, and a separate metadata Parquet file. This module reads the entire\\noffset vector (small relative to spectra), but NEVER scans peak values.\\n\\nLayout verification confirms file lengths, offsets, row counts and per-row\\npeak counts. It does not establish chemical correctness, non-finite peak\\ncontent, upstream Kaggle provenance, or source-library independence.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom dataclasses import dataclass\\nfrom pathlib import Path\\n\\nimport numpy as np\\n\\n\\n@dataclass(frozen=True)\\nclass StoreLayout:\\n    \\"\\"\\"Checked store offsets with metadata-only count diagnostics.\\"\\"\\"\\n    offsets: np.ndarray\\n    spectrum_count: int\\n    peak_count: int\\n    empty_spectra: int\\n    max_peaks_per_spectrum: int\\n\\n    def summary(self) -> dict:\\n        return {\\n            \\"status\\": \\"STRUCTURE_VERIFIED\\",\\n            \\"spectrum_count\\": self.spectrum_count,\\n            \\"peak_count\\": self.peak_count,\\n            \\"empty_spectra\\": self.empty_spectra,\\n            \\"max_peaks_per_spectrum\\": self.max_peaks_per_spectrum,\\n            \\"method\\": \\"offsets and file byte lengths; not a full peak-content scan\\",\\n        }\\n\\n\\ndef inspect_store_layout(root, *, expected_spectra=None,\\n                         metadata_n_peaks=None) -> StoreLayout:\\n    \\"\\"\\"Validate raw files before mmap; raise ValueError on structural drift.\\n\\n    Integers in offsets.i64 are native-endian signed int64 by store design,\\n    and mz.f32 / it.f32 are native-endian float32. Check byte alignment\\n    before trusting them; avoid Python assertions (which can be disabled).\\n    \\"\\"\\"\\n    root = Path(root)\\n    offsets_file = root / \\"offsets.i64\\"\\n    mz_file = root / \\"mz.f32\\"\\n    it_file = root / \\"it.f32\\"\\n    offset_bytes = offsets_file.stat().st_size\\n    mz_bytes = mz_file.stat().st_size\\n    it_bytes = it_file.stat().st_size\\n    if offset_bytes < 16 or offset_bytes % 8:\\n        raise ValueError(\\"Invalid offsets.i64 byte length: at least two aligned offsets required\\")\\n    if mz_bytes % 4 or it_bytes % 4:\\n        raise ValueError(\\"Spectrum peak files must have float32-aligned byte lengths\\")\\n    if mz_bytes != it_bytes:\\n        raise ValueError(\\"Spectrum m/z and intensity files have different peak counts\\")\\n    offsets = np.fromfile(offsets_file, dtype=np.int64)\\n    n_spectra = len(offsets) - 1\\n    n_peaks = mz_bytes // 4\\n    if expected_spectra is not None:\\n        if type(expected_spectra) is not int or expected_spectra < 0:\\n            raise ValueError(\\"Expected spectrum row count must be a nonnegative integer\\")\\n        if n_spectra != expected_spectra:\\n            raise ValueError(\\"Spectrum metadata row count and offset count differ\\")\\n    if offsets[0] != 0:\\n        raise ValueError(\\"First spectrum offset must equal zero\\")\\n    if np.any(offsets < 0) or np.any(offsets[1:] < offsets[:-1]):\\n        raise ValueError(\\"Spectrum offsets must be nonnegative and nondecreasing\\")\\n    if int(offsets[-1]) != n_peaks:\\n        raise ValueError(\\"Final spectrum offset does not equal flat peak count\\")\\n    counts = offsets[1:] - offsets[:-1]\\n\\n    if metadata_n_peaks is not None:\\n        try:\\n            declared = np.asarray(metadata_n_peaks, dtype=np.float64)\\n        except (TypeError, ValueError) as exc:\\n            raise ValueError(\\"Invalid metadata n_peaks values\\") from exc\\n        if declared.ndim != 1 or len(declared) != n_spectra:\\n            raise ValueError(\\"Metadata n_peaks must align with spectrum rows\\")\\n        if (not np.all(np.isfinite(declared)) or np.any(declared < 0)\\n                or np.any(declared != np.floor(declared))):\\n            raise ValueError(\\"Metadata n_peaks contains missing or invalid counts\\")\\n        if not np.array_equal(declared, counts):\\n            raise ValueError(\\"Metadata n_peaks disagrees with spectrum offsets\\")\\n\\n    return StoreLayout(\\n        offsets=offsets, spectrum_count=n_spectra, peak_count=n_peaks,\\n        empty_spectra=int(np.count_nonzero(counts == 0)),\\n        max_peaks_per_spectrum=int(np.max(counts, initial=0)),\\n    )\\n\\ndef scan_store_peak_content(root, *, chunk_peaks: int = 262144) -> dict:\\n    \\"\\"\\"Optionally scan every stored m/z and intensity using bounded memory.\\n\\n    The standard store reader and inspect_store_layout deliberately avoid the\\n    multi-GB content scan. This separate diagnostic does not load Parquet,\\n    mutate peak files, or reveal individual spectra. Only aggregate counts\\n    leave the analysis. A \\"clean\\" result is about numeric peak admissibility,\\n    not chemical correctness or the origin of the source files.\\n    \\"\\"\\"\\n    if type(chunk_peaks) is not int or not 1 <= chunk_peaks <= 1048576:\\n        raise ValueError(\\"chunk_peaks must be an integer from 1 to 1048576\\")\\n    root = Path(root)\\n    layout = inspect_store_layout(root)\\n    nonfinite_mz = nonpositive_mz = 0\\n    nonfinite_intensity = nonpositive_intensity = 0\\n    invalid_peak_pairs = 0\\n    read = 0\\n\\n    with (root / \\"mz.f32\\").open(\\"rb\\") as mz_file, (root / \\"it.f32\\").open(\\"rb\\") as it_file:\\n        remaining = layout.peak_count\\n        while remaining:\\n            count = min(chunk_peaks, remaining)\\n            mz = np.fromfile(mz_file, dtype=np.float32, count=count)\\n            it = np.fromfile(it_file, dtype=np.float32, count=count)\\n            if len(mz) != count or len(it) != count:\\n                raise ValueError(\\"Peak file truncated during content scan\\")\\n            mz_finite = np.isfinite(mz)\\n            it_finite = np.isfinite(it)\\n            mz_nonpositive = mz_finite & (mz <= 0)\\n            it_nonpositive = it_finite & (it <= 0)\\n            nonfinite_mz += int(np.count_nonzero(~mz_finite))\\n            nonpositive_mz += int(np.count_nonzero(mz_nonpositive))\\n            nonfinite_intensity += int(np.count_nonzero(~it_finite))\\n            nonpositive_intensity += int(np.count_nonzero(it_nonpositive))\\n            invalid_peak_pairs += int(np.count_nonzero(\\n                ~(mz_finite & it_finite) | mz_nonpositive | it_nonpositive\\n            ))\\n            read += count\\n            remaining -= count\\n\\n        if mz_file.read(1) or it_file.read(1):\\n            raise ValueError(\\"Peak file grew during content scan\\")\\n\\n    invalid = invalid_peak_pairs > 0\\n    return {\\n        \\"status\\": \\"INVALID_CLEANED_PEAKS\\" if invalid else \\"VALID_CLEANED_PEAKS\\",\\n        \\"total_peaks_scanned\\": read,\\n        \\"nonfinite_mz\\": nonfinite_mz,\\n        \\"nonpositive_mz\\": nonpositive_mz,\\n        \\"nonfinite_intensity\\": nonfinite_intensity,\\n        \\"nonpositive_intensity\\": nonpositive_intensity,\\n        \\"invalid_peak_pairs\\": invalid_peak_pairs,\\n        \\"chunk_peaks\\": chunk_peaks,\\n        \\"method\\": \\"full numeric content scan in bounded float32 chunks\\",\\n        \\"interpretation\\": (\\n            \\"Aggregate cleaned-peak numeric validity only; not chemistry, \\"\\n            \\"data-lineage, leakage or instrument accuracy\\"\\n        ),\\n    }\\n", "src/casmi/submit/__init__.py": "", "src/casmi/validate/__init__.py": "", "src/casmi/validate/analog_index_evidence.py": "\\"\\"\\"Decision-grade D2 index evidence for signed CASMI validation reports.\\n\\nLegacy cached analog index files were created without source provenance.\\nA run without a *freshly* rebuilt reference matrix remains exploratory even\\nwhen the source manifest and candidate Morgan bits have been verified.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom collections.abc import Mapping\\n\\nINDEX_SCHEMA = \\"casmi-d2-fresh-index-v1\\"\\nINDEX_STATUS = \\"REBUILT_WITHOUT_CACHE\\"\\n\\n\\ndef verified_analog_index_evidence(run: Mapping) -> bool:\\n    \\"\\"\\"Validate reported cache-free index provenance, not external data truth.\\"\\"\\"\\n    proof = run.get(\\"analog_index_provenance\\")\\n    if proof is None:\\n        return False\\n    if not isinstance(proof, Mapping):\\n        raise ValueError(\\"Malformed D2 index provenance\\")\\n    if proof.get(\\"schema\\") != INDEX_SCHEMA or proof.get(\\"status\\") != INDEX_STATUS:\\n        raise ValueError(\\"Unverified D2 index provenance schema or status\\")\\n    if proof.get(\\"cache_used\\") is not False:\\n        raise ValueError(\\"D2 index provenance must explicitly bypass the cache\\")\\n    digest = proof.get(\\"index_sha256\\")\\n    if not isinstance(digest, str) or len(digest) != 64 or any(\\n        char not in \\"0123456789abcdef\\" for char in digest\\n    ):\\n        raise ValueError(\\"Invalid D2 index content digest\\")\\n    rows = proof.get(\\"n_references\\")\\n    nnz = proof.get(\\"n_nonzero\\")\\n    if type(rows) is not int or rows < 1:\\n        raise ValueError(\\"Invalid D2 index reference count\\")\\n    if type(nnz) is not int or nnz < 0:\\n        raise ValueError(\\"Invalid D2 index nonzero count\\")\\n    cfg = run.get(\\"config\\")\\n    analog = cfg.get(\\"analog\\") if isinstance(cfg, Mapping) else None\\n    policy = analog.get(\\"reference_policy\\") if isinstance(analog, Mapping) else None\\n    if not isinstance(policy, str) or policy not in (\\"legacy\\", \\"source_balanced\\"):\\n        raise ValueError(\\"Missing or unsupported D2 reference-selection policy\\")\\n    if proof.get(\\"reference_policy\\") != policy:\\n        raise ValueError(\\"D2 index policy contradicts model configuration\\")\\n    return True\\n", "src/casmi/validate/budget_features.py": "\\"\\"\\"Recompute list-relative features after selecting a declared candidate budget.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom casmi.rank.features import add_context\\n\\n\\ndef budget_features(frame, policy):\\n    if policy not in (\\"legacy\\", \\"expanded\\"):\\n        raise ValueError(\\"Unknown declared candidate-budget policy\\")\\n    if frame.empty:\\n        return frame.copy()\\n    if \\"audit_legacy\\" not in frame or not frame.audit_legacy.isin([0, 1]).all():\\n        raise ValueError(\\"Candidate origin audit is missing or invalid\\")\\n    selected = frame if policy == \\"expanded\\" else frame[frame.audit_legacy == 1]\\n    selected = selected.drop(columns=[c for c in selected if c.startswith(\\"audit_\\")]).reset_index(drop=True).copy()\\n    if selected.empty:\\n        return selected\\n    selected[\\"n_cand\\"] = len(selected)\\n    selected = add_context(selected)\\n    for column in (\\"d3_ll\\", \\"d3_cos\\", \\"d4_int\\", \\"d4_pk\\", \\"d4x_int\\", \\"d4x_pk\\"):\\n        if column not in selected:\\n            continue\\n        values = selected[column].to_numpy(dtype=float)\\n        sd = values.std()\\n        selected[column + \\"_rank\\"] = pd.Series(values).rank(ascending=False, method=\\"min\\").to_numpy()\\n        selected[column + \\"_gap\\"] = values.max() - values\\n        selected[column + \\"_z\\"] = (values - values.mean()) / sd if sd > 0 else np.zeros_like(values)\\n    return selected\\n", "src/casmi/validate/cohort_bootstrap.py": "\\"\\"\\"Joint fixed-family uncertainty with independent resampling across source cohorts.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport numpy as np\\n\\nfrom .rank_inputs import validated_bootstrap\\n\\n\\ndef joint_cohort_intervals(effects_by_cohort, *, n_boot=4000, seed=0):\\n    validated_bootstrap(n_boot, seed)\\n    if not effects_by_cohort:\\n        raise ValueError(\\"At least one declared source cohort required\\")\\n    groups, names_seen = {}, set()\\n    for cohort, effects in sorted(effects_by_cohort.items()):\\n        if not effects or set(effects) & names_seen:\\n            raise ValueError(\\"Empty or duplicate secondary contrast family\\")\\n        names_seen.update(effects)\\n        arrays = [np.asarray(value, dtype=float) for value in effects.values()]\\n        if any(a.ndim != 1 or len(a) < 2 or not np.isfinite(a).all() or np.any(np.abs(a) > 1) for a in arrays):\\n            raise ValueError(\\"Finite bounded paired effects from at least two molecules required\\")\\n        if len({len(a) for a in arrays}) != 1:\\n            raise ValueError(\\"Every contrast in one cohort must share paired molecules\\")\\n        groups[cohort] = (list(effects), np.column_stack(arrays))\\n    maximum = np.zeros(n_boot)\\n    outcomes, counts = {}, {}\\n    for child_seed, (cohort, (names, values)) in zip(np.random.SeedSequence(seed).spawn(len(groups)), groups.items()):\\n        mean = values.mean(axis=0)\\n        counts[cohort] = len(values)\\n        rng = np.random.default_rng(child_seed)\\n        for start in range(0, n_boot, 100):\\n            ids = rng.integers(len(values), size=(min(100, n_boot - start), len(values)))\\n            deviations = np.max(np.abs(values[ids].mean(axis=1) - mean), axis=1)\\n            maximum[start:start + len(ids)] = np.maximum(maximum[start:start + len(ids)], deviations)\\n        outcomes.update({name: {\\"cohort\\": cohort, \\"delta\\": float(value)} for name, value in zip(names, mean)})\\n    radius = float(np.quantile(maximum, .95))\\n    for value in outcomes.values():\\n        value.update(ci_low=value[\\"delta\\"] - radius, ci_high=value[\\"delta\\"] + radius)\\n    return {\\"method\\": \\"Independent molecule resampling across cohorts; paired within cohort; fixed-family max-deviation bootstrap\\",\\n            \\"n_bootstrap\\": n_boot, \\"seed\\": seed, \\"family_size\\": len(outcomes), \\"cohort_counts\\": counts,\\n            \\"joint_radius\\": radius, \\"outcomes\\": outcomes}\\n", "src/casmi/validate/control_alignment.py": "\\"\\"\\"Identity-verified, diagnostic-only pairing of spectral negative controls.\\n\\nThis module deliberately does not accept legacy/unverified runs or produce\\nmodel-improvement ACCEPT/REJECT judgments. It verifies that the reference and\\nperturbed reports use the same molecule IDs, underlying original spectra,\\nnovelty exclusion scope, model configuration and query sampling.\\n\\nSeparate because align_runs must continue to reject altered MS/MS evidence.\\nNo numerical/chemistry dependency is required.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom collections.abc import Mapping\\nfrom copy import deepcopy\\n\\nfrom casmi.validate.run_alignment import CLASSES, SIGNATURE_VERSION, _check_ranks\\n\\n\\ndef _signatures(run, ids, key):\\n    values = run.get(key)\\n    if not isinstance(values, list) or len(values) != len(ids):\\n        raise ValueError(f\\"{key}: one spectrum SHA-256 required per query\\")\\n    if any(not isinstance(v, str) or len(v) != 64\\n           or any(c not in \\"0123456789abcdef\\" for c in v) for v in values):\\n        raise ValueError(f\\"{key}: invalid SHA-256 fingerprint\\")\\n    return dict(zip(ids, values))\\n\\n\\ndef _identities(run, title):\\n    count = run.get(\\"n_queries\\")\\n    ids = run.get(\\"query_ids\\")\\n    if type(count) is not int or count < 1 or not isinstance(ids, list) or len(ids) != count:\\n        raise ValueError(f\\"{title}: invalid query count or identity manifest\\")\\n    if any(not isinstance(item, str) or not item for item in ids) or len(set(ids)) != count:\\n        raise ValueError(f\\"{title}: duplicate or missing query identities\\")\\n    version = run.get(\\"query_signature_version\\")\\n    if version != SIGNATURE_VERSION:\\n        raise ValueError(f\\"{title}: unsupported query signature recipe\\")\\n    ranks = _check_ranks(run, count)\\n    return ids, _signatures(run, ids, \\"query_signatures\\"), ranks\\n\\n\\ndef align_control_runs(reference: Mapping, treatment: Mapping) -> dict:\\n    \\"\\"\\"Align diagnostic ranks by query ID, verify source fingerprints.\\n\\n    `reference` must be an unperturbed validation report. `treatment` must\\n    be a negative-control run with original signatures captured *before*\\n    treatment. A changed model/config/sampling invalidates the diagnostic.\\n    \\"\\"\\"\\n    if not isinstance(reference, Mapping) or not isinstance(treatment, Mapping):\\n        raise ValueError(\\"Both reports must be mappings\\")\\n    rc, tc = reference.get(\\"spectral_control\\"), treatment.get(\\"spectral_control\\")\\n    if not isinstance(rc, Mapping) or rc.get(\\"mode\\") != \\"none\\":\\n        raise ValueError(\\"Reference must be a signed unperturbed control=none run\\")\\n    if not isinstance(tc, Mapping) or tc.get(\\"mode\\") not in (\\"blank\\", \\"intensity_shuffle\\"):\\n        raise ValueError(\\"Treatment must be a signed negative-control run\\")\\n    if type(tc.get(\\"seed\\")) is not int or tc[\\"seed\\"] < 0:\\n        raise ValueError(\\"Treatment seed must be a nonnegative integer\\")\\n    a_ids, a_signatures, a_ranks = _identities(reference, \\"Reference\\")\\n    b_ids, b_signatures, b_ranks = _identities(treatment, \\"Treatment\\")\\n    if len(a_ids) != len(b_ids) or set(a_ids) != set(b_ids):\\n        raise ValueError(\\"Control population/query ID mismatch\\")\\n    source_a = reference.get(\\"source_manifest_sha256\\")\\n    source_b = treatment.get(\\"source_manifest_sha256\\")\\n    if source_a is not None or source_b is not None:\\n        if (not isinstance(source_a, str) or len(source_a) != 64\\n                or any(ch not in \\"0123456789abcdef\\" for ch in source_a)\\n                or source_a != source_b):\\n            raise ValueError(\\"Negative control source artifact snapshots differ\\")\\n    if reference.get(\\"config\\") != treatment.get(\\"config\\"):\\n        raise ValueError(\\"Control cannot be compared across model configurations\\")\\n    sampling_a, sampling_b = reference.get(\\"query_sampling\\"), treatment.get(\\"query_sampling\\")\\n    if not isinstance(sampling_a, Mapping) or not isinstance(sampling_b, Mapping):\\n        raise ValueError(\\"Missing query sampling provenance\\")\\n    keep_a = {k: v for k, v in sampling_a.items()\\n              if k not in (\\"spectral_control\\", \\"control_seed\\")}\\n    keep_b = {k: v for k, v in sampling_b.items()\\n              if k not in (\\"spectral_control\\", \\"control_seed\\")}\\n    if keep_a != keep_b:\\n        raise ValueError(\\"Control query sampling differs\\")\\n    if sampling_a.get(\\"spectral_control\\") != \\"none\\":\\n        raise ValueError(\\"Reference sampling not marked as unperturbed\\")\\n    if (sampling_b.get(\\"spectral_control\\") != tc[\\"mode\\"]\\n            or sampling_b.get(\\"control_seed\\") != tc[\\"seed\\"]):\\n        raise ValueError(\\"Treatment mode or seed contradicts query sampling\\")\\n\\n    source_signatures = _signatures(\\n        {\\"query_signatures\\": tc.get(\\"source_query_signatures\\",\\n                                  treatment.get(\\"spectral_control\\", {}).get(\\n                                      \\"source_query_signatures\\"))},\\n        b_ids, \\"query_signatures\\",\\n    )\\n    if source_signatures != a_signatures:\\n        raise ValueError(\\"Control source spectra or reference-exclusion scope differs\\")\\n    if not any(b_signatures[qid] != source_signatures[qid] for qid in a_ids):\\n        raise ValueError(\\"No actual spectral perturbation detected\\")\\n    index_b = {qid: i for i, qid in enumerate(b_ids)}\\n    return {\\n        \\"n_queries\\": len(a_ids),\\n        \\"control_mode\\": tc[\\"mode\\"],\\n        \\"control_seed\\": tc[\\"seed\\"],\\n        \\"verified_source_pairing\\": True,\\n        \\"changed_queries\\": sum(\\n            b_signatures[qid] != source_signatures[qid] for qid in a_ids\\n        ),\\n        \\"reference_ranks\\": deepcopy(a_ranks),\\n        \\"treatment_ranks\\": {\\n            cls: [b_ranks[cls][index_b[qid]] for qid in a_ids] for cls in CLASSES\\n        },\\n    }\\n", "src/casmi/validate/coverage_budget.py": "\\"\\"\\"Descriptive candidate-coverage and ranking-loss accounting from aggregates.\\n\\nNo per-query outcomes are inferred. Conditional means describe the observed\\ncovered subset and are unsuitable for comparing models with different pools.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport math\\n\\n\\ndef coverage_budget(summary, truth_candidates_present):\\n    n = summary[\\"n\\"]\\n    if type(n) is not int or n < 1 or type(truth_candidates_present) is not int or not 0 <= truth_candidates_present <= n:\\n        raise ValueError(\\"Positive integer population and bounded integer coverage count required\\")\\n    values = {}\\n    for field in (\\"mrr\\", \\"top1\\", \\"top10\\", \\"recall_at_25\\"):\\n        value = summary[field]\\n        if type(value) not in (int, float) or not math.isfinite(value) or not 0 <= value <= 1:\\n            raise ValueError(\\"Finite bounded aggregate metrics required\\")\\n        values[field] = value\\n    counts = {}\\n    for field in (\\"top1\\", \\"top10\\", \\"recall_at_25\\"):\\n        count = round(values[field] * n)\\n        if not math.isclose(count, values[field] * n, abs_tol=1e-9, rel_tol=0):\\n            raise ValueError(\\"Aggregate hit rate does not correspond to integer molecules\\")\\n        counts[field] = count\\n    if not 0 <= counts[\\"top1\\"] <= counts[\\"top10\\"] <= counts[\\"recall_at_25\\"] <= truth_candidates_present:\\n        raise ValueError(\\"Ranked hits exceed candidate coverage or violate nested cutoffs\\")\\n    ceiling = truth_candidates_present / n\\n    mrr = values[\\"mrr\\"]\\n    # Non-top-1 hits contribute at most 1/2; top-10 and top-25\\n    # hit groups contribute at least 1/10 and 1/25 respectively.\\n    lower = (counts[\\"top1\\"] + (counts[\\"top10\\"] - counts[\\"top1\\"]) / 10\\n             + (counts[\\"recall_at_25\\"] - counts[\\"top10\\"]) / 25) / n\\n    upper = (counts[\\"top1\\"] + (counts[\\"top10\\"] - counts[\\"top1\\"]) / 2\\n             + (counts[\\"recall_at_25\\"] - counts[\\"top10\\"]) / 11) / n\\n    if not lower - 1e-12 <= mrr <= upper + 1e-12:\\n        raise ValueError(\\"MRR contradicts the observed top-1/top-10/top-25 counts\\")\\n    return {\\n        \\"n_molecules\\": n,\\n        \\"truth_candidates_present\\": truth_candidates_present,\\n        \\"truth_candidates_absent\\": n - truth_candidates_present,\\n        \\"top1_hits\\": counts[\\"top1\\"], \\"top10_hits\\": counts[\\"top10\\"], \\"top25_hits\\": counts[\\"recall_at_25\\"],\\n        \\"covered_outside_top25\\": truth_candidates_present - counts[\\"recall_at_25\\"],\\n        \\"coverage_ceiling\\": ceiling, \\"observed_mrr\\": mrr,\\n        \\"coverage_loss\\": 1 - ceiling, \\"within_coverage_ranking_loss\\": max(0.0, ceiling - mrr),\\n        \\"conditional_mrr_given_coverage\\": mrr / ceiling if truth_candidates_present else None,\\n        \\"conditional_hit25_given_coverage\\": counts[\\"recall_at_25\\"] / truth_candidates_present if truth_candidates_present else None,\\n    }\\n", "src/casmi/validate/error_slices.py": "\\"\\"\\"Prespecified molecule-level diagnostic slices for CASMI research validation.\\n\\nGroupings use only input metadata. All outcomes are descriptive, with small\\ncells suppressed and no post-hoc statistical ACCEPT/REJECT claims. This module\\nrequires only the Python standard library and signed validation reports.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport math\\nfrom collections import defaultdict\\nfrom collections.abc import Mapping\\nfrom statistics import median\\n\\nfrom casmi.validate.run_alignment import CLASSES, align_runs\\n\\nFIELDS = (\\"adduct\\", \\"ionization_mode\\", \\"spectra_bin\\", \\"precursor_mz_bin\\")\\nMASS_BINS = (\\"<300\\", \\"300-600\\", \\">=600\\", \\"unknown\\")\\n\\n\\ndef describe_query(spectra) -> dict:\\n    \\"\\"\\"Compute one query\'s *original* non-label metadata, before perturbation.\\"\\"\\"\\n    if not isinstance(spectra, (list, tuple)) or not spectra:\\n        raise ValueError(\\"Query must contain at least one selected spectrum\\")\\n    def shared(field):\\n        values = {str(s[field]) for s in spectra}\\n        return next(iter(values)) if len(values) == 1 else \\"mixed\\"\\n\\n    n = len(spectra)\\n    masses = [float(s[\\"precursor_mz\\"]) for s in spectra]\\n    masses = [m for m in masses if math.isfinite(m)]\\n    midpoint = median(masses) if masses else math.nan\\n    mass_bin = (\\"unknown\\" if not math.isfinite(midpoint) else\\n                \\"<300\\" if midpoint < 300 else\\n                \\"300-600\\" if midpoint < 600 else \\">=600\\")\\n    return {\\n        \\"adduct\\": shared(\\"adduct\\"),\\n        \\"ionization_mode\\": shared(\\"mode\\"),\\n        \\"spectra_bin\\": \\"1\\" if n == 1 else \\"2-3\\" if n <= 3 else \\"4+\\",\\n        \\"precursor_mz_bin\\": mass_bin,\\n    }\\n\\n\\ndef _checked_contexts(run, ids, label):\\n    context = run.get(\\"query_context\\")\\n    if not isinstance(context, list) or len(context) != len(ids):\\n        raise ValueError(f\\"{label}: missing or incomplete query_context\\")\\n    by_id = {}\\n    for qid, item in zip(ids, context):\\n        if not isinstance(item, Mapping) or set(item) != set(FIELDS):\\n            raise ValueError(f\\"{label}: invalid query_context fields\\")\\n        row = dict(item)\\n        if any(not isinstance(row[field], str) or not row[field]\\n               for field in FIELDS):\\n            raise ValueError(f\\"{label}: empty or malformed query_context value\\")\\n        if row[\\"spectra_bin\\"] not in (\\"1\\", \\"2-3\\", \\"4+\\"):\\n            raise ValueError(f\\"{label}: invalid spectra-count bin\\")\\n        if row[\\"precursor_mz_bin\\"] not in MASS_BINS:\\n            raise ValueError(f\\"{label}: invalid precursor-mass bin\\")\\n        by_id[qid] = row\\n    return by_id\\n\\n\\ndef _mrr(ranks, indices):\\n    return sum(1.0 / ranks[i] for i in indices if ranks[i]) / len(indices)\\n\\n\\ndef diagnostic_slices(reference: Mapping, candidate: Mapping, *, min_n: int = 15) -> dict:\\n    \\"\\"\\"Compare known, query-aligned context slices; NEVER infer superiority.\\"\\"\\"\\n    if type(min_n) is not int or min_n < 2:\\n        raise ValueError(\\"min_n must be an integer >= 2\\")\\n    aligned = align_runs(reference, candidate)\\n    ids = reference[\\"query_ids\\"]\\n    b_ids = candidate[\\"query_ids\\"]\\n    context_a = _checked_contexts(reference, ids, \\"Reference\\")\\n    context_b = _checked_contexts(candidate, b_ids, \\"Candidate\\")\\n    if context_a != context_b:\\n        raise ValueError(\\"Query input metadata differ across validation runs\\")\\n    groups = {}\\n    for field in FIELDS:\\n        cells = defaultdict(list)\\n        for index, qid in enumerate(ids):\\n            cells[context_a[qid][field]].append(index)\\n        records = []\\n        for value in sorted(cells):\\n            subset = cells[value]\\n            row = {\\"value\\": value, \\"n_molecules\\": len(subset)}\\n            if len(subset) < min_n:\\n                row.update(status=\\"SUPPRESSED_SMALL_CELL\\")\\n            else:\\n                classes = {}\\n                for cls in CLASSES:\\n                    a = _mrr(aligned[\\"a\\"][cls], subset)\\n                    b = _mrr(aligned[\\"b\\"][cls], subset)\\n                    classes[cls] = {\\n                        \\"reference_mrr\\": a,\\n                        \\"candidate_mrr\\": b,\\n                        \\"candidate_minus_reference\\": b - a,\\n                    }\\n                row.update(status=\\"DESCRIPTIVE_ONLY\\", classes=classes)\\n            records.append(row)\\n        groups[field] = records\\n    return {\\n        \\"status\\": \\"DESCRIPTIVE_ONLY\\",\\n        \\"source_evidence\\": (\\n            \\"VERIFIED_REQUIRED_INPUTS\\" if aligned[\\"decision_eligible\\"]\\n            else \\"QUERY_PAIRED_ONLY_SOURCE_UNVERIFIED\\"\\n        ),\\n        \\"n_molecules\\": aligned[\\"n\\"],\\n        \\"min_cell_size\\": min_n,\\n        \\"slices\\": groups,\\n        \\"interpretation\\": (\\n            \\"Exploratory error characterization, not a model-improvement \\"\\n            \\"verdict. Multiple groups and novelty classes are inspected \\"\\n            \\"without multiplicity correction or independent holdout.\\"\\n        ),\\n    }\\n", "src/casmi/validate/external_population.py": "\\"\\"\\"Outcome-blind follow-on populations, preserving previously opened test families.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom .scaffold_study import family_key, partition, digest\\nfrom casmi.chem.canon import canonical_ik14\\n\\n\\ndef prepare_internal(meta, truth_pool, previous_protocol, previous_audit, protocol, *, protected=None, return_protected=False):\\n    keys = meta.train_ik14.astype(str).to_numpy()\\n    labels = dict(zip(truth_pool.key, truth_pool.smiles))\\n    masses = dict(zip(truth_pool.key, truth_pool.mass))\\n    eligible = (meta.ingest_lib.isin(previous_protocol[\\"query_libraries\\"]).to_numpy()\\n                & meta.test_adduct.to_numpy() & (meta.n_peaks.to_numpy() >= 5)\\n                & (np.abs(meta.ppm.to_numpy()) <= 20))\\n    rows = np.flatnonzero(eligible)\\n    choices = pd.DataFrame({\\"row\\": rows, \\"key\\": keys[rows], \\"library\\": meta.ingest_lib.astype(str).to_numpy()[rows]})\\n    choices = choices[choices.key.map(masses).between(*previous_protocol[\\"query_mass_range\\"])]\\n    counts = choices.groupby([\\"key\\", \\"library\\"]).size().reset_index(name=\\"count\\")\\n    best = counts.sort_values([\\"count\\", \\"key\\", \\"library\\"], ascending=[False, True, True]).drop_duplicates(\\"key\\")\\n    groups = {key: group.row.to_numpy() for key, group in choices.merge(best[[\\"key\\", \\"library\\"]], on=[\\"key\\", \\"library\\"]).groupby(\\"key\\")}\\n    historical_vin = {family_key(labels[key]) for key in pd.unique(keys[(meta.ingest_lib == \\"enveda-np-examples\\").to_numpy()]) if key in labels}\\n    if protected is not None:\\n        if protected[\\"previous_partitions\\"] != {name: previous_audit[name] for name in (\\"train\\", \\"dev\\", \\"test\\")}:\\n            raise ValueError(\\"Pre-purge historical partition evidence differs\\")\\n        historical_vin.update(protected[\\"historical_vin\\"])\\n    representatives = {}\\n    for row in best.itertuples():\\n        family = family_key(labels[row.key])\\n        if family and family not in historical_vin and (family not in representatives or row.key < representatives[family][\\"key\\"]):\\n            representatives[family] = {\\"key\\": row.key, \\"family\\": family, \\"library\\": row.library}\\n    records = []\\n    for value in representatives.values():\\n        metric = canonical_ik14(labels[value[\\"key\\"]])\\n        if metric:\\n            records.append({**value, \\"metric\\": metric})\\n    if protected is None:\\n        previous = partition(records, previous_protocol[\\"split_counts\\"], previous_protocol[\\"split_salt\\"])\\n        for name, values in previous.items():\\n            for field, private_values in ((\\"identity_sha256\\", [v[\\"key\\"] for v in values]),\\n                                          (\\"family_sha256\\", [v[\\"group\\"] for v in values]),\\n                                          (\\"metric_sha256\\", [v[\\"metric\\"] for v in values])):\\n                if digest(private_values) != previous_audit[name][field]:\\n                    raise ValueError(\\"Historical partitions could not be reconstructed exactly\\")\\n        old_values = [v for values in previous.values() for v in values]\\n        old_metrics = {v[\\"metric\\"] for v in old_values}\\n        old_families = {v[field] for v in old_values for field in (\\"family\\", \\"group\\")}\\n        protected = {\\"old_metrics\\": sorted(old_metrics), \\"old_families\\": sorted(old_families),\\n                     \\"historical_vin\\": sorted(v for v in historical_vin if v),\\n                     \\"previous_partitions\\": {name: previous_audit[name] for name in (\\"train\\", \\"dev\\", \\"test\\")}}\\n    else:\\n        old_metrics, old_families = set(protected[\\"old_metrics\\"]), set(protected[\\"old_families\\"])\\n    fresh_records = [v for v in records if v[\\"metric\\"] not in old_metrics and v[\\"family\\"] not in old_families]\\n    splits = partition(fresh_records, protocol[\\"internal_counts\\"], protocol[\\"split_salt\\"])\\n    audit = {}\\n    for name, values in splits.items():\\n        audit[name] = {\\"n_molecules\\": len(values), \\"identity_sha256\\": digest([r[\\"key\\"] for r in values]),\\n                       \\"family_sha256\\": digest([r[\\"group\\"] for r in values]), \\"metric_sha256\\": digest([r[\\"metric\\"] for r in values]),\\n                       \\"libraries\\": pd.Series([r[\\"library\\"] for r in values]).value_counts().to_dict()}\\n    selected = [v for values in splits.values() for v in values]\\n    metric_overlap = len({v[\\"metric\\"] for v in selected} & old_metrics)\\n    family_overlap = len({v[field] for v in selected for field in (\\"family\\", \\"group\\")} & old_families)\\n    cross_split_overlap = sum(len({v[\\"key\\"] for v in splits[a]} & {v[\\"key\\"] for v in splits[b]})\\n                              for a, b in ((\\"train\\", \\"dev\\"), (\\"train\\", \\"test\\"), (\\"dev\\", \\"test\\")))\\n    if metric_overlap or family_overlap or cross_split_overlap:\\n        raise ValueError(\\"New population overlaps a protected historical or current partition\\")\\n    audit.update(previous_population_reconstructed=True, previous_selected_metric_overlap=metric_overlap,\\n                 previous_selected_family_overlap=family_overlap, cross_split_identity_overlaps=cross_split_overlap)\\n    result = (splits, groups, labels, audit)\\n    return (*result, protected) if return_protected else result\\n\\n\\ndef select_external(novel, splits, salt, *, maximum=150, minimum=30):\\n    blocked = {v[field] for name in (\\"train\\", \\"dev\\") for v in splits[name] for field in (\\"family\\", \\"group\\")}\\n    excluded_metrics = set(novel.loc[novel.family.isin(blocked), \\"metric_key\\"])\\n    eligible = novel[~novel.metric_key.isin(excluded_metrics)]\\n    ordered = sorted(set(eligible.metric_key), key=lambda metric: hashlib.sha256((salt + \\"|external|\\" + metric).encode()).hexdigest())[:maximum]\\n    if len(ordered) < minimum:\\n        raise ValueError(\\"Insufficient independent external molecules after supervised-family exclusion\\")\\n    selected = []\\n    for metric in ordered:\\n        spectra = eligible[eligible.metric_key == metric].sort_values(\\"accession\\").head(5)\\n        first = spectra.iloc[0]\\n        selected.append({\\"metric\\": metric, \\"key\\": first.standard_ik14, \\"smiles\\": first.smiles,\\n                         \\"family\\": first.family, \\"records\\": spectra})\\n    overlap = len({v[\\"family\\"] for v in selected} & blocked)\\n    if overlap:\\n        raise ValueError(\\"External population shares supervised training/development families\\")\\n    audit = {\\"n_molecules\\": len(selected), \\"metric_sha256\\": digest(ordered),\\n             \\"family_sha256\\": digest([v[\\"family\\"] for v in selected]),\\n             \\"accessions_sha256\\": digest([str(a) for v in selected for a in v[\\"records\\"].accession]),\\n             \\"supervised_family_overlap\\": overlap, \\"excluded_supervised_family_molecules\\": len(excluded_metrics)}\\n    return selected, audit\\n", "src/casmi/validate/external_purge.py": "\\"\\"\\"Deterministic external-family selection and whole-reference training exclusion.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\n\\n\\ndef select_external_families(source, salt, *, maximum=150, minimum=30):\\n    \\"\\"\\"Choose source chemistry before candidate or model inspection.\\n\\n    Every family annotation of a chosen metric is protected, including variants\\n    outside the five spectra eventually used as query input.\\n    \\"\\"\\"\\n    if type(maximum) is not int or type(minimum) is not int or not 1 <= minimum <= maximum:\\n        raise ValueError(\\"Positive bounded external population required\\")\\n    required = {\\"family\\", \\"metric_key\\", \\"accession\\", \\"standard_ik14\\"}\\n    if not required <= set(source) or source[list(required)].isna().any().any() or source.accession.duplicated().any():\\n        raise ValueError(\\"Invalid external source identity contract\\")\\n    family_of = source.groupby(\\"metric_key\\").family.agg(lambda values: set(values)).to_dict()\\n    chosen, blocked_families = [], set()\\n    ordered = sorted(set(source.family), key=lambda family: hashlib.sha256((salt + \\"|external-family|\\" + family).encode()).hexdigest())\\n    for family in ordered:\\n        if family in blocked_families:\\n            continue\\n        for metric in sorted(set(source.loc[source.family == family, \\"metric_key\\"])):\\n            if not family_of[metric] & blocked_families:\\n                chosen.append(metric); blocked_families.update(family_of[metric]); break\\n        if len(chosen) == maximum:\\n            break\\n    if len(chosen) < minimum:\\n        raise ValueError(\\"Insufficient independently grouped external molecules\\")\\n    return source[source.metric_key.isin(chosen)].sort_values(\\"accession\\").reset_index(drop=True), sorted(blocked_families)\\n\\n\\ndef blocked_training_identities(rows, selected_source, selected_families, *, return_audit=False):\\n    \\"\\"\\"Inspect full labels; a blocked label blocks every row of its declared key.\\"\\"\\"\\n    from casmi.chem.canon import canonical_ik14, skeleton_key\\n    from casmi.validate.scaffold_study import family_key\\n\\n    external_metrics = set(selected_source.metric_key)\\n    external_standard = set(selected_source.standard_ik14)\\n    external_skeletons = {skeleton_key(s) for s in selected_source.smiles}\\n    selected_families = set(selected_families)\\n    blocked, cache = set(external_standard), {}\\n    unassigned_labels, unassigned_keys = set(), set()\\n    for key, smiles in rows:\\n        if not isinstance(key, str) or not key:\\n            raise ValueError(\\"Missing original training identity\\")\\n        if smiles not in cache:\\n            family = family_key(smiles)\\n            if family is None:\\n                # Undefined grouping cannot establish family separation. Remove\\n                # every row of this declared identity rather than retain it.\\n                unassigned_labels.add(smiles)\\n                cache[smiles] = True\\n            else:\\n                by_family = family in selected_families\\n                skeleton = skeleton_key(smiles)\\n                if skeleton is None:\\n                    raise ValueError(\\"Unparseable original training connectivity\\")\\n                by_metric = False\\n                if skeleton in external_skeletons:\\n                    metric = canonical_ik14(smiles)\\n                    if metric is None:\\n                        raise ValueError(\\"Unverifiable matching original metric\\")\\n                    by_metric = metric in external_metrics\\n                cache[smiles] = by_family or by_metric\\n        if smiles in unassigned_labels:\\n            unassigned_keys.add(key)\\n        if key in external_standard or cache[smiles]:\\n            blocked.add(key)\\n    audit = {\\"unassignable_family_labels_removed\\": len(unassigned_labels),\\n             \\"unassignable_family_training_identities_removed\\": len(unassigned_keys)}\\n    return (blocked, audit) if return_audit else blocked\\n", "src/casmi/validate/fingerprint_evidence.py": "\\"\\"\\"Standard-library D2 fingerprint evidence gate for scientific run comparisons.\\n\\nA verified local input inventory alone does not prove that an analog model\\nused Morgan fingerprints computed for the *same SMILES and masses*. Reports\\nwithout a verified D2 sidecar remain descriptive/exploratory; they must not\\nsupport a model ACCEPT/REJECT decision.\\n\\nThis validates report evidence, not the source data itself. The upstream\\nrun_validation step must cryptographically verify the actual artifacts.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom collections.abc import Mapping\\n\\nFINGERPRINT_SCHEMA = \\"casmi-morgan-fingerprint-provenance-v1\\"\\nFINGERPRINT_STATUS = \\"POOL_FINGERPRINTS_VERIFIED\\"\\n\\n\\ndef _valid_sha(value) -> bool:\\n    return (isinstance(value, str) and len(value) == 64\\n            and all(char in \\"0123456789abcdef\\" for char in value))\\n\\n\\ndef verified_d2_fingerprint_evidence(a: Mapping, b: Mapping) -> bool:\\n    \\"\\"\\"Return whether all D2 runs use verified, mutually compatible vectors.\\n\\n    D1-only records do not consume D2 fingerprints. A D1/D2 ablation is\\n    permitted if the D2 run has verified provenance and BOTH runs share a\\n    complete encompassing input manifest as enforced by run_alignment.\\n    \\"\\"\\"\\n    from casmi.validate.analog_index_evidence import verified_analog_index_evidence\\n\\n    proofs = []\\n    missing = False\\n    for run in (a, b):\\n        cfg = run.get(\\"config\\")\\n        channels = cfg.get(\\"channels\\") if isinstance(cfg, Mapping) else None\\n        if not isinstance(channels, list):\\n            raise ValueError(\\"Missing model channel provenance\\")\\n        if \\"d2\\" not in channels:\\n            continue\\n        proof = run.get(\\"fingerprint_provenance\\")\\n        if proof is None:\\n            missing = True  # historical report: descriptive only\\n            continue\\n        if not isinstance(proof, Mapping):\\n            raise ValueError(\\"Malformed D2 fingerprint provenance\\")\\n        if (proof.get(\\"status\\") != FINGERPRINT_STATUS\\n                or proof.get(\\"schema\\") != FINGERPRINT_SCHEMA):\\n            raise ValueError(\\"Unverified D2 fingerprint provenance schema or status\\")\\n        count = proof.get(\\"n_structures\\")\\n        if type(count) is not int or count < 1:\\n            raise ValueError(\\"Invalid D2 fingerprint row count\\")\\n        for key in (\\"pool_rows_sha256\\", \\"fingerprint_bytes_sha256\\"):\\n            if not _valid_sha(proof.get(key)):\\n                raise ValueError(f\\"Invalid D2 fingerprint provenance: {key}\\")\\n        proofs.append(proof)\\n        # A valid fingerprint sidecar cannot certify a cached D2 reference\\n        # matrix. Unverified historical/cached runs remain exploratory.\\n        if not verified_analog_index_evidence(run):\\n            missing = True\\n\\n    if missing:\\n        return False\\n    if len(proofs) == 2 and any(\\n        proofs[0][key] != proofs[1][key] for key in\\n        (\\"n_structures\\", \\"pool_rows_sha256\\", \\"fingerprint_bytes_sha256\\")\\n    ):\\n        raise ValueError(\\"D2 fingerprint source mismatch between validation runs\\")\\n    return True\\n", "src/casmi/validate/knockout.py": "\\"\\"\\"Structure-level knockout validation.\\n\\nA query is one structure with all its spectra from the query library (e.g. enveda-np-examples),\\nmimicking a test molecule. Each query is scored under three simulated novelty classes:\\n\\n  C1  library minus the query\'s own spectra        pool keeps the structure\\n  C2  library minus EVERY spectrum of the structure, in every library   pool keeps it\\n  C3  as C2, and the structure is removed from the pool                 (retrieval must score 0)\\n\\nKnockout is by metric key (tautomer-canonical InChIKey14), never by library name \\u2014 holding a\\nlibrary out by name leaks, because the same structures appear in other libraries.\\n\\"\\"\\"\\nfrom dataclasses import dataclass, field\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nCLASSES = (\\"C1\\", \\"C2\\", \\"C3\\")\\n\\n\\n@dataclass\\nclass Query:\\n    qid: str\\n    pool_key: str                 # the structure\'s pool key (standard InChIKey14)\\n    spectra: list                 # list of dicts: mz, it, precursor_mz, adduct, ce_ev, mode\\n    spectrum_rows: np.ndarray     # selected source spectra used as query input\\n    source_library_rows: np.ndarray | None = None  # ALL same-key source-library spectra for C1 exclusion\\n\\n\\n@dataclass\\nclass Knockout:\\n    \\"\\"\\"What the engine must not see for one (query, class).\\"\\"\\"\\n    excluded_rows: np.ndarray                 # store rows hidden from library search\\n    excluded_keys: set = field(default_factory=set)   # structures hidden from the pool\\n\\n\\ndef build_queries(store, key_of_row, library=\\"enveda-np-examples\\", keys_subset=None,\\n                  max_spectra=None, max_queries=None, seed=0):\\n    \\"\\"\\"Group one library\'s test-adduct spectra by structure into molecule-like queries.\\n    `keys_subset` restricts the structures; `max_spectra` subsamples each query (test median is 3).\\n    A subsampled query still excludes ALL of that structure\'s reference rows in its own\\n    source library. Otherwise an unselected duplicate spectrum leaks into C1 scoring.\\"\\"\\"\\n    meta = store.meta\\n    rng = np.random.default_rng(seed)\\n    rows = np.flatnonzero((meta.ingest_lib == library).to_numpy() & meta.test_adduct.to_numpy()\\n                          & (meta.n_peaks.to_numpy() >= 3))\\n    keys = key_of_row[rows]\\n    if keys_subset is not None:\\n        m = np.isin(keys, list(keys_subset))\\n        rows, keys = rows[m], keys[m]\\n    order = np.argsort(keys.astype(str), kind=\\"stable\\")\\n    rows, keys = rows[order], keys[order]\\n    starts = np.r_[0, np.flatnonzero(keys[1:] != keys[:-1]) + 1] if len(keys) else np.empty(0, np.int64)\\n    ends = np.r_[starts[1:], len(keys)] if len(keys) else np.empty(0, np.int64)\\n    groups = list(zip(starts, ends))\\n    if max_queries and len(groups) > max_queries:\\n        groups = [groups[i] for i in sorted(rng.choice(len(groups), max_queries, replace=False))]\\n    # Collect all source-library rows for the selected query keys, not only\\n    # test-adduct/peak-filtered inputs. Reference indices have their own\\n    # filtering rules and cannot safely be inferred from sampled inputs.\\n    wanted = {str(keys[s_]) for s_, _ in groups}\\n    source_rows = np.flatnonzero((meta.ingest_lib == library).to_numpy())\\n    reference_rows = {key: [] for key in wanted}\\n    for index in source_rows:\\n        key = str(key_of_row[index])\\n        if key in reference_rows:\\n            reference_rows[key].append(int(index))\\n\\n    queries = []\\n    for s_, e_ in groups:\\n        r = rows[s_:e_]\\n        if max_spectra and len(r) > max_spectra:\\n            r = np.sort(rng.choice(r, max_spectra, replace=False))\\n        spectra = []\\n        for i in r:\\n            mz, it = store.peaks(i)\\n            m = meta.iloc[i]\\n            spectra.append(dict(mz=mz, it=it, precursor_mz=float(m.precursor_mz), adduct=str(m.adduct),\\n                                ce_ev=float(m.ce_ev), mode=str(m.ionization_mode)))\\n        own_rows = np.asarray(reference_rows[str(keys[s_])], dtype=np.int64)\\n        queries.append(Query(\\n            qid=f\\"{library}:{keys[s_]}\\", pool_key=keys[s_],\\n            spectra=spectra, spectrum_rows=r,\\n            source_library_rows=own_rows,\\n        ))\\n    return queries\\n\\n\\ndef knockout_for(query: Query, cls: str, rows_of_key, equivalent_keys) -> Knockout:\\n    \\"\\"\\"`equivalent_keys`: every pool key whose metric key equals the truth\'s (tautomer variants\\n    stored under a different standard key). Knocking out only the query\'s own key would leak.\\"\\"\\"\\n    if cls not in CLASSES:\\n        raise ValueError(f\\"Unknown knockout class: {cls}\\")\\n    query_rows = np.asarray(query.spectrum_rows, dtype=np.int64)\\n    own_rows = (query_rows if query.source_library_rows is None\\n                else np.union1d(query_rows, query.source_library_rows))\\n    if cls == \\"C1\\":\\n        return Knockout(own_rows)\\n    rows = [rows_of_key.get(k, np.empty(0, np.int64)) for k in equivalent_keys]\\n    excluded = np.union1d(\\n        np.concatenate(rows) if rows else np.empty(0, np.int64),\\n        own_rows,\\n    )\\n    return Knockout(excluded, set(equivalent_keys) if cls == \\"C3\\" else set())\\n\\n\\n\\ndef include_equivalent_source_rows(query: Query, equivalent_keys, source_rows, key_of_row):\\n    \\"\\"\\"Extend the query\'s C1 exclusions with its same-metric-key source refs.\\n\\n    `source_rows` must contain only indices from the query\'s own library.\\n    `key_of_row` maps every store index to its *standard* structure key.\\n    Every equivalent-key variant must be removed, not just the sampled peaks.\\n    This mutates only the optional reference exclusion field, not spectra.\\n    \\"\\"\\"\\n    source_rows = np.asarray(source_rows, dtype=np.int64)\\n    key_of_row = np.asarray(key_of_row)\\n    if np.any(source_rows < 0) or np.any(source_rows >= len(key_of_row)):\\n        raise ValueError(\\"Source-library reference row outside store\\")\\n    if not equivalent_keys:\\n        raise ValueError(\\"Expected at least the query\'s own structure key\\")\\n    matching = source_rows[np.isin(key_of_row[source_rows], list(equivalent_keys))]\\n    base = query.spectrum_rows if query.source_library_rows is None else query.source_library_rows\\n    query.source_library_rows = np.union1d(base, matching).astype(np.int64, copy=False)\\n    return query.source_library_rows\\n\\n\\n\\ndef same_library_reference_rows(rows_of_key, equivalent_keys, lib_of_row, library):\\n    \\"\\"\\"D1/analog-indexed same-structure references in the query\'s library.\\n\\n    Use alongside full source-library exclusions on build_queries outputs.\\n    The caller decides which novelty class requires this cross-library\\n    knockout; do not affect intentionally retained legacy training modes.\\n    \\"\\"\\"\\n    arrays = [np.asarray(rows_of_key.get(k, []), dtype=np.int64)\\n              for k in equivalent_keys]\\n    arrays = [arr for arr in arrays if len(arr)]\\n    if not arrays:\\n        return np.empty(0, dtype=np.int64)\\n    refs = np.concatenate(arrays)\\n    libraries = np.asarray(lib_of_row)\\n    if np.any(refs < 0) or np.any(refs >= len(libraries)):\\n        raise ValueError(\\"Reference row outside library metadata\\")\\n    return np.unique(refs[libraries[refs] == library])\\n\\n\\n\\ndef equivalent_keys(query_key, query_mass, pool, keys, window_ppm=30.0, truth_smiles=None):\\n    \\"\\"\\"Metric truth and every pool key in the mass window that is the same structure under the\\n    metric (tautomer variants stored under another standard key). The skeleton key is an exact\\n    necessary condition, so only skeleton matches pay for tautomer canonicalisation.\\n    `truth_smiles` is required when the query structure itself is not in this pool.\\"\\"\\"\\n    from casmi.chem.canon import skeleton_key\\n    if truth_smiles is None:\\n        truth_smiles = pool.smiles_of(query_key)\\n    truth, skel = keys.get(query_key, truth_smiles), skeleton_key(truth_smiles)\\n    same = {query_key} if query_key in pool.row_of else set()\\n    for k in pool.key[pool.window(query_mass, window_ppm).idx]:\\n        if k != query_key and skeleton_key(pool.smiles_of(k)) == skel and keys.get(k, pool.smiles_of(k)) == truth:\\n            same.add(k)\\n    return truth, same\\n\\n\\ndef mixture(mrr_by_class: dict, f1: float, f3: float) -> float:\\n    \\"\\"\\"Expected leaderboard score for class shares f1, f2 = 1 - f1 - f3, f3.\\"\\"\\"\\n    f2 = 1.0 - f1 - f3\\n    return f1 * mrr_by_class[\\"C1\\"] + f2 * mrr_by_class[\\"C2\\"] + f3 * mrr_by_class[\\"C3\\"]\\n\\n\\ndef mixture_grid(mrr_by_class, f1_values=(0.15, 0.20, 0.25, 0.30), f3_values=(0.2, 0.3, 0.4)):\\n    rows = [(f1, f3, mixture(mrr_by_class, f1, f3)) for f1 in f1_values for f3 in f3_values if f1 + f3 < 1]\\n    return pd.DataFrame(rows, columns=[\\"f1\\", \\"f3\\", \\"score\\"])\\n", "src/casmi/validate/massbank_records.py": "\\"\\"\\"Strict MassBank 2.x CC0 MS2 record ingestion for external research inputs.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport math\\n\\n\\nclass RecordRejected(ValueError):\\n    pass\\n\\n\\ndef parse_record(text):\\n    tags, peaks, in_peaks = {}, [], False\\n    if not text.rstrip().endswith(\\"//\\"):\\n        raise RecordRejected(\\"unterminated_record\\")\\n    for line in text.splitlines():\\n        if line == \\"//\\":\\n            break\\n        if line.startswith(\\"PK$PEAK:\\"):\\n            in_peaks = True\\n            continue\\n        if in_peaks:\\n            fields = line.split()\\n            if len(fields) != 3:\\n                raise RecordRejected(\\"malformed_peak_row\\")\\n            try:\\n                mz, intensity, relative = map(float, fields)\\n            except ValueError as exc:\\n                raise RecordRejected(\\"nonnumeric_peak\\") from exc\\n            if not all(math.isfinite(v) for v in (mz, intensity, relative)) or mz <= 0 or intensity <= 0 or relative < 0:\\n                raise RecordRejected(\\"invalid_peak\\")\\n            peaks.append((mz, intensity))\\n            continue\\n        if \\": \\" in line:\\n            name, value = line.split(\\": \\", 1)\\n            if name in {\\"AC$MASS_SPECTROMETRY\\", \\"MS$FOCUSED_ION\\"}:\\n                key, separator, value = value.partition(\\" \\")\\n                if not separator:\\n                    raise RecordRejected(\\"malformed_subtag\\")\\n                name += \\":\\" + key\\n            tags.setdefault(name, []).append(value.strip())\\n\\n    def one(name):\\n        values = tags.get(name, [])\\n        if len(values) != 1 or not values[0]:\\n            raise RecordRejected(\\"missing_or_duplicate_\\" + name)\\n        return values[0]\\n\\n    if one(\\"LICENSE\\") != \\"CC0\\":\\n        raise RecordRejected(\\"non_cc0_license\\")\\n    if one(\\"AC$MASS_SPECTROMETRY:MS_TYPE\\") != \\"MS2\\" or one(\\"AC$MASS_SPECTROMETRY:IONIZATION\\") != \\"ESI\\":\\n        raise RecordRejected(\\"unsupported_acquisition\\")\\n    mode = one(\\"AC$MASS_SPECTROMETRY:ION_MODE\\").lower()\\n    if mode not in (\\"positive\\", \\"negative\\"):\\n        raise RecordRejected(\\"unsupported_mode\\")\\n    try:\\n        precursor = float(one(\\"MS$FOCUSED_ION:PRECURSOR_M/Z\\"))\\n        declared_peaks = int(one(\\"PK$NUM_PEAK\\"))\\n    except ValueError as exc:\\n        raise RecordRejected(\\"invalid_numeric_header\\") from exc\\n    if not math.isfinite(precursor) or precursor <= 0 or declared_peaks != len(peaks) or len(peaks) < 5:\\n        raise RecordRejected(\\"invalid_precursor_or_peak_count\\")\\n    if any(a[0] >= b[0] for a, b in zip(peaks, peaks[1:])):\\n        raise RecordRejected(\\"unordered_or_duplicate_mz\\")\\n    maximum = max(v for _, v in peaks)\\n    return {\\"accession\\": one(\\"ACCESSION\\"), \\"authors\\": one(\\"AUTHORS\\"), \\"license\\": \\"CC0\\",\\n            \\"record_sha256\\": hashlib.sha256(text.encode(\\"utf-8\\")).hexdigest(),\\n            \\"smiles\\": one(\\"CH$SMILES\\"), \\"precursor_mz\\": precursor,\\n            \\"adduct\\": one(\\"MS$FOCUSED_ION:PRECURSOR_TYPE\\"), \\"ionization_mode\\": mode,\\n            \\"instrument\\": one(\\"AC$INSTRUMENT_TYPE\\"), \\"ms2_mzs\\": [m for m, _ in peaks],\\n            \\"ms2_normalized_intensities\\": [100 * v / maximum for _, v in peaks]}\\n", "src/casmi/validate/metric.py": "\\"\\"\\"MRR@25 with the official metric\'s semantics, plus the per-molecule detail we need for analysis.\\n\\nSemantics reproduced from the official scoring notebook (`metric/casmi-mean-reciprocal-rank`):\\n  * rank = position in the list; an unparseable or duplicate guess still consumes its position,\\n  * only the first correct guess counts (reciprocal rank, not average precision),\\n  * a molecule absent from the submission scores 0.\\n\\"\\"\\"\\nfrom dataclasses import dataclass\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom casmi.chem.canon import canonical_ik14, heavy_atom_key\\nfrom casmi.validate.paired_bootstrap import paired_difference\\nfrom casmi.validate.ranked_slots import split_ranked_guesses\\nfrom casmi.validate.rank_inputs import validated_ranks, validated_bootstrap\\n\\nMAX_RANK = 25\\n\\n\\ndef reciprocal_rank(guesses, answer_ik14s, answer_heavy_keys, max_rank=MAX_RANK):\\n    for rank, guess in enumerate(guesses[:max_rank], 1):\\n        if heavy_atom_key(guess) not in answer_heavy_keys:\\n            continue\\n        if canonical_ik14(guess) in answer_ik14s:\\n            return 1.0 / rank\\n    return 0.0\\n\\n\\ndef score_submission(solution: pd.DataFrame, submission: pd.DataFrame, max_rank=MAX_RANK) -> pd.Series:\\n    \\"\\"\\"Per-molecule reciprocal rank. `solution` has molecule_id + smiles (comma-separated if several\\n    structures are accepted); `submission` has molecule_id + semicolon-joined guesses.\\"\\"\\"\\n    if submission[\\"molecule_id\\"].duplicated().any():\\n        raise ValueError(\\"Each molecule_id must appear exactly once\\")\\n    # Never compress malformed empty slots: a blank candidate still occupies\\n    # its original rank, even though a valid competition submission forbids it.\\n    guesses = {m: split_ranked_guesses(str(s))\\n               for m, s in zip(submission[\\"molecule_id\\"], submission[\\"smiles\\"])}\\n    too_many = [m for m, g in guesses.items() if len(g) > max_rank]\\n    if too_many:\\n        raise ValueError(f\\"More than {max_rank} guesses for {too_many[:3]}\\")\\n    out = {}\\n    for m, answer in zip(solution[\\"molecule_id\\"], solution[\\"smiles\\"]):\\n        accepted = [s for s in str(answer).split(\\",\\") if s]\\n        ik14s = {k for k in map(canonical_ik14, accepted) if k}\\n        heavy = {k for k in map(heavy_atom_key, accepted) if k}\\n        if not ik14s:\\n            raise ValueError(f\\"Unparseable solution SMILES for {m}\\")\\n        out[m] = reciprocal_rank(guesses.get(m, []), ik14s, heavy, max_rank)\\n    return pd.Series(out, name=\\"rr\\")\\n\\n\\ndef mrr(solution, submission, max_rank=MAX_RANK) -> float:\\n    return float(score_submission(solution, submission, max_rank).mean())\\n\\n\\n# ----------------------------------------------------------------------------- key-level scoring\\ndef rank_of_truth(ranked_ik14s, true_ik14, max_rank=MAX_RANK):\\n    \\"\\"\\"1-based rank of the true key in an already canonicalised, deduplicated list (0 = absent).\\n    Validation runs on keys directly, which is exact once the output writer guarantees\\n    canonical, deduplicated lists \\u2014 and ~1000x faster than re-parsing SMILES.\\"\\"\\"\\n    for rank, key in enumerate(ranked_ik14s[:max_rank], 1):\\n        if key == true_ik14:\\n            return rank\\n    return 0\\n\\n\\n@dataclass\\nclass Summary:\\n    mrr: float\\n    ci_low: float\\n    ci_high: float\\n    top1: float\\n    top10: float\\n    recall_at_25: float\\n    n: int\\n\\n    def __str__(self):\\n        return (f\\"MRR {self.mrr:.4f} [{self.ci_low:.4f}, {self.ci_high:.4f}]  top1 {self.top1:.3f}  \\"\\n                f\\"top10 {self.top10:.3f}  hit@25 {self.recall_at_25:.3f}  n={self.n}\\")\\n\\n\\ndef summarize(ranks, n_boot=2000, seed=0) -> Summary:\\n    \\"\\"\\"MRR with a molecule-level bootstrap 95% CI, from 1-based ranks (0 = not in top 25).\\"\\"\\"\\n    ranks = validated_ranks(ranks)\\n    validated_bootstrap(n_boot, seed)\\n    rr = np.where(ranks > 0, 1.0 / np.maximum(ranks, 1), 0.0)\\n    rng = np.random.default_rng(seed)\\n    boots = rr[rng.integers(0, len(rr), size=(n_boot, len(rr)))].mean(axis=1)\\n    return Summary(float(rr.mean()), float(np.quantile(boots, .025)), float(np.quantile(boots, .975)),\\n                   float((ranks == 1).mean()), float(((ranks > 0) & (ranks <= 10)).mean()),\\n                   float((ranks > 0).mean()), len(rr))\\n\\n\\n", "src/casmi/validate/overlap_audit.py": "\\"\\"\\"Aggregate-only, exact-content public-example overlap audit.\\n\\nRaw identifiers and peak arrays stay in memory and are never exported. Equality\\nof stored numeric arrays is distinct from proof of acquisition provenance.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom collections import defaultdict\\n\\nfrom .spectrum_overlap import coarse_spectrum_bucket, full_spectrum_match\\n\\n\\ndef _spectrum(row):\\n    return (row.get(\\"precursor_mz\\"), row.get(\\"ms2_mzs\\"),\\n            row.get(\\"ms2_normalized_intensities\\"))\\n\\n\\ndef _identifier(value):\\n    if value is None or not str(value).strip():\\n        raise ValueError(\\"Public example identifiers must be present\\")\\n    return str(value)\\n\\n\\nclass PublicOverlapAudit:\\n    def __init__(self, public_rows):\\n        self.targets = list(public_rows)\\n        if not self.targets:\\n            raise ValueError(\\"Public example input must not be empty\\")\\n        self.buckets = defaultdict(list)\\n        self.molecules = defaultdict(set)\\n        seen = set()\\n        for index, row in enumerate(self.targets):\\n            sid = _identifier(row.get(\\"spectrum_id\\"))\\n            molecule = _identifier(row.get(\\"molecule_id\\"))\\n            if sid in seen:\\n                raise ValueError(\\"Duplicate public spectrum identifiers\\")\\n            seen.add(sid)\\n            spectrum = _spectrum(row)\\n            if not full_spectrum_match(*spectrum, *spectrum):\\n                raise ValueError(\\"Public spectrum must have finite aligned nonempty arrays\\")\\n            self.buckets[coarse_spectrum_bucket(spectrum[0], spectrum[1])].append(index)\\n            self.molecules[molecule].add(index)\\n        self.precursors = {float(row[\\"precursor_mz\\"]) for row in self.targets}\\n        self.scanned = 0\\n        self.candidate_rows = 0\\n        self.coarse_pairs = 0\\n        self.exact_pairs = 0\\n        self.false_pairs = 0\\n        self.matched = set()\\n        self.metadata_matched = set()\\n        self.library_matches = defaultdict(set)\\n        self.metadata_disagreement_pairs = 0\\n\\n    def scan_batch(self, candidate_rows, *, scanned_count=None):\\n        \\"\\"\\"Count all scanned rows, considering only exact-precursor candidates.\\n\\n        A caller can filter Arrow batches before converting peak lists to Python.\\n        `scanned_count` must include filtered-out rows; finish checks full coverage.\\n        \\"\\"\\"\\n        rows = list(candidate_rows)\\n        count = len(rows) if scanned_count is None else scanned_count\\n        if isinstance(count, bool) or not isinstance(count, int) or count < len(rows):\\n            raise ValueError(\\"Scanned row count must cover every considered row\\")\\n        self.scanned += count\\n        for row in rows:\\n            try:\\n                precursor = float(row.get(\\"precursor_mz\\"))\\n            except (TypeError, ValueError):\\n                continue\\n            if precursor not in self.precursors:\\n                continue\\n            self.candidate_rows += 1\\n            spectrum = _spectrum(row)\\n            bucket = coarse_spectrum_bucket(spectrum[0], spectrum[1])\\n            for index in self.buckets.get(bucket, ()):\\n                self.coarse_pairs += 1\\n                target = self.targets[index]\\n                if not full_spectrum_match(*spectrum, *_spectrum(target)):\\n                    self.false_pairs += 1\\n                    continue\\n                library = row.get(\\"ingest_lib\\")\\n                if not isinstance(library, str) or not library.strip():\\n                    raise ValueError(\\"Matching training spectra need a source-library label\\")\\n                self.exact_pairs += 1\\n                self.matched.add(index)\\n                self.library_matches[library].add(index)\\n                aligned = all(target.get(key) is not None and row.get(key) == target[key]\\n                              for key in (\\"adduct\\", \\"ionization_mode\\"))\\n                if aligned:\\n                    self.metadata_matched.add(index)\\n                else:\\n                    self.metadata_disagreement_pairs += 1\\n\\n    def finish(self, *, expected_train_rows):\\n        if (isinstance(expected_train_rows, bool) or not isinstance(expected_train_rows, int)\\n                or expected_train_rows <= 0 or self.scanned != expected_train_rows):\\n            raise ValueError(\\"A complete scan of the declared training row count is required\\")\\n        return {\\n            \\"schema_version\\": 1,\\n            \\"status\\": \\"COMPLETE_CONTENT_AUDIT\\",\\n            \\"matching_method\\": \\"exact stored precursor and full ordered m/z/intensity numeric arrays; no rounding or tolerance\\",\\n            \\"metadata_fields\\": [\\"adduct\\", \\"ionization_mode\\"],\\n            \\"train_rows_scanned\\": self.scanned,\\n            \\"public_spectra\\": len(self.targets),\\n            \\"public_molecules\\": len(self.molecules),\\n            \\"exact_precursor_candidate_rows\\": self.candidate_rows,\\n            \\"coarse_candidate_pairs_after_precursor_filter\\": self.coarse_pairs,\\n            \\"coarse_false_match_pairs\\": self.false_pairs,\\n            \\"exact_matching_pairs\\": self.exact_pairs,\\n            \\"content_matched_spectra\\": len(self.matched),\\n            \\"metadata_consistent_matched_spectra\\": len(self.metadata_matched),\\n            \\"metadata_disagreement_pairs\\": self.metadata_disagreement_pairs,\\n            \\"fully_content_matched_molecules\\": sum(indices <= self.matched for indices in self.molecules.values()),\\n            \\"fully_metadata_consistent_molecules\\": sum(indices <= self.metadata_matched for indices in self.molecules.values()),\\n            \\"content_matches_by_library\\": {key: len(value) for key, value in sorted(self.library_matches.items())},\\n            \\"limitations\\": [\\n                \\"Source-library match counts overlap and must not be summed as unique public spectra.\\",\\n                \\"Stored numerical equality does not independently prove common acquisition identity.\\",\\n                \\"This audit concerns mounted public examples, not the hidden competition evaluation set.\\",\\n                \\"No model evaluation, training or competition submission was performed.\\",\\n            ],\\n        }\\n", "src/casmi/validate/paired_bootstrap.py": "\\"\\"\\"Minimal numerical bootstrap for CASMI ranking comparisons.\\n\\nSeparate from RDKit/pandas scoring so synthetic CI regression tests and\\nexperiment comparisons need only NumPy, not the entire inference stack.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport numpy as np\\nfrom casmi.validate.rank_inputs import validated_ranks, validated_bootstrap\\n\\n\\ndef paired_difference(ranks_a, ranks_b, n_boot=4000, seed=0):\\n    \\"\\"\\"Molecule-level bootstrap CI of MRR(B)-MRR(A).\\n\\n    Input sequences MUST already have been aligned on verified query IDs and\\n    spectra. This routine does not infer or validate pairing.\\n    \\"\\"\\"\\n    a, b = validated_ranks(ranks_a), validated_ranks(ranks_b)\\n    if a.shape != b.shape:\\n        raise ValueError(\\"Paired ranks must be nonempty, one-dimensional and equal length\\")\\n    validated_bootstrap(n_boot, seed)\\n    to_rr = lambda r: np.where(r > 0, 1.0 / np.maximum(r, 1), 0.0)\\n    d = to_rr(b) - to_rr(a)\\n    rng = np.random.default_rng(seed)\\n    boots = d[rng.integers(0, len(d), size=(n_boot, len(d)))].mean(axis=1)\\n    return float(d.mean()), float(np.quantile(boots, .025)), float(np.quantile(boots, .975))\\n", "src/casmi/validate/rank_inputs.py": "\\"\\"\\"Shared numerical contracts for fixed MRR@25 research summaries.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport numpy as np\\n\\n\\ndef validated_ranks(values):\\n    ranks = np.asarray(values)\\n    if ranks.ndim != 1 or not len(ranks):\\n        raise ValueError(\\"Ranks must be a nonempty one-dimensional integer sequence\\")\\n    if not np.issubdtype(ranks.dtype, np.integer):\\n        raise ValueError(\\"Ranks must be integers from 0 to 25; 0 denotes a miss\\")\\n    if np.any((ranks < 0) | (ranks > 25)):\\n        raise ValueError(\\"Ranks must lie between 0 and 25; out-of-list hits are encoded as 0\\")\\n    return ranks\\n\\n\\ndef validated_bootstrap(n_boot, seed):\\n    if not isinstance(n_boot, (int, np.integer)) or isinstance(n_boot, (bool, np.bool_)) or n_boot < 100:\\n        raise ValueError(\\"At least 100 integer bootstrap draws are required\\")\\n    if not isinstance(seed, (int, np.integer)) or isinstance(seed, (bool, np.bool_)) or seed < 0:\\n        raise ValueError(\\"Bootstrap seed must be a nonnegative integer\\")\\n", "src/casmi/validate/ranked_slots.py": "\\"\\"\\"Position-preserving parsing for CASMI ranked SMILES predictions.\\n\\nA semicolon defines the next ranked position even when the candidate is\\nblank or invalid. Never filter an empty field when computing MRR.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\n\\ndef split_ranked_guesses(text: str) -> list[str]:\\n    \\"\\"\\"Keep every guess slot, including interior/leading/trailing blanks.\\"\\"\\"\\n    if not isinstance(text, str):\\n        raise TypeError(\\"Ranked SMILES must be a string\\")\\n    return [part.strip() for part in text.split(\\";\\")]\\n", "src/casmi/validate/run_alignment.py": "\\"\\"\\"Fail-closed identity checks for paired CASMI validation comparisons.\\n\\nPaired confidence intervals require exactly the same query molecules and the\\nsame source spectra, not merely the same number of queries. Unlike rank\\nsummaries, this module needs no numerical or chemistry dependencies.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\nimport math\\nfrom collections.abc import Mapping, Sequence\\n\\nCLASSES = (\\"C1\\", \\"C2\\", \\"C3\\")\\nSIGNATURE_VERSION = \\"casmi-query-spectra-v2\\"\\n\\n\\ndef query_signature(query) -> str:\\n    \\"\\"\\"Fingerprint *actual* source spectra and exclusion rows for one query.\\n\\n    Signatures are validation provenance, not a tamper-proof dataset checksum.\\n    This prevents pairing runs that resample spectra, change source-library\\n    exclusion scope, or change store row order. Version the signature recipe.\\n    \\"\\"\\"\\n    payload = {\\n        \\"version\\": SIGNATURE_VERSION,\\n        \\"qid\\": str(query.qid),\\n        \\"pool_key\\": str(query.pool_key),\\n        \\"selected_query_rows\\": sorted(int(i) for i in query.spectrum_rows),\\n        \\"source_library_excluded_rows\\": (\\n            sorted(int(i) for i in query.source_library_rows)\\n            if getattr(query, \\"source_library_rows\\", None) is not None else None\\n        ),\\n        \\"spectra\\": [\\n            {\\n                \\"mz\\": [float(x).hex() for x in spec[\\"mz\\"]],\\n                \\"intensity\\": [float(x).hex() for x in spec[\\"it\\"]],\\n                \\"precursor_mz\\": float(spec[\\"precursor_mz\\"]).hex(),\\n                \\"adduct\\": str(spec[\\"adduct\\"]),\\n                \\"ce_ev\\": float(spec[\\"ce_ev\\"]).hex(),\\n                \\"mode\\": str(spec[\\"mode\\"]),\\n            }\\n            for spec in query.spectra\\n        ],\\n    }\\n    encoded = json.dumps(payload, sort_keys=True, separators=(\\",\\", \\":\\"), ensure_ascii=True)\\n    return hashlib.sha256(encoded.encode(\\"utf-8\\")).hexdigest()\\n\\n\\ndef _check_ranks(run: Mapping, n: int) -> dict[str, list[int]]:\\n    classes = run.get(\\"classes\\")\\n    if not isinstance(classes, dict):\\n        raise ValueError(\\"Missing classes in validation run\\")\\n    result = {}\\n    for cls in CLASSES:\\n        block = classes.get(cls)\\n        if not isinstance(block, dict):\\n            raise ValueError(f\\"Missing class {cls}\\")\\n        ranks = block.get(\\"ranks\\")\\n        if not isinstance(ranks, list) or len(ranks) != n:\\n            raise ValueError(f\\"{cls}: expected {n} per-query ranks\\")\\n        for rank in ranks:\\n            if type(rank) is not int or not 0 <= rank <= 25:\\n                raise ValueError(f\\"{cls}: rank must be an integer from 0 to 25\\")\\n        computed = sum(1.0 / rank for rank in ranks if rank) / n\\n        recorded = block.get(\\"mrr\\")\\n        if recorded is not None and (\\n            isinstance(recorded, bool) or not isinstance(recorded, (float, int))\\n            or not math.isfinite(recorded)\\n            or abs(computed - recorded) > 0.000005\\n        ):\\n            raise ValueError(f\\"{cls}: MRR summary disagrees with per-query ranks\\")\\n        result[cls] = ranks\\n    return result\\n\\n\\n\\ndef _source_snapshot_verified(a: Mapping, b: Mapping) -> bool:\\n    \\"\\"\\"Check local input inventory identity and complete known input coverage.\\n\\n    Historical reports without inventory hashes remain exploratory. Merely\\n    naming an identical SHA-256 does not prove that a subset manifest covers\\n    the store and active candidate pool used by either run.\\n    \\"\\"\\"\\n    from casmi.validate.source_manifest import (\\n        MANIFEST_SCHEMA, required_validation_inputs,\\n    )\\n\\n    hashes = (a.get(\\"source_manifest_sha256\\"), b.get(\\"source_manifest_sha256\\"))\\n    if hashes == (None, None):\\n        return False\\n    if None in hashes:\\n        raise ValueError(\\"Source artifact snapshot mismatch: one run lacks a manifest\\")\\n    if any(not isinstance(s, str) or len(s) != 64 or\\n           any(ch not in \\"0123456789abcdef\\" for ch in s) for s in hashes):\\n        raise ValueError(\\"Malformed source artifact snapshot SHA-256\\")\\n    if hashes[0] != hashes[1]:\\n        raise ValueError(\\"Source artifact snapshot mismatch: input files differ\\")\\n\\n    for run in (a, b):\\n        # Old signed reports did not yet validate required-file coverage.\\n        # They can be inspected, but never yield conclusive model verdicts.\\n        if run.get(\\"source_manifest_coverage\\") != \\"required-validation-inputs-v1\\":\\n            return False\\n        if run.get(\\"source_manifest_schema\\") != MANIFEST_SCHEMA:\\n            raise ValueError(\\"Invalid source artifact manifest schema\\")\\n        pool_tag = run.get(\\"source_manifest_pool_tag\\")\\n        channels = run.get(\\"config\\", {}).get(\\"channels\\") if isinstance(run.get(\\"config\\"), Mapping) else None\\n        if not isinstance(channels, list) or \\"d1\\" not in channels:\\n            raise ValueError(\\"Missing validation channel provenance\\")\\n        expected = required_validation_inputs(\\n            pool_tag=pool_tag, analog=(\\"d2\\" in channels),\\n        )\\n        provided = run.get(\\"source_manifest_required_inputs\\")\\n        if not isinstance(provided, list) or provided != list(expected):\\n            raise ValueError(\\"Incomplete or contradictory source artifact inventory\\")\\n        count = run.get(\\"source_manifest_file_count\\")\\n        if type(count) is not int or count < len(expected):\\n            raise ValueError(\\"Invalid source artifact file count\\")\\n    from casmi.validate.fingerprint_evidence import verified_d2_fingerprint_evidence\\n\\n    # An unchanged pool/fingerprint FILE inventory does not establish that\\n    # D2 vectors belong to the pool\'s exact SMILES, masses and row ordering.\\n    # Historical D2 runs without a verified sidecar remain exploratory.\\n    return verified_d2_fingerprint_evidence(a, b)\\n\\n\\ndef align_runs(a: Mapping, b: Mapping, *, allow_unverified_legacy_order: bool = False) -> dict:\\n    \\"\\"\\"Validate evidence and align B to A by query ID.\\n\\n    New-format reports must have unique IDs, per-query signatures, and the\\n    same ID->spectrum-signature mapping. Older runs can only be compared with\\n    an explicit unsafe override, and should not justify acceptance claims.\\n    \\"\\"\\"\\n    # Paired improvement inference requires identical *actual* MS/MS inputs.\\n    # Negative controls intentionally change spectra, so they must never\\n    # receive model-improvement ACCEPT/REJECT judgments from this comparator.\\n    for run in (a, b):\\n        control = run.get(\\"spectral_control\\")\\n        if control is not None:\\n            if not isinstance(control, Mapping):\\n                raise ValueError(\\"Malformed spectral-control provenance\\")\\n            if control.get(\\"mode\\", \\"none\\") != \\"none\\":\\n                raise ValueError(\\n                    \\"Negative-control reports are diagnostic only; cannot \\"\\n                    \\"be compared as model-improvement runs\\"\\n                )\\n\\n    n_a, n_b = a.get(\\"n_queries\\"), b.get(\\"n_queries\\")\\n    if type(n_a) is not int or type(n_b) is not int or n_a < 1 or n_a != n_b:\\n        raise ValueError(\\"Runs must contain the same positive number of queries\\")\\n    n = n_a\\n    ar, br = _check_ranks(a, n), _check_ranks(b, n)\\n    a_ids, b_ids = a.get(\\"query_ids\\"), b.get(\\"query_ids\\")\\n    a_sig, b_sig = a.get(\\"query_signatures\\"), b.get(\\"query_signatures\\")\\n    has_manifest = all(isinstance(x, list) for x in (a_ids, b_ids, a_sig, b_sig))\\n    if not has_manifest:\\n        if not allow_unverified_legacy_order:\\n            raise ValueError(\\n                \\"Cannot validate paired query identities/spectra in legacy runs. \\"\\n                \\"Regenerate reports with query_ids + query_signatures; \\"\\n                \\"use --allow-unverified-legacy-order only for exploratory comparison.\\"\\n            )\\n        return {\\"a\\": ar, \\"b\\": br, \\"n\\": n, \\"verified\\": False,\\n                \\"source_verified\\": False, \\"decision_eligible\\": False}\\n\\n    for run_ids, run_sigs, name in ((a_ids, a_sig, \\"A\\"), (b_ids, b_sig, \\"B\\")):\\n        if len(run_ids) != n or len(run_sigs) != n:\\n            raise ValueError(f\\"Run {name}: manifest length differs from n_queries\\")\\n        if any(not isinstance(i, str) or not i for i in run_ids):\\n            raise ValueError(f\\"Run {name}: empty or invalid query ID\\")\\n        if len(set(run_ids)) != n:\\n            raise ValueError(f\\"Run {name}: duplicate query IDs\\")\\n        if any(not isinstance(s, str) or len(s) != 64 or\\n               any(c not in \\"0123456789abcdef\\" for c in s) for s in run_sigs):\\n            raise ValueError(f\\"Run {name}: malformed query signatures\\")\\n\\n    # A changed signature recipe makes otherwise comparable digests\\n    # uninterpretable; old records without version tags remain supported.\\n    version_a = a.get(\\"query_signature_version\\")\\n    version_b = b.get(\\"query_signature_version\\")\\n    if version_a != version_b:\\n        raise ValueError(\\"Query signature algorithm versions differ\\")\\n    if version_a is not None and (not isinstance(version_a, str) or not version_a):\\n        raise ValueError(\\"Malformed query signature version\\")\\n\\n    if set(a_ids) != set(b_ids):\\n        raise ValueError(\\"Query identity mismatch: validation populations differ\\")\\n    a_by_id = dict(zip(a_ids, a_sig))\\n    b_by_id = dict(zip(b_ids, b_sig))\\n    if any(a_by_id[mid] != b_by_id[mid] for mid in a_ids):\\n        raise ValueError(\\"Spectrum/knockout mismatch: query inputs differ\\")\\n    source_verified = _source_snapshot_verified(a, b)\\n    b_positions = {mid: pos for pos, mid in enumerate(b_ids)}\\n    return {\\n        \\"a\\": ar,\\n        \\"b\\": {cls: [br[cls][b_positions[mid]] for mid in a_ids] for cls in CLASSES},\\n        \\"n\\": n,\\n        \\"verified\\": True,\\n        \\"source_verified\\": source_verified,\\n        \\"decision_eligible\\": source_verified,\\n    }\\n", "src/casmi/validate/scaffold_study.py": "\\"\\"\\"Predeclared internal chemical-family partitions and joint paired uncertainty.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\n\\nimport numpy as np\\n\\n\\ndef digest(value):\\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(\\",\\", \\":\\"), default=str).encode()).hexdigest()\\n\\n\\ndef family_key(smiles):\\n    \\"\\"\\"Generic Murcko ring topology; generic whole connectivity for acyclics.\\n\\n    Atomic/bond types are intentionally removed to make exclusion conservative.\\n    This is a computational partition, not a chemical similarity guarantee.\\n    \\"\\"\\"\\n    from rdkit import Chem\\n    from rdkit.Chem.Scaffolds import MurckoScaffold\\n\\n    try:\\n        mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None\\n        if mol is None or not mol.GetNumHeavyAtoms():\\n            return None\\n        parts = Chem.GetMolFrags(mol, asMols=True)\\n        mol = max(parts, key=lambda part: (part.GetNumHeavyAtoms(), Chem.MolToSmiles(part)))\\n        core = MurckoScaffold.GetScaffoldForMol(mol)\\n        prefix = \\"ring:\\" if core.GetNumHeavyAtoms() else \\"acyclic:\\"\\n        core = core if core.GetNumHeavyAtoms() else mol\\n        return prefix + Chem.MolToSmiles(MurckoScaffold.MakeScaffoldGeneric(core), canonical=True)\\n    except (ValueError, RuntimeError, TypeError):\\n        return None\\n\\n\\ndef partition(records, sizes, salt):\\n    \\"\\"\\"One representative per family; union metric identities across families.\\n\\n    Input: key/family/metric plus query metadata. No model scores are accepted.\\n    The union prevents metric-equivalent labels crossing an unusual core change.\\n    \\"\\"\\"\\n    parents = {}\\n    def find(value):\\n        parents.setdefault(value, value)\\n        while parents[value] != value:\\n            parents[value] = parents[parents[value]]\\n            value = parents[value]\\n        return value\\n    first = {}\\n    for row in records:\\n        if not row.get(\\"family\\") or not row.get(\\"metric\\"):\\n            raise ValueError(\\"Every partition row requires a valid family and metric identity\\")\\n        family = find(row[\\"family\\"])\\n        previous = first.setdefault(row[\\"metric\\"], family)\\n        a, b = find(previous), find(family)\\n        if a != b:\\n            parents[max(a, b)] = min(a, b)\\n    groups = {}\\n    for row in records:\\n        groups.setdefault(find(row[\\"family\\"]), []).append(row)\\n    ordered = sorted(groups, key=lambda family: hashlib.sha256((salt + \\"|\\" + family).encode()).hexdigest())\\n    if set(sizes) != {\\"train\\", \\"dev\\", \\"test\\"} or any(type(n) is not int or n < 1 for n in sizes.values()):\\n        raise ValueError(\\"Positive train/dev/test counts required\\")\\n    if len(ordered) < sum(sizes.values()):\\n        raise ValueError(\\"Insufficient independent family groups for declared population\\")\\n    selected, offset = {}, 0\\n    for split in (\\"train\\", \\"dev\\", \\"test\\"):\\n        rows = []\\n        for family in ordered[offset:offset + sizes[split]]:\\n            representative = min(groups[family], key=lambda row: row[\\"key\\"])\\n            rows.append({**representative, \\"group\\": family,\\n                         \\"fold\\": int(hashlib.sha256((salt + \\"|fold|\\" + family).encode()).hexdigest(), 16) % 2})\\n        selected[split] = rows\\n        offset += sizes[split]\\n    for identity in (\\"family\\", \\"group\\", \\"metric\\", \\"key\\"):\\n        sets = {name: {row[identity] for row in values} for name, values in selected.items()}\\n        if any(sets[a] & sets[b] for a, b in ((\\"train\\", \\"dev\\"), (\\"train\\", \\"test\\"), (\\"dev\\", \\"test\\"))):\\n            raise ValueError(\\"Partition identity overlap\\")\\n    return selected\\n\\n\\ndef joint_intervals(effects, *, n_boot=4000, seed=0):\\n    \\"\\"\\"Joint max-absolute-deviation paired bootstrap across a fixed outcome family.\\"\\"\\"\\n    names = list(effects)\\n    values = np.column_stack([effects[name] for name in names])\\n    if values.ndim != 2 or not len(values) or not np.isfinite(values).all() or n_boot < 100:\\n        raise ValueError(\\"Finite paired effect vectors and at least 100 draws required\\")\\n    mean = values.mean(axis=0)\\n    rng = np.random.default_rng(seed)\\n    maximum = np.empty(n_boot)\\n    for start in range(0, n_boot, 100):\\n        ids = rng.integers(len(values), size=(min(100, n_boot - start), len(values)))\\n        maximum[start:start + len(ids)] = np.max(np.abs(values[ids].mean(axis=1) - mean), axis=1)\\n    radius = float(np.quantile(maximum, .95))\\n    return {\\"method\\": \\"molecule-paired max-deviation bootstrap over all prespecified secondary contrasts\\",\\n            \\"n_molecules\\": len(values), \\"n_bootstrap\\": n_boot, \\"seed\\": seed,\\n            \\"family_size\\": len(names), \\"joint_radius\\": radius,\\n            \\"outcomes\\": {name: {\\"delta_mrr\\": float(value), \\"ci_low\\": float(value - radius),\\n                                \\"ci_high\\": float(value + radius)} for name, value in zip(names, mean)},\\n            \\"limitations\\": \\"Approximate fixed-family uncertainty; no correction for domain shift or historical hypothesis selection\\"}\\n", "src/casmi/validate/simultaneous_ci.py": "\\"\\"\\"Simultaneous paired-bootstrap uncertainty for CASMI novelty mixtures.\\n\\nThe old comparator calculated 12 pointwise 95% intervals and inspected\\nall of them. They are not a joint 95% confidence statement.\\nEach bootstrap draw here reuses the same molecules across C1/C2/C3;\\na max-deviation statistic gives approximately simultaneous intervals\\nover the predeclared novelty-share grid.\\n\\nThese bounds capture validation sampling variation, not Kaggle domain shift,\\nhyperparameter search multiplicity, model bias, or data leakage.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport numpy as np\\n\\nDEFAULT_GRID = tuple(\\n    (f1, f3) for f1 in (0.15, 0.20, 0.25, 0.30)\\n    for f3 in (0.2, 0.3, 0.4)\\n)\\n\\n\\ndef _ranks_to_rr(values, cls, side):\\n    ranks = np.asarray(values)\\n    if ranks.ndim != 1 or not len(ranks):\\n        raise ValueError(f\\"{side} {cls}: ranks must be nonempty 1D\\")\\n    if not np.issubdtype(ranks.dtype, np.integer):\\n        raise ValueError(f\\"{side} {cls}: ranks must be integers\\")\\n    if np.any((ranks < 0) | (ranks > 25)):\\n        raise ValueError(f\\"{side} {cls}: invalid rank (valid: 0 to 25)\\")\\n    return np.where(ranks > 0, 1.0 / np.maximum(ranks, 1), 0.0)\\n\\n\\ndef paired_novelty_deltas(a: dict, b: dict) -> np.ndarray:\\n    \\"\\"\\"Molecule x class effect matrix, with preverified aligned query ordering.\\"\\"\\"\\n    columns = []\\n    n = None\\n    for cls in (\\"C1\\", \\"C2\\", \\"C3\\"):\\n        if cls not in a or cls not in b:\\n            raise ValueError(f\\"Missing novelty class {cls}\\")\\n        x, y = _ranks_to_rr(a[cls], cls, \\"A\\"), _ranks_to_rr(b[cls], cls, \\"B\\")\\n        if x.shape != y.shape or (n is not None and len(x) != n):\\n            raise ValueError(\\"Unpaired class ranks, unequal lengths\\")\\n        n = len(x)\\n        columns.append(y - x)\\n    return np.stack(columns, axis=1)\\n\\n\\ndef compare_mixtures(\\n    a: dict,\\n    b: dict,\\n    *,\\n    grid=DEFAULT_GRID,\\n    n_boot: int = 4000,\\n    seed: int = 0,\\n    confidence: float = 0.95,\\n) -> dict:\\n    \\"\\"\\"Approximate simultaneous CI over a fixed mixture grid.\\n\\n    Must follow align_runs; this function is not an identity check.\\n    Bootstrap rows remain paired across all three novelty classes.\\n    \\"\\"\\"\\n    if type(n_boot) is not int or n_boot < 100:\\n        raise ValueError(\\"Use at least 100 bootstrap resamples\\")\\n    if not 0 < confidence < 1:\\n        raise ValueError(\\"Confidence must lie strictly between 0 and 1\\")\\n    if not isinstance(seed, (int, np.integer)) or seed < 0:\\n        raise ValueError(\\"Seed must be a nonnegative integer\\")\\n    shares = np.asarray(grid, dtype=float)\\n    if shares.ndim != 2 or shares.shape[1] != 2 or len(shares) == 0:\\n        raise ValueError(\\"Grid must contain (f1, f3) pairs\\")\\n    if not np.isfinite(shares).all():\\n        raise ValueError(\\"Mixture shares must be finite\\")\\n    f1, f3 = shares[:, 0], shares[:, 1]\\n    f2 = 1.0 - f1 - f3\\n    if np.any(f1 < 0) or np.any(f3 < 0) or np.any(f2 < -1e-12):\\n        raise ValueError(\\"Mixture fractions must be nonnegative and sum to 1\\")\\n    f2 = np.maximum(f2, 0.0)\\n    weights = np.stack([f1, f2, f3], axis=1)\\n\\n    effects = paired_novelty_deltas(a, b)\\n    mean = effects.mean(axis=0) @ weights.T\\n    rng = np.random.default_rng(int(seed))\\n    indices = rng.integers(0, effects.shape[0], size=(n_boot, effects.shape[0]))\\n    sampled = effects[indices].mean(axis=1) @ weights.T\\n    max_abs_deviation = np.max(np.abs(sampled - mean), axis=1)\\n    radius = float(np.quantile(max_abs_deviation, confidence))\\n    lower, upper = mean - radius, mean + radius\\n    verdict = (\\"ACCEPT\\" if np.all(lower > 0) else\\n               \\"REJECT\\" if np.all(upper < 0) else \\"NOT PROVEN\\")\\n    scenarios = [\\n        {\\"f1\\": float(f1[i]), \\"f2\\": float(f2[i]), \\"f3\\": float(f3[i]),\\n         \\"delta_mrr\\": float(mean[i]), \\"ci_low\\": float(lower[i]),\\n         \\"ci_high\\": float(upper[i])}\\n        for i in range(len(mean))\\n    ]\\n    return {\\n        \\"method\\": \\"paired bootstrap; maximum absolute deviation over fixed mixture grid\\",\\n        \\"confidence\\": confidence,\\n        \\"familywise\\": \\"approximate for a fixed prespecified grid; not post-selection corrected\\",\\n        \\"n_molecules\\": int(len(effects)),\\n        \\"n_boot\\": n_boot,\\n        \\"seed\\": int(seed),\\n        \\"half_width\\": radius,\\n        \\"verdict\\": verdict,\\n        \\"scenarios\\": scenarios,\\n    }\\n", "src/casmi/validate/source_manifest.py": "\\"\\"\\"Privacy-preserving SHA-256 manifests for approved local CASMI artifacts.\\n\\nNo dataset payload, raw spectra, private labels, credentials or absolute paths\\nare written into a manifest. This is a provenance check, not a security\\nsignature or an assertion that upstream Kaggle/BigQuery data are correct.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\nfrom pathlib import Path, PurePosixPath\\n\\nMANIFEST_SCHEMA = \\"casmi-local-artifact-manifest-v1\\"\\nCHUNK_BYTES = 1024 * 1024\\n\\n\\ndef _safe_relative(root: Path, name: str) -> Path:\\n    if not isinstance(name, str) or not name or \\"\\\\\\\\\\" in name:\\n        raise ValueError(\\"Manifest entries must be normalized relative POSIX paths\\")\\n    normalized = PurePosixPath(name)\\n    if (normalized.is_absolute() or name != normalized.as_posix()\\n            or any(part in (\\".\\", \\"..\\") for part in name.split(\\"/\\"))):\\n        raise ValueError(f\\"Unsafe artifact path: {name!r}\\")\\n    path = (root / name).resolve(strict=True)\\n    if not path.is_relative_to(root.resolve()):\\n        raise ValueError(f\\"Artifact resolves outside source root: {name!r}\\")\\n    if not path.is_file():\\n        raise ValueError(f\\"Expected an artifact file: {name!r}\\")\\n    return path\\n\\n\\ndef sha256_file(path: Path) -> str:\\n    digest = hashlib.sha256()\\n    with path.open(\\"rb\\") as file:\\n        for chunk in iter(lambda: file.read(CHUNK_BYTES), b\\"\\"):\\n            digest.update(chunk)\\n    return digest.hexdigest()\\n\\n\\ndef _canonical_files_digest(files: list[dict]) -> str:\\n    payload = json.dumps(files, ensure_ascii=True, sort_keys=True,\\n                         separators=(\\",\\", \\":\\"))\\n    return hashlib.sha256(payload.encode(\\"utf-8\\")).hexdigest()\\n\\n\\ndef create_manifest(root: Path, relative_paths) -> dict:\\n    \\"\\"\\"Hash existing explicit input artifacts; no directory glob or traversal.\\"\\"\\"\\n    root = Path(root).resolve(strict=True)\\n    if not root.is_dir():\\n        raise ValueError(\\"Source root must be a directory\\")\\n    if not isinstance(relative_paths, (list, tuple)) or not relative_paths:\\n        raise ValueError(\\"Provide at least one explicit input-artifact path\\")\\n    if len(set(relative_paths)) != len(relative_paths):\\n        raise ValueError(\\"Duplicate artifact manifest path\\")\\n    files = []\\n    for name in sorted(relative_paths):\\n        path = _safe_relative(root, name)\\n        files.append({\\n            \\"path\\": name, \\"bytes\\": path.stat().st_size,\\n            \\"sha256\\": sha256_file(path),\\n        })\\n    return {\\n        \\"schema\\": MANIFEST_SCHEMA,\\n        \\"method\\": \\"streaming SHA-256 per local input file\\",\\n        \\"files\\": files,\\n        \\"files_digest_sha256\\": _canonical_files_digest(files),\\n        \\"source_assertion\\": (\\n            \\"local byte identity only; no independent Kaggle/BigQuery validation\\"\\n        ),\\n    }\\n\\n\\ndef verify_manifest(root: Path, manifest: dict) -> dict:\\n    \\"\\"\\"Fail closed when a local artifact differs from the recorded snapshot.\\"\\"\\"\\n    if not isinstance(manifest, dict) or manifest.get(\\"schema\\") != MANIFEST_SCHEMA:\\n        raise ValueError(\\"Unsupported artifact manifest schema\\")\\n    entries = manifest.get(\\"files\\")\\n    if not isinstance(entries, list) or not entries:\\n        raise ValueError(\\"Manifest must contain artifact entries\\")\\n    paths = [x.get(\\"path\\") for x in entries if isinstance(x, dict)]\\n    if len(paths) != len(entries) or len(paths) != len(set(paths)):\\n        raise ValueError(\\"Invalid or duplicate manifest paths\\")\\n    for row in entries:\\n        if type(row.get(\\"bytes\\")) is not int or row[\\"bytes\\"] < 0:\\n            raise ValueError(\\"Invalid manifest file size\\")\\n        sha = row.get(\\"sha256\\")\\n        if not isinstance(sha, str) or len(sha) != 64 or any(c not in \\"0123456789abcdef\\" for c in sha):\\n            raise ValueError(\\"Invalid manifest file hash\\")\\n    if entries != sorted(entries, key=lambda e: e[\\"path\\"]):\\n        raise ValueError(\\"Manifest entries must be sorted\\")\\n    digest = manifest.get(\\"files_digest_sha256\\")\\n    if digest != _canonical_files_digest(entries):\\n        raise ValueError(\\"Manifest file list/digest mismatch\\")\\n    fresh = create_manifest(root, paths)\\n    if fresh[\\"files\\"] != entries:\\n        raise ValueError(\\"Local input artifacts differ from the signed-inventory snapshot\\")\\n    return {\\n        \\"status\\": \\"VERIFIED_LOCAL_ARTIFACTS\\",\\n        \\"schema\\": MANIFEST_SCHEMA,\\n        \\"files_digest_sha256\\": fresh[\\"files_digest_sha256\\"],\\n        \\"files_checked\\": len(entries),\\n    }\\n\\ndef required_validation_inputs(*, pool_tag: str = \\"\\", analog: bool = False) -> tuple[str, ...]:\\n    \\"\\"\\"Enumerate the minimum immutable data inputs used by validation.\\n\\n    Only the required static store and pool artifacts are included here.\\n    Derived caches and mutable metric-key memoization remain outside this\\n    minimal source-input attestation.\\n    \\"\\"\\"\\n    import re\\n\\n    if not isinstance(pool_tag, str) or not re.fullmatch(r\\"(?:_[A-Za-z0-9_-]+)?\\", pool_tag):\\n        raise ValueError(\\"Invalid candidate-pool tag\\")\\n    files = [\\n        \\"store/meta.parquet\\", \\"store/offsets.i64\\", \\"store/mz.f32\\",\\n        \\"store/it.f32\\", f\\"pool{pool_tag}.parquet\\",\\n    ]\\n    if analog:\\n        files.extend((\\n            f\\"pool_fp2048{pool_tag}.npy\\",\\n            f\\"pool_fp_keys{pool_tag}.parquet\\",\\n        ))\\n    return tuple(sorted(files))\\n\\n\\ndef require_validation_coverage(manifest: dict, *, pool_tag: str = \\"\\",\\n                                analog: bool = False) -> tuple[str, ...]:\\n    \\"\\"\\"Fail closed if an opt-in inventory omits an input consumed by this run.\\n\\n    Must be followed by verify_manifest to validate file contents and hashes.\\n    \\"\\"\\"\\n    if not isinstance(manifest, dict) or not isinstance(manifest.get(\\"files\\"), list):\\n        raise ValueError(\\"Invalid local artifact inventory\\")\\n    paths = set()\\n    for entry in manifest[\\"files\\"]:\\n        if not isinstance(entry, dict) or not isinstance(entry.get(\\"path\\"), str):\\n            raise ValueError(\\"Malformed local artifact path entry\\")\\n        paths.add(entry[\\"path\\"])\\n    expected = required_validation_inputs(pool_tag=pool_tag, analog=analog)\\n    missing = sorted(set(expected) - paths)\\n    if missing:\\n        raise ValueError(\\n            \\"Source manifest omits required validation inputs: \\" + \\", \\".join(missing)\\n        )\\n    return expected\\n\\n", "src/casmi/validate/spectral_controls.py": "\\"\\"\\"Reproducible negative-control perturbations of query MS/MS peaks.\\n\\nThese transformations are *diagnostics*, never submission-model features:\\n- blank: suppress all fragment evidence while preserving precursor mass,\\n  adduct, ionization mode and collision energy; this exposes non-spectral\\n  candidate-pool/mass priors.\\n- intensity_shuffle: independently permute fragment intensities within each\\n  spectrum. m/z and precursor stay fixed, while intensity/mz relationships\\n  are intentionally broken. Identical intensities cannot be changed.\\n\\nOriginal query spectra are never mutated. For molecule-level reproducibility\\npermutations derive from (seed, query ID, spectrum index), not list position.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\n\\nimport numpy as np\\n\\nMODES = (\\"none\\", \\"blank\\", \\"intensity_shuffle\\")\\n\\n\\ndef perturb_spectra(spectra, *, mode: str = \\"none\\",\\n                    seed: int = 0, query_id: str = \\"\\") -> list[dict]:\\n    \\"\\"\\"Make an independent copy of each spectrum, perturbing only peak arrays.\\"\\"\\"\\n    if mode not in MODES:\\n        raise ValueError(f\\"Unsupported spectral control: {mode}\\")\\n    if type(seed) is not int or seed < 0:\\n        raise ValueError(\\"Control seed must be a nonnegative integer\\")\\n    if not isinstance(query_id, str) or not query_id:\\n        raise ValueError(\\"query_id must identify one molecule\\")\\n    controlled = []\\n    for index, spectrum in enumerate(spectra):\\n        original_mz = np.array(spectrum[\\"mz\\"], copy=True)\\n        original_it = np.array(spectrum[\\"it\\"], copy=True)\\n        if original_mz.ndim != 1 or original_it.ndim != 1 or len(original_mz) != len(original_it):\\n            raise ValueError(\\"Fragment m/z and intensity arrays must have matching 1D shapes\\")\\n        item = dict(spectrum)\\n        if mode == \\"blank\\":\\n            item[\\"mz\\"] = np.empty(0, dtype=original_mz.dtype)\\n            item[\\"it\\"] = np.empty(0, dtype=original_it.dtype)\\n        elif mode == \\"intensity_shuffle\\" and len(original_it) > 1:\\n            digest = hashlib.sha256(\\n                f\\"casmi-negative-control-v1:{seed}:{query_id}:{index}\\".encode()\\n            ).digest()\\n            rng = np.random.default_rng(int.from_bytes(digest[:8], \\"big\\"))\\n            permutation = rng.permutation(len(original_it))\\n            if np.array_equal(permutation, np.arange(len(original_it))):\\n                permutation = np.roll(permutation, 1)\\n            item[\\"mz\\"] = original_mz\\n            item[\\"it\\"] = original_it[permutation]\\n        else:\\n            item[\\"mz\\"], item[\\"it\\"] = original_mz, original_it\\n        controlled.append(item)\\n    return controlled\\n", "src/casmi/validate/spectrum_overlap.py": "\\"\\"\\"Exact-peak overlap helpers for CASMI validation leakage checks.\\n\\nA precursor mass, number of peaks and first m/z are a *bucket*, not evidence\\nof a duplicate spectrum. Only full arrays, including intensities, can confirm\\na same-spectrum match. This module deliberately has no numerical dependencies.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport math\\nfrom collections.abc import Sequence\\n\\n\\ndef coarse_spectrum_bucket(precursor_mz: float, ms2_mzs: Sequence) -> tuple | None:\\n    \\"\\"\\"Cheap candidate retrieval; NEVER use the bucket as a match on its own.\\"\\"\\"\\n    if ms2_mzs is None or len(ms2_mzs) == 0:\\n        return None\\n    try:\\n        precursor, first = float(precursor_mz), float(ms2_mzs[0])\\n    except (TypeError, ValueError):\\n        return None\\n    if not math.isfinite(precursor) or not math.isfinite(first):\\n        return None\\n    return round(precursor, 4), len(ms2_mzs), round(first, 4)\\n\\n\\ndef full_spectrum_match(\\n    precursor_a, mz_a: Sequence, intensities_a: Sequence,\\n    precursor_b, mz_b: Sequence, intensities_b: Sequence,\\n) -> bool:\\n    \\"\\"\\"Require full precursor, fragment m/z and normalized intensity equality.\\n\\n    Converts array scalar dtypes to Python float without rounding. An equal\\n    coarse bucket with shifted *any* peak or intensity is not a verified match.\\n    \\"\\"\\"\\n    if mz_a is None or mz_b is None or intensities_a is None or intensities_b is None:\\n        return False\\n    if len(mz_a) == 0 or len(mz_b) == 0:\\n        return False\\n    if len(mz_a) != len(mz_b) or len(intensities_a) != len(intensities_b):\\n        return False\\n    if len(mz_a) != len(intensities_a):\\n        return False\\n    try:\\n        values_a = [float(precursor_a)] + [float(x) for x in mz_a] + [float(x) for x in intensities_a]\\n        values_b = [float(precursor_b)] + [float(x) for x in mz_b] + [float(x) for x in intensities_b]\\n    except (TypeError, ValueError):\\n        return False\\n    if not all(math.isfinite(x) for x in values_a + values_b):\\n        return False\\n    return values_a == values_b\\n", "scripts/build_structures.py": "\\"\\"\\"A2 \\u2014 structure table: one row per training structure.\\n\\nOutput: artifacts/structures.parquet\\n  pool_key    inchikey14 as shipped in train.parquet (standard InChIKey14; also the pool identity)\\n  smiles, formula, mass (monoisotopic, from formula)\\n  libs        bitmask of ingest_lib values the structure appears in\\n  n_spectra\\n\\nThe metric key (tautomer-canonical InChIKey14) is computed lazily by casmi.chem.keys.MetricKeys.\\n\\"\\"\\"\\nimport sys\\nimport time\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\nimport pyarrow.parquet as pq\\n\\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \\"src\\"))\\nfrom casmi.config import ARTIFACT_DIR, DATA_DIR  # noqa: E402\\nfrom casmi.chem.masses import formula_mass  # noqa: E402\\n\\nLIBS = [\\"enveda-180\\", \\"pluskal_ms2\\", \\"riken\\", \\"gnps\\", \\"massbank\\", \\"mona\\", \\"spectraverse\\",\\n        \\"msdial\\", \\"drug_plus\\", \\"enveda-np-examples\\", \\"masaryk\\"]\\nLIB_BIT = {lib: 1 << i for i, lib in enumerate(LIBS)}\\n\\n\\ndef main():\\n    t0 = time.time()\\n    pf = pq.ParquetFile(DATA_DIR / \\"train.parquet\\")\\n    frames = []\\n    for rg in range(pf.num_row_groups):\\n        t = pf.read_row_group(rg, columns=[\\"inchikey14\\", \\"normalized_smiles\\", \\"molecular_formula\\", \\"ingest_lib\\"]).to_pandas()\\n        t[\\"bit\\"] = t.ingest_lib.map(LIB_BIT).astype(\\"int64\\")\\n        g = t.groupby(\\"inchikey14\\")\\n        frames.append(pd.DataFrame({\\"smiles\\": g.normalized_smiles.first(), \\"formula\\": g.molecular_formula.first(),\\n                                    \\"libs\\": g.bit.agg(np.bitwise_or.reduce), \\"n_spectra\\": g.size()}))\\n    s = pd.concat(frames)\\n    g = s.groupby(level=0)\\n    s = pd.DataFrame({\\"smiles\\": g.smiles.first(), \\"formula\\": g.formula.first(),\\n                      \\"libs\\": g.libs.agg(np.bitwise_or.reduce), \\"n_spectra\\": g.n_spectra.sum()})\\n    s.index.name = \\"pool_key\\"\\n    s = s.reset_index()\\n    s[\\"mass\\"] = s.formula.map(formula_mass)\\n    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)\\n    s.to_parquet(ARTIFACT_DIR / \\"structures.parquet\\", index=False)\\n    print(f\\"{len(s):,} structures, mass missing {s.mass.isna().sum()}, built in {time.time() - t0:.0f}s\\")\\n    for lib, bit in LIB_BIT.items():\\n        print(f\\"  {lib:<20} {((s.libs & bit) > 0).sum():>7,}\\")\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "scripts/build_store.py": "\\"\\"\\"Build the on-disk spectrum store (A1) from train.parquet, one row group at a time.\\n\\nPeaks are read through Arrow\'s flat value/offset buffers instead of Python lists, which keeps\\npeak memory at a few hundred MB per row group. Masses and ppm errors are recomputed from the\\nformula and adduct, because the shipped precursor_error_ppm column follows different conventions\\nin different libraries (see reports/).\\n\\"\\"\\"\\nimport argparse\\nimport sys\\nimport time\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\nimport pyarrow as pa\\nimport pyarrow.parquet as pq\\n\\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \\"src\\"))\\nfrom casmi.config import ARTIFACT_DIR, DATA_DIR  # noqa: E402\\nfrom casmi.chem.masses import TEST_ADDUCTS, formula_mass, neutral_mass, parse_adduct  # noqa: E402\\nfrom casmi.spectra.clean import CleanConfig, clean_peaks_fast  # noqa: E402\\nfrom casmi.spectra.store_integrity import inspect_store_layout  # noqa: E402\\nfrom casmi.spectra.build_safety import fresh_store_target  # noqa: E402\\n\\nSTORE_CFG = CleanConfig(min_rel_intensity=0.001, max_peaks=150, top_k_per_window=0)\\nMETA_COLS = [\\"ingest_lib\\", \\"inchikey14\\", \\"molecular_formula\\", \\"ionization_mode\\", \\"instrument_type\\",\\n             \\"adduct\\", \\"precursor_mz\\", \\"base_peak_intensity\\", \\"collision_energy_orig_units\\"]\\n\\n\\ndef _flat(col):\\n    \\"\\"\\"(values, offsets) of a list column as NumPy, without materialising Python lists.\\"\\"\\"\\n    arr = col.combine_chunks() if isinstance(col, pa.ChunkedArray) else col\\n    return arr.values.to_numpy(zero_copy_only=False), arr.offsets.to_numpy()\\n\\n\\ndef _build_store(out: Path, train_file: Path):\\n    \\"\\"\\"Write and validate a brand-new staging store, never the frozen one.\\"\\"\\"\\n    pf = pq.ParquetFile(train_file)\\n    f_mz, f_it = open(out / \\"mz.f32\\", \\"wb\\"), open(out / \\"it.f32\\", \\"wb\\")\\n    offsets, metas, n_peaks_total = [0], [], 0\\n    t0 = time.time()\\n\\n    for rg in range(pf.num_row_groups):\\n        tbl = pf.read_row_group(rg, columns=META_COLS + [\\"ms2_mzs\\", \\"ms2_normalized_intensities\\", \\"collision_energy_ev\\"])\\n        if tbl.num_rows == 0:            # a filtered train.parquet (validation hold-out) can carry empty row groups\\n            continue\\n        meta = tbl.select(META_COLS).to_pandas()\\n        mz_v, mz_o = _flat(tbl.column(\\"ms2_mzs\\"))\\n        it_v, it_o = _flat(tbl.column(\\"ms2_normalized_intensities\\"))\\n        ce_v, ce_o = _flat(tbl.column(\\"collision_energy_ev\\"))\\n        prec = meta.precursor_mz.to_numpy()\\n\\n        buf_mz, buf_it, lens = [], [], np.zeros(len(meta), np.int32)\\n        for i in range(len(meta)):\\n            cm, ci = clean_peaks_fast(mz_v[mz_o[i]:mz_o[i + 1]], it_v[it_o[i]:it_o[i + 1]], prec[i], STORE_CFG)\\n            buf_mz.append(cm)\\n            buf_it.append(ci)\\n            lens[i] = cm.size\\n        np.concatenate(buf_mz).tofile(f_mz)\\n        np.concatenate(buf_it).tofile(f_it)\\n        offsets.extend((n_peaks_total + np.cumsum(lens)).tolist())\\n        n_peaks_total += int(lens.sum())\\n\\n        # collision energy: mean of the listed eV values, count of merged acquisitions\\n        # (bincount over segment ids, not np.add.reduceat, which mishandles empty trailing lists)\\n        ce_n = np.diff(ce_o)\\n        seg = np.repeat(np.arange(len(meta)), ce_n)\\n        ce_sum = np.bincount(seg, weights=np.nan_to_num(ce_v[ce_o[0]:ce_o[-1]]), minlength=len(meta))\\n        ce_sum = np.where(ce_n > 0, ce_sum, np.nan)\\n        meta[\\"ce_ev\\"] = (ce_sum / np.maximum(ce_n, 1)).astype(np.float32)\\n        meta[\\"ce_count\\"] = ce_n.astype(np.int8)\\n        meta[\\"n_peaks\\"] = lens\\n        meta[\\"row_group\\"] = np.int16(rg)\\n        metas.append(meta)\\n        print(f\\"row group {rg + 1}/{pf.num_row_groups}: {len(meta):,} spectra, {lens.mean():.0f} peaks avg, \\"\\n              f\\"{time.time() - t0:.0f}s\\", flush=True)\\n        del tbl, buf_mz, buf_it\\n\\n    f_mz.close()\\n    f_it.close()\\n    np.asarray(offsets, dtype=np.int64).tofile(out / \\"offsets.i64\\")\\n\\n    meta = pd.concat(metas, ignore_index=True)\\n    meta = meta.rename(columns={\\"inchikey14\\": \\"train_ik14\\", \\"molecular_formula\\": \\"formula\\"})\\n    meta[\\"is_timstof\\"] = meta.ingest_lib.isin([\\"enveda-180\\", \\"enveda-np-examples\\"])\\n    meta[\\"test_adduct\\"] = meta.adduct.isin(TEST_ADDUCTS)\\n\\n    # our own masses: neutral mass implied by precursor + adduct, and signed ppm vs the formula\\n    meta[\\"formula_mass\\"] = meta.formula.map(formula_mass).astype(np.float64)\\n    nm = np.full(len(meta), np.nan)\\n    for adduct, idx in meta.groupby(\\"adduct\\").indices.items():\\n        if parse_adduct(adduct) is not None:\\n            nm[idx] = neutral_mass(meta.precursor_mz.to_numpy()[idx], adduct)\\n    meta[\\"neutral_mass\\"] = nm\\n    meta[\\"ppm\\"] = ((nm - meta.formula_mass) / meta.formula_mass * 1e6).astype(np.float32)\\n\\n    for c in [\\"ingest_lib\\", \\"ionization_mode\\", \\"instrument_type\\", \\"adduct\\", \\"collision_energy_orig_units\\"]:\\n        meta[c] = meta[c].astype(\\"category\\")\\n    meta.to_parquet(out / \\"meta.parquet\\", index=False)\\n    # Reject silently truncated/shifted flat arrays immediately after the build.\\n    # This audits structural alignment, not the chemical validity of each peak.\\n    layout = inspect_store_layout(\\n        out, expected_spectra=len(meta),\\n        metadata_n_peaks=meta[\\"n_peaks\\"].to_numpy(),\\n    )\\n    if layout.peak_count != n_peaks_total:\\n        raise ValueError(\\"Builder peak total disagrees with persisted store\\")\\n    print(f\\"\\\\nstore: {len(meta):,} spectra, {n_peaks_total:,} peaks \\"\\n          f\\"({n_peaks_total * 8 / 1e9:.2f} GB), built in {time.time() - t0:.0f}s\\")\\n\\n\\n\\ndef main(argv=None):\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\n        \\"--output-dir\\", type=Path, default=ARTIFACT_DIR / \\"store\\",\\n        help=\\"new, nonexistent A1 store directory; existing snapshots are immutable\\",\\n    )\\n    parser.add_argument(\\n        \\"--train\\", type=Path, default=DATA_DIR / \\"train.parquet\\",\\n        help=\\"authorized local training Parquet source\\",\\n    )\\n    args = parser.parse_args(argv)\\n    with fresh_store_target(args.output_dir) as staging:\\n        _build_store(staging, args.train)\\n    print(f\\"Published immutable A1 store to {args.output_dir}\\", flush=True)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "scripts/build_pool.py": "\\"\\"\\"A3 \\u2014 candidate pool = training structures \\u222a COCONUT 2.0, keyed on standard InChIKey14.\\n\\nCOCONUT (CC0, coconut.naturalproducts.net, 09-2026 lite CSV) contributes natural-product\\nstructures that have no reference spectra \\u2014 exactly the Class-2 search space. Per-structure\\nCOCONUT annotations (NP-likeness, NPClassifier pathway, sugars) are kept as ranker priors.\\n\\nOutput: artifacts/pool.parquet\\n\\"\\"\\"\\nimport sys\\nimport json\\nimport time\\nimport zipfile\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \\"src\\"))\\nfrom casmi.config import ARTIFACT_DIR, POOL_FILE, POOL_FP_FILE, POOL_FP_KEYS_FILE  # noqa: E402\\nfrom casmi.chem.masses import formula_mass  # noqa: E402\\nfrom casmi.pool.pool import SOURCE_BITS  # noqa: E402\\nfrom casmi.pool.structure_identity import neutral_parent_identity  # noqa: E402\\nfrom casmi.pool.source_reconciliation import reconcile_source_rows  # noqa: E402\\nfrom casmi.pool.build_safety import fresh_pool_file  # noqa: E402\\n\\nimport os  # noqa: E402\\nCOCONUT_ZIP = Path(os.environ.get(\\"CASMI_COCONUT\\", \\"C:/Users/MONSTER/kaggle-data/external/coconut/coconut_csv_lite-09-2026.zip\\"))\\nPUBCHEMLITE = os.environ.get(\\"CASMI_PUBCHEMLITE\\")   # optional: PubChemLite for Exposomics CSV (CC BY 4.0)\\n# optional: directory with extra_lipidmaps.parquet (LIPID MAPS, CC0) / extra_chebi.parquet (ChEBI, CC BY 4.0),\\n# produced by scripts/build_extra_sources.py\\nEXTRA_DIR = os.environ.get(\\"CASMI_EXTRA_SOURCES\\")\\n# optional: PubChem literature subset (extra_pubchem.parquet in EXTRA_DIR, scripts/build_pubchem_subset.py),\\n# capped to the N best-documented structures (by number of PubMed articles)\\nPUBCHEM_MAX = int(os.environ.get(\\"CASMI_PUBCHEM_MAX\\", \\"0\\"))\\nMASS_RANGE = (100.0, 1300.0)          # test monoisotopic masses span 157\\u20131,159 Da\\nENVEDA180_BIT = 1                     # bit 0 in build_structures.LIBS\\n\\n\\ndef load_coconut():\\n    cols = [\\"identifier\\", \\"canonical_smiles\\", \\"standard_inchi_key\\", \\"molecular_formula\\",\\n            \\"exact_molecular_weight\\", \\"formal_charge\\", \\"np_likeness\\", \\"np_classifier_pathway\\",\\n            \\"np_classifier_superclass\\", \\"contains_sugar\\"]\\n    if COCONUT_ZIP.suffix == \\".zip\\":\\n        with zipfile.ZipFile(COCONUT_ZIP).open(\\"coconut_csv_lite-09-2026.csv\\") as f:\\n            c = pd.read_csv(f, usecols=cols, low_memory=False)\\n    else:   # Kaggle extracts uploaded archives, so the dataset holds the CSV itself\\n        c = pd.read_csv(COCONUT_ZIP, usecols=cols, low_memory=False)\\n    n0 = len(c)\\n    c = c[(c.formal_charge == 0) & c.canonical_smiles.notna() & c.standard_inchi_key.notna()].copy()\\n    # Preserve the original standard key for unchanged single-component\\n    # molecules. For every salt/mixture, derive SMILES, standard InChIKey14\\n    # and exact mass from the SAME neutral parent. Keeping the old compound\\n    # InChIKey after stripping fragments silently mislabels candidate rows.\\n    c[\\"pool_key\\"] = c.standard_inchi_key.str[:14]\\n    c[\\"mass\\"] = c.molecular_formula.fillna(\\"\\").map(formula_mass)\\n    multi = c.canonical_smiles.str.contains(\\".\\", regex=False)\\n    c[\\"_derived_parent\\"] = multi\\n    rejected_parents = 0\\n    retained_parents = 0\\n    if multi.any():\\n        normalized = c.loc[multi, \\"canonical_smiles\\"].map(neutral_parent_identity)\\n        valid = normalized.map(lambda item: item is not None)\\n        rejected_parents = int((~valid).sum())\\n        accepted = normalized.loc[valid]\\n        retained_parents = len(accepted)\\n        if len(accepted):\\n            parent = pd.DataFrame(\\n                accepted.tolist(), index=accepted.index,\\n                columns=[\\"canonical_smiles\\", \\"pool_key\\", \\"mass\\", \\"molecular_formula\\"],\\n            )\\n            c.loc[parent.index, [\\n                \\"canonical_smiles\\", \\"pool_key\\", \\"mass\\", \\"molecular_formula\\",\\n            ]] = parent[[\\n                \\"canonical_smiles\\", \\"pool_key\\", \\"mass\\", \\"molecular_formula\\",\\n            ]]\\n        c = c.drop(index=normalized.index[~valid])\\n    # If a source supplies both the original neutral parent and a salt of\\n    # the same parent, prefer the original single-component annotation.\\n    # Never let a transformed salt displace a directly identified molecule.\\n    c = (c[c.mass.between(*MASS_RANGE) & c.pool_key.notna()]\\n         .sort_values(\\"_derived_parent\\", kind=\\"stable\\")\\n         .drop_duplicates(\\"pool_key\\")\\n         .drop(columns=\\"_derived_parent\\"))\\n    print(f\\"COCONUT: {n0:,} rows -> {len(c):,} neutral structures in {MASS_RANGE} Da \\"\\n          f\\"({retained_parents:,} neutral parents recovered, {rejected_parents:,} invalid/charged parents excluded)\\")\\n    return c.rename(columns={\\"canonical_smiles\\": \\"smiles\\"})\\n\\n\\ndef load_pubchemlite():\\n    \\"\\"\\"PubChemLite for Exposomics (Schymanski et al., Zenodo 22143470, CC BY 4.0). Only structure\\n    columns are kept: its annotation counts exist for PubChemLite rows alone and would act as a\\n    source flag in the ranker.\\"\\"\\"\\n    c = pd.read_csv(PUBCHEMLITE, usecols=[\\"FirstBlock\\", \\"SMILES\\", \\"MolecularFormula\\", \\"MonoisotopicMass\\"])\\n    n0 = len(c)\\n    c = c.dropna()\\n    c = c[~c.MolecularFormula.str.contains(r\\"[+-]\\") & ~c.SMILES.str.contains(\\".\\", regex=False)]\\n    c = c[c.MonoisotopicMass.between(*MASS_RANGE)].drop_duplicates(\\"FirstBlock\\")\\n    print(f\\"PubChemLite: {n0:,} rows -> {len(c):,} neutral single-component structures in {MASS_RANGE} Da\\")\\n    return pd.DataFrame({\\"pool_key\\": c.FirstBlock, \\"smiles\\": c.SMILES, \\"mass\\": c.MonoisotopicMass,\\n                         \\"sources\\": SOURCE_BITS[\\"pubchem\\"], \\"n_spectra\\": 0, \\"libs\\": 0})\\n\\n\\ndef main():\\n    # The candidate pool is an immutable research snapshot. In particular,\\n    # COCONUT parent-identity fixes must not silently replace the old pool\\n    # while leaving stale Morgan fingerprints and model artifacts behind.\\n    output = ARTIFACT_DIR / POOL_FILE\\n    if os.path.lexists(output):\\n        raise FileExistsError(\\n            f\\"Refusing to overwrite frozen candidate pool {output}; \\"\\n            \\"set a new CASMI_POOL_TAG for a separately evaluated variant\\"\\n        )\\n    t0 = time.time()\\n    s = pd.read_parquet(ARTIFACT_DIR / \\"structures.parquet\\")\\n    train = pd.DataFrame({\\"pool_key\\": s.pool_key, \\"smiles\\": s.smiles, \\"mass\\": s.mass,\\n                          \\"sources\\": SOURCE_BITS[\\"train\\"] | np.where(s.libs == ENVEDA180_BIT, SOURCE_BITS[\\"enveda180_only\\"], 0),\\n                          \\"n_spectra\\": s.n_spectra, \\"libs\\": s.libs})\\n    coco = load_coconut()\\n    coco_part = pd.DataFrame({\\"pool_key\\": coco.pool_key, \\"smiles\\": coco.smiles, \\"mass\\": coco.mass,\\n                              \\"sources\\": SOURCE_BITS[\\"coconut\\"], \\"n_spectra\\": 0, \\"libs\\": 0,\\n                              \\"np_likeness\\": coco.np_likeness, \\"np_pathway\\": coco.np_classifier_pathway,\\n                              \\"np_superclass\\": coco.np_classifier_superclass, \\"has_sugar\\": coco.contains_sugar})\\n    parts = [train, coco_part]\\n    if PUBCHEMLITE:\\n        parts.append(load_pubchemlite())\\n    if EXTRA_DIR:\\n        names = [n for n in os.environ.get(\\"CASMI_EXTRA_NAMES\\", \\"chebi,lipidmaps\\").split(\\",\\") if n]\\n        for name in names:\\n            bit = SOURCE_BITS[name]\\n            e = pd.read_parquet(Path(EXTRA_DIR) / f\\"extra_{name}.parquet\\")\\n            parts.append(e.assign(sources=bit, n_spectra=0, libs=0))\\n            print(f\\"{name}: {len(e):,} structures\\")\\n        if PUBCHEM_MAX:\\n            e = pd.read_parquet(Path(EXTRA_DIR) / \\"extra_pubchem.parquet\\", columns=[\\"pool_key\\", \\"smiles\\", \\"mass\\", \\"n_pmid\\"])\\n            e = e.nlargest(PUBCHEM_MAX, \\"n_pmid\\", keep=\\"first\\").drop(columns=\\"n_pmid\\")\\n            parts.append(e.assign(sources=SOURCE_BITS[\\"pubchem_lit\\"], n_spectra=0, libs=0))\\n            print(f\\"PubChem literature subset: {len(e):,} structures\\")\\n    both = pd.concat(parts, ignore_index=True)\\n    # Never let groupby.first() assemble SMILES from one source and mass\\n    # from another. All identity fields must coexist in the chosen row.\\n    # Reject material mass contradictions for a shared InChIKey14.\\n    both, quality = reconcile_source_rows(\\n        both, quarantine_conflicts=os.environ.get(\\"CASMI_QUARANTINE_MASS_CONFLICTS\\") == \\"1\\",\\n    )\\n    quality_path = ARTIFACT_DIR / f\\"pool_source_quality{os.environ.get(\'CASMI_POOL_TAG\', \'\')}.json\\"\\n    print(\\"Source reconciliation: \\"\\n          f\\"{quality[\'complete_rows\']:,}/{quality[\'input_rows\']:,} complete rows, \\"\\n          f\\"{quality[\'incomplete_rows_excluded\']:,} excluded, \\"\\n          f\\"{quality[\'shared_pool_keys\']:,} multiply-sourced keys, \\"\\n          f\\"max mass spread {quality[\'max_source_mass_spread_da\']:.5f} Da\\")\\n    g = both.groupby(\\"pool_key\\", sort=False)\\n    pool = pd.DataFrame({\\n        \\"smiles\\": g.smiles.first(),                       # train SMILES win: they are the labels\' form\\n        \\"mass\\": g.mass.first(),\\n        \\"sources\\": g.sources.agg(np.bitwise_or.reduce),\\n        \\"n_spectra\\": g.n_spectra.sum(), \\"libs\\": g.libs.agg(np.bitwise_or.reduce),\\n        \\"np_likeness\\": g.np_likeness.first(), \\"np_pathway\\": g.np_pathway.first(),\\n        \\"np_superclass\\": g.np_superclass.first(), \\"has_sugar\\": g.has_sugar.first(),\\n    }).reset_index()\\n    pool = pool.dropna(subset=[\\"mass\\"]).sort_values(\\"mass\\", kind=\\"stable\\").reset_index(drop=True)\\n    if os.environ.get(\\"CASMI_QUARANTINE_INVALID_STRUCTURES\\") == \\"1\\":\\n        from casmi.pool.structure_quality import quarantine_invalid_candidates\\n        pool, structure_quality = quarantine_invalid_candidates(\\n            pool, workers=max(1, min(3, (os.cpu_count() or 2) - 1)),\\n        )\\n        quality[\\"structure_quarantine\\"] = structure_quality\\n    with quality_path.open(\\"x\\", encoding=\\"utf-8\\") as stream:\\n        json.dump(quality, stream, indent=2, allow_nan=False)\\n        stream.write(\\"\\\\n\\")\\n    # Historical pools are immutable. Complete, verify and publish a\\n    # new Parquet before making the final output name visible.\\n    from pyarrow import parquet as pq\\n    with fresh_pool_file(output) as staged:\\n        pool.to_parquet(staged, index=False)\\n        metadata = pq.ParquetFile(staged).metadata\\n        if metadata.num_rows != len(pool):\\n            raise ValueError(\\"Persisted candidate pool row count differs from build\\")\\n\\n    in_train = (pool.sources & SOURCE_BITS[\\"train\\"]) > 0\\n    in_coco = (pool.sources & SOURCE_BITS[\\"coconut\\"]) > 0\\n    print(f\\"pool: {len(pool):,} structures | train {in_train.sum():,} | COCONUT {in_coco.sum():,} | \\"\\n          f\\"both {(in_train & in_coco).sum():,} | COCONUT-only {(in_coco & ~in_train).sum():,}\\")\\n    np_ex = (pool.libs & (1 << 9)) > 0\\n    print(f\\"enveda-np-examples structures covered by COCONUT: {(np_ex & in_coco).sum()}/{np_ex.sum()}\\")\\n    if PUBCHEMLITE:\\n        in_pcl = (pool.sources & SOURCE_BITS[\\"pubchem\\"]) > 0\\n        print(f\\"PubChemLite adds {(in_pcl & ~in_train & ~in_coco).sum():,} structures found in neither train nor COCONUT\\")\\n    print(f\\"built in {time.time() - t0:.0f}s\\")\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "scripts/build_fingerprints.py": "\\"\\"\\"Morgan fingerprints (radius 2, 2048 bits) for every pool structure, packed and row-aligned\\nwith artifacts/pool.parquet. Used for analog propagation (Tanimoto candidate \\u2194 analog) and later\\nas the target of the spectrum \\u2192 fingerprint model.\\n\\nOutput: artifacts/pool_fp2048.npy  (uint8, shape [n_pool, 256]), zero rows where RDKit fails\\n\\"\\"\\"\\nimport json\\nimport os\\nimport sys\\nimport time\\nfrom concurrent.futures import ProcessPoolExecutor\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \\"src\\"))\\nfrom casmi.config import ARTIFACT_DIR, POOL_FILE, POOL_FP_FILE, POOL_FP_KEYS_FILE, POOL_TAG  # noqa: E402\\nfrom casmi.pool.fingerprint_provenance import make_fingerprint_provenance  # noqa: E402\\n\\nNBITS = 2048\\n\\n\\ndef _fp_chunk(job):\\n    \\"\\"\\"Produce packed Morgan bits, or fail on the first invalid candidate.\\n\\n    Do not silently turn a malformed/empty SMILES into an all-zero vector.\\n    Such rows otherwise become apparently valid signed research features.\\n    Report only the row number: raw licensed molecule strings stay private.\\n    \\"\\"\\"\\n    start, smiles_list = job\\n    from rdkit import Chem, DataStructs, RDLogger\\n    from rdkit.Chem import rdFingerprintGenerator\\n    RDLogger.DisableLog(\\"rdApp.*\\")\\n    gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=NBITS)\\n    out = np.zeros((len(smiles_list), NBITS // 8), np.uint8)\\n    arr = np.zeros(NBITS, np.uint8)\\n    for i, s in enumerate(smiles_list):\\n        try:\\n            mol = Chem.MolFromSmiles(s) if isinstance(s, str) else None\\n            if mol is None or mol.GetNumHeavyAtoms() == 0:\\n                raise ValueError(\\"Invalid candidate SMILES\\")\\n            arr.fill(0)\\n            DataStructs.ConvertToNumpyArray(gen.GetFingerprint(mol), arr)\\n            if not np.any(arr):\\n                raise ValueError(\\"Zero-vector Morgan fingerprint\\")\\n        except (ValueError, TypeError, RuntimeError) as exc:\\n            raise ValueError(\\n                f\\"Cannot create a valid Morgan fingerprint for pool row {start + i}\\"\\n            ) from None\\n        out[i] = np.packbits(arr)\\n    return out\\n\\n\\ndef main():\\n    t0 = time.time()\\n    # Frozen scientific artifacts must never be silently replaced when\\n    # COCONUT normalization or the source pool has changed.\\n    manifest_path = ARTIFACT_DIR / f\\"pool_fp_provenance{POOL_TAG}.json\\"\\n    outputs = [ARTIFACT_DIR / POOL_FP_FILE, ARTIFACT_DIR / POOL_FP_KEYS_FILE,\\n               manifest_path]\\n    existing = [str(path.name) for path in outputs if os.path.lexists(path)]\\n    if existing:\\n        raise FileExistsError(\\n            \\"Refusing to replace existing fingerprints: \\" + \\", \\".join(existing)\\n            + \\"; use a new CASMI_POOL_TAG for an independent research variant\\"\\n        )\\n    # Use the actual inference pool constructor rather than maintaining a\\n    # second sorting/deduplication implementation that can drift.\\n    from casmi.pool.pool import CandidatePool\\n    source = pd.read_parquet(\\n        ARTIFACT_DIR / POOL_FILE, columns=[\\"pool_key\\", \\"smiles\\", \\"mass\\"]\\n    )\\n    pool = CandidatePool(source).table\\n    if pool.empty:\\n        raise ValueError(\\"Candidate pool is empty; cannot generate fingerprints\\")\\n    smiles = pool.smiles.tolist()\\n    chunks = [(i, smiles[i:i + 5000]) for i in range(0, len(smiles), 5000)]\\n    with ProcessPoolExecutor(max_workers=max(1, (os.cpu_count() or 2) - 1)) as ex:\\n        fps = np.concatenate(list(ex.map(_fp_chunk, chunks)))\\n    if not np.all(fps.any(axis=1)):\\n        raise ValueError(\\"Zero-vector candidate fingerprints are prohibited\\")\\n    provenance = make_fingerprint_provenance(pool, pool.pool_key.to_numpy(), fps)\\n    # Exclusive output files close the race between the initial existence\\n    # check and publication, protecting frozen historical arrays.\\n    with (ARTIFACT_DIR / POOL_FP_FILE).open(\\"xb\\") as output:\\n        np.save(output, fps)\\n    with (ARTIFACT_DIR / POOL_FP_KEYS_FILE).open(\\"xb\\") as output:\\n        pool[[\\"pool_key\\"]].to_parquet(output, index=False)\\n    with manifest_path.open(\\"x\\", encoding=\\"utf-8\\") as output:\\n        json.dump(provenance, output, indent=2)\\n        output.write(\\"\\\\n\\")\\n    empty = (fps.sum(axis=1) == 0).sum()\\n    print(f\\"{len(fps):,} fingerprints ({fps.nbytes / 1e6:.0f} MB), {empty:,} empty, built in {time.time() - t0:.0f}s\\")\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "scripts/run_external_audit.py": "\\"\\"\\"Audit pinned external CC0 structures and stored spectra against full CASMI train.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\nimport sys\\nfrom datetime import UTC, datetime\\n\\nROOT = Path(__file__).resolve().parents[1]\\nsys.path[:0] = [str(ROOT), str(ROOT / \\"src\\")]\\n\\n\\ndef canonical_digest(value):\\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(\\",\\", \\":\\")).encode()).hexdigest()\\n\\n\\ndef audit(train_path, external_path, manifest_path, protocol_path):\\n    import pandas as pd\\n    import pyarrow as pa\\n    import pyarrow.compute as pc\\n    import pyarrow.parquet as pq\\n    from casmi.chem.canon import heavy_atom_key, skeleton_key, canonical_ik14\\n    from casmi.validate.overlap_audit import PublicOverlapAudit\\n    from casmi.validate.source_manifest import sha256_file\\n\\n    protocol = json.loads(Path(protocol_path).read_text(encoding=\\"utf-8\\"))\\n    source = json.loads(Path(manifest_path).read_text(encoding=\\"utf-8\\"))\\n    external_path, train_path = Path(external_path), Path(train_path)\\n    if source[\\"source\\"] != protocol[\\"external_source\\"] or sha256_file(external_path) != source[\\"parquet_sha256\\"]:\\n        raise ValueError(\\"External source manifest differs from declared inputs\\")\\n    external = pd.read_parquet(external_path)\\n    if set(external.license) != {\\"CC0\\"} or external.accession.duplicated().any():\\n        raise ValueError(\\"External license or accession contract failed\\")\\n    targets = external.rename(columns={\\"accession\\": \\"spectrum_id\\", \\"metric_key\\": \\"molecule_id\\"}).to_dict(\\"records\\")\\n    content = PublicOverlapAudit(targets)\\n    # Metric InChIKey14 discards isotopic layers. The legacy connectivity helper\\n    # retains isotope labels, so it is not a safe necessary-condition filter here.\\n    # Heavy-element composition is deliberately broader and isotope/H agnostic.\\n    target_compositions = {heavy_atom_key(smiles) for smiles in external.smiles}\\n    isotope_safe_connectivity = bool(protocol.get(\\"identity_prefilter\\"))\\n    target_skeletons = {skeleton_key(smiles) for smiles in external.smiles} if isotope_safe_connectivity else set()\\n    target_metrics = set(external.metric_key)\\n    training_standard, overlapping_metrics, seen_smiles = set(), set(), set()\\n    invalid_structures = matching_compositions = 0\\n    train = pq.ParquetFile(train_path)\\n    precursors = pa.array(sorted(content.precursors), type=pa.float64())\\n    columns = [\\"normalized_smiles\\", \\"inchikey14\\", \\"precursor_mz\\", \\"ms2_mzs\\", \\"ms2_normalized_intensities\\", \\"adduct\\", \\"ionization_mode\\", \\"ingest_lib\\"]\\n    for index, batch in enumerate(train.iter_batches(batch_size=8192, columns=columns)):\\n        metadata = batch.select([\\"normalized_smiles\\", \\"inchikey14\\"]).to_pylist()\\n        for row in metadata:\\n            key, smiles = row[\\"inchikey14\\"], row[\\"normalized_smiles\\"]\\n            if not isinstance(key, str) or not key:\\n                raise ValueError(\\"Training identity is missing\\")\\n            training_standard.add(key)\\n            if smiles in seen_smiles:\\n                continue\\n            seen_smiles.add(smiles)\\n            composition = heavy_atom_key(smiles)\\n            if composition is None:\\n                invalid_structures += 1\\n            elif composition in target_compositions:\\n                matching_compositions += 1\\n                if isotope_safe_connectivity and skeleton_key(smiles) not in target_skeletons:\\n                    continue\\n                metric = canonical_ik14(smiles)\\n                if metric is None:\\n                    raise ValueError(\\"Matching training composition has unverifiable metric identity\\")\\n                if metric in target_metrics:\\n                    overlapping_metrics.add(metric)\\n        mask = pc.is_in(pc.cast(batch.column(\\"precursor_mz\\"), pa.float64()), value_set=precursors)\\n        content.scan_batch(batch.filter(mask).to_pylist(), scanned_count=batch.num_rows)\\n        if index % 25 == 0:\\n            print(f\\"External audit scanned {content.scanned}/{train.metadata.num_rows} training rows\\", flush=True)\\n    content_summary = content.finish(expected_train_rows=train.metadata.num_rows)\\n    # All spectra belonging to a matched external molecule are excluded together.\\n    content_metrics = {targets[index][\\"molecule_id\\"] for index in content.matched}\\n    standard_metrics = set(external.loc[external.standard_ik14.isin(training_standard), \\"metric_key\\"])\\n    rejected = overlapping_metrics | content_metrics | standard_metrics\\n    novel = external[~external.metric_key.isin(rejected)].copy()\\n    novel_metrics = sorted(set(novel.metric_key))\\n    verified = invalid_structures == 0\\n    enough = len(novel_metrics) >= 30\\n    report = {\\n        \\"schema_version\\": 1,\\n        \\"status\\": \\"EXTERNAL_INDEPENDENCE_FEASIBLE\\" if verified and enough else \\"EXTERNAL_INDEPENDENCE_NOT_ESTABLISHED\\",\\n        \\"checked_at\\": datetime.now(UTC).isoformat(),\\n        \\"protocol_sha256\\": sha256_file(Path(protocol_path)),\\n        \\"inputs\\": {\\"train.parquet\\": {\\"sha256\\": sha256_file(train_path), \\"bytes\\": train_path.stat().st_size},\\n                   external_path.name: {\\"sha256\\": sha256_file(external_path), \\"bytes\\": external_path.stat().st_size}},\\n        \\"external_source\\": source,\\n        \\"identity_audit\\": {\\"training_rows_scanned\\": content.scanned, \\"training_standard_identities\\": len(training_standard),\\n                           \\"distinct_training_smiles\\": len(seen_smiles), \\"unparseable_training_labels\\": invalid_structures,\\n                           \\"training_smiles_sharing_external_composition\\": matching_compositions,\\n                           \\"external_standard_overlap_molecules\\": len(standard_metrics),\\n                           \\"external_metric_overlap_molecules\\": len(overlapping_metrics),\\n                           \\"external_exact_content_overlap_molecules\\": len(content_metrics),\\n                           \\"excluded_external_molecules_union\\": len(rejected),\\n                           \\"novel_external_molecules\\": len(novel_metrics), \\"novel_external_spectra\\": len(novel),\\n                           \\"novel_metric_identities_sha256\\": canonical_digest(novel_metrics),\\n                           \\"novel_accessions_sha256\\": canonical_digest(sorted(novel.accession))},\\n        \\"stored_content_audit\\": content_summary,\\n        \\"model_performance_inspected\\": False,\\n        \\"limitations\\": [\\"External source acquisition is independently retrieved public reference data, not a new laboratory replication.\\",\\n                        \\"Identity checks rely on source chemical annotations and the official computational canonicalizer.\\",\\n                        \\"Exact stored-content comparison cannot detect arbitrary preprocessing/normalization variants of a common acquisition.\\",\\n                        \\"This feasibility stage does not establish supervised-family separation, model accuracy or a C3 coverage improvement.\\"]}\\n    return report, novel\\n\\n\\nif __name__ == \\"__main__\\":\\n    import argparse\\n    parser = argparse.ArgumentParser()\\n    parser.add_argument(\\"--train\\", required=True, type=Path)\\n    parser.add_argument(\\"--external\\", required=True, type=Path)\\n    parser.add_argument(\\"--manifest\\", required=True, type=Path)\\n    parser.add_argument(\\"--protocol\\", required=True, type=Path)\\n    parser.add_argument(\\"--output\\", required=True, type=Path)\\n    args = parser.parse_args()\\n    report, _ = audit(args.train, args.external, args.manifest, args.protocol)\\n    with args.output.open(\\"x\\", encoding=\\"utf-8\\") as destination:\\n        json.dump(report, destination, indent=2, allow_nan=False)\\n        destination.write(\\"\\\\n\\")\\n    print(\\"External metadata/content feasibility audit completed; no model outcomes inspected\\", flush=True)\\n", "scripts/run_advanced_study.py": "\\"\\"\\"Execute the predeclared internal family-disjoint study; export aggregates only.\\n\\nAll supervised weights are fitted from scratch. No archived model is loaded.\\nThe test phase begins only after every ranker and its feature list is sealed.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport importlib.metadata\\nimport json\\nimport os\\nimport sys\\nimport time\\nfrom dataclasses import asdict\\nfrom datetime import UTC, datetime\\nfrom pathlib import Path\\n\\nROOT = Path(__file__).resolve().parents[1]\\nsys.path[:0] = [str(ROOT), str(ROOT / \\"src\\")]\\n\\n\\ndef ablation_columns(columns, name):\\n    \\"\\"\\"Remove a channel and its derived context together; retain fixed candidates.\\"\\"\\"\\n    membership = (\\"prior_\\",)\\n    if name == \\"membership_only\\":\\n        selected = [c for c in columns if c.startswith(membership)]\\n    else:\\n        prefixes = {\\"full\\": (), \\"no_d1\\": (\\"d1_\\",), \\"no_d2\\": (\\"an_\\",),\\n                    \\"no_d3\\": (\\"d3_\\",), \\"no_d4\\": (\\"d4_\\", \\"d4x_\\"),\\n                    \\"no_gen_features\\": (\\"gen\\",), \\"no_np_score\\": (\\"np_score\\",)}\\n        if name not in prefixes:\\n            raise ValueError(\\"Unknown prespecified ablation\\")\\n        selected = [c for c in columns if not c.startswith(membership + prefixes[name])]\\n    if not selected:\\n        raise ValueError(\\"Ablation has no usable features\\")\\n    return selected\\n\\n\\nclass SearchMemo:\\n    \\"\\"\\"Per-query memoization keyed by full cleaned evidence and reference exclusions.\\"\\"\\"\\n    def __init__(self, index):\\n        self.index, self.cache = index, {}\\n        self.hits = self.misses = 0\\n\\n    def reset(self):\\n        self.cache.clear()\\n\\n    def search(self, spectra, mass, excluded_rows=None):\\n        import numpy as np\\n        key = hashlib.sha256()\\n        key.update(float(mass).hex().encode())\\n        for s in spectra:\\n            key.update(str((s[\\"adduct\\"], s[\\"mode\\"], float(s[\\"precursor_mz\\"]).hex())).encode())\\n            for field in (\\"mz_c\\", \\"it_c\\"):\\n                data = np.asarray(s[field], dtype=np.float64)\\n                key.update(len(data).to_bytes(8, \\"little\\"))\\n                key.update(data.tobytes())\\n        key.update(np.unique(np.asarray(excluded_rows if excluded_rows is not None else [], dtype=np.int64)).tobytes())\\n        token = key.hexdigest()\\n        if token not in self.cache:\\n            self.cache[token] = self.index.search(spectra, mass, excluded_rows)\\n            self.misses += 1\\n        else:\\n            self.hits += 1\\n        return dict(self.cache[token])\\n\\n\\ndef main():\\n    import numpy as np\\n    import pandas as pd\\n    import rdkit\\n    import torch\\n    from rdkit import Chem, DataStructs\\n    from rdkit.Chem import rdFingerprintGenerator\\n\\n    from casmi.config import ARTIFACT_DIR, POOL_FILE, POOL_FP_FILE, POOL_FP_KEYS_FILE, POOL_TAG\\n    from casmi.channels.library import LibraryIndex\\n    from casmi.channels.analog import AnalogIndex\\n    from casmi.channels.fpmodel import FPModel, FPModelConfig, save\\n    from casmi.channels.fpinfer import FP_CLEAN, MAX_PEAKS, ADDUCT_ID\\n    from casmi.chem.keys import MetricKeys\\n    from casmi.chem.canon import canonical_ik14\\n    from casmi.spectra.clean import clean_peaks_fast\\n    from casmi.spectra.store import SpectrumStore\\n    from casmi.pool.pool import CandidatePool\\n    from casmi.pool.fingerprint_provenance import verify_fingerprint_provenance\\n    from casmi.engine import BaselineEngine, EngineConfig, molecule_mass\\n    from casmi.predict import Predictor\\n    from casmi.rank.features import feature_columns\\n    from casmi.rank.ranker import Ranker, RankerConfig\\n    from casmi.validate.knockout import Query, CLASSES, knockout_for, include_equivalent_source_rows, equivalent_keys\\n    from casmi.validate.metric import rank_of_truth, summarize\\n    from casmi.validate.paired_bootstrap import paired_difference\\n    from casmi.validate.scaffold_study import family_key, partition, digest, joint_intervals\\n    from casmi.validate.spectral_controls import perturb_spectra\\n    from casmi.validate.run_alignment import query_signature\\n    from casmi.validate.source_manifest import create_manifest, required_validation_inputs, verify_manifest, sha256_file\\n\\n    started = time.time()\\n    protocol_path = ROOT / \\"protocols/advanced-internal-v1.json\\"\\n    protocol = json.loads(protocol_path.read_text())\\n    if rdkit.__version__ != \\"2026.03.3\\":\\n        raise RuntimeError(\\"Official metric RDKit is required\\")\\n    seed = protocol[\\"seed\\"]\\n    np.random.seed(seed)\\n    torch.manual_seed(seed)\\n    torch.set_num_threads(2)\\n    torch.use_deterministic_algorithms(True)\\n    device = \\"cuda\\" if torch.cuda.is_available() else \\"cpu\\"\\n    store = SpectrumStore(ARTIFACT_DIR / \\"store\\")\\n    meta = store.meta\\n    key_of_row = meta.train_ik14.astype(str).to_numpy()\\n    pool = CandidatePool(pd.read_parquet(ARTIFACT_DIR / POOL_FILE))\\n    truth_pool = CandidatePool(pd.read_parquet(ARTIFACT_DIR / \\"structures.parquet\\"))\\n    labels = dict(zip(truth_pool.key, truth_pool.smiles))\\n    source_quality = json.loads((ARTIFACT_DIR / f\\"pool_source_quality{POOL_TAG}.json\\").read_text())\\n    static_names = list(required_validation_inputs(pool_tag=POOL_TAG, analog=True))\\n    static_names += [\\"structures.parquet\\", f\\"pool_source_quality{POOL_TAG}.json\\"]\\n    manifest = create_manifest(ARTIFACT_DIR, static_names)\\n    fps = np.load(ARTIFACT_DIR / POOL_FP_FILE, mmap_mode=\\"r\\", allow_pickle=False)\\n    fp_keys = pd.read_parquet(ARTIFACT_DIR / POOL_FP_KEYS_FILE).pool_key.to_numpy()\\n    fp_status = verify_fingerprint_provenance(pool.table, fp_keys, fps,\\n        json.loads((ARTIFACT_DIR / f\\"pool_fp_provenance{POOL_TAG}.json\\").read_text()))\\n    cfg = EngineConfig(generate=True, gen_consensus=True, np_score=True)\\n    library = LibraryIndex(store, key_of_row, cfg.library)\\n    raw_analog = AnalogIndex(store, key_of_row, cfg.analog, cache=None)\\n    analog_status = raw_analog.fresh_index_provenance()\\n    memo = SearchMemo(raw_analog)\\n    keys = MetricKeys(ARTIFACT_DIR / \\"advanced_metric_keys.parquet\\")\\n    print(\\"Prepared fresh artifact inventory and D2 index\\", flush=True)\\n\\n    # Population construction is label/metadata-only and cannot consult scores.\\n    eligible = (meta.ingest_lib.isin(protocol[\\"query_libraries\\"]).to_numpy()\\n                & meta.test_adduct.to_numpy() & (meta.n_peaks.to_numpy() >= 5)\\n                & (np.abs(meta.ppm.to_numpy()) <= 20))\\n    rows = np.flatnonzero(eligible)\\n    choices = pd.DataFrame({\\"row\\": rows, \\"key\\": key_of_row[rows],\\n                            \\"library\\": meta.ingest_lib.astype(str).to_numpy()[rows]})\\n    mass_of = dict(zip(truth_pool.key, truth_pool.mass))\\n    choices = choices[choices.key.map(mass_of).between(*protocol[\\"query_mass_range\\"])]\\n    counts = choices.groupby([\\"key\\", \\"library\\"]).size().reset_index(name=\\"count\\")\\n    best = counts.sort_values([\\"count\\", \\"key\\", \\"library\\"], ascending=[False, True, True]).drop_duplicates(\\"key\\")\\n    groups = {key: group.row.to_numpy() for key, group in\\n              choices.merge(best[[\\"key\\", \\"library\\"]], on=[\\"key\\", \\"library\\"]).groupby(\\"key\\")}\\n    excluded_families = {family_key(labels[key]) for key in pd.unique(key_of_row[\\n        (meta.ingest_lib == \\"enveda-np-examples\\").to_numpy()]) if key in labels}\\n    representatives = {}\\n    invalid_families = blocked_families = 0\\n    for row in best.itertuples():\\n        family = family_key(labels[row.key])\\n        if family is None:\\n            invalid_families += 1\\n        elif family in excluded_families:\\n            blocked_families += 1\\n        elif family not in representatives or row.key < representatives[family][\\"key\\"]:\\n            representatives[family] = {\\"key\\": row.key, \\"family\\": family, \\"library\\": row.library}\\n    records = []\\n    # Only family representatives need expensive tautomer canonicalization.\\n    for value in representatives.values():\\n        metric = canonical_ik14(labels[value[\\"key\\"]])\\n        if metric:\\n            records.append({**value, \\"metric\\": metric})\\n    splits = partition(records, protocol[\\"split_counts\\"], protocol[\\"split_salt\\"])\\n    split_audit = {name: {\\"n_molecules\\": len(values), \\"identity_sha256\\": digest([r[\\"key\\"] for r in values]),\\n                          \\"family_sha256\\": digest([r[\\"group\\"] for r in values]),\\n                          \\"metric_sha256\\": digest([r[\\"metric\\"] for r in values]),\\n                          \\"libraries\\": pd.Series([r[\\"library\\"] for r in values]).value_counts().to_dict()}\\n                   for name, values in splits.items()}\\n    split_audit[\\"cross_split_identity_overlaps\\"] = 0\\n    split_audit[\\"invalid_family_labels_excluded\\"] = invalid_families\\n    split_audit[\\"historical_vin_family_rows_excluded\\"] = blocked_families\\n    split_audit[\\"eligible_representative_families\\"] = len(records)\\n    print(\\"Prepared declared 600/150/250 family-disjoint population\\", flush=True)\\n\\n    # Both D3 models train on selected training identities only, on opposite folds.\\n    train_lookup = {row[\\"key\\"]: row for row in splits[\\"train\\"]}\\n    fp_eligible = (meta.test_adduct.to_numpy() & (meta.n_peaks.to_numpy() >= 5)\\n                   & (np.abs(meta.ppm.to_numpy()) <= 20))\\n    training_rows = {}\\n    for index in np.flatnonzero(fp_eligible & np.isin(key_of_row, list(train_lookup))):\\n        training_rows.setdefault(key_of_row[index], []).append(int(index))\\n    target_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)\\n    fp_models, fp_lineage = {}, []\\n    settings = protocol[\\"fp_model\\"]\\n    (ARTIFACT_DIR / \\"models\\").mkdir(exist_ok=True)\\n    for fold in (0, 1):\\n        rng = np.random.default_rng(seed + fold)\\n        selected_rows, selected_keys = [], []\\n        for key, item in sorted(train_lookup.items()):\\n            if item[\\"fold\\"] != fold:\\n                continue\\n            available = np.asarray(training_rows.get(key, []), dtype=np.int64)\\n            cap = protocol[\\"max_training_spectra_per_structure\\"]\\n            if len(available) > cap:\\n                available = np.sort(rng.choice(available, cap, replace=False))\\n            selected_rows.extend(available.tolist())\\n            selected_keys.extend([key] * len(available))\\n        if len(selected_rows) < 100:\\n            raise ValueError(\\"Insufficient fingerprint training spectra in declared fold\\")\\n        if not set(selected_keys) <= {r[\\"key\\"] for r in splits[\\"train\\"]}:\\n            raise ValueError(\\"Fingerprint training crossed the declared training partition\\")\\n        n = len(selected_rows)\\n        mz = np.zeros((n, MAX_PEAKS), np.float32)\\n        it = np.zeros_like(mz)\\n        npk = np.zeros(n, np.int64)\\n        unique_keys = sorted(set(selected_keys))\\n        targets = {}\\n        for key in unique_keys:\\n            mol = Chem.MolFromSmiles(labels[key])\\n            bits = np.zeros(2048, np.uint8)\\n            DataStructs.ConvertToNumpyArray(target_gen.GetFingerprint(mol), bits)\\n            if not bits.any():\\n                raise ValueError(\\"Invalid fingerprint training target\\")\\n            targets[key] = bits\\n        for j, index in enumerate(selected_rows):\\n            cm, ci = clean_peaks_fast(*store.peaks(index), float(meta.precursor_mz.iat[index]), FP_CLEAN)\\n            mz[j, :len(cm)], it[j, :len(ci)], npk[j] = cm, ci, len(cm)\\n        side = meta.iloc[selected_rows]\\n        inputs = {\\"mz\\": mz, \\"it\\": it, \\"npk\\": npk, \\"prec\\": side.precursor_mz.to_numpy(np.float32),\\n                  \\"adduct\\": side.adduct.map(ADDUCT_ID).to_numpy(np.int64),\\n                  \\"ce\\": side.ce_ev.to_numpy(np.float32),\\n                  \\"positive\\": (side.ionization_mode == \\"positive\\").to_numpy(np.float32)}\\n        y = np.asarray([targets[key] for key in selected_keys], dtype=np.float32)\\n        torch.manual_seed(seed + fold)\\n        model = FPModel(FPModelConfig(d_model=settings[\\"d_model\\"], n_layers=settings[\\"n_layers\\"],\\n                                      d_ff=settings[\\"d_ff\\"])).to(device)\\n        optimizer = torch.optim.AdamW(model.parameters(), lr=settings[\\"lr\\"], weight_decay=.01)\\n        losses = []\\n        model.train()\\n        for epoch in range(settings[\\"epochs\\"]):\\n            total = 0.0\\n            order = rng.permutation(n)\\n            for start in range(0, n, settings[\\"batch_size\\"]):\\n                ids = order[start:start + settings[\\"batch_size\\"]]\\n                x = {name: torch.as_tensor(np.ascontiguousarray(value[ids]), device=device,\\n                     dtype=torch.long if name in (\\"npk\\", \\"adduct\\") else torch.float32) for name, value in inputs.items()}\\n                target = torch.as_tensor(y[ids], device=device)\\n                loss = torch.nn.functional.binary_cross_entropy_with_logits(model(**x), target)\\n                if not torch.isfinite(loss):\\n                    raise ValueError(\\"Nonfinite fingerprint training loss\\")\\n                optimizer.zero_grad(set_to_none=True)\\n                loss.backward()\\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\\n                optimizer.step()\\n                total += float(loss.detach().cpu()) * len(ids)\\n            losses.append(total / n)\\n            print(f\\"Trained fresh fingerprint fold {fold}, epoch {epoch + 1}/{settings[\'epochs\']}\\", flush=True)\\n        model.eval()\\n        model_path = ARTIFACT_DIR / \\"models\\" / f\\"advanced_fp_fold{fold}.pt\\"\\n        lineage = {\\"fold\\": fold, \\"n_spectra\\": n, \\"n_structures\\": len(unique_keys),\\n                   \\"training_rows_sha256\\": digest(selected_rows), \\"training_identities_sha256\\": digest(unique_keys),\\n                   \\"training_families_sha256\\": digest(sorted({train_lookup[k][\\"group\\"] for k in unique_keys})),\\n                   \\"dev_test_identities_used\\": 0, \\"epochs_fixed_before_execution\\": settings[\\"epochs\\"],\\n                   \\"training_bce_by_epoch\\": losses, \\"initialization\\": \\"fresh seeded random; no archived weights\\"}\\n        save(model, model_path, {\\"lineage\\": lineage, \\"protocol_sha256\\": sha256_file(protocol_path)})\\n        fp_lineage.append({**lineage, \\"model_sha256\\": sha256_file(model_path)})\\n        fp_models[fold] = model\\n        del mz, it, npk, inputs, y\\n\\n    predictor = Predictor(BaselineEngine(pool, library, cfg, memo, fps), pool, keys,\\n                          fp_models=list(fp_models.values()), device=device)\\n    # Trigger the actual inference D4 implementation without fitting any archived model.\\n    predictor.ranker = type(\\"FeatureDeclaration\\", (), {\\"columns\\": [\\"d4_int\\"]})()\\n    source_rows = {name: np.flatnonzero((meta.ingest_lib == name).to_numpy()) for name in protocol[\\"query_libraries\\"]}\\n    private_query_hashes = {name: [] for name in splits}\\n    truth_by_qid, query_context = {}, {}\\n\\n    def prepare_query(item):\\n        available = groups[item[\\"key\\"]]\\n        rng = np.random.default_rng(int(hashlib.sha256(item[\\"key\\"].encode()).hexdigest()[:16], 16))\\n        chosen = np.sort(rng.choice(available, protocol[\\"max_spectra\\"], replace=False)) if len(available) > protocol[\\"max_spectra\\"] else available\\n        spectra = []\\n        for index in chosen:\\n            mz, intensity = store.peaks(index)\\n            row = meta.iloc[index]\\n            spectra.append({\\"mz\\": mz, \\"it\\": intensity, \\"precursor_mz\\": float(row.precursor_mz),\\n                            \\"adduct\\": str(row.adduct), \\"ce_ev\\": float(row.ce_ev), \\"mode\\": str(row.ionization_mode)})\\n        query = Query(f\\"{item[\'library\']}:{item[\'key\']}\\", item[\\"key\\"], spectra, chosen)\\n        truth, same = equivalent_keys(item[\\"key\\"], molecule_mass(spectra), truth_pool, keys,\\n                                       truth_smiles=labels[item[\\"key\\"]])\\n        candidate_truth, candidates = equivalent_keys(item[\\"key\\"], molecule_mass(spectra), pool, keys,\\n                                                       truth_smiles=labels[item[\\"key\\"]])\\n        if truth != item[\\"metric\\"] or candidate_truth != truth:\\n            raise ValueError(\\"Split labels and inference canonical truth disagree\\")\\n        same.update(candidates)\\n        same.add(item[\\"key\\"])\\n        include_equivalent_source_rows(query, same, source_rows[item[\\"library\\"]], key_of_row)\\n        truth_by_qid[query.qid] = truth\\n        m = molecule_mass(spectra)\\n        unique = lambda field: next(iter(values)) if len(values := {s[field] for s in spectra}) == 1 else \\"mixed\\"\\n        query_context[query.qid] = {\\"library\\": item[\\"library\\"], \\"adduct\\": unique(\\"adduct\\"), \\"mode\\": unique(\\"mode\\"),\\n                                  \\"mass\\": \\"150-300\\" if m < 300 else \\"300-600\\" if m < 600 else \\"600-1170\\",\\n                                  \\"spectra\\": \\"1\\" if len(spectra) == 1 else \\"2-3\\" if len(spectra) < 4 else \\"4-5\\",\\n                                  \\"instrument\\": \\"timsTOF\\" if meta.is_timstof.iloc[chosen].all() else \\"other_or_mixed\\"}\\n        return query, same\\n\\n    def frames(query, same, mode):\\n        spectra = perturb_spectra(query.spectra, mode=mode, seed=seed, query_id=query.qid)\\n        for cls in CLASSES:\\n            knockout = knockout_for(query, cls, library.rows_of, same)\\n            frame = predictor.features(spectra, excluded_rows=knockout.excluded_rows,\\n                                       excluded_keys=knockout.excluded_keys,\\n                                       knocked_keys=same if cls != \\"C1\\" else frozenset())\\n            if not frame.empty:\\n                frame = frame.assign(qid=query.qid, cls=cls)\\n                frame[\\"label\\"] = [int(keys.get(key, pool.smiles_of(key)) == truth_by_qid[query.qid]) for key in frame.pool_key]\\n            yield cls, frame\\n\\n    tables = {}\\n    for split in (\\"train\\", \\"dev\\"):\\n        blocks = []\\n        for index, item in enumerate(splits[split]):\\n            memo.reset()\\n            query, same = prepare_query(item)\\n            private_query_hashes[split].append(query_signature(query))\\n            # No training spectrum may be scored by its own fitted D3 fold.\\n            predictor.fp_models = [fp_models[1 - item[\\"fold\\"]]] if split == \\"train\\" else list(fp_models.values())\\n            blocks.extend(frame for _, frame in frames(query, same, \\"none\\") if not frame.empty)\\n            if (index + 1) % 25 == 0:\\n                print(f\\"Prepared {split} features {index + 1}/{len(splits[split])}\\", flush=True)\\n        tables[split] = pd.concat(blocks, ignore_index=True)\\n    columns = [name for name in feature_columns(tables[\\"train\\"]) if name in tables[\\"dev\\"]]\\n    models, training_audit = {}, {}\\n    for name in protocol[\\"ablations\\"]:\\n        selected = ablation_columns(columns, name)\\n        base = RankerConfig()\\n        config = RankerConfig(seeds=tuple(protocol[\\"ranker\\"][\\"seeds\\"]),\\n            num_boost_round=protocol[\\"ranker\\"][\\"rounds\\"], early_stopping=protocol[\\"ranker\\"][\\"early_stopping\\"],\\n            params={**base.params, \\"num_threads\\": 2, \\"deterministic\\": True, \\"force_col_wise\\": True})\\n        model = Ranker(config)\\n        model.fit(tables[\\"train\\"], tables[\\"dev\\"], selected)\\n        model_hashes = []\\n        for index, fitted in enumerate(model.models):\\n            path = ARTIFACT_DIR / \\"models\\" / f\\"advanced_{name}_{index}.txt\\"\\n            fitted.save_model(str(path))\\n            model_hashes.append(sha256_file(path))\\n        models[name] = model\\n        training_audit[name] = {\\"columns\\": selected, \\"best_iterations\\": model.best_iters,\\n                                \\"model_sha256\\": model_hashes, \\"config\\": asdict(config),\\n                                \\"positive_training_groups\\": int(tables[\\"train\\"].groupby([\\"qid\\", \\"cls\\"]).label.max().sum()),\\n                                \\"zero_positive_groups_retained_in_test_denominator\\": True}\\n        print(f\\"Fitted prespecified ranker {name}\\", flush=True)\\n    model_files = {path.name: sha256_file(path) for path in sorted((ARTIFACT_DIR / \\"models\\").iterdir())}\\n    if len(model_files) != 2 + len(protocol[\\"ablations\\"]) * len(protocol[\\"ranker\\"][\\"seeds\\"]):\\n        raise ValueError(\\"Unexpected model artifact inventory; fresh training required\\")\\n    seal = {\\"status\\": \\"ALL_MODELS_SEALED_BEFORE_TEST_FEATURES\\", \\"sealed_at\\": datetime.now(UTC).isoformat(),\\n            \\"model_files\\": model_files,\\n            \\"models\\": training_audit, \\"fingerprint_training\\": fp_lineage,\\n            \\"query_input_sha256\\": {name: digest(values) for name, values in private_query_hashes.items() if name != \\"test\\"},\\n            \\"test_identities_sha256\\": split_audit[\\"test\\"][\\"identity_sha256\\"], \\"protocol_sha256\\": sha256_file(protocol_path)}\\n    seal_sha256 = digest(seal)\\n    (ARTIFACT_DIR / \\"training_seal.json\\").write_text(json.dumps(seal, indent=2, allow_nan=False))\\n    del tables\\n    print(\\"Sealed all learned models; starting final test phase\\", flush=True)\\n\\n    names = [\\"mass_only\\", \\"d1\\", \\"d1_d2\\"] + list(models)\\n    ranks = {name: {mode: {cls: [] for cls in CLASSES} for mode in protocol[\\"controls\\"]} for name in names}\\n    coverage = {cls: {\\"pool_truth_present\\": 0, \\"with_generation_truth_present\\": 0} for cls in CLASSES}\\n    test_ids = []\\n    predictor.fp_models = list(fp_models.values())\\n\\n    def ranked(frame, order, truth):\\n        guesses = keys.dedupe_ranked(frame.pool_key.to_numpy()[order].tolist()[:60], pool.smiles_of, 25)\\n        return int(rank_of_truth([metric for _, metric in guesses], truth))\\n\\n    for index, item in enumerate(splits[\\"test\\"]):\\n        memo.reset()\\n        query, same = prepare_query(item)\\n        test_ids.append(query.qid)\\n        private_query_hashes[\\"test\\"].append(query_signature(query))\\n        for mode in protocol[\\"controls\\"]:\\n            for cls, frame in frames(query, same, mode):\\n                if frame.empty:\\n                    for name in names:\\n                        ranks[name][mode][cls].append(0)\\n                    continue\\n                truth = truth_by_qid[query.qid]\\n                fixed = frame[frame.gen == 0].reset_index(drop=True)\\n                for name in (\\"mass_only\\", \\"d1\\", \\"d1_d2\\"):\\n                    if fixed.empty:\\n                        value = 0\\n                    elif name == \\"mass_only\\":\\n                        value = ranked(fixed, np.argsort(np.abs(fixed.ppm), kind=\\"stable\\"), truth)\\n                    else:\\n                        primary = fixed.d1_max.to_numpy()\\n                        secondary = fixed.d1_mean.to_numpy()\\n                        if name == \\"d1_d2\\":\\n                            primary = np.maximum(primary, fixed.an_p4.fillna(0).to_numpy()) if \\"an_p4\\" in fixed else primary\\n                            secondary = fixed.an_p2.fillna(0).to_numpy() if \\"an_p2\\" in fixed else secondary\\n                        value = ranked(fixed, np.lexsort((np.abs(fixed.ppm.to_numpy()), -secondary, -primary)), truth)\\n                    ranks[name][mode][cls].append(value)\\n                for name, model in models.items():\\n                    ranks[name][mode][cls].append(ranked(frame, np.argsort(-model.predict(frame), kind=\\"stable\\"), truth))\\n                if mode == \\"none\\":\\n                    coverage[cls][\\"pool_truth_present\\"] += int(fixed.label.max()) if len(fixed) else 0\\n                    coverage[cls][\\"with_generation_truth_present\\"] += int(frame.label.max())\\n        if (index + 1) % 10 == 0:\\n            print(f\\"Completed sealed test {index + 1}/{len(splits[\'test\'])} with controls\\", flush=True)\\n    if digest(json.loads((ARTIFACT_DIR / \\"training_seal.json\\").read_text())) != seal_sha256:\\n        raise ValueError(\\"Model seal changed during test\\")\\n    if {path.name: sha256_file(path) for path in sorted((ARTIFACT_DIR / \\"models\\").iterdir())} != model_files:\\n        raise ValueError(\\"Learned artifact bytes changed during test\\")\\n    verify_manifest(ARTIFACT_DIR, manifest)\\n    summaries = {name: {mode: {cls: asdict(summarize(values, n_boot=protocol[\\"bootstrap_draws\\"], seed=seed))\\n                               for cls, values in classes.items()} for mode, classes in conditions.items()}\\n                 for name, conditions in ranks.items()}\\n    rr = lambda values: np.where(np.asarray(values) > 0, 1.0 / np.maximum(np.asarray(values), 1), 0.0)\\n    effects = {}\\n    for name in protocol[\\"ablations\\"]:\\n        if name != \\"full\\":\\n            for cls in CLASSES:\\n                effects[f\\"{name}_minus_full:{cls}\\"] = rr(ranks[name][\\"none\\"][cls]) - rr(ranks[\\"full\\"][\\"none\\"][cls])\\n    for mode in protocol[\\"controls\\"][1:]:\\n        for cls in CLASSES:\\n            effects[f\\"full_{mode}_minus_none:{cls}\\"] = rr(ranks[\\"full\\"][mode][cls]) - rr(ranks[\\"full\\"][\\"none\\"][cls])\\n    primary = paired_difference(ranks[\\"d1_d2\\"][\\"none\\"][\\"C2\\"], ranks[\\"full\\"][\\"none\\"][\\"C2\\"],\\n                                n_boot=protocol[\\"bootstrap_draws\\"], seed=seed)\\n    slices = {}\\n    for dimension in (\\"library\\", \\"adduct\\", \\"mode\\", \\"mass\\", \\"spectra\\", \\"instrument\\"):\\n        slices[dimension] = {}\\n        for value in sorted({query_context[qid][dimension] for qid in test_ids}):\\n            ids = [i for i, qid in enumerate(test_ids) if query_context[qid][dimension] == value]\\n            if len(ids) < protocol[\\"slice_min_n\\"]:\\n                slices[dimension][value] = {\\"n\\": len(ids), \\"status\\": \\"SUPPRESSED_SMALL_CELL\\"}\\n            else:\\n                cells = {}\\n                for cls in CLASSES:\\n                    a = np.asarray(ranks[\\"d1_d2\\"][\\"none\\"][cls])[ids]\\n                    b = np.asarray(ranks[\\"full\\"][\\"none\\"][cls])[ids]\\n                    delta, low, high = paired_difference(a, b, n_boot=protocol[\\"bootstrap_draws\\"], seed=seed)\\n                    cells[cls] = {\\"baseline_mrr\\": float(rr(a).mean()), \\"full_mrr\\": float(rr(b).mean()),\\n                                  \\"delta_mrr\\": delta, \\"ci_low\\": low, \\"ci_high\\": high}\\n                slices[dimension][value] = {\\"n\\": len(ids), \\"status\\": \\"DESCRIPTIVE_ONLY\\", \\"classes\\": cells}\\n    result = {\\"schema_version\\": 1, \\"status\\": \\"COMPLETE_INTERNAL_FAMILY_HOLDOUT\\",\\n              \\"checked_at\\": datetime.now(UTC).isoformat(), \\"elapsed_seconds\\": round(time.time() - started, 1),\\n              \\"protocol\\": protocol, \\"protocol_sha256\\": sha256_file(protocol_path), \\"partitions\\": split_audit,\\n              \\"training_seal\\": seal, \\"training_seal_sha256\\": seal_sha256,\\n              \\"source_manifest\\": manifest, \\"candidate_source_quality\\": source_quality,\\n              \\"fingerprint_provenance\\": fp_status, \\"analog_index_provenance\\": analog_status,\\n              \\"search_memo\\": {\\"scope\\": \\"per-query full-cleaned-content plus exact exclusion set; no index file cache\\",\\n                              \\"hits\\": memo.hits, \\"misses\\": memo.misses},\\n              \\"engine_config\\": asdict(cfg), \\"models\\": summaries, \\"candidate_coverage\\": coverage,\\n              \\"primary\\": {\\"contrast\\": protocol[\\"primary\\"], \\"delta_mrr\\": primary[0], \\"ci_low\\": primary[1], \\"ci_high\\": primary[2],\\n                          \\"interpretation\\": \\"Single predeclared paired diagnostic; no automatic promotion\\"},\\n              \\"secondary_joint_intervals\\": joint_intervals(effects, n_boot=protocol[\\"bootstrap_draws\\"], seed=seed),\\n              \\"error_slices\\": slices, \\"private_details_sha256\\": digest({\\"ranks\\": ranks, \\"test_ids\\": test_ids, \\"signatures\\": private_query_hashes}),\\n              \\"environment\\": {\\"python\\": sys.version.split()[0], \\"device\\": device,\\n                              **{name: importlib.metadata.version(name) for name in (\\"rdkit\\", \\"numpy\\", \\"pandas\\", \\"pyarrow\\", \\"scipy\\", \\"numba\\", \\"torch\\", \\"lightgbm\\")}},\\n              \\"limitations\\": [\\"Internal source data, not newly acquired external spectra or an independent laboratory replication.\\",\\n                              \\"Family-disjoint supervised query fitting does not remove the task-defined retrieval reference corpus.\\",\\n                              \\"Generic Murcko topology and acyclic connectivity are computational grouping rules, not a guarantee of distant chemistry.\\",\\n                              \\"Fixed training sample and small fresh fingerprint architecture trade capacity for auditable zero-cost execution; no state-of-the-art claim.\\",\\n                              \\"Feature ablations reuse a fixed generated candidate universe; no_gen_features does not remove candidate generation.\\",\\n                              \\"Slices are descriptive; subgroup intervals do not establish multiplicity-adjusted subgroup effects.\\",\\n                              \\"No raw spectra, structures, query identifiers, per-query ranks, model weights or submitted predictions are exported.\\"]}\\n    output = Path(os.environ.get(\\"CASMI_ADVANCED_OUTPUT\\", \\"/kaggle/working/advanced_study.json\\"))\\n    with output.open(\\"x\\", encoding=\\"utf-8\\") as stream:\\n        json.dump(result, stream, indent=2, allow_nan=False)\\n        stream.write(\\"\\\\n\\")\\n    print(\\"Completed sealed internal family-holdout study with fresh weights and all declared controls\\", flush=True)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "scripts/external_study_training.py": "\\"\\"\\"Fresh supervised fingerprint fitting shared by the external study runner.\\"\\"\\"\\nfrom pathlib import Path\\n\\ndef fit_fingerprints(store, labels, splits, protocol, protocol_path, artifact_dir, device):\\n    import numpy as np\\n    import torch\\n    from rdkit import Chem, DataStructs\\n    from rdkit.Chem import rdFingerprintGenerator\\n    from casmi.channels.fpmodel import FPModel, FPModelConfig, save\\n    from casmi.channels.fpinfer import FP_CLEAN, MAX_PEAKS, ADDUCT_ID\\n    from casmi.spectra.clean import clean_peaks_fast\\n    from casmi.validate.scaffold_study import digest\\n    from casmi.validate.source_manifest import sha256_file\\n    meta = store.meta\\n    key_of_row = meta.train_ik14.astype(str).to_numpy()\\n    seed = protocol[\'seed\']\\n    artifact_dir = Path(artifact_dir)\\n    # Both D3 models train on selected training identities only, on opposite folds.\\n    train_lookup = {row[\\"key\\"]: row for row in splits[\\"train\\"]}\\n    fp_eligible = (meta.test_adduct.to_numpy() & (meta.n_peaks.to_numpy() >= 5)\\n                   & (np.abs(meta.ppm.to_numpy()) <= 20))\\n    training_rows = {}\\n    for index in np.flatnonzero(fp_eligible & np.isin(key_of_row, list(train_lookup))):\\n        training_rows.setdefault(key_of_row[index], []).append(int(index))\\n    target_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)\\n    fp_models, fp_lineage = {}, []\\n    settings = protocol[\\"fp_model\\"]\\n    (artifact_dir / \\"models\\").mkdir(exist_ok=True)\\n    for fold in (0, 1):\\n        rng = np.random.default_rng(seed + fold)\\n        selected_rows, selected_keys = [], []\\n        for key, item in sorted(train_lookup.items()):\\n            if item[\\"fold\\"] != fold:\\n                continue\\n            available = np.asarray(training_rows.get(key, []), dtype=np.int64)\\n            cap = protocol[\\"max_training_spectra_per_structure\\"]\\n            if len(available) > cap:\\n                available = np.sort(rng.choice(available, cap, replace=False))\\n            selected_rows.extend(available.tolist())\\n            selected_keys.extend([key] * len(available))\\n        if len(selected_rows) < 100:\\n            raise ValueError(\\"Insufficient fingerprint training spectra in declared fold\\")\\n        if not set(selected_keys) <= {r[\\"key\\"] for r in splits[\\"train\\"]}:\\n            raise ValueError(\\"Fingerprint training crossed the declared training partition\\")\\n        n = len(selected_rows)\\n        mz = np.zeros((n, MAX_PEAKS), np.float32)\\n        it = np.zeros_like(mz)\\n        npk = np.zeros(n, np.int64)\\n        unique_keys = sorted(set(selected_keys))\\n        targets = {}\\n        for key in unique_keys:\\n            mol = Chem.MolFromSmiles(labels[key])\\n            bits = np.zeros(2048, np.uint8)\\n            DataStructs.ConvertToNumpyArray(target_gen.GetFingerprint(mol), bits)\\n            if not bits.any():\\n                raise ValueError(\\"Invalid fingerprint training target\\")\\n            targets[key] = bits\\n        for j, index in enumerate(selected_rows):\\n            cm, ci = clean_peaks_fast(*store.peaks(index), float(meta.precursor_mz.iat[index]), FP_CLEAN)\\n            mz[j, :len(cm)], it[j, :len(ci)], npk[j] = cm, ci, len(cm)\\n        side = meta.iloc[selected_rows]\\n        inputs = {\\"mz\\": mz, \\"it\\": it, \\"npk\\": npk, \\"prec\\": side.precursor_mz.to_numpy(np.float32),\\n                  \\"adduct\\": side.adduct.map(ADDUCT_ID).to_numpy(np.int64),\\n                  \\"ce\\": side.ce_ev.to_numpy(np.float32),\\n                  \\"positive\\": (side.ionization_mode == \\"positive\\").to_numpy(np.float32)}\\n        y = np.asarray([targets[key] for key in selected_keys], dtype=np.float32)\\n        torch.manual_seed(seed + fold)\\n        model = FPModel(FPModelConfig(d_model=settings[\\"d_model\\"], n_layers=settings[\\"n_layers\\"],\\n                                      d_ff=settings[\\"d_ff\\"])).to(device)\\n        optimizer = torch.optim.AdamW(model.parameters(), lr=settings[\\"lr\\"], weight_decay=.01)\\n        losses = []\\n        model.train()\\n        for epoch in range(settings[\\"epochs\\"]):\\n            total = 0.0\\n            order = rng.permutation(n)\\n            for start in range(0, n, settings[\\"batch_size\\"]):\\n                ids = order[start:start + settings[\\"batch_size\\"]]\\n                x = {name: torch.as_tensor(np.ascontiguousarray(value[ids]), device=device,\\n                     dtype=torch.long if name in (\\"npk\\", \\"adduct\\") else torch.float32) for name, value in inputs.items()}\\n                target = torch.as_tensor(y[ids], device=device)\\n                loss = torch.nn.functional.binary_cross_entropy_with_logits(model(**x), target)\\n                if not torch.isfinite(loss):\\n                    raise ValueError(\\"Nonfinite fingerprint training loss\\")\\n                optimizer.zero_grad(set_to_none=True)\\n                loss.backward()\\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\\n                optimizer.step()\\n                total += float(loss.detach().cpu()) * len(ids)\\n            losses.append(total / n)\\n            print(f\\"Trained fresh fingerprint fold {fold}, epoch {epoch + 1}/{settings[\'epochs\']}\\", flush=True)\\n        model.eval()\\n        model_path = artifact_dir / \\"models\\" / f\\"external_fp_fold{fold}.pt\\"\\n        lineage = {\\"fold\\": fold, \\"n_spectra\\": n, \\"n_structures\\": len(unique_keys),\\n                   \\"training_rows_sha256\\": digest(selected_rows), \\"training_identities_sha256\\": digest(unique_keys),\\n                   \\"training_families_sha256\\": digest(sorted({train_lookup[k][\\"group\\"] for k in unique_keys})),\\n                   \\"dev_test_identities_used\\": 0, \\"epochs_fixed_before_execution\\": settings[\\"epochs\\"],\\n                   \\"training_bce_by_epoch\\": losses, \\"initialization\\": \\"fresh seeded random; no archived weights\\"}\\n        save(model, model_path, {\\"lineage\\": lineage, \\"protocol_sha256\\": sha256_file(protocol_path)})\\n        fp_lineage.append({**lineage, \\"model_sha256\\": sha256_file(model_path)})\\n        fp_models[fold] = model\\n        del mz, it, npk, inputs, y\\n\\n    return fp_models, fp_lineage\\n", "scripts/run_external_c3_study.py": "\\"\\"\\"Execute fixed C3 budgets with fresh weights and independently audited external data.\\n\\nFail closed on insufficient external novelty. Test features are computed only after\\nboth policies, all learned artifacts and the development choice have been sealed.\\nOnly aggregate outcomes and private-detail hashes may leave scratch storage.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom dataclasses import asdict\\nfrom datetime import UTC, datetime\\nimport hashlib\\nimport importlib.metadata\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport sys\\nimport time\\n\\nROOT = Path(__file__).resolve().parents[1]\\nsys.path[:0] = [str(ROOT), str(ROOT / \\"src\\")]\\nPOLICIES = (\\"legacy\\", \\"expanded\\")\\n\\n\\ndef primary_coverage_interval(legacy, expanded, *, seed, n_boot=4000):\\n    import numpy as np\\n    from casmi.validate.rank_inputs import validated_bootstrap\\n    validated_bootstrap(n_boot, seed)\\n    a, b = np.asarray(legacy), np.asarray(expanded)\\n    if a.ndim != 1 or a.shape != b.shape or len(a) < 2 or not np.isin(a, [0, 1]).all() or not np.isin(b, [0, 1]).all():\\n        raise ValueError(\\"Paired binary molecule coverage required\\")\\n    delta = b.astype(float) - a.astype(float)\\n    rng = np.random.default_rng(seed)\\n    draws = np.empty(n_boot)\\n    for start in range(0, n_boot, 100):\\n        ids = rng.integers(len(delta), size=(min(100, n_boot - start), len(delta)))\\n        draws[start:start + len(ids)] = delta[ids].mean(axis=1)\\n    low, high = np.quantile(draws, [.025, .975])\\n    return {\\"n_molecules\\": len(a), \\"legacy_covered\\": int(a.sum()), \\"expanded_covered\\": int(b.sum()),\\n            \\"gained\\": int(((a == 0) & (b == 1)).sum()), \\"lost\\": int(((a == 1) & (b == 0)).sum()),\\n            \\"delta_coverage\\": float(delta.mean()), \\"ci_low\\": float(low), \\"ci_high\\": float(high),\\n            \\"n_bootstrap\\": n_boot, \\"seed\\": seed, \\"method\\": \\"Molecule-paired binary coverage percentile bootstrap\\"}\\n\\n\\ndef development_choice(legacy, expanded):\\n    import numpy as np\\n    a, b = np.asarray(legacy), np.asarray(expanded)\\n    if a.ndim != 1 or a.shape != b.shape or not len(a) or not np.isin(a, [0, 1]).all() or not np.isin(b, [0, 1]).all():\\n        raise ValueError(\\"Paired development coverage required\\")\\n    gained, lost = int(((a == 0) & (b == 1)).sum()), int(((a == 1) & (b == 0)).sum())\\n    return {\\"selected_policy\\": \\"expanded\\" if lost == 0 and gained >= 1 else \\"legacy\\",\\n            \\"n_molecules\\": len(a), \\"legacy_covered\\": int(a.sum()), \\"expanded_covered\\": int(b.sum()),\\n            \\"gained\\": gained, \\"lost\\": lost, \\"policy\\": \\"Prespecified development-only C3 coverage gate; both test policies reported\\"}\\n\\n\\ndef main():\\n    import numpy as np\\n    import pandas as pd\\n    import rdkit\\n    import torch\\n    from casmi.config import ARTIFACT_DIR, POOL_FILE, POOL_FP_FILE, POOL_FP_KEYS_FILE, POOL_TAG\\n    from casmi.channels.library import LibraryIndex\\n    from casmi.channels.analog import AnalogIndex\\n    from casmi.chem.keys import MetricKeys\\n    from casmi.engine import BaselineEngine, EngineConfig, molecule_mass\\n    from casmi.pool.pool import CandidatePool\\n    from casmi.pool.fingerprint_provenance import verify_fingerprint_provenance\\n    from casmi.spectra.store import SpectrumStore\\n    from casmi.predict import Predictor\\n    from casmi.rank.features import feature_columns\\n    from casmi.rank.ranker import Ranker, RankerConfig\\n    from casmi.validate.budget_features import budget_features\\n    from casmi.validate.external_population import prepare_internal, select_external\\n    from casmi.validate.cohort_bootstrap import joint_cohort_intervals\\n    from casmi.validate.knockout import Query, CLASSES, knockout_for, equivalent_keys, include_equivalent_source_rows\\n    from casmi.validate.spectral_controls import perturb_spectra\\n    from casmi.validate.metric import rank_of_truth, summarize\\n    from casmi.validate.scaffold_study import digest\\n    from casmi.validate.source_manifest import create_manifest, required_validation_inputs, verify_manifest, sha256_file\\n    from scripts.run_external_audit import audit\\n    from scripts.run_advanced_study import SearchMemo\\n    from scripts.external_study_training import fit_fingerprints\\n\\n    started = time.time()\\n    protocol_path = ROOT / os.environ.get(\\"CASMI_STUDY_PROTOCOL\\", \\"protocols/external-c3-v1.json\\")\\n    execution_path = ROOT / os.environ.get(\\"CASMI_EXECUTION_PROTOCOL\\", \\"protocols/external-c3-v1-execution.json\\")\\n    source_protocol_path = ROOT / os.environ.get(\\"CASMI_EXTERNAL_PROTOCOL\\", \\"protocols/external-source-v2.json\\")\\n    protocol = json.loads(protocol_path.read_text())\\n    execution = json.loads(execution_path.read_text())\\n    seed, device = protocol[\\"seed\\"], \\"cpu\\"\\n    if rdkit.__version__ != \\"2026.03.3\\":\\n        raise ValueError(\\"Official metric RDKit required\\")\\n    np.random.seed(seed); torch.manual_seed(seed); torch.set_num_threads(2)\\n    torch.use_deterministic_algorithms(True)\\n    independence, novel = audit(Path(os.environ[\\"CASMI_DATA\\"]) / \\"train.parquet\\",\\n        Path(os.environ[\\"CASMI_EXTERNAL_INPUT\\"]), Path(os.environ[\\"CASMI_EXTERNAL_MANIFEST\\"]), source_protocol_path)\\n    if independence[\\"status\\"] != \\"EXTERNAL_INDEPENDENCE_FEASIBLE\\":\\n        raise ValueError(\\"External independence gate failed; no supervised fitting or predictions permitted\\")\\n    store = SpectrumStore(ARTIFACT_DIR / \\"store\\")\\n    meta = store.meta\\n    row_keys = meta.train_ik14.astype(str).to_numpy()\\n    if len(meta) != independence[\\"identity_audit\\"][\\"training_rows_scanned\\"]:\\n        raise ValueError(\\"Spectrum store row count differs from independently audited training snapshot\\")\\n    if set(novel.standard_ik14) & set(row_keys):\\n        raise ValueError(\\"External source identity remains in the actual spectrum store\\")\\n    pool = CandidatePool(pd.read_parquet(ARTIFACT_DIR / POOL_FILE))\\n    truth_pool = CandidatePool(pd.read_parquet(ARTIFACT_DIR / \\"structures.parquet\\"))\\n    previous_protocol = json.loads((ROOT / \\"protocols/advanced-internal-v1.json\\").read_text())\\n    previous_audit = json.loads((ROOT / \\"reports/advanced_study.json\\").read_text())[\\"partitions\\"]\\n    protected, purge_report = None, None\\n    if os.environ.get(\\"CASMI_PURGE_REPORT\\"):\\n        purge_report = json.loads(Path(os.environ[\\"CASMI_PURGE_REPORT\\"]).read_text())\\n        protected_path = Path(os.environ[\\"CASMI_PROTECTED_POPULATION\\"])\\n        if (purge_report[\\"status\\"] != \\"EXTERNAL_FAMILIES_PURGED_AND_INDEPENDENCE_VERIFIED\\"\\n                or purge_report[\\"model_performance_inspected\\"] is not False\\n                or purge_report[\\"protocol_sha256\\"] != sha256_file(protocol_path)\\n                or purge_report[\\"protected_population_sha256\\"] != sha256_file(protected_path)\\n                or purge_report[\\"post_purge_independence\\"][\\"inputs\\"] != independence[\\"inputs\\"]\\n                or purge_report[\\"retained_external_family_overlap\\"] != 0):\\n            raise ValueError(\\"Purged training and protected historical population provenance differs\\")\\n        protected = json.loads(protected_path.read_text())\\n    splits, groups, labels, split_audit = prepare_internal(meta, truth_pool, previous_protocol, previous_audit, protocol, protected=protected)\\n    external_items, external_population = select_external(novel, splits, protocol[\\"split_salt\\"])\\n    if purge_report is not None and (len(external_items) != purge_report[\\"selected_external_molecules\\"]\\n            or digest(sorted(item[\\"metric\\"] for item in external_items)) != purge_report[\\"selected_metric_sha256\\"]):\\n        raise ValueError(\\"Declared external population changed after source-family selection\\")\\n    static_names = list(required_validation_inputs(pool_tag=POOL_TAG, analog=True)) + [\\"structures.parquet\\", f\\"pool_source_quality{POOL_TAG}.json\\"]\\n    manifest = create_manifest(ARTIFACT_DIR, static_names)\\n    fps = np.load(ARTIFACT_DIR / POOL_FP_FILE, mmap_mode=\\"r\\", allow_pickle=False)\\n    fp_keys = pd.read_parquet(ARTIFACT_DIR / POOL_FP_KEYS_FILE).pool_key.to_numpy()\\n    fp_status = verify_fingerprint_provenance(pool.table, fp_keys, fps,\\n        json.loads((ARTIFACT_DIR / f\\"pool_fp_provenance{POOL_TAG}.json\\").read_text()))\\n    fp_models, fp_lineage = fit_fingerprints(store, labels, splits, {**protocol, **execution}, protocol_path, ARTIFACT_DIR, device)\\n    cfg = EngineConfig(generate=True, gen_consensus=True, gen_expand=True, gen_max_analogs=30, gen_cap=900, np_score=True)\\n    library = LibraryIndex(store, row_keys, cfg.library)\\n    analog = AnalogIndex(store, row_keys, cfg.analog, cache=None)\\n    analog_status = analog.fresh_index_provenance()\\n    memo = SearchMemo(analog)\\n    keys = MetricKeys(ARTIFACT_DIR / \\"external_metric_keys.parquet\\")\\n    predictor = Predictor(BaselineEngine(pool, library, cfg, memo, fps), pool, keys,\\n                          fp_models=list(fp_models.values()), device=device)\\n    predictor.ranker = type(\\"FeatureDeclaration\\", (), {\\"columns\\": [\\"d4_int\\"]})()\\n    source_rows = {name: np.flatnonzero((meta.ingest_lib == name).to_numpy()) for name in previous_protocol[\\"query_libraries\\"]}\\n\\n    def prepare(item, external=False):\\n        if external:\\n            spectra = [{\\"mz\\": np.asarray(r.ms2_mzs), \\"it\\": np.asarray(r.ms2_normalized_intensities),\\n                        \\"precursor_mz\\": float(r.precursor_mz), \\"adduct\\": str(r.adduct), \\"mode\\": str(r.ionization_mode), \\"ce_ev\\": 0.0}\\n                       for r in item[\\"records\\"].itertuples()]\\n            query = Query(\\"external:\\" + item[\\"metric\\"], item[\\"key\\"], spectra, np.empty(0, dtype=np.int64))\\n            smiles = item[\\"smiles\\"]\\n        else:\\n            available = groups[item[\\"key\\"]]\\n            rng = np.random.default_rng(int(hashlib.sha256(item[\\"key\\"].encode()).hexdigest()[:16], 16))\\n            chosen = np.sort(rng.choice(available, 5, replace=False)) if len(available) > 5 else available\\n            spectra = []\\n            for index in chosen:\\n                mz, intensity = store.peaks(index); row = meta.iloc[index]\\n                spectra.append({\\"mz\\": mz, \\"it\\": intensity, \\"precursor_mz\\": float(row.precursor_mz),\\n                                \\"adduct\\": str(row.adduct), \\"mode\\": str(row.ionization_mode), \\"ce_ev\\": float(row.ce_ev)})\\n            query = Query(item[\\"library\\"] + \\":\\" + item[\\"key\\"], item[\\"key\\"], spectra, chosen)\\n            smiles = labels[item[\\"key\\"]]\\n        truth, same = equivalent_keys(item[\\"key\\"], molecule_mass(spectra), truth_pool, keys, truth_smiles=smiles)\\n        candidate_truth, candidates = equivalent_keys(item[\\"key\\"], molecule_mass(spectra), pool, keys, truth_smiles=smiles)\\n        if truth != item[\\"metric\\"] or candidate_truth != truth:\\n            raise ValueError(\\"Population metric and query truth disagree\\")\\n        same.update(candidates); same.add(item[\\"key\\"])\\n        if not external:\\n            include_equivalent_source_rows(query, same, source_rows[item[\\"library\\"]], row_keys)\\n        elif any(key in library.rows_of for key in same):\\n            raise ValueError(\\"A verified external truth unexpectedly has training reference spectra\\")\\n        return query, same, truth\\n\\n    def frames(query, same, truth, classes, mode):\\n        spectra = perturb_spectra(query.spectra, mode=mode, seed=seed, query_id=query.qid)\\n        for cls in classes:\\n            knockout = knockout_for(query, cls, library.rows_of, same)\\n            union = predictor.features(spectra, excluded_rows=knockout.excluded_rows,\\n                excluded_keys=knockout.excluded_keys, knocked_keys=same if cls != \\"C1\\" else frozenset())\\n            if not union.empty:\\n                union = union.assign(qid=query.qid, cls=cls)\\n                union[\\"label\\"] = [int(keys.get(key, pool.smiles_of(key)) == truth) for key in union.pool_key]\\n            yield cls, {policy: budget_features(union, policy) for policy in POLICIES}\\n\\n    tables, development = {}, {p: [] for p in POLICIES}\\n    for split in (\\"train\\", \\"dev\\"):\\n        blocks = {p: [] for p in POLICIES}\\n        for index, item in enumerate(splits[split]):\\n            memo.reset(); query, same, truth = prepare(item)\\n            predictor.fp_models = [fp_models[1 - item[\\"fold\\"]]] if split == \\"train\\" else list(fp_models.values())\\n            for cls, budgets in frames(query, same, truth, CLASSES, \\"none\\"):\\n                for policy, frame in budgets.items():\\n                    if len(frame): blocks[policy].append(frame)\\n                    if split == \\"dev\\" and cls == \\"C3\\":\\n                        development[policy].append(int(frame.label.max()) if len(frame) else 0)\\n            if (index + 1) % 10 == 0:\\n                print(f\\"Prepared {split} budget features {index + 1}/{len(splits[split])}\\", flush=True)\\n        tables[split] = {p: pd.concat(blocks[p], ignore_index=True) for p in POLICIES}\\n    choice = development_choice(development[\\"legacy\\"], development[\\"expanded\\"])\\n    if choice[\\"lost\\"]:\\n        raise ValueError(\\"Legacy-preservation invariant failed on development\\")\\n    models, training_audit = {}, {}\\n    for policy in POLICIES:\\n        columns = [c for c in feature_columns(tables[\\"train\\"][policy]) if c in tables[\\"dev\\"][policy] and not c.startswith((\\"prior_\\", \\"audit_\\"))]\\n        base = RankerConfig()\\n        config = RankerConfig(seeds=tuple(execution[\\"ranker\\"][\\"seeds\\"]), num_boost_round=execution[\\"ranker\\"][\\"rounds\\"],\\n            early_stopping=execution[\\"ranker\\"][\\"early_stopping\\"],\\n            params={**base.params, \\"num_threads\\": 2, \\"deterministic\\": True, \\"force_col_wise\\": True})\\n        model = Ranker(config).fit(tables[\\"train\\"][policy], tables[\\"dev\\"][policy], columns)\\n        for index, fitted in enumerate(model.models):\\n            fitted.save_model(str(ARTIFACT_DIR / \\"models\\" / f\\"external_{policy}_{index}.txt\\"))\\n        models[policy] = model\\n        training_audit[policy] = {\\"columns\\": columns, \\"best_iterations\\": model.best_iters, \\"config\\": asdict(config),\\n            \\"positive_training_groups\\": int(tables[\\"train\\"][policy].groupby([\\"qid\\", \\"cls\\"]).label.max().sum())}\\n        print(f\\"Fitted prespecified budget ranker {policy}\\", flush=True)\\n    model_hashes = {p.name: sha256_file(p) for p in sorted((ARTIFACT_DIR / \\"models\\").iterdir())}\\n    if len(model_hashes) != 8:\\n        raise ValueError(\\"Unexpected fresh learned artifact inventory\\")\\n    seal = {\\"status\\": \\"ALL_MODELS_AND_DEVELOPMENT_CHOICE_SEALED_BEFORE_TEST_FEATURES\\", \\"sealed_at\\": datetime.now(UTC).isoformat(),\\n            \\"model_files\\": model_hashes, \\"models\\": training_audit, \\"fingerprint_training\\": fp_lineage,\\n            \\"development_choice\\": choice, \\"partitions\\": split_audit, \\"external_population\\": external_population,\\n            \\"protocol_sha256\\": sha256_file(protocol_path), \\"execution_protocol_sha256\\": sha256_file(execution_path),\\n            \\"source_protocol_sha256\\": sha256_file(source_protocol_path)}\\n    if purge_report is not None:\\n        seal[\\"purged_source_sha256\\"] = {name: sha256_file(Path(os.environ[variable])) for name, variable in\\n            ((\\"purge_report\\", \\"CASMI_PURGE_REPORT\\"), (\\"protected_population\\", \\"CASMI_PROTECTED_POPULATION\\"))}\\n    seal_hash = digest(seal)\\n    seal_path = ARTIFACT_DIR / \\"external_training_seal.json\\"\\n    seal_path.write_text(json.dumps(seal, indent=2, allow_nan=False))\\n    del tables\\n    print(\\"Sealed all weights, populations and development choice; starting test features\\", flush=True)\\n    predictor.fp_models = list(fp_models.values())\\n    ranks, coverage, sizes, private_ids = {}, {}, {}, {}\\n    for cohort, items, classes, modes in ((\\"internal\\", splits[\\"test\\"], CLASSES, (\\"none\\",)),\\n                                          (\\"external\\", external_items, (\\"C2\\", \\"C3\\"), (\\"none\\", \\"blank\\"))):\\n        names = (*POLICIES, \\"mass_only\\", \\"d1\\", \\"d1_d2\\")\\n        ranks[cohort] = {p: {m: {c: [] for c in classes} for m in modes} for p in names}\\n        coverage[cohort] = {p: {c: [] for c in classes} for p in POLICIES}\\n        sizes[cohort] = {p: {c: [] for c in classes} for p in POLICIES}\\n        private_ids[cohort] = []\\n        for index, item in enumerate(items):\\n            memo.reset(); query, same, truth = prepare(item, external=cohort == \\"external\\")\\n            private_ids[cohort].append(query.qid)\\n            def ranked(frame, order):\\n                guesses = keys.dedupe_ranked(frame.pool_key.to_numpy()[order].tolist()[:60], pool.smiles_of, 25)\\n                return int(rank_of_truth([metric for _, metric in guesses], truth))\\n            for mode in modes:\\n                for cls, budgets in frames(query, same, truth, classes, mode):\\n                    for policy, frame in budgets.items():\\n                        value = ranked(frame, np.argsort(-models[policy].predict(frame), kind=\\"stable\\")) if len(frame) else 0\\n                        ranks[cohort][policy][mode][cls].append(value)\\n                        if mode == \\"none\\":\\n                            coverage[cohort][policy][cls].append(int(frame.label.max()) if len(frame) else 0)\\n                            sizes[cohort][policy][cls].append(len(frame))\\n                    legacy = budgets[\\"legacy\\"]\\n                    fixed = legacy[legacy.gen == 0].reset_index(drop=True) if len(legacy) else legacy\\n                    for name in (\\"mass_only\\", \\"d1\\", \\"d1_d2\\"):\\n                        value = 0\\n                        if len(fixed):\\n                            if name == \\"mass_only\\": order = np.argsort(np.abs(fixed.ppm), kind=\\"stable\\")\\n                            else:\\n                                primary, secondary = fixed.d1_max.to_numpy(), fixed.d1_mean.to_numpy()\\n                                if name == \\"d1_d2\\":\\n                                    primary = np.maximum(primary, fixed.an_p4.fillna(0).to_numpy()) if \\"an_p4\\" in fixed else primary\\n                                    secondary = fixed.an_p2.fillna(0).to_numpy() if \\"an_p2\\" in fixed else secondary\\n                                order = np.lexsort((np.abs(fixed.ppm.to_numpy()), -secondary, -primary))\\n                            value = ranked(fixed, order)\\n                        ranks[cohort][name][mode][cls].append(value)\\n            if (index + 1) % 10 == 0:\\n                print(f\\"Completed sealed {cohort} test {index + 1}/{len(items)}\\", flush=True)\\n    if digest(json.loads(seal_path.read_text())) != seal_hash or model_hashes != {p.name: sha256_file(p) for p in sorted((ARTIFACT_DIR / \\"models\\").iterdir())}:\\n        raise ValueError(\\"Learned artifacts or seal changed during test\\")\\n    verify_manifest(ARTIFACT_DIR, manifest)\\n    if purge_report is not None and seal[\\"purged_source_sha256\\"] != {name: sha256_file(Path(os.environ[variable])) for name, variable in\\n            ((\\"purge_report\\", \\"CASMI_PURGE_REPORT\\"), (\\"protected_population\\", \\"CASMI_PROTECTED_POPULATION\\"))}:\\n        raise ValueError(\\"Purged-source or historical-protection evidence changed during test\\")\\n    rr = lambda v: np.where(np.asarray(v) > 0, 1 / np.maximum(np.asarray(v), 1), 0.)\\n    effects = {\\"internal\\": {}, \\"external\\": {}}\\n    for cohort in effects:\\n        for cls in coverage[cohort][\\"legacy\\"]:\\n            effects[cohort][f\\"{cohort}_mrr_expanded_minus_legacy:{cls}\\"] = rr(ranks[cohort][\\"expanded\\"][\\"none\\"][cls]) - rr(ranks[cohort][\\"legacy\\"][\\"none\\"][cls])\\n            if cohort == \\"external\\":\\n                effects[cohort][f\\"external_coverage_expanded_minus_legacy:{cls}\\"] = np.asarray(coverage[cohort][\\"expanded\\"][cls]) - np.asarray(coverage[cohort][\\"legacy\\"][cls])\\n                for policy in POLICIES:\\n                    effects[cohort][f\\"external_{policy}_blank_minus_none:{cls}\\"] = rr(ranks[cohort][policy][\\"blank\\"][cls]) - rr(ranks[cohort][policy][\\"none\\"][cls])\\n    if {name for cohort in effects.values() for name in cohort} != set(execution[\\"secondary_family\\"]):\\n        raise ValueError(\\"Prespecified secondary family differs\\")\\n    summaries = {cohort: {p: {mode: {cls: asdict(summarize(values, n_boot=4000, seed=seed)) for cls, values in classes.items()}\\n        for mode, classes in conditions.items()} for p, conditions in policies.items()} for cohort, policies in ranks.items()}\\n    primary = primary_coverage_interval(coverage[\\"internal\\"][\\"legacy\\"][\\"C3\\"], coverage[\\"internal\\"][\\"expanded\\"][\\"C3\\"], seed=seed)\\n    if primary[\\"lost\\"]:\\n        raise ValueError(\\"Legacy-preserving expansion lost internal test truths\\")\\n    result = {\\"schema_version\\": 1, \\"status\\": \\"COMPLETE_C3_AND_PURGED_REFERENCE_EXTERNAL_HOLDOUT\\" if purge_report else \\"COMPLETE_C3_BUDGET_AND_EXTERNAL_VALIDATION\\", \\"checked_at\\": datetime.now(UTC).isoformat(),\\n        \\"elapsed_seconds\\": round(time.time() - started, 1), \\"protocol\\": protocol, \\"execution_protocol\\": execution,\\n        \\"independence_audit\\": independence, \\"purged_training\\": purge_report, \\"partitions\\": split_audit, \\"external_population\\": external_population,\\n        \\"training_seal\\": seal, \\"training_seal_sha256\\": seal_hash, \\"model_bytes_unchanged_after_test\\": True,\\n        \\"source_manifest\\": manifest, \\"fingerprint_provenance\\": fp_status, \\"analog_index_provenance\\": analog_status,\\n        \\"engine_config\\": asdict(cfg), \\"models\\": summaries, \\"primary\\": primary,\\n        \\"secondary_joint_intervals\\": joint_cohort_intervals(effects, n_boot=4000, seed=seed),\\n        \\"candidate_coverage\\": {cohort: {p: {cls: {\\"n_molecules\\": len(v), \\"covered\\": int(sum(v)),\\n            \\"oracle_mrr_ceiling\\": float(np.mean(v)), \\"mean_candidates\\": float(np.mean(sizes[cohort][p][cls])),\\n            \\"maximum_candidates\\": int(max(sizes[cohort][p][cls]))} for cls, v in classes.items()} for p, classes in policies.items()} for cohort, policies in coverage.items()},\\n        \\"private_details_sha256\\": digest({\\"ranks\\": ranks, \\"coverage\\": coverage, \\"ids\\": private_ids, \\"development\\": development}),\\n        \\"environment\\": {\\"python\\": sys.version.split()[0], \\"device\\": device,\\n            **{name: importlib.metadata.version(name) for name in (\\"rdkit\\", \\"numpy\\", \\"pandas\\", \\"pyarrow\\", \\"torch\\", \\"lightgbm\\")}},\\n        \\"limitations\\": [\\"External standard/metric novelty is computationally audited against the executed training snapshot; source annotations are not independently authenticated laboratory identity.\\",\\n            \\"When a family-purged snapshot is used, original-snapshot source novelty feasibility remains unsuccessful; external-family generalization is relative to the rebuilt filtered corpus.\\",\\n            \\"External collision energy is not harmonized to eV; unavailable numeric eV is encoded zero. This is a distribution-shift limitation.\\",\\n            \\"External libraries have a restricted chemical/instrument distribution and are not a representative clinical or environmental population.\\",\\n            \\"Generic-family supervised separation does not remove task-defined shared candidate knowledge or fixed NP-score priors.\\",\\n            \\"Both predeclared budgets and every adverse outcome are retained. Positive generator improvement is not guaranteed.\\",\\n            \\"Historical study populations, fitting and architecture differ; direct score comparison does not isolate improvement.\\",\\n            \\"No competition submission, paid infrastructure, service deployment or raw competition-record export.\\"]}\\n    with Path(os.environ[\\"CASMI_EXTERNAL_STUDY_OUTPUT\\"]).open(\\"x\\", encoding=\\"utf-8\\") as stream:\\n        json.dump(result, stream, indent=2, allow_nan=False); stream.write(\\"\\\\n\\")\\n    print(\\"Completed sealed C3 and external study with all declared outcomes\\", flush=True)\\n\\n\\nif __name__ == \\"__main__\\": main()\\n", "scripts/prepare_purged_external.py": "\\"\\"\\"Create a new training snapshot with every selected external family removed.\\n\\nOriginal inputs stay read-only. Selected public source rows and historical\\nexclusion labels remain in private scratch storage; export aggregates only.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport sys\\n\\nROOT = Path(__file__).resolve().parents[1]\\nsys.path[:0] = [str(ROOT), str(ROOT / \\"src\\")]\\n\\n\\ndef prepare():\\n    import pandas as pd\\n    import pyarrow as pa\\n    import pyarrow.compute as pc\\n    import pyarrow.parquet as pq\\n    from casmi.spectra.store import SpectrumStore\\n    from casmi.pool.pool import CandidatePool\\n    from casmi.validate.external_population import prepare_internal\\n    from casmi.validate.external_purge import select_external_families, blocked_training_identities\\n    from casmi.validate.scaffold_study import digest\\n    from casmi.validate.source_manifest import sha256_file\\n    from scripts.run_external_audit import audit\\n\\n    protocol_path = ROOT / \\"protocols/purged-external-v1.json\\"\\n    protocol = json.loads(protocol_path.read_text())\\n    original = Path(os.environ[\\"CASMI_ORIGINAL_TRAIN\\"])\\n    source_path = Path(os.environ[\\"CASMI_ORIGINAL_EXTERNAL\\"])\\n    source_manifest = json.loads(Path(os.environ[\\"CASMI_ORIGINAL_EXTERNAL_MANIFEST\\"]).read_text())\\n    previous = json.loads((ROOT / \\"reports/advanced_study.json\\").read_text())\\n    expected_train = previous[\\"raw_inputs\\"][\\"train.parquet\\"]\\n    original_hash = sha256_file(original)\\n    if original_hash != expected_train[\\"sha256\\"] or original.stat().st_size != expected_train[\\"bytes\\"]:\\n        raise ValueError(\\"Original training snapshot differs from protected historical input\\")\\n    if (source_manifest[\\"source\\"] != protocol[\\"external_source\\"]\\n            or sha256_file(source_path) != source_manifest[\\"parquet_sha256\\"]):\\n        raise ValueError(\\"Pinned external source or source policy differs\\")\\n    artifacts = Path(os.environ[\\"CASMI_ARTIFACTS\\"])\\n    original_store = SpectrumStore(artifacts / \\"store\\")\\n    truth_pool = CandidatePool(pd.read_parquet(artifacts / \\"structures.parquet\\"))\\n    previous_protocol = json.loads((ROOT / \\"protocols/advanced-internal-v1.json\\").read_text())\\n    *_, protected = prepare_internal(original_store.meta, truth_pool, previous_protocol,\\n        previous[\\"partitions\\"], protocol, return_protected=True)\\n    protected[\\"original_training_sha256\\"] = original_hash\\n    print(\\"Prepared verified original historical population exclusions before source filtering\\", flush=True)\\n    source = pd.read_parquet(source_path)\\n    selected, families = select_external_families(source, protocol[\\"split_salt\\"])\\n    print(f\\"Prepared {selected.metric_key.nunique()} external molecules before fitting or candidate inspection\\", flush=True)\\n    unique_labels = pd.read_parquet(original, columns=[\\"inchikey14\\", \\"normalized_smiles\\"]).drop_duplicates()\\n    blocked, grouping_audit = blocked_training_identities(unique_labels.itertuples(index=False, name=None), selected, families, return_audit=True)\\n    actual_blocked = set(unique_labels.inchikey14) & blocked\\n    output = Path(os.environ[\\"CASMI_PURGED_DATA\\"])\\n    output.mkdir(parents=True, exist_ok=False)\\n    protected_path = output / \\"protected_population.json\\"\\n    protected_path.write_text(json.dumps(protected, sort_keys=True, allow_nan=False) + \\"\\\\n\\", encoding=\\"utf-8\\")\\n    selected_path = output / \\"selected_external.parquet\\"\\n    selected.to_parquet(selected_path, index=False, compression=\\"zstd\\")\\n    counts = dict(source_manifest[\\"counts\\"])\\n    counts.update(selection_excluded_eligible_spectra=len(source) - len(selected),\\n                  eligible_spectra=len(selected), eligible_metric_molecules=int(selected.metric_key.nunique()))\\n    derived_manifest = {**source_manifest, \\"counts\\": counts, \\"parquet_sha256\\": sha256_file(selected_path),\\n                        \\"parquet_bytes\\": selected_path.stat().st_size,\\n                        \\"scope\\": \\"Outcome-blind source-family selection; independence is relative to a separately purged training snapshot\\"}\\n    manifest_path = output / \\"selected_external_manifest.json\\"\\n    manifest_path.write_text(json.dumps(derived_manifest, indent=2, allow_nan=False) + \\"\\\\n\\", encoding=\\"utf-8\\")\\n    train = pq.ParquetFile(original)\\n    retained, removed, seen = 0, 0, 0\\n    clean_path = output / \\"train.parquet\\"\\n    blocked_array = pa.array(sorted(blocked), type=pa.string())\\n    with pq.ParquetWriter(clean_path, train.schema_arrow, compression=\\"zstd\\") as writer:\\n        for index, batch in enumerate(train.iter_batches(batch_size=16384)):\\n            mask = pc.invert(pc.is_in(batch.column(\\"inchikey14\\"), value_set=blocked_array))\\n            clean = batch.filter(mask)\\n            if clean.num_rows:\\n                writer.write_table(pa.Table.from_batches([clean]))\\n            retained += clean.num_rows; removed += batch.num_rows - clean.num_rows; seen += batch.num_rows\\n            if index % 25 == 0:\\n                print(f\\"Prepared purged reference rows {seen}/{train.metadata.num_rows}\\", flush=True)\\n    if retained < 100000 or retained + removed != train.metadata.num_rows or removed == 0:\\n        raise ValueError(\\"Purged training row accounting or minimum size failed\\")\\n    independence, novel = audit(clean_path, selected_path, manifest_path, ROOT / \\"protocols/external-source-v3.json\\")\\n    identities = independence[\\"identity_audit\\"]\\n    if (independence[\\"status\\"] != \\"EXTERNAL_INDEPENDENCE_FEASIBLE\\"\\n            or identities[\\"excluded_external_molecules_union\\"] != 0 or len(novel) != len(selected)):\\n        raise ValueError(\\"Selected external source retains training identity/content overlap after purge\\")\\n    remaining = unique_labels[~unique_labels.inchikey14.isin(blocked)]\\n    from casmi.validate.scaffold_study import family_key\\n    retained_family_overlap = sum(family_key(s) in set(families) for s in set(remaining.normalized_smiles))\\n    if retained_family_overlap:\\n        raise ValueError(\\"Selected external family remains in filtered training\\")\\n    report = {\\"schema_version\\": 1, \\"status\\": \\"EXTERNAL_FAMILIES_PURGED_AND_INDEPENDENCE_VERIFIED\\",\\n        \\"protocol_sha256\\": sha256_file(protocol_path),\\n        \\"source_amendment_sha256\\": sha256_file(ROOT / \\"protocols/purged-external-v1-source-amendment.json\\"),\\n        \\"original_training\\": expected_train,\\n        \\"original_source_preparation\\": source_manifest, \\"selected_source_preparation\\": derived_manifest,\\n        \\"original_training_rows\\": train.metadata.num_rows, \\"retained_training_rows\\": retained, \\"removed_training_rows\\": removed,\\n        \\"removed_training_identities\\": len(actual_blocked), \\"family_grouping_quarantine\\": grouping_audit,\\n        \\"selected_external_molecules\\": int(selected.metric_key.nunique()),\\n        \\"selected_external_families\\": len(families), \\"selected_external_spectra_available\\": len(selected),\\n        \\"selected_metric_sha256\\": digest(sorted(set(selected.metric_key))), \\"selected_family_sha256\\": digest(families),\\n        \\"selected_accessions_sha256\\": digest(sorted(selected.accession)), \\"retained_external_family_overlap\\": retained_family_overlap,\\n        \\"protected_population_sha256\\": sha256_file(protected_path), \\"historical_population_reconstructed_before_purge\\": True,\\n        \\"post_purge_independence\\": independence, \\"model_performance_inspected\\": False,\\n        \\"scope\\": \\"External-family holdout relative to newly purged training/reference data; original snapshot novelty feasibility remains unsuccessful\\"}\\n    (output / \\"purge_report.json\\").write_text(json.dumps(report, indent=2, allow_nan=False) + \\"\\\\n\\", encoding=\\"utf-8\\")\\n    print(\\"Prepared independently audited family-purged training snapshot; no model outcomes inspected\\", flush=True)\\n\\n\\nif __name__ == \\"__main__\\": prepare()\\n", "protocols/purged-external-v1.json": "{\\n  \\"schema_version\\": 1,\\n  \\"id\\": \\"casmi-purged-external-v1\\",\\n  \\"registered_on\\": \\"2026-10-28\\",\\n  \\"scope\\": \\"New experiment after failed original-snapshot novelty feasibility: freshly fitted models with external chemical families purged from the entire training/reference corpus; no claim that these molecules were absent from the original CASMI snapshot.\\",\\n  \\"external_source\\": {\\n    \\"repository\\": \\"https://library.gnps2.org/libraries\\",\\n    \\"retrieved_on\\": \\"2026-10-27\\",\\n    \\"license_policy\\": \\"Directly contributed GNPS reference spectra default to CC0; exclude third-party imported and propagated/MassQL/suspect libraries.\\",\\n    \\"license_reference\\": \\"https://ccms-ucsd.github.io/GNPSDocumentation/browselibraries/#license\\",\\n    \\"allowed_record_licenses\\": [\\n      \\"CC0\\"\\n    ],\\n    \\"files\\": {\\n      \\"CEMBIO_OXYLIPINS.json\\": {\\n        \\"url\\": \\"https://library.gnps2.org/api/libraries/CEMBIO_OXYLIPINS/json\\",\\n        \\"sha256\\": \\"5bb461648eb111304b8e47765a177a01b443b9ad0589038ea5f261428814130e\\",\\n        \\"bytes\\": 192299\\n      },\\n      \\"USER-CONTRIBUTED-GNPS2.json\\": {\\n        \\"url\\": \\"https://library.gnps2.org/api/libraries/USER-CONTRIBUTED-GNPS2/json\\",\\n        \\"sha256\\": \\"3146f450ba71009edac2375041a05de3fb1dae282903119074701a84fbc15e7a\\",\\n        \\"bytes\\": 302638\\n      },\\n      \\"GNPS-LIBRARY.json\\": {\\n        \\"url\\": \\"https://library.gnps2.org/api/libraries/GNPS-LIBRARY/json\\",\\n        \\"sha256\\": \\"d463bc39362359329a520d8fdd1b352abc4807188fc6ef465501183b1a261264\\",\\n        \\"bytes\\": 334137083\\n      }\\n    },\\n    \\"eligibility\\": \\"Source-classified standard spectra; MS2, ESI evidenced in source/instrument annotation; supported adduct and matching mode; connected valid molecular structure, 150-1170 Da, precursor error <=20 ppm, at least five finite positive ordered peaks; reject explicit low-quality or inferred/synthetic acquisition annotations. Preserve raw record hash, original metadata and source URL. GNPS2 snapshot-relative record IDs are derived hashes, not claimed official accessions.\\"\\n  },\\n  \\"feasibility_stage\\": \\"Audit the selected external inputs against every row of the newly purged training snapshot; require zero standard/metric/exact-content overlap, zero retained selected generic families, at least 30 external molecules, and at least 100000 retained training rows. These requirements do not retroactively establish novelty against the original snapshot.\\",\\n  \\"external_selection\\": \\"Before any fitting, hash-order eligible generic families with salt NEXUS-CASMI-PURGED-EXTERNAL-20261028-v1|external-family|family; choose at most 150 molecules, at least 30, one representative metric per family component. Choose lexical metric representatives, retain at most five accession-sorted spectra. Do not use predictions, candidate presence, or the previously observed seven-novel-molecule subset.\\",\\n  \\"internal_counts\\": {\\n    \\"train\\": 300,\\n    \\"dev\\": 100,\\n    \\"test\\": 150\\n  },\\n  \\"internal_selection\\": \\"Reconstruct and verify the original 600/150/250 population on the original unpurged metadata before filtering. Preserve its selected family/metric exclusion sets privately. Select fresh 300/100/150 internal partitions from the purged corpus, excluding those original selected families/metrics and every historical V-in family. Source filtering cannot authorize reusing a previously opened test family.\\",\\n  \\"split_salt\\": \\"NEXUS-CASMI-PURGED-EXTERNAL-20261028-v1\\",\\n  \\"seed\\": 20261028,\\n  \\"generator_policies\\": {\\n    \\"legacy\\": {\\n      \\"analog_seeds\\": 15,\\n      \\"candidate_cap\\": 300,\\n      \\"transform_pair_order\\": \\"historical single order\\"\\n    },\\n    \\"expanded\\": {\\n      \\"analog_seeds\\": 30,\\n      \\"candidate_cap\\": 900,\\n      \\"transform_pair_order\\": \\"both orders\\",\\n      \\"preserve_legacy_candidates_and_features\\": true\\n    }\\n  },\\n  \\"generator_choice\\": \\"Only two fixed policies; expanded is retained for test only if development coverage does not decrease and at least one new development truth is recovered. Freeze the choice before test inference. Always report both policies and development gains/losses.\\",\\n  \\"fresh_supervised_fitting\\": \\"Two cross-fitted fingerprint networks, dimension128/layers2/ff512, six fixed epochs, batch256, AdamW lr0.0003; two three-seed LambdaRank models, <=400 rounds with development-only early stopping40; no archived weight attachment\\",\\n  \\"feature_policy\\": \\"D1/D2/D3/D4 and generator features; exclude all prior_* and audit_* columns from scientific models; shared candidate-intrinsic computation may be reused only if each candidate subset\'s list-context features are recomputed\\",\\n  \\"test_controls\\": \\"Unperturbed internal C1/C2/C3 and external C2/C3; blank-fragment external C2/C3 diagnostic; same selected molecules retained including empty candidate lists\\",\\n  \\"primary\\": \\"Paired expanded-minus-legacy C3 candidate truth-coverage difference on the new internal test\\",\\n  \\"external_validation\\": \\"Report every external C2/C3 outcome with both budgets and blank controls, using the same predeclared estimators as the earlier design. All selected external molecules stay in denominators. Generalization is relative to the purged training/reference snapshot; source chemical annotations and original shared-archive acquisition lineage are not independently authenticated.\\",\\n  \\"uncertainty\\": \\"4000 molecule-paired bootstrap draws; primary single contrast; secondary fixed-family joint max-deviation intervals; no inference from small descriptive slices\\",\\n  \\"seal\\": \\"Seal all supervised weights, features, protocol, population hashes and development choice before internal or external test feature generation\\",\\n  \\"budget\\": \\"Existing free Kaggle CPU/GPU only; no paid compute, external LLM calls, deployment or competition submissions\\",\\n  \\"failure_policy\\": \\"Fail closed if any purge, family, metric, content, population or seal gate fails. Preserve every execution and adverse result. Do not call this a passing original-snapshot novelty study or laboratory replication. Do not change the selected external molecules after prediction/candidate inspection.\\",\\n  \\"parent_feasibility_outcome\\": \\"MassBank 0/491 novel; two GNPS2 libraries 7/85; expanded GNPS 7/936. Preserve all failed gates. This is a separately defined purged-training experiment, not a passing result for external-c3-v1.\\",\\n  \\"external_training_purge\\": \\"Remove all original training rows belonging to any selected external generic family, standard identity, or official tautomer metric identity, including every row sharing a blocked declared training identity. Compute full-label family/identity checks before filtering. Rebuild structures, spectrum store, candidate pool, fingerprints and D2 index from the filtered snapshot. Keep independent COCONUT/PubChemLite candidate knowledge under ordinary C2 semantics; exclude all membership priors from fitted features.\\"\\n}\\n", "protocols/purged-external-v1-execution.json": "{\\n  \\"schema_version\\": 1,\\n  \\"parent_protocol\\": \\"purged-external-v1.json\\",\\n  \\"registered_before_any_model_performance\\": true,\\n  \\"purpose\\": \\"Fixed execution details for the separate purged-training study, registered before any model performance\\",\\n  \\"max_spectra\\": 5,\\n  \\"max_training_spectra_per_structure\\": 128,\\n  \\"fp_model\\": {\\n    \\"d_model\\": 128,\\n    \\"n_layers\\": 2,\\n    \\"d_ff\\": 512,\\n    \\"epochs\\": 6,\\n    \\"batch_size\\": 256,\\n    \\"lr\\": 0.0003\\n  },\\n  \\"ranker\\": {\\n    \\"seeds\\": [\\n      0,\\n      1,\\n      2\\n    ],\\n    \\"rounds\\": 400,\\n    \\"early_stopping\\": 40\\n  },\\n  \\"external_spectrum_selection\\": \\"First five accession-sorted eligible spectra per selected metric identity; no performance or candidate-coverage filtering\\",\\n  \\"external_missing_energy\\": \\"Collision energy is not harmonized from percent/NCE to eV. Encode unavailable numeric eV as zero using the existing model input convention and disclose this distribution-shift limitation.\\",\\n  \\"primary_bootstrap\\": \\"Percentile paired bootstrap of per-molecule expanded-minus-legacy binary C3 coverage on new internal test, 4000 draws\\",\\n  \\"secondary_family\\": [\\n    \\"internal_mrr_expanded_minus_legacy:C1\\",\\n    \\"internal_mrr_expanded_minus_legacy:C2\\",\\n    \\"internal_mrr_expanded_minus_legacy:C3\\",\\n    \\"external_mrr_expanded_minus_legacy:C2\\",\\n    \\"external_mrr_expanded_minus_legacy:C3\\",\\n    \\"external_coverage_expanded_minus_legacy:C2\\",\\n    \\"external_coverage_expanded_minus_legacy:C3\\",\\n    \\"external_legacy_blank_minus_none:C2\\",\\n    \\"external_legacy_blank_minus_none:C3\\",\\n    \\"external_expanded_blank_minus_none:C2\\",\\n    \\"external_expanded_blank_minus_none:C3\\"\\n  ],\\n  \\"secondary_resampling\\": \\"Joint max-deviation bootstrap over the fixed 11-contrast family. Resample molecules independently within internal/external cohorts, keeping every policy/class/control paired within its cohort. 4000 draws, uniform joint radius.\\",\\n  \\"baseline_diagnostics\\": \\"Report mass-only, D1 and D1+D2 rules on original pool candidates in each cohort/condition; no post-test baseline selection\\",\\n  \\"generation_feature_reuse\\": \\"Score candidate-intrinsic channels once on the legacy-preserving union; recompute n_cand, log_n_cand and all D1/D2/D3/D4 rank/gap/z features for each candidate subset. Audit membership never enters fitted features.\\",\\n  \\"runtime\\": \\"Free CPU execution permitted; GPU optional. Two Torch/BLAS/LightGBM threads, deterministic seeds. No deployment or submission.\\",\\n  \\"development_choice_interpretation\\": \\"The gate selects the recommended policy only. Both fixed budgets are evaluated on the declared test populations for the primary and all secondary comparisons, even if development selects legacy. No policy or test outcome is suppressed.\\"\\n}\\n", "protocols/external-source-v3.json": "{\\n  \\"schema_version\\": 1,\\n  \\"parent_protocol\\": \\"external-source-v2.json\\",\\n  \\"registered_on\\": \\"2026-10-27\\",\\n  \\"reason\\": \\"Before any model performance or candidate-coverage inspection, broaden the source beyond two small GNPS2-native libraries to the directly contributed GNPS community standard library. Retain all previous source snapshots and strict novelty gates.\\",\\n  \\"external_source\\": {\\n    \\"repository\\": \\"https://library.gnps2.org/libraries\\",\\n    \\"retrieved_on\\": \\"2026-10-27\\",\\n    \\"license_policy\\": \\"Directly contributed GNPS reference spectra default to CC0; exclude third-party imported and propagated/MassQL/suspect libraries.\\",\\n    \\"license_reference\\": \\"https://ccms-ucsd.github.io/GNPSDocumentation/browselibraries/#license\\",\\n    \\"allowed_record_licenses\\": [\\n      \\"CC0\\"\\n    ],\\n    \\"files\\": {\\n      \\"CEMBIO_OXYLIPINS.json\\": {\\n        \\"url\\": \\"https://library.gnps2.org/api/libraries/CEMBIO_OXYLIPINS/json\\",\\n        \\"sha256\\": \\"5bb461648eb111304b8e47765a177a01b443b9ad0589038ea5f261428814130e\\",\\n        \\"bytes\\": 192299\\n      },\\n      \\"USER-CONTRIBUTED-GNPS2.json\\": {\\n        \\"url\\": \\"https://library.gnps2.org/api/libraries/USER-CONTRIBUTED-GNPS2/json\\",\\n        \\"sha256\\": \\"3146f450ba71009edac2375041a05de3fb1dae282903119074701a84fbc15e7a\\",\\n        \\"bytes\\": 302638\\n      },\\n      \\"GNPS-LIBRARY.json\\": {\\n        \\"url\\": \\"https://library.gnps2.org/api/libraries/GNPS-LIBRARY/json\\",\\n        \\"sha256\\": \\"d463bc39362359329a520d8fdd1b352abc4807188fc6ef465501183b1a261264\\",\\n        \\"bytes\\": 334137083\\n      }\\n    },\\n    \\"eligibility\\": \\"Source-classified standard spectra; MS2, ESI evidenced in source/instrument annotation; supported adduct and matching mode; connected valid molecular structure, 150-1170 Da, precursor error <=20 ppm, at least five finite positive ordered peaks; reject explicit low-quality or inferred/synthetic acquisition annotations. Preserve raw record hash, original metadata and source URL. GNPS2 snapshot-relative record IDs are derived hashes, not claimed official accessions.\\"\\n  },\\n  \\"feasibility_stage\\": \\"Audit every eligible source molecule against every CASMI training row using the unchanged standard/metric/exact-content exclusion policy. At least 30 novel molecules are required. Counts alone may determine feasibility; no prediction or candidate-coverage inspection.\\",\\n  \\"failure_policy\\": \\"Preserve the failed MassBank source and this outcome-blind addition separately. Insufficient population cannot be relabeled as external validation. No model study starts on this source unless all original independence requirements pass.\\",\\n  \\"identity_prefilter\\": \\"Heavy-element composition followed by isotope-stripped heavy connectivity as two necessary conditions before official canonicalization. Clear atom isotope labels before explicit-H removal. Retain metric-equivalent isotope/tautomer pairs verified by regression; no change to standard/metric/content rejection or the 30-molecule minimum.\\"\\n}\\n", "protocols/advanced-internal-v1.json": "{\\n  \\"schema_version\\": 1,\\n  \\"id\\": \\"casmi-advanced-internal-v1\\",\\n  \\"scope\\": \\"Newly partitioned internal chemical-family benchmark with fresh supervised weights; not an external dataset\\",\\n  \\"split_salt\\": \\"NEXUS-CASMI-ADVANCED-20261025-v1\\",\\n  \\"split_counts\\": {\\"train\\": 600, \\"dev\\": 150, \\"test\\": 250},\\n  \\"query_libraries\\": [\\"riken\\", \\"gnps\\", \\"massbank\\", \\"mona\\", \\"msdial\\", \\"spectraverse\\", \\"pluskal_ms2\\"],\\n  \\"query_mass_range\\": [150.0, 1170.0],\\n  \\"max_spectra\\": 5,\\n  \\"max_training_spectra_per_structure\\": 128,\\n  \\"seed\\": 20261025,\\n  \\"fp_model\\": {\\"d_model\\": 128, \\"n_layers\\": 2, \\"d_ff\\": 512, \\"epochs\\": 6, \\"batch_size\\": 256, \\"lr\\": 0.0003},\\n  \\"ranker\\": {\\"seeds\\": [0, 1, 2], \\"rounds\\": 400, \\"early_stopping\\": 40},\\n  \\"ablations\\": [\\"full\\", \\"no_d1\\", \\"no_d2\\", \\"no_d3\\", \\"no_d4\\", \\"no_gen_features\\", \\"no_np_score\\", \\"membership_only\\"],\\n  \\"controls\\": [\\"none\\", \\"blank\\", \\"intensity_shuffle\\"],\\n  \\"primary\\": \\"Unperturbed test C2 MRR@25: fresh full model minus D1+D2 baseline\\",\\n  \\"secondary\\": \\"Joint paired bootstrap over all prespecified ablation/control contrasts and C1/C2/C3\\",\\n  \\"bootstrap_draws\\": 4000,\\n  \\"slice_min_n\\": 20,\\n  \\"test_policy\\": \\"All rankers, feature lists and model hashes sealed before test feature generation. Test never used for training, stopping or hyperparameter selection.\\",\\n  \\"retrieval_policy\\": \\"The retrieval corpus may contain evaluation-family references according to the C1 task definition. Per-query C2/C3 exclude metric-equivalent truth references. Query-family disjointness concerns supervised fitting, not a reference-free discovery task.\\",\\n  \\"promotion_policy\\": \\"Report all outcomes, including failures and negative controls; no automatic model promotion or competition submission\\",\\n  \\"budget\\": \\"Existing free Kaggle execution only; no hosted inference, paid infrastructure or external LLM calls\\"\\n}\\n", "protocols/purged-external-v1-source-amendment.json": "{\\n  \\"schema_version\\": 1,\\n  \\"registered_on\\": \\"2026-10-28\\",\\n  \\"parent_protocol\\": \\"purged-external-v1.json\\",\\n  \\"reason\\": \\"Attempt 1 failed closed in prepare_purged_external.py before any supervised fitting or candidate/model-performance inspection because a training label had undefined generic-family grouping.\\",\\n  \\"correction\\": \\"Quarantine every declared training identity with any label for which generic-family grouping is undefined, removing all rows sharing that identity. Report aggregate counts of undefined labels and affected identities. Continue to require zero retained selected external families and zero standard/metric/content overlap before fitting.\\",\\n  \\"unchanged_design\\": \\"Keep source snapshot, external family/metric selection, salts, 300/100/150 internal counts, at-most-150/at-least-30 external population, both generator budgets, all fitting settings, development recommendation gate, model seal, controls and 11 secondary contrasts unchanged. Do not select replacement external molecules using candidate or prediction outcomes.\\",\\n  \\"evidence_policy\\": \\"Preserve attempt 1 and its exact source hash, sanitized failure and absence of fitting/outcome inspection. Bind this amendment to the new executed source, purge report and publication record. The original-snapshot novelty failure remains preserved.\\"\\n}\\n", "reports/advanced_study.json": "{\\n  \\"schema_version\\": 1,\\n  \\"status\\": \\"COMPLETE_INTERNAL_FAMILY_HOLDOUT\\",\\n  \\"checked_at\\": \\"2026-10-26T01:11:23.040467+00:00\\",\\n  \\"elapsed_seconds\\": 7792.3,\\n  \\"protocol\\": {\\n    \\"schema_version\\": 1,\\n    \\"id\\": \\"casmi-advanced-internal-v1\\",\\n    \\"scope\\": \\"Newly partitioned internal chemical-family benchmark with fresh supervised weights; not an external dataset\\",\\n    \\"split_salt\\": \\"NEXUS-CASMI-ADVANCED-20261025-v1\\",\\n    \\"split_counts\\": {\\n      \\"train\\": 600,\\n      \\"dev\\": 150,\\n      \\"test\\": 250\\n    },\\n    \\"query_libraries\\": [\\n      \\"riken\\",\\n      \\"gnps\\",\\n      \\"massbank\\",\\n      \\"mona\\",\\n      \\"msdial\\",\\n      \\"spectraverse\\",\\n      \\"pluskal_ms2\\"\\n    ],\\n    \\"query_mass_range\\": [\\n      150.0,\\n      1170.0\\n    ],\\n    \\"max_spectra\\": 5,\\n    \\"max_training_spectra_per_structure\\": 128,\\n    \\"seed\\": 20261025,\\n    \\"fp_model\\": {\\n      \\"d_model\\": 128,\\n      \\"n_layers\\": 2,\\n      \\"d_ff\\": 512,\\n      \\"epochs\\": 6,\\n      \\"batch_size\\": 256,\\n      \\"lr\\": 0.0003\\n    },\\n    \\"ranker\\": {\\n      \\"seeds\\": [\\n        0,\\n        1,\\n        2\\n      ],\\n      \\"rounds\\": 400,\\n      \\"early_stopping\\": 40\\n    },\\n    \\"ablations\\": [\\n      \\"full\\",\\n      \\"no_d1\\",\\n      \\"no_d2\\",\\n      \\"no_d3\\",\\n      \\"no_d4\\",\\n      \\"no_gen_features\\",\\n      \\"no_np_score\\",\\n      \\"membership_only\\"\\n    ],\\n    \\"controls\\": [\\n      \\"none\\",\\n      \\"blank\\",\\n      \\"intensity_shuffle\\"\\n    ],\\n    \\"primary\\": \\"Unperturbed test C2 MRR@25: fresh full model minus D1+D2 baseline\\",\\n    \\"secondary\\": \\"Joint paired bootstrap over all prespecified ablation/control contrasts and C1/C2/C3\\",\\n    \\"bootstrap_draws\\": 4000,\\n    \\"slice_min_n\\": 20,\\n    \\"test_policy\\": \\"All rankers, feature lists and model hashes sealed before test feature generation. Test never used for training, stopping or hyperparameter selection.\\",\\n    \\"retrieval_policy\\": \\"The retrieval corpus may contain evaluation-family references according to the C1 task definition. Per-query C2/C3 exclude metric-equivalent truth references. Query-family disjointness concerns supervised fitting, not a reference-free discovery task.\\",\\n    \\"promotion_policy\\": \\"Report all outcomes, including failures and negative controls; no automatic model promotion or competition submission\\",\\n    \\"budget\\": \\"Existing free Kaggle execution only; no hosted inference, paid infrastructure or external LLM calls\\"\\n  },\\n  \\"protocol_sha256\\": \\"f2cbbfce99d532958f49b420f70103d537ec216ef68bb67012c1bf239789a46a\\",\\n  \\"partitions\\": {\\n    \\"train\\": {\\n      \\"n_molecules\\": 600,\\n      \\"identity_sha256\\": \\"1891e223685fcb69cae15b5ba637ded61a769addba2131af6b61bc18bba61731\\",\\n      \\"family_sha256\\": \\"fb20af82eaea0f7fed0c2259cf4dd53b3c950d0c5d3c5912f57cc19f5f05ed42\\",\\n      \\"metric_sha256\\": \\"e3026f89140d3bb84add656aeb0ad5aca61c3229be29c48fc19eb2563f29148f\\",\\n      \\"libraries\\": {\\n        \\"pluskal_ms2\\": 320,\\n        \\"gnps\\": 158,\\n        \\"spectraverse\\": 53,\\n        \\"riken\\": 21,\\n        \\"msdial\\": 17,\\n        \\"mona\\": 17,\\n        \\"massbank\\": 14\\n      }\\n    },\\n    \\"dev\\": {\\n      \\"n_molecules\\": 150,\\n      \\"identity_sha256\\": \\"65923389f6c0a098cc6515ca70570bd830ad4794bbf2400ace141fc41a7c9c71\\",\\n      \\"family_sha256\\": \\"a1d261fefd085d577e1279fd3cde951783a07c77b0e32a0d1a1f0cbbc42c6a01\\",\\n      \\"metric_sha256\\": \\"75ccf13c1545c188e2b4ec34224b4f7c89f3892c249ef8ef7d71f45bc3f01d73\\",\\n      \\"libraries\\": {\\n        \\"pluskal_ms2\\": 90,\\n        \\"gnps\\": 38,\\n        \\"spectraverse\\": 8,\\n        \\"mona\\": 5,\\n        \\"massbank\\": 5,\\n        \\"riken\\": 4\\n      }\\n    },\\n    \\"test\\": {\\n      \\"n_molecules\\": 250,\\n      \\"identity_sha256\\": \\"be29f22054bea6230c5aaf16647cd903b4071401442fc5a54a8a3efd4f909259\\",\\n      \\"family_sha256\\": \\"d6f8d6acc028752127210d5bd8bc83f43c85d44e6ffab2a3e9ddcd1292c82278\\",\\n      \\"metric_sha256\\": \\"632cc221852afc13d4d16275e07f2a5e1bc4b19a23191fa8ca6ec68cf6f7ee12\\",\\n      \\"libraries\\": {\\n        \\"pluskal_ms2\\": 141,\\n        \\"gnps\\": 56,\\n        \\"spectraverse\\": 27,\\n        \\"massbank\\": 11,\\n        \\"riken\\": 7,\\n        \\"mona\\": 6,\\n        \\"msdial\\": 2\\n      }\\n    },\\n    \\"cross_split_identity_overlaps\\": 0,\\n    \\"invalid_family_labels_excluded\\": 0,\\n    \\"historical_vin_family_rows_excluded\\": 18499,\\n    \\"eligible_representative_families\\": 26581\\n  },\\n  \\"training_seal\\": {\\n    \\"status\\": \\"ALL_MODELS_SEALED_BEFORE_TEST_FEATURES\\",\\n    \\"sealed_at\\": \\"2026-10-26T00:24:45.432928+00:00\\",\\n    \\"model_files\\": {\\n      \\"advanced_fp_fold0.pt\\": \\"44a00c1db403703b685ad9f3513cea51d4a3d27a9ab92c89cac90334d795f87f\\",\\n      \\"advanced_fp_fold1.pt\\": \\"085f8c8fcea2ca7fd9393cc53010d091c51eb338364aa96e505937497c087b41\\",\\n      \\"advanced_full_0.txt\\": \\"0b61db3b8b5e6da1b5c037d394aba18956320b59b746dddeecfc7e3bad651c76\\",\\n      \\"advanced_full_1.txt\\": \\"c9fa9807e206e27e9905f25310091be24422fb0c52a0c2c8f62295d42fe8fe7f\\",\\n      \\"advanced_full_2.txt\\": \\"22547b4f56064430140fec31f0ed6ce5b884c990a8d92abf6e91a95334e0cce6\\",\\n      \\"advanced_membership_only_0.txt\\": \\"b3ef37c1ed60cc26639fcf8b24f1d330edd5a3b93e27963ddc7c38b5f9c7f1f7\\",\\n      \\"advanced_membership_only_1.txt\\": \\"c2423456b86b8b76e18fd0d69d7386058220cf6cedc1c9e2d738b9a5815cf4a7\\",\\n      \\"advanced_membership_only_2.txt\\": \\"2c2a4484f9d9cee15c0008dda287808d771429c53c43d41671e5639bfb207447\\",\\n      \\"advanced_no_d1_0.txt\\": \\"a9ddf30fd4792f431cb52faffdc75e6ce409366b7898335ae48b81855ae9b330\\",\\n      \\"advanced_no_d1_1.txt\\": \\"1610aeae77c4e67ecc2ca667c8c492b06eee7f75a4b4eeba1f3535b9c162d2bc\\",\\n      \\"advanced_no_d1_2.txt\\": \\"eb6a96463839eea1961ca32a8d98ee5c911d19c8fabc4b69391871ae5bea414c\\",\\n      \\"advanced_no_d2_0.txt\\": \\"5ae1cb50b282588fe3efa3fac92e481c5a0ddae5dfd690bfa5f97be6f029ad8f\\",\\n      \\"advanced_no_d2_1.txt\\": \\"dd2667276efca84daba0bc4cab0c2da17ad61f6c674eb7b5a1c0f114b4ca8fc5\\",\\n      \\"advanced_no_d2_2.txt\\": \\"b3d83200f373ee98ce88f9d3c3fe2d01a90e0d4ca846faba431a082dc3a76ae3\\",\\n      \\"advanced_no_d3_0.txt\\": \\"fd06dac34d6b2b1346215b3060139f347a8291b8399ff5a02ccbc0a0982316a8\\",\\n      \\"advanced_no_d3_1.txt\\": \\"d6de69649be34aff8e3f9862b146c566c64b73822b9f16767ec9491d2f239d45\\",\\n      \\"advanced_no_d3_2.txt\\": \\"f765114802251807f98c87bbae0fd4c4803596850d585af86e3658746dfaaabf\\",\\n      \\"advanced_no_d4_0.txt\\": \\"f8a87e6930d234e60379ea9c70d665767286853de7b7630e941649cd49275172\\",\\n      \\"advanced_no_d4_1.txt\\": \\"98b805437a58f7142500a2116cb007213369769b1f30c05f30e3ae63420282a2\\",\\n      \\"advanced_no_d4_2.txt\\": \\"512b231fb42e6d0d445961f98ecd5666ac8f640f140e16938ae473c6b1da45a8\\",\\n      \\"advanced_no_gen_features_0.txt\\": \\"49d2b3378256c4a4b6f1e962a5f36695f1ca267022b82af2c18792c6ed721e75\\",\\n      \\"advanced_no_gen_features_1.txt\\": \\"b06a71443915bc2ebf09c5e4077d2320055fdd1afbbe51c2942336cba4b6cd02\\",\\n      \\"advanced_no_gen_features_2.txt\\": \\"c96b6e4deebe27166a625ed6e1c5aa9633a179fca01cc54038a20a899061e186\\",\\n      \\"advanced_no_np_score_0.txt\\": \\"752c25ac7c41b58668f9100fef8529ed74ebb84a9409dca7202c0c7b1af7217b\\",\\n      \\"advanced_no_np_score_1.txt\\": \\"8d08cad47d4ad136ddec5a6de9e64c28eea8d01451c58ee7ebaebc5243571ca6\\",\\n      \\"advanced_no_np_score_2.txt\\": \\"5fdf5b2feae501a519827f66d8fb56c9d81f5761f47a4747bb608316f114fec3\\"\\n    },\\n    \\"models\\": {\\n      \\"full\\": {\\n        \\"columns\\": [\\n          \\"d1_max\\",\\n          \\"d1_mean\\",\\n          \\"d1_nref\\",\\n          \\"an_p1\\",\\n          \\"an_p2\\",\\n          \\"an_p4\\",\\n          \\"an_sim\\",\\n          \\"an_tani\\",\\n          \\"an_dmass\\",\\n          \\"an_n_close\\",\\n          \\"an_self\\",\\n          \\"gen\\",\\n          \\"gen_sim\\",\\n          \\"gen_depth\\",\\n          \\"gen_sim_sum\\",\\n          \\"gen_n_seeds\\",\\n          \\"gen_n_paths\\",\\n          \\"gen_t1\\",\\n          \\"gen_t2\\",\\n          \\"np_score\\",\\n          \\"abs_ppm\\",\\n          \\"d1_max_rank\\",\\n          \\"d1_max_gap\\",\\n          \\"d1_max_z\\",\\n          \\"an_p4_rank\\",\\n          \\"an_p4_gap\\",\\n          \\"an_p4_z\\",\\n          \\"an_p2_rank\\",\\n          \\"an_p2_gap\\",\\n          \\"an_p2_z\\",\\n          \\"an_tani_rank\\",\\n          \\"an_tani_gap\\",\\n          \\"an_tani_z\\",\\n          \\"log_n_cand\\",\\n          \\"d3_ll\\",\\n          \\"d3_cos\\",\\n          \\"d3_ll_rank\\",\\n          \\"d3_ll_gap\\",\\n          \\"d3_ll_z\\",\\n          \\"d3_cos_rank\\",\\n          \\"d3_cos_gap\\",\\n          \\"d3_cos_z\\",\\n          \\"d4_int\\",\\n          \\"d4_int_max\\",\\n          \\"d4_pk\\",\\n          \\"d4_pk_max\\",\\n          \\"d4_int_rank\\",\\n          \\"d4_int_gap\\",\\n          \\"d4_int_z\\",\\n          \\"d4_pk_rank\\",\\n          \\"d4_pk_gap\\",\\n          \\"d4_pk_z\\"\\n        ],\\n        \\"best_iterations\\": [\\n          115,\\n          106,\\n          91\\n        ],\\n        \\"model_sha256\\": [\\n          \\"0b61db3b8b5e6da1b5c037d394aba18956320b59b746dddeecfc7e3bad651c76\\",\\n          \\"c9fa9807e206e27e9905f25310091be24422fb0c52a0c2c8f62295d42fe8fe7f\\",\\n          \\"22547b4f56064430140fec31f0ed6ce5b884c990a8d92abf6e91a95334e0cce6\\"\\n        ],\\n        \\"config\\": {\\n          \\"seeds\\": [\\n            0,\\n            1,\\n            2\\n          ],\\n          \\"params\\": {\\n            \\"objective\\": \\"lambdarank\\",\\n            \\"metric\\": \\"map\\",\\n            \\"eval_at\\": [\\n              25\\n            ],\\n            \\"lambdarank_truncation_level\\": 25,\\n            \\"learning_rate\\": 0.05,\\n            \\"num_leaves\\": 31,\\n            \\"min_data_in_leaf\\": 50,\\n            \\"feature_fraction\\": 0.8,\\n            \\"bagging_fraction\\": 0.8,\\n            \\"bagging_freq\\": 1,\\n            \\"lambda_l2\\": 1.0,\\n            \\"verbose\\": -1,\\n            \\"num_threads\\": 2,\\n            \\"deterministic\\": true,\\n            \\"force_col_wise\\": true\\n          },\\n          \\"num_boost_round\\": 400,\\n          \\"early_stopping\\": 40\\n        },\\n        \\"positive_training_groups\\": 1260,\\n        \\"zero_positive_groups_retained_in_test_denominator\\": true\\n      },\\n      \\"no_d1\\": {\\n        \\"columns\\": [\\n          \\"an_p1\\",\\n          \\"an_p2\\",\\n          \\"an_p4\\",\\n          \\"an_sim\\",\\n          \\"an_tani\\",\\n          \\"an_dmass\\",\\n          \\"an_n_close\\",\\n          \\"an_self\\",\\n          \\"gen\\",\\n          \\"gen_sim\\",\\n          \\"gen_depth\\",\\n          \\"gen_sim_sum\\",\\n          \\"gen_n_seeds\\",\\n          \\"gen_n_paths\\",\\n          \\"gen_t1\\",\\n          \\"gen_t2\\",\\n          \\"np_score\\",\\n          \\"abs_ppm\\",\\n          \\"an_p4_rank\\",\\n          \\"an_p4_gap\\",\\n          \\"an_p4_z\\",\\n          \\"an_p2_rank\\",\\n          \\"an_p2_gap\\",\\n          \\"an_p2_z\\",\\n          \\"an_tani_rank\\",\\n          \\"an_tani_gap\\",\\n          \\"an_tani_z\\",\\n          \\"log_n_cand\\",\\n          \\"d3_ll\\",\\n          \\"d3_cos\\",\\n          \\"d3_ll_rank\\",\\n          \\"d3_ll_gap\\",\\n          \\"d3_ll_z\\",\\n          \\"d3_cos_rank\\",\\n          \\"d3_cos_gap\\",\\n          \\"d3_cos_z\\",\\n          \\"d4_int\\",\\n          \\"d4_int_max\\",\\n          \\"d4_pk\\",\\n          \\"d4_pk_max\\",\\n          \\"d4_int_rank\\",\\n          \\"d4_int_gap\\",\\n          \\"d4_int_z\\",\\n          \\"d4_pk_rank\\",\\n          \\"d4_pk_gap\\",\\n          \\"d4_pk_z\\"\\n        ],\\n        \\"best_iterations\\": [\\n          114,\\n          104,\\n          115\\n        ],\\n        \\"model_sha256\\": [\\n          \\"a9ddf30fd4792f431cb52faffdc75e6ce409366b7898335ae48b81855ae9b330\\",\\n          \\"1610aeae77c4e67ecc2ca667c8c492b06eee7f75a4b4eeba1f3535b9c162d2bc\\",\\n          \\"eb6a96463839eea1961ca32a8d98ee5c911d19c8fabc4b69391871ae5bea414c\\"\\n        ],\\n        \\"config\\": {\\n          \\"seeds\\": [\\n            0,\\n            1,\\n            2\\n          ],\\n          \\"params\\": {\\n            \\"objective\\": \\"lambdarank\\",\\n            \\"metric\\": \\"map\\",\\n            \\"eval_at\\": [\\n              25\\n            ],\\n            \\"lambdarank_truncation_level\\": 25,\\n            \\"learning_rate\\": 0.05,\\n            \\"num_leaves\\": 31,\\n            \\"min_data_in_leaf\\": 50,\\n            \\"feature_fraction\\": 0.8,\\n            \\"bagging_fraction\\": 0.8,\\n            \\"bagging_freq\\": 1,\\n            \\"lambda_l2\\": 1.0,\\n            \\"verbose\\": -1,\\n            \\"num_threads\\": 2,\\n            \\"deterministic\\": true,\\n            \\"force_col_wise\\": true\\n          },\\n          \\"num_boost_round\\": 400,\\n          \\"early_stopping\\": 40\\n        },\\n        \\"positive_training_groups\\": 1260,\\n        \\"zero_positive_groups_retained_in_test_denominator\\": true\\n      },\\n      \\"no_d2\\": {\\n        \\"columns\\": [\\n          \\"d1_max\\",\\n          \\"d1_mean\\",\\n          \\"d1_nref\\",\\n          \\"gen\\",\\n          \\"gen_sim\\",\\n          \\"gen_depth\\",\\n          \\"gen_sim_sum\\",\\n          \\"gen_n_seeds\\",\\n          \\"gen_n_paths\\",\\n          \\"gen_t1\\",\\n          \\"gen_t2\\",\\n          \\"np_score\\",\\n          \\"abs_ppm\\",\\n          \\"d1_max_rank\\",\\n          \\"d1_max_gap\\",\\n          \\"d1_max_z\\",\\n          \\"log_n_cand\\",\\n          \\"d3_ll\\",\\n          \\"d3_cos\\",\\n          \\"d3_ll_rank\\",\\n          \\"d3_ll_gap\\",\\n          \\"d3_ll_z\\",\\n          \\"d3_cos_rank\\",\\n          \\"d3_cos_gap\\",\\n          \\"d3_cos_z\\",\\n          \\"d4_int\\",\\n          \\"d4_int_max\\",\\n          \\"d4_pk\\",\\n          \\"d4_pk_max\\",\\n          \\"d4_int_rank\\",\\n          \\"d4_int_gap\\",\\n          \\"d4_int_z\\",\\n          \\"d4_pk_rank\\",\\n          \\"d4_pk_gap\\",\\n          \\"d4_pk_z\\"\\n        ],\\n        \\"best_iterations\\": [\\n          130,\\n          182,\\n          133\\n        ],\\n        \\"model_sha256\\": [\\n          \\"5ae1cb50b282588fe3efa3fac92e481c5a0ddae5dfd690bfa5f97be6f029ad8f\\",\\n          \\"dd2667276efca84daba0bc4cab0c2da17ad61f6c674eb7b5a1c0f114b4ca8fc5\\",\\n          \\"b3d83200f373ee98ce88f9d3c3fe2d01a90e0d4ca846faba431a082dc3a76ae3\\"\\n        ],\\n        \\"config\\": {\\n          \\"seeds\\": [\\n            0,\\n            1,\\n            2\\n          ],\\n          \\"params\\": {\\n            \\"objective\\": \\"lambdarank\\",\\n            \\"metric\\": \\"map\\",\\n            \\"eval_at\\": [\\n              25\\n            ],\\n            \\"lambdarank_truncation_level\\": 25,\\n            \\"learning_rate\\": 0.05,\\n            \\"num_leaves\\": 31,\\n            \\"min_data_in_leaf\\": 50,\\n            \\"feature_fraction\\": 0.8,\\n            \\"bagging_fraction\\": 0.8,\\n            \\"bagging_freq\\": 1,\\n            \\"lambda_l2\\": 1.0,\\n            \\"verbose\\": -1,\\n            \\"num_threads\\": 2,\\n            \\"deterministic\\": true,\\n            \\"force_col_wise\\": true\\n          },\\n          \\"num_boost_round\\": 400,\\n          \\"early_stopping\\": 40\\n        },\\n        \\"positive_training_groups\\": 1260,\\n        \\"zero_positive_groups_retained_in_test_denominator\\": true\\n      },\\n      \\"no_d3\\": {\\n        \\"columns\\": [\\n          \\"d1_max\\",\\n          \\"d1_mean\\",\\n          \\"d1_nref\\",\\n          \\"an_p1\\",\\n          \\"an_p2\\",\\n          \\"an_p4\\",\\n          \\"an_sim\\",\\n          \\"an_tani\\",\\n          \\"an_dmass\\",\\n          \\"an_n_close\\",\\n          \\"an_self\\",\\n          \\"gen\\",\\n          \\"gen_sim\\",\\n          \\"gen_depth\\",\\n          \\"gen_sim_sum\\",\\n          \\"gen_n_seeds\\",\\n          \\"gen_n_paths\\",\\n          \\"gen_t1\\",\\n          \\"gen_t2\\",\\n          \\"np_score\\",\\n          \\"abs_ppm\\",\\n          \\"d1_max_rank\\",\\n          \\"d1_max_gap\\",\\n          \\"d1_max_z\\",\\n          \\"an_p4_rank\\",\\n          \\"an_p4_gap\\",\\n          \\"an_p4_z\\",\\n          \\"an_p2_rank\\",\\n          \\"an_p2_gap\\",\\n          \\"an_p2_z\\",\\n          \\"an_tani_rank\\",\\n          \\"an_tani_gap\\",\\n          \\"an_tani_z\\",\\n          \\"log_n_cand\\",\\n          \\"d4_int\\",\\n          \\"d4_int_max\\",\\n          \\"d4_pk\\",\\n          \\"d4_pk_max\\",\\n          \\"d4_int_rank\\",\\n          \\"d4_int_gap\\",\\n          \\"d4_int_z\\",\\n          \\"d4_pk_rank\\",\\n          \\"d4_pk_gap\\",\\n          \\"d4_pk_z\\"\\n        ],\\n        \\"best_iterations\\": [\\n          102,\\n          5,\\n          123\\n        ],\\n        \\"model_sha256\\": [\\n          \\"fd06dac34d6b2b1346215b3060139f347a8291b8399ff5a02ccbc0a0982316a8\\",\\n          \\"d6de69649be34aff8e3f9862b146c566c64b73822b9f16767ec9491d2f239d45\\",\\n          \\"f765114802251807f98c87bbae0fd4c4803596850d585af86e3658746dfaaabf\\"\\n        ],\\n        \\"config\\": {\\n          \\"seeds\\": [\\n            0,\\n            1,\\n            2\\n          ],\\n          \\"params\\": {\\n            \\"objective\\": \\"lambdarank\\",\\n            \\"metric\\": \\"map\\",\\n            \\"eval_at\\": [\\n              25\\n            ],\\n            \\"lambdarank_truncation_level\\": 25,\\n            \\"learning_rate\\": 0.05,\\n            \\"num_leaves\\": 31,\\n            \\"min_data_in_leaf\\": 50,\\n            \\"feature_fraction\\": 0.8,\\n            \\"bagging_fraction\\": 0.8,\\n            \\"bagging_freq\\": 1,\\n            \\"lambda_l2\\": 1.0,\\n            \\"verbose\\": -1,\\n            \\"num_threads\\": 2,\\n            \\"deterministic\\": true,\\n            \\"force_col_wise\\": true\\n          },\\n          \\"num_boost_round\\": 400,\\n          \\"early_stopping\\": 40\\n        },\\n        \\"positive_training_groups\\": 1260,\\n        \\"zero_positive_groups_retained_in_test_denominator\\": true\\n      },\\n      \\"no_d4\\": {\\n        \\"columns\\": [\\n          \\"d1_max\\",\\n          \\"d1_mean\\",\\n          \\"d1_nref\\",\\n          \\"an_p1\\",\\n          \\"an_p2\\",\\n          \\"an_p4\\",\\n          \\"an_sim\\",\\n          \\"an_tani\\",\\n          \\"an_dmass\\",\\n          \\"an_n_close\\",\\n          \\"an_self\\",\\n          \\"gen\\",\\n          \\"gen_sim\\",\\n          \\"gen_depth\\",\\n          \\"gen_sim_sum\\",\\n          \\"gen_n_seeds\\",\\n          \\"gen_n_paths\\",\\n          \\"gen_t1\\",\\n          \\"gen_t2\\",\\n          \\"np_score\\",\\n          \\"abs_ppm\\",\\n          \\"d1_max_rank\\",\\n          \\"d1_max_gap\\",\\n          \\"d1_max_z\\",\\n          \\"an_p4_rank\\",\\n          \\"an_p4_gap\\",\\n          \\"an_p4_z\\",\\n          \\"an_p2_rank\\",\\n          \\"an_p2_gap\\",\\n          \\"an_p2_z\\",\\n          \\"an_tani_rank\\",\\n          \\"an_tani_gap\\",\\n          \\"an_tani_z\\",\\n          \\"log_n_cand\\",\\n          \\"d3_ll\\",\\n          \\"d3_cos\\",\\n          \\"d3_ll_rank\\",\\n          \\"d3_ll_gap\\",\\n          \\"d3_ll_z\\",\\n          \\"d3_cos_rank\\",\\n          \\"d3_cos_gap\\",\\n          \\"d3_cos_z\\"\\n        ],\\n        \\"best_iterations\\": [\\n          44,\\n          108,\\n          62\\n        ],\\n        \\"model_sha256\\": [\\n          \\"f8a87e6930d234e60379ea9c70d665767286853de7b7630e941649cd49275172\\",\\n          \\"98b805437a58f7142500a2116cb007213369769b1f30c05f30e3ae63420282a2\\",\\n          \\"512b231fb42e6d0d445961f98ecd5666ac8f640f140e16938ae473c6b1da45a8\\"\\n        ],\\n        \\"config\\": {\\n          \\"seeds\\": [\\n            0,\\n            1,\\n            2\\n          ],\\n          \\"params\\": {\\n            \\"objective\\": \\"lambdarank\\",\\n            \\"metric\\": \\"map\\",\\n            \\"eval_at\\": [\\n              25\\n            ],\\n            \\"lambdarank_truncation_level\\": 25,\\n            \\"learning_rate\\": 0.05,\\n            \\"num_leaves\\": 31,\\n            \\"min_data_in_leaf\\": 50,\\n            \\"feature_fraction\\": 0.8,\\n            \\"bagging_fraction\\": 0.8,\\n            \\"bagging_freq\\": 1,\\n            \\"lambda_l2\\": 1.0,\\n            \\"verbose\\": -1,\\n            \\"num_threads\\": 2,\\n            \\"deterministic\\": true,\\n            \\"force_col_wise\\": true\\n          },\\n          \\"num_boost_round\\": 400,\\n          \\"early_stopping\\": 40\\n        },\\n        \\"positive_training_groups\\": 1260,\\n        \\"zero_positive_groups_retained_in_test_denominator\\": true\\n      },\\n      \\"no_gen_features\\": {\\n        \\"columns\\": [\\n          \\"d1_max\\",\\n          \\"d1_mean\\",\\n          \\"d1_nref\\",\\n          \\"an_p1\\",\\n          \\"an_p2\\",\\n          \\"an_p4\\",\\n          \\"an_sim\\",\\n          \\"an_tani\\",\\n          \\"an_dmass\\",\\n          \\"an_n_close\\",\\n          \\"an_self\\",\\n          \\"np_score\\",\\n          \\"abs_ppm\\",\\n          \\"d1_max_rank\\",\\n          \\"d1_max_gap\\",\\n          \\"d1_max_z\\",\\n          \\"an_p4_rank\\",\\n          \\"an_p4_gap\\",\\n          \\"an_p4_z\\",\\n          \\"an_p2_rank\\",\\n          \\"an_p2_gap\\",\\n          \\"an_p2_z\\",\\n          \\"an_tani_rank\\",\\n          \\"an_tani_gap\\",\\n          \\"an_tani_z\\",\\n          \\"log_n_cand\\",\\n          \\"d3_ll\\",\\n          \\"d3_cos\\",\\n          \\"d3_ll_rank\\",\\n          \\"d3_ll_gap\\",\\n          \\"d3_ll_z\\",\\n          \\"d3_cos_rank\\",\\n          \\"d3_cos_gap\\",\\n          \\"d3_cos_z\\",\\n          \\"d4_int\\",\\n          \\"d4_int_max\\",\\n          \\"d4_pk\\",\\n          \\"d4_pk_max\\",\\n          \\"d4_int_rank\\",\\n          \\"d4_int_gap\\",\\n          \\"d4_int_z\\",\\n          \\"d4_pk_rank\\",\\n          \\"d4_pk_gap\\",\\n          \\"d4_pk_z\\"\\n        ],\\n        \\"best_iterations\\": [\\n          94,\\n          170,\\n          125\\n        ],\\n        \\"model_sha256\\": [\\n          \\"49d2b3378256c4a4b6f1e962a5f36695f1ca267022b82af2c18792c6ed721e75\\",\\n          \\"b06a71443915bc2ebf09c5e4077d2320055fdd1afbbe51c2942336cba4b6cd02\\",\\n          \\"c96b6e4deebe27166a625ed6e1c5aa9633a179fca01cc54038a20a899061e186\\"\\n        ],\\n        \\"config\\": {\\n          \\"seeds\\": [\\n            0,\\n            1,\\n            2\\n          ],\\n          \\"params\\": {\\n            \\"objective\\": \\"lambdarank\\",\\n            \\"metric\\": \\"map\\",\\n            \\"eval_at\\": [\\n              25\\n            ],\\n            \\"lambdarank_truncation_level\\": 25,\\n            \\"learning_rate\\": 0.05,\\n            \\"num_leaves\\": 31,\\n            \\"min_data_in_leaf\\": 50,\\n            \\"feature_fraction\\": 0.8,\\n            \\"bagging_fraction\\": 0.8,\\n            \\"bagging_freq\\": 1,\\n            \\"lambda_l2\\": 1.0,\\n            \\"verbose\\": -1,\\n            \\"num_threads\\": 2,\\n            \\"deterministic\\": true,\\n            \\"force_col_wise\\": true\\n          },\\n          \\"num_boost_round\\": 400,\\n          \\"early_stopping\\": 40\\n        },\\n        \\"positive_training_groups\\": 1260,\\n        \\"zero_positive_groups_retained_in_test_denominator\\": true\\n      },\\n      \\"no_np_score\\": {\\n        \\"columns\\": [\\n          \\"d1_max\\",\\n          \\"d1_mean\\",\\n          \\"d1_nref\\",\\n          \\"an_p1\\",\\n          \\"an_p2\\",\\n          \\"an_p4\\",\\n          \\"an_sim\\",\\n          \\"an_tani\\",\\n          \\"an_dmass\\",\\n          \\"an_n_close\\",\\n          \\"an_self\\",\\n          \\"gen\\",\\n          \\"gen_sim\\",\\n          \\"gen_depth\\",\\n          \\"gen_sim_sum\\",\\n          \\"gen_n_seeds\\",\\n          \\"gen_n_paths\\",\\n          \\"gen_t1\\",\\n          \\"gen_t2\\",\\n          \\"abs_ppm\\",\\n          \\"d1_max_rank\\",\\n          \\"d1_max_gap\\",\\n          \\"d1_max_z\\",\\n          \\"an_p4_rank\\",\\n          \\"an_p4_gap\\",\\n          \\"an_p4_z\\",\\n          \\"an_p2_rank\\",\\n          \\"an_p2_gap\\",\\n          \\"an_p2_z\\",\\n          \\"an_tani_rank\\",\\n          \\"an_tani_gap\\",\\n          \\"an_tani_z\\",\\n          \\"log_n_cand\\",\\n          \\"d3_ll\\",\\n          \\"d3_cos\\",\\n          \\"d3_ll_rank\\",\\n          \\"d3_ll_gap\\",\\n          \\"d3_ll_z\\",\\n          \\"d3_cos_rank\\",\\n          \\"d3_cos_gap\\",\\n          \\"d3_cos_z\\",\\n          \\"d4_int\\",\\n          \\"d4_int_max\\",\\n          \\"d4_pk\\",\\n          \\"d4_pk_max\\",\\n          \\"d4_int_rank\\",\\n          \\"d4_int_gap\\",\\n          \\"d4_int_z\\",\\n          \\"d4_pk_rank\\",\\n          \\"d4_pk_gap\\",\\n          \\"d4_pk_z\\"\\n        ],\\n        \\"best_iterations\\": [\\n          110,\\n          106,\\n          80\\n        ],\\n        \\"model_sha256\\": [\\n          \\"752c25ac7c41b58668f9100fef8529ed74ebb84a9409dca7202c0c7b1af7217b\\",\\n          \\"8d08cad47d4ad136ddec5a6de9e64c28eea8d01451c58ee7ebaebc5243571ca6\\",\\n          \\"5fdf5b2feae501a519827f66d8fb56c9d81f5761f47a4747bb608316f114fec3\\"\\n        ],\\n        \\"config\\": {\\n          \\"seeds\\": [\\n            0,\\n            1,\\n            2\\n          ],\\n          \\"params\\": {\\n            \\"objective\\": \\"lambdarank\\",\\n            \\"metric\\": \\"map\\",\\n            \\"eval_at\\": [\\n              25\\n            ],\\n            \\"lambdarank_truncation_level\\": 25,\\n            \\"learning_rate\\": 0.05,\\n            \\"num_leaves\\": 31,\\n            \\"min_data_in_leaf\\": 50,\\n            \\"feature_fraction\\": 0.8,\\n            \\"bagging_fraction\\": 0.8,\\n            \\"bagging_freq\\": 1,\\n            \\"lambda_l2\\": 1.0,\\n            \\"verbose\\": -1,\\n            \\"num_threads\\": 2,\\n            \\"deterministic\\": true,\\n            \\"force_col_wise\\": true\\n          },\\n          \\"num_boost_round\\": 400,\\n          \\"early_stopping\\": 40\\n        },\\n        \\"positive_training_groups\\": 1260,\\n        \\"zero_positive_groups_retained_in_test_denominator\\": true\\n      },\\n      \\"membership_only\\": {\\n        \\"columns\\": [\\n          \\"prior_in_train\\",\\n          \\"prior_in_coconut\\",\\n          \\"prior_enveda180_only\\",\\n          \\"prior_log_spectra\\",\\n          \\"prior_np_likeness\\",\\n          \\"prior_has_sugar\\"\\n        ],\\n        \\"best_iterations\\": [\\n          18,\\n          12,\\n          19\\n        ],\\n        \\"model_sha256\\": [\\n          \\"b3ef37c1ed60cc26639fcf8b24f1d330edd5a3b93e27963ddc7c38b5f9c7f1f7\\",\\n          \\"c2423456b86b8b76e18fd0d69d7386058220cf6cedc1c9e2d738b9a5815cf4a7\\",\\n          \\"2c2a4484f9d9cee15c0008dda287808d771429c53c43d41671e5639bfb207447\\"\\n        ],\\n        \\"config\\": {\\n          \\"seeds\\": [\\n            0,\\n            1,\\n            2\\n          ],\\n          \\"params\\": {\\n            \\"objective\\": \\"lambdarank\\",\\n            \\"metric\\": \\"map\\",\\n            \\"eval_at\\": [\\n              25\\n            ],\\n            \\"lambdarank_truncation_level\\": 25,\\n            \\"learning_rate\\": 0.05,\\n            \\"num_leaves\\": 31,\\n            \\"min_data_in_leaf\\": 50,\\n            \\"feature_fraction\\": 0.8,\\n            \\"bagging_fraction\\": 0.8,\\n            \\"bagging_freq\\": 1,\\n            \\"lambda_l2\\": 1.0,\\n            \\"verbose\\": -1,\\n            \\"num_threads\\": 2,\\n            \\"deterministic\\": true,\\n            \\"force_col_wise\\": true\\n          },\\n          \\"num_boost_round\\": 400,\\n          \\"early_stopping\\": 40\\n        },\\n        \\"positive_training_groups\\": 1260,\\n        \\"zero_positive_groups_retained_in_test_denominator\\": true\\n      }\\n    },\\n    \\"fingerprint_training\\": [\\n      {\\n        \\"fold\\": 0,\\n        \\"n_spectra\\": 2996,\\n        \\"n_structures\\": 279,\\n        \\"training_rows_sha256\\": \\"89bd8b95a6979c97ca8ffa201cf9910a7d8b6d3073d6a25cd0749a0b0a2ec672\\",\\n        \\"training_identities_sha256\\": \\"e2d07a7b80b5a178254abc7bc7aa2e845df26a5e8894178ac121b8a08c12be67\\",\\n        \\"training_families_sha256\\": \\"b3d468872901e097363a680ec28719e3f81bba271f97b0969f176d6810be1b96\\",\\n        \\"dev_test_identities_used\\": 0,\\n        \\"epochs_fixed_before_execution\\": 6,\\n        \\"training_bce_by_epoch\\": [\\n          0.6554662227153142,\\n          0.5354613629934466,\\n          0.37260346132540734,\\n          0.22326410791584264,\\n          0.1362699523508469,\\n          0.10212109853651559\\n        ],\\n        \\"initialization\\": \\"fresh seeded random; no archived weights\\",\\n        \\"model_sha256\\": \\"44a00c1db403703b685ad9f3513cea51d4a3d27a9ab92c89cac90334d795f87f\\"\\n      },\\n      {\\n        \\"fold\\": 1,\\n        \\"n_spectra\\": 3087,\\n        \\"n_structures\\": 321,\\n        \\"training_rows_sha256\\": \\"1bcd6c856534606a41fbc60987018e3cad8a4d708408b396619740e9b266b132\\",\\n        \\"training_identities_sha256\\": \\"61ec8576bcb3772d5cc96b99cb1d825dbdb675357aee0982a6c912bdfeedd44f\\",\\n        \\"training_families_sha256\\": \\"b89d6c2d036b259792cd3bcaf0ec3b433077096af296c85288c45406c4824b0d\\",\\n        \\"dev_test_identities_used\\": 0,\\n        \\"epochs_fixed_before_execution\\": 6,\\n        \\"training_bce_by_epoch\\": [\\n          0.6545156509813692,\\n          0.5072951461259393,\\n          0.3212682252510257,\\n          0.1817651104446287,\\n          0.11923359916371978,\\n          0.09921987550975746\\n        ],\\n        \\"initialization\\": \\"fresh seeded random; no archived weights\\",\\n        \\"model_sha256\\": \\"085f8c8fcea2ca7fd9393cc53010d091c51eb338364aa96e505937497c087b41\\"\\n      }\\n    ],\\n    \\"query_input_sha256\\": {\\n      \\"train\\": \\"bf014858bef931e023a5eef3322ddc4335b19b6a3dd6fcd7e525c8b411ea62c0\\",\\n      \\"dev\\": \\"50e1fbb29a392d64aa47106cdc4859f80cab5ba6f05ea4bbad607771219f4b37\\"\\n    },\\n    \\"test_identities_sha256\\": \\"be29f22054bea6230c5aaf16647cd903b4071401442fc5a54a8a3efd4f909259\\",\\n    \\"protocol_sha256\\": \\"f2cbbfce99d532958f49b420f70103d537ec216ef68bb67012c1bf239789a46a\\"\\n  },\\n  \\"training_seal_sha256\\": \\"db8ff659c18a06b0cdb756536e618a1d64bbd640e21f3b15b620de5806262466\\",\\n  \\"source_manifest\\": {\\n    \\"schema\\": \\"casmi-local-artifact-manifest-v1\\",\\n    \\"method\\": \\"streaming SHA-256 per local input file\\",\\n    \\"files\\": [\\n      {\\n        \\"path\\": \\"pool_advanced_internal.parquet\\",\\n        \\"bytes\\": 48469329,\\n        \\"sha256\\": \\"940d904c1c4ccd9fbb4f4e7dd07e1697c1d8271d76ef61ffb50700e7b4d28438\\"\\n      },\\n      {\\n        \\"path\\": \\"pool_fp2048_advanced_internal.npy\\",\\n        \\"bytes\\": 274397056,\\n        \\"sha256\\": \\"4104b0c963c90f18451c48185436406c394360b3e576af3f4bd05627f91ce8cb\\"\\n      },\\n      {\\n        \\"path\\": \\"pool_fp_keys_advanced_internal.parquet\\",\\n        \\"bytes\\": 16979467,\\n        \\"sha256\\": \\"2a7cfabbb2d6b51c1b173661f66fb5f9f8bf10cdb33c6a6099d2cae7fa46c4a3\\"\\n      },\\n      {\\n        \\"path\\": \\"pool_source_quality_advanced_internal.json\\",\\n        \\"bytes\\": 992,\\n        \\"sha256\\": \\"30ef62c64455afdfc9bdb7612c6dbdd69a45f1581eefad1bd0d2621517649837\\"\\n      },\\n      {\\n        \\"path\\": \\"store/it.f32\\",\\n        \\"bytes\\": 436098548,\\n        \\"sha256\\": \\"f1a1bbe0c143761ff06da6d7e028fbb7b50034ef2cada82fb119b3cae0e90187\\"\\n      },\\n      {\\n        \\"path\\": \\"store/meta.parquet\\",\\n        \\"bytes\\": 48766764,\\n        \\"sha256\\": \\"7185e15986520d4b57a8309fc0f962f911783800b2660f3d627b3fbb87265453\\"\\n      },\\n      {\\n        \\"path\\": \\"store/mz.f32\\",\\n        \\"bytes\\": 436098548,\\n        \\"sha256\\": \\"859c76e6ef623c2b03dad0ebf9c171e719d6fc0bc96b6e26ed79257142406291\\"\\n      },\\n      {\\n        \\"path\\": \\"store/offsets.i64\\",\\n        \\"bytes\\": 20316872,\\n        \\"sha256\\": \\"91b67e92f84e85fe40b800abcb4b4dacfdee498ba0e85f29cf6bf143a2f26935\\"\\n      },\\n      {\\n        \\"path\\": \\"structures.parquet\\",\\n        \\"bytes\\": 12571432,\\n        \\"sha256\\": \\"22dc7592f604ddd608568319254f88bb21d2a8a8c42e858d2148919b5f9f4aec\\"\\n      }\\n    ],\\n    \\"files_digest_sha256\\": \\"05cb68622f288d5af083af032da21097593bb15c733f3de399590894387dc078\\",\\n    \\"source_assertion\\": \\"local byte identity only; no independent Kaggle/BigQuery validation\\"\\n  },\\n  \\"candidate_source_quality\\": {\\n    \\"status\\": \\"SOURCE_ROWS_RECONCILED\\",\\n    \\"input_rows\\": 1182943,\\n    \\"complete_rows\\": 1179052,\\n    \\"incomplete_rows_excluded\\": 0,\\n    \\"distinct_pool_keys\\": 1071876,\\n    \\"shared_pool_keys\\": 93716,\\n    \\"mass_conflict_groups\\": 0,\\n    \\"mass_conflict_policy\\": \\"quarantine_entire_identity\\",\\n    \\"quarantined_mass_conflict_groups\\": 1710,\\n    \\"quarantined_source_rows\\": 3891,\\n    \\"max_source_mass_spread_da\\": 9.05649058751004,\\n    \\"mass_threshold_da\\": 0.01,\\n    \\"mass_threshold_ppm\\": 20.0,\\n    \\"interpretation\\": \\"Pre-aggregation identity consistency only; not RDKit verification\\",\\n    \\"structure_quarantine\\": {\\n      \\"policy\\": \\"quarantine_invalid_selected_candidate_identity\\",\\n      \\"input_candidates\\": 1071876,\\n      \\"retained_candidates\\": 1071863,\\n      \\"quarantined_candidates\\": 13,\\n      \\"reasons\\": {\\n        \\"unparseable\\": 13,\\n        \\"no_heavy_atoms\\": 0,\\n        \\"zero_morgan\\": 0,\\n        \\"fingerprint_error\\": 0\\n      },\\n      \\"scope\\": \\"Parseable, heavy-atom-containing, nonzero Morgan structures; not provider identity authentication\\"\\n    }\\n  },\\n  \\"fingerprint_provenance\\": {\\n    \\"status\\": \\"POOL_FINGERPRINTS_VERIFIED\\",\\n    \\"schema\\": \\"casmi-morgan-fingerprint-provenance-v1\\",\\n    \\"n_structures\\": 1071863,\\n    \\"pool_rows_sha256\\": \\"4c346290e112f2a92a4ae62238d31df491e8259e4a0121ef398138325944d9d8\\",\\n    \\"fingerprint_bytes_sha256\\": \\"b0bba8ed06e19690cb1c6093c225bb4fa4e9bd594132628493dd7dd670d18c32\\"\\n  },\\n  \\"analog_index_provenance\\": {\\n    \\"schema\\": \\"casmi-d2-fresh-index-v1\\",\\n    \\"status\\": \\"REBUILT_WITHOUT_CACHE\\",\\n    \\"index_sha256\\": \\"6700236d619238ec893c803690b2e240658b835b37bffe64cb17b15c5b7f421f\\",\\n    \\"n_references\\": 1062995,\\n    \\"n_nonzero\\": 34076549,\\n    \\"reference_policy\\": \\"legacy\\",\\n    \\"cache_used\\": false\\n  },\\n  \\"search_memo\\": {\\n    \\"scope\\": \\"per-query full-cleaned-content plus exact exclusion set; no index file cache\\",\\n    \\"hits\\": 2527,\\n    \\"misses\\": 1973\\n  },\\n  \\"engine_config\\": {\\n    \\"tol_ppm\\": 10.0,\\n    \\"mass_offset_ppm\\": 0.0,\\n    \\"top_n\\": 25,\\n    \\"library\\": {\\n      \\"tol_da\\": 0.02,\\n      \\"max_ppm\\": 20.0,\\n      \\"max_refs_per_candidate\\": 40,\\n      \\"clean\\": {\\n        \\"max_above_precursor\\": 2.0,\\n        \\"min_rel_intensity\\": 0.01,\\n        \\"window_da\\": 50,\\n        \\"top_k_per_window\\": 6,\\n        \\"max_peaks\\": 64,\\n        \\"merge_tol_da\\": 0.005\\n      }\\n    },\\n    \\"analog\\": {\\n      \\"max_shift_da\\": 200.0,\\n      \\"prefilter_top\\": 200,\\n      \\"keep_structures\\": 100,\\n      \\"per_structure_adduct\\": 3,\\n      \\"reference_policy\\": \\"legacy\\",\\n      \\"tol_da\\": 0.02,\\n      \\"powers\\": [\\n        1.0,\\n        2.0,\\n        4.0\\n      ],\\n      \\"clean\\": {\\n        \\"max_above_precursor\\": 2.0,\\n        \\"min_rel_intensity\\": 0.01,\\n        \\"window_da\\": 50,\\n        \\"top_k_per_window\\": 6,\\n        \\"max_peaks\\": 64,\\n        \\"merge_tol_da\\": 0.005\\n      }\\n    },\\n    \\"generate\\": true,\\n    \\"gen_max_analogs\\": 15,\\n    \\"gen_cap\\": 300,\\n    \\"gen_consensus\\": true,\\n    \\"np_score\\": true\\n  },\\n  \\"models\\": {\\n    \\"mass_only\\": {\\n      \\"none\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.3863205196997107,\\n          \\"ci_low\\": 0.33907778383119846,\\n          \\"ci_high\\": 0.4355216073079528,\\n          \\"top1\\": 0.264,\\n          \\"top10\\": 0.64,\\n          \\"recall_at_25\\": 0.816,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.3863205196997107,\\n          \\"ci_low\\": 0.33907778383119846,\\n          \\"ci_high\\": 0.4355216073079528,\\n          \\"top1\\": 0.264,\\n          \\"top10\\": 0.64,\\n          \\"recall_at_25\\": 0.816,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.0,\\n          \\"ci_low\\": 0.0,\\n          \\"ci_high\\": 0.0,\\n          \\"top1\\": 0.0,\\n          \\"top10\\": 0.0,\\n          \\"recall_at_25\\": 0.0,\\n          \\"n\\": 250\\n        }\\n      },\\n      \\"blank\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.3863205196997107,\\n          \\"ci_low\\": 0.33907778383119846,\\n          \\"ci_high\\": 0.4355216073079528,\\n          \\"top1\\": 0.264,\\n          \\"top10\\": 0.64,\\n          \\"recall_at_25\\": 0.816,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.3863205196997107,\\n          \\"ci_low\\": 0.33907778383119846,\\n          \\"ci_high\\": 0.4355216073079528,\\n          \\"top1\\": 0.264,\\n          \\"top10\\": 0.64,\\n          \\"recall_at_25\\": 0.816,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.0,\\n          \\"ci_low\\": 0.0,\\n          \\"ci_high\\": 0.0,\\n          \\"top1\\": 0.0,\\n          \\"top10\\": 0.0,\\n          \\"recall_at_25\\": 0.0,\\n          \\"n\\": 250\\n        }\\n      },\\n      \\"intensity_shuffle\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.3863205196997107,\\n          \\"ci_low\\": 0.33907778383119846,\\n          \\"ci_high\\": 0.4355216073079528,\\n          \\"top1\\": 0.264,\\n          \\"top10\\": 0.64,\\n          \\"recall_at_25\\": 0.816,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.3863205196997107,\\n          \\"ci_low\\": 0.33907778383119846,\\n          \\"ci_high\\": 0.4355216073079528,\\n          \\"top1\\": 0.264,\\n          \\"top10\\": 0.64,\\n          \\"recall_at_25\\": 0.816,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.0,\\n          \\"ci_low\\": 0.0,\\n          \\"ci_high\\": 0.0,\\n          \\"top1\\": 0.0,\\n          \\"top10\\": 0.0,\\n          \\"recall_at_25\\": 0.0,\\n          \\"n\\": 250\\n        }\\n      }\\n    },\\n    \\"d1\\": {\\n      \\"none\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.43048938446322077,\\n          \\"ci_low\\": 0.3751673012351388,\\n          \\"ci_high\\": 0.48607883245881706,\\n          \\"top1\\": 0.364,\\n          \\"top10\\": 0.6,\\n          \\"recall_at_25\\": 0.704,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.17405454111599353,\\n          \\"ci_low\\": 0.13893088522735697,\\n          \\"ci_high\\": 0.2103140821233664,\\n          \\"top1\\": 0.084,\\n          \\"top10\\": 0.396,\\n          \\"recall_at_25\\": 0.552,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.0,\\n          \\"ci_low\\": 0.0,\\n          \\"ci_high\\": 0.0,\\n          \\"top1\\": 0.0,\\n          \\"top10\\": 0.0,\\n          \\"recall_at_25\\": 0.0,\\n          \\"n\\": 250\\n        }\\n      },\\n      \\"blank\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.3863205196997107,\\n          \\"ci_low\\": 0.33907778383119846,\\n          \\"ci_high\\": 0.4355216073079528,\\n          \\"top1\\": 0.264,\\n          \\"top10\\": 0.64,\\n          \\"recall_at_25\\": 0.816,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.3863205196997107,\\n          \\"ci_low\\": 0.33907778383119846,\\n          \\"ci_high\\": 0.4355216073079528,\\n          \\"top1\\": 0.264,\\n          \\"top10\\": 0.64,\\n          \\"recall_at_25\\": 0.816,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.0,\\n          \\"ci_low\\": 0.0,\\n          \\"ci_high\\": 0.0,\\n          \\"top1\\": 0.0,\\n          \\"top10\\": 0.0,\\n          \\"recall_at_25\\": 0.0,\\n          \\"n\\": 250\\n        }\\n      },\\n      \\"intensity_shuffle\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.39505991697418813,\\n          \\"ci_low\\": 0.34230143951132697,\\n          \\"ci_high\\": 0.44889526889940123,\\n          \\"top1\\": 0.32,\\n          \\"top10\\": 0.584,\\n          \\"recall_at_25\\": 0.696,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.17835428926856037,\\n          \\"ci_low\\": 0.1424987084604086,\\n          \\"ci_high\\": 0.21587626179929764,\\n          \\"top1\\": 0.088,\\n          \\"top10\\": 0.4,\\n          \\"recall_at_25\\": 0.552,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.0,\\n          \\"ci_low\\": 0.0,\\n          \\"ci_high\\": 0.0,\\n          \\"top1\\": 0.0,\\n          \\"top10\\": 0.0,\\n          \\"recall_at_25\\": 0.0,\\n          \\"n\\": 250\\n        }\\n      }\\n    },\\n    \\"d1_d2\\": {\\n      \\"none\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.46082545284780574,\\n          \\"ci_low\\": 0.40682865508021393,\\n          \\"ci_high\\": 0.5143597478991596,\\n          \\"top1\\": 0.376,\\n          \\"top10\\": 0.656,\\n          \\"recall_at_25\\": 0.752,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.20658579723338238,\\n          \\"ci_low\\": 0.16924229617664757,\\n          \\"ci_high\\": 0.2446944949403881,\\n          \\"top1\\": 0.1,\\n          \\"top10\\": 0.46,\\n          \\"recall_at_25\\": 0.6,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.0,\\n          \\"ci_low\\": 0.0,\\n          \\"ci_high\\": 0.0,\\n          \\"top1\\": 0.0,\\n          \\"top10\\": 0.0,\\n          \\"recall_at_25\\": 0.0,\\n          \\"n\\": 250\\n        }\\n      },\\n      \\"blank\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.3863205196997107,\\n          \\"ci_low\\": 0.33907778383119846,\\n          \\"ci_high\\": 0.4355216073079528,\\n          \\"top1\\": 0.264,\\n          \\"top10\\": 0.64,\\n          \\"recall_at_25\\": 0.816,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.3863205196997107,\\n          \\"ci_low\\": 0.33907778383119846,\\n          \\"ci_high\\": 0.4355216073079528,\\n          \\"top1\\": 0.264,\\n          \\"top10\\": 0.64,\\n          \\"recall_at_25\\": 0.816,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.0,\\n          \\"ci_low\\": 0.0,\\n          \\"ci_high\\": 0.0,\\n          \\"top1\\": 0.0,\\n          \\"top10\\": 0.0,\\n          \\"recall_at_25\\": 0.0,\\n          \\"n\\": 250\\n        }\\n      },\\n      \\"intensity_shuffle\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.36591843360809034,\\n          \\"ci_low\\": 0.3140513982741769,\\n          \\"ci_high\\": 0.4187098691521421,\\n          \\"top1\\": 0.288,\\n          \\"top10\\": 0.556,\\n          \\"recall_at_25\\": 0.68,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.12918857901823577,\\n          \\"ci_low\\": 0.10112551458178232,\\n          \\"ci_high\\": 0.15846500893493745,\\n          \\"top1\\": 0.044,\\n          \\"top10\\": 0.356,\\n          \\"recall_at_25\\": 0.524,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.0,\\n          \\"ci_low\\": 0.0,\\n          \\"ci_high\\": 0.0,\\n          \\"top1\\": 0.0,\\n          \\"top10\\": 0.0,\\n          \\"recall_at_25\\": 0.0,\\n          \\"n\\": 250\\n        }\\n      }\\n    },\\n    \\"full\\": {\\n      \\"none\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.8665075884899415,\\n          \\"ci_low\\": 0.8288914298446651,\\n          \\"ci_high\\": 0.9024854275952806,\\n          \\"top1\\": 0.824,\\n          \\"top10\\": 0.948,\\n          \\"recall_at_25\\": 0.976,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.8388353662677193,\\n          \\"ci_low\\": 0.7970543943638062,\\n          \\"ci_high\\": 0.8782056947627536,\\n          \\"top1\\": 0.8,\\n          \\"top10\\": 0.928,\\n          \\"recall_at_25\\": 0.964,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.037349759419324634,\\n          \\"ci_low\\": 0.018833808751634835,\\n          \\"ci_high\\": 0.05876470780427302,\\n          \\"top1\\": 0.024,\\n          \\"top10\\": 0.064,\\n          \\"recall_at_25\\": 0.096,\\n          \\"n\\": 250\\n        }\\n      },\\n      \\"blank\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.2608974928977635,\\n          \\"ci_low\\": 0.21876093301465666,\\n          \\"ci_high\\": 0.3041455393628447,\\n          \\"top1\\": 0.156,\\n          \\"top10\\": 0.528,\\n          \\"recall_at_25\\": 0.728,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.32697745077524587,\\n          \\"ci_low\\": 0.28160935544086046,\\n          \\"ci_high\\": 0.3718827113284705,\\n          \\"top1\\": 0.212,\\n          \\"top10\\": 0.612,\\n          \\"recall_at_25\\": 0.808,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.0,\\n          \\"ci_low\\": 0.0,\\n          \\"ci_high\\": 0.0,\\n          \\"top1\\": 0.0,\\n          \\"top10\\": 0.0,\\n          \\"recall_at_25\\": 0.0,\\n          \\"n\\": 250\\n        }\\n      },\\n      \\"intensity_shuffle\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.7251134326457855,\\n          \\"ci_low\\": 0.6755973264154147,\\n          \\"ci_high\\": 0.7727542659791189,\\n          \\"top1\\": 0.64,\\n          \\"top10\\": 0.888,\\n          \\"recall_at_25\\": 0.924,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.7452362229557882,\\n          \\"ci_low\\": 0.6974693628834934,\\n          \\"ci_high\\": 0.7917301702584855,\\n          \\"top1\\": 0.668,\\n          \\"top10\\": 0.9,\\n          \\"recall_at_25\\": 0.944,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.020813797313797313,\\n          \\"ci_low\\": 0.00770874542124542,\\n          \\"ci_high\\": 0.037080815018315026,\\n          \\"top1\\": 0.012,\\n          \\"top10\\": 0.036,\\n          \\"recall_at_25\\": 0.056,\\n          \\"n\\": 250\\n        }\\n      }\\n    },\\n    \\"no_d1\\": {\\n      \\"none\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.819452798663325,\\n          \\"ci_low\\": 0.7795697034252298,\\n          \\"ci_high\\": 0.8586806307435254,\\n          \\"top1\\": 0.744,\\n          \\"top10\\": 0.952,\\n          \\"recall_at_25\\": 0.968,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.7659121817922733,\\n          \\"ci_low\\": 0.7212131773273763,\\n          \\"ci_high\\": 0.808532656278377,\\n          \\"top1\\": 0.68,\\n          \\"top10\\": 0.92,\\n          \\"recall_at_25\\": 0.956,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.03149105852867463,\\n          \\"ci_low\\": 0.015820924922600618,\\n          \\"ci_high\\": 0.05079624760430489,\\n          \\"top1\\": 0.016,\\n          \\"top10\\": 0.068,\\n          \\"recall_at_25\\": 0.092,\\n          \\"n\\": 250\\n        }\\n      },\\n      \\"blank\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.26846784565025916,\\n          \\"ci_low\\": 0.22346512099635907,\\n          \\"ci_high\\": 0.3122387536056733,\\n          \\"top1\\": 0.172,\\n          \\"top10\\": 0.544,\\n          \\"recall_at_25\\": 0.7,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.26846784565025916,\\n          \\"ci_low\\": 0.22346512099635907,\\n          \\"ci_high\\": 0.3122387536056733,\\n          \\"top1\\": 0.172,\\n          \\"top10\\": 0.544,\\n          \\"recall_at_25\\": 0.7,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.0,\\n          \\"ci_low\\": 0.0,\\n          \\"ci_high\\": 0.0,\\n          \\"top1\\": 0.0,\\n          \\"top10\\": 0.0,\\n          \\"recall_at_25\\": 0.0,\\n          \\"n\\": 250\\n        }\\n      },\\n      \\"intensity_shuffle\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.6743903406475219,\\n          \\"ci_low\\": 0.6237921750041063,\\n          \\"ci_high\\": 0.7234229928394283,\\n          \\"top1\\": 0.576,\\n          \\"top10\\": 0.872,\\n          \\"recall_at_25\\": 0.936,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.6517494442889785,\\n          \\"ci_low\\": 0.6016757070359121,\\n          \\"ci_high\\": 0.7010975059413785,\\n          \\"top1\\": 0.552,\\n          \\"top10\\": 0.852,\\n          \\"recall_at_25\\": 0.924,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.01600631386157702,\\n          \\"ci_low\\": 0.006393060776942355,\\n          \\"ci_high\\": 0.02796523721161879,\\n          \\"top1\\": 0.004,\\n          \\"top10\\": 0.036,\\n          \\"recall_at_25\\": 0.056,\\n          \\"n\\": 250\\n        }\\n      }\\n    },\\n    \\"no_d2\\": {\\n      \\"none\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.8099140098463629,\\n          \\"ci_low\\": 0.7676652218614719,\\n          \\"ci_high\\": 0.8517407273363892,\\n          \\"top1\\": 0.748,\\n          \\"top10\\": 0.916,\\n          \\"recall_at_25\\": 0.956,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.7512181260200441,\\n          \\"ci_low\\": 0.7058233863184982,\\n          \\"ci_high\\": 0.7966296422620431,\\n          \\"top1\\": 0.676,\\n          \\"top10\\": 0.88,\\n          \\"recall_at_25\\": 0.948,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.03442429196190806,\\n          \\"ci_low\\": 0.017139489362546636,\\n          \\"ci_high\\": 0.055363949646844386,\\n          \\"top1\\": 0.02,\\n          \\"top10\\": 0.056,\\n          \\"recall_at_25\\": 0.084,\\n          \\"n\\": 250\\n        }\\n      },\\n      \\"blank\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.3076155251646246,\\n          \\"ci_low\\": 0.26098273738475297,\\n          \\"ci_high\\": 0.35486353135999665,\\n          \\"top1\\": 0.208,\\n          \\"top10\\": 0.564,\\n          \\"recall_at_25\\": 0.748,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.35620859227295487,\\n          \\"ci_low\\": 0.3069547620873652,\\n          \\"ci_high\\": 0.4049219390156515,\\n          \\"top1\\": 0.248,\\n          \\"top10\\": 0.636,\\n          \\"recall_at_25\\": 0.804,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.0,\\n          \\"ci_low\\": 0.0,\\n          \\"ci_high\\": 0.0,\\n          \\"top1\\": 0.0,\\n          \\"top10\\": 0.0,\\n          \\"recall_at_25\\": 0.0,\\n          \\"n\\": 250\\n        }\\n      },\\n      \\"intensity_shuffle\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.6743831484769848,\\n          \\"ci_low\\": 0.6254633951243415,\\n          \\"ci_high\\": 0.7233581186205099,\\n          \\"top1\\": 0.588,\\n          \\"top10\\": 0.86,\\n          \\"recall_at_25\\": 0.94,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.6873774017116727,\\n          \\"ci_low\\": 0.6386360264039739,\\n          \\"ci_high\\": 0.7363058135889402,\\n          \\"top1\\": 0.608,\\n          \\"top10\\": 0.872,\\n          \\"recall_at_25\\": 0.948,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.016262210012210013,\\n          \\"ci_low\\": 0.005484378815628814,\\n          \\"ci_high\\": 0.029622817460317467,\\n          \\"top1\\": 0.008,\\n          \\"top10\\": 0.032,\\n          \\"recall_at_25\\": 0.056,\\n          \\"n\\": 250\\n        }\\n      }\\n    },\\n    \\"no_d3\\": {\\n      \\"none\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.8616981684981685,\\n          \\"ci_low\\": 0.8235713431013432,\\n          \\"ci_high\\": 0.8978717796092797,\\n          \\"top1\\": 0.82,\\n          \\"top10\\": 0.944,\\n          \\"recall_at_25\\": 0.972,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.8261942723942722,\\n          \\"ci_low\\": 0.7840456035631035,\\n          \\"ci_high\\": 0.866051943056943,\\n          \\"top1\\": 0.784,\\n          \\"top10\\": 0.92,\\n          \\"recall_at_25\\": 0.964,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.03720347985347985,\\n          \\"ci_low\\": 0.018919537545787547,\\n          \\"ci_high\\": 0.0585267673992674,\\n          \\"top1\\": 0.024,\\n          \\"top10\\": 0.064,\\n          \\"recall_at_25\\": 0.092,\\n          \\"n\\": 250\\n        }\\n      },\\n      \\"blank\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.3764935904198403,\\n          \\"ci_low\\": 0.3256076544704155,\\n          \\"ci_high\\": 0.42801763347026356,\\n          \\"top1\\": 0.28,\\n          \\"top10\\": 0.616,\\n          \\"recall_at_25\\": 0.78,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.4937681525896965,\\n          \\"ci_low\\": 0.4395891509913562,\\n          \\"ci_high\\": 0.5451375303922944,\\n          \\"top1\\": 0.388,\\n          \\"top10\\": 0.74,\\n          \\"recall_at_25\\": 0.856,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.0,\\n          \\"ci_low\\": 0.0,\\n          \\"ci_high\\": 0.0,\\n          \\"top1\\": 0.0,\\n          \\"top10\\": 0.0,\\n          \\"recall_at_25\\": 0.0,\\n          \\"n\\": 250\\n        }\\n      },\\n      \\"intensity_shuffle\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.7245122615370899,\\n          \\"ci_low\\": 0.6746688267354943,\\n          \\"ci_high\\": 0.7717417874186581,\\n          \\"top1\\": 0.64,\\n          \\"top10\\": 0.888,\\n          \\"recall_at_25\\": 0.936,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.7420578366078366,\\n          \\"ci_low\\": 0.6944675213675213,\\n          \\"ci_high\\": 0.7882578968253967,\\n          \\"top1\\": 0.66,\\n          \\"top10\\": 0.912,\\n          \\"recall_at_25\\": 0.948,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.017813697413697413,\\n          \\"ci_low\\": 0.0066138406038406036,\\n          \\"ci_high\\": 0.032211865911865915,\\n          \\"top1\\": 0.008,\\n          \\"top10\\": 0.032,\\n          \\"recall_at_25\\": 0.056,\\n          \\"n\\": 250\\n        }\\n      }\\n    },\\n    \\"no_d4\\": {\\n      \\"none\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.7952300382048666,\\n          \\"ci_low\\": 0.7515441229968862,\\n          \\"ci_high\\": 0.836689655215113,\\n          \\"top1\\": 0.72,\\n          \\"top10\\": 0.92,\\n          \\"recall_at_25\\": 0.956,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.7398885519133803,\\n          \\"ci_low\\": 0.6932745475389726,\\n          \\"ci_high\\": 0.7849814884997407,\\n          \\"top1\\": 0.652,\\n          \\"top10\\": 0.912,\\n          \\"recall_at_25\\": 0.944,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.030624675324675322,\\n          \\"ci_low\\": 0.015034199134199134,\\n          \\"ci_high\\": 0.048335974025974025,\\n          \\"top1\\": 0.012,\\n          \\"top10\\": 0.064,\\n          \\"recall_at_25\\": 0.088,\\n          \\"n\\": 250\\n        }\\n      },\\n      \\"blank\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.31511638973610706,\\n          \\"ci_low\\": 0.2661305972075553,\\n          \\"ci_high\\": 0.36297916786679474,\\n          \\"top1\\": 0.216,\\n          \\"top10\\": 0.556,\\n          \\"recall_at_25\\": 0.732,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.41674678286027994,\\n          \\"ci_low\\": 0.364414650774974,\\n          \\"ci_high\\": 0.46613509389288565,\\n          \\"top1\\": 0.304,\\n          \\"top10\\": 0.656,\\n          \\"recall_at_25\\": 0.812,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.0,\\n          \\"ci_low\\": 0.0,\\n          \\"ci_high\\": 0.0,\\n          \\"top1\\": 0.0,\\n          \\"top10\\": 0.0,\\n          \\"recall_at_25\\": 0.0,\\n          \\"n\\": 250\\n        }\\n      },\\n      \\"intensity_shuffle\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.6096692579883883,\\n          \\"ci_low\\": 0.5561514656273677,\\n          \\"ci_high\\": 0.6601137803042368,\\n          \\"top1\\": 0.5,\\n          \\"top10\\": 0.78,\\n          \\"recall_at_25\\": 0.888,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.6001716075711729,\\n          \\"ci_low\\": 0.5476562556248099,\\n          \\"ci_high\\": 0.6496373907758907,\\n          \\"top1\\": 0.488,\\n          \\"top10\\": 0.804,\\n          \\"recall_at_25\\": 0.912,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.013141891143809302,\\n          \\"ci_low\\": 0.0057196044524529185,\\n          \\"ci_high\\": 0.022141451946391208,\\n          \\"top1\\": 0.0,\\n          \\"top10\\": 0.032,\\n          \\"recall_at_25\\": 0.056,\\n          \\"n\\": 250\\n        }\\n      }\\n    },\\n    \\"no_gen_features\\": {\\n      \\"none\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.7797657898757588,\\n          \\"ci_low\\": 0.735379010571214,\\n          \\"ci_high\\": 0.8249175613498989,\\n          \\"top1\\": 0.716,\\n          \\"top10\\": 0.88,\\n          \\"recall_at_25\\": 0.94,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.7073418632141852,\\n          \\"ci_low\\": 0.6570839793534637,\\n          \\"ci_high\\": 0.7557740680991695,\\n          \\"top1\\": 0.636,\\n          \\"top10\\": 0.828,\\n          \\"recall_at_25\\": 0.912,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.056511484593837535,\\n          \\"ci_low\\": 0.03212358076563959,\\n          \\"ci_high\\": 0.08336156395891692,\\n          \\"top1\\": 0.044,\\n          \\"top10\\": 0.08,\\n          \\"recall_at_25\\": 0.096,\\n          \\"n\\": 250\\n        }\\n      },\\n      \\"blank\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.24939026386933238,\\n          \\"ci_low\\": 0.20925235492189118,\\n          \\"ci_high\\": 0.2908220767409555,\\n          \\"top1\\": 0.136,\\n          \\"top10\\": 0.556,\\n          \\"recall_at_25\\": 0.708,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.2977393300198254,\\n          \\"ci_low\\": 0.2544577355174066,\\n          \\"ci_high\\": 0.34088607201364163,\\n          \\"top1\\": 0.176,\\n          \\"top10\\": 0.632,\\n          \\"recall_at_25\\": 0.772,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.0,\\n          \\"ci_low\\": 0.0,\\n          \\"ci_high\\": 0.0,\\n          \\"top1\\": 0.0,\\n          \\"top10\\": 0.0,\\n          \\"recall_at_25\\": 0.0,\\n          \\"n\\": 250\\n        }\\n      },\\n      \\"intensity_shuffle\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.6626812727013642,\\n          \\"ci_low\\": 0.6122271161550121,\\n          \\"ci_high\\": 0.7124498891623252,\\n          \\"top1\\": 0.572,\\n          \\"top10\\": 0.824,\\n          \\"recall_at_25\\": 0.888,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.6571814730239175,\\n          \\"ci_low\\": 0.6059503124500647,\\n          \\"ci_high\\": 0.707896621439814,\\n          \\"top1\\": 0.568,\\n          \\"top10\\": 0.816,\\n          \\"recall_at_25\\": 0.892,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.03214005602240896,\\n          \\"ci_low\\": 0.014238025210084033,\\n          \\"ci_high\\": 0.05337574696545286,\\n          \\"top1\\": 0.024,\\n          \\"top10\\": 0.048,\\n          \\"recall_at_25\\": 0.06,\\n          \\"n\\": 250\\n        }\\n      }\\n    },\\n    \\"no_np_score\\": {\\n      \\"none\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.8712570010665663,\\n          \\"ci_low\\": 0.834165057625949,\\n          \\"ci_high\\": 0.9058767085764476,\\n          \\"top1\\": 0.828,\\n          \\"top10\\": 0.952,\\n          \\"recall_at_25\\": 0.98,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.8439632353551535,\\n          \\"ci_low\\": 0.8031159461975932,\\n          \\"ci_high\\": 0.8829685195506397,\\n          \\"top1\\": 0.804,\\n          \\"top10\\": 0.932,\\n          \\"recall_at_25\\": 0.972,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.03813607106089943,\\n          \\"ci_low\\": 0.019624616464770928,\\n          \\"ci_high\\": 0.059593988647698604,\\n          \\"top1\\": 0.024,\\n          \\"top10\\": 0.064,\\n          \\"recall_at_25\\": 0.096,\\n          \\"n\\": 250\\n        }\\n      },\\n      \\"blank\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.2596963615624889,\\n          \\"ci_low\\": 0.2170529780406586,\\n          \\"ci_high\\": 0.30303715730436237,\\n          \\"top1\\": 0.148,\\n          \\"top10\\": 0.532,\\n          \\"recall_at_25\\": 0.708,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.3358921033739596,\\n          \\"ci_low\\": 0.2893759294216276,\\n          \\"ci_high\\": 0.38168399476340653,\\n          \\"top1\\": 0.216,\\n          \\"top10\\": 0.632,\\n          \\"recall_at_25\\": 0.804,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.0,\\n          \\"ci_low\\": 0.0,\\n          \\"ci_high\\": 0.0,\\n          \\"top1\\": 0.0,\\n          \\"top10\\": 0.0,\\n          \\"recall_at_25\\": 0.0,\\n          \\"n\\": 250\\n        }\\n      },\\n      \\"intensity_shuffle\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.7205022832637665,\\n          \\"ci_low\\": 0.6719723968346232,\\n          \\"ci_high\\": 0.7682360457368719,\\n          \\"top1\\": 0.636,\\n          \\"top10\\": 0.872,\\n          \\"recall_at_25\\": 0.928,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.7441926278864641,\\n          \\"ci_low\\": 0.6959861988961591,\\n          \\"ci_high\\": 0.7901192875872352,\\n          \\"top1\\": 0.668,\\n          \\"top10\\": 0.884,\\n          \\"recall_at_25\\": 0.948,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.018336507936507936,\\n          \\"ci_low\\": 0.006857142857142857,\\n          \\"ci_high\\": 0.03292547619047619,\\n          \\"top1\\": 0.008,\\n          \\"top10\\": 0.036,\\n          \\"recall_at_25\\": 0.056,\\n          \\"n\\": 250\\n        }\\n      }\\n    },\\n    \\"membership_only\\": {\\n      \\"none\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.42040647451365576,\\n          \\"ci_low\\": 0.37726497381335333,\\n          \\"ci_high\\": 0.4667900321421075,\\n          \\"top1\\": 0.252,\\n          \\"top10\\": 0.792,\\n          \\"recall_at_25\\": 0.96,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.10612767557101517,\\n          \\"ci_low\\": 0.08076080150222116,\\n          \\"ci_high\\": 0.1336987102821312,\\n          \\"top1\\": 0.04,\\n          \\"top10\\": 0.268,\\n          \\"recall_at_25\\": 0.552,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.005407849168375485,\\n          \\"ci_low\\": 0.0022544494759626343,\\n          \\"ci_high\\": 0.009381573536113009,\\n          \\"top1\\": 0.0,\\n          \\"top10\\": 0.016,\\n          \\"recall_at_25\\": 0.048,\\n          \\"n\\": 250\\n        }\\n      },\\n      \\"blank\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.42040647451365576,\\n          \\"ci_low\\": 0.37726497381335333,\\n          \\"ci_high\\": 0.4667900321421075,\\n          \\"top1\\": 0.252,\\n          \\"top10\\": 0.792,\\n          \\"recall_at_25\\": 0.96,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.11226676478245733,\\n          \\"ci_low\\": 0.0864701165958561,\\n          \\"ci_high\\": 0.13927388920680583,\\n          \\"top1\\": 0.04,\\n          \\"top10\\": 0.284,\\n          \\"recall_at_25\\": 0.596,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.0,\\n          \\"ci_low\\": 0.0,\\n          \\"ci_high\\": 0.0,\\n          \\"top1\\": 0.0,\\n          \\"top10\\": 0.0,\\n          \\"recall_at_25\\": 0.0,\\n          \\"n\\": 250\\n        }\\n      },\\n      \\"intensity_shuffle\\": {\\n        \\"C1\\": {\\n          \\"mrr\\": 0.42040647451365576,\\n          \\"ci_low\\": 0.37726497381335333,\\n          \\"ci_high\\": 0.4667900321421075,\\n          \\"top1\\": 0.252,\\n          \\"top10\\": 0.792,\\n          \\"recall_at_25\\": 0.96,\\n          \\"n\\": 250\\n        },\\n        \\"C2\\": {\\n          \\"mrr\\": 0.10720545334879296,\\n          \\"ci_low\\": 0.08174313240472826,\\n          \\"ci_high\\": 0.13454225647247642,\\n          \\"top1\\": 0.04,\\n          \\"top10\\": 0.272,\\n          \\"recall_at_25\\": 0.56,\\n          \\"n\\": 250\\n        },\\n        \\"C3\\": {\\n          \\"mrr\\": 0.0052029429634692795,\\n          \\"ci_low\\": 0.0022185079175210754,\\n          \\"ci_high\\": 0.008665330940988834,\\n          \\"top1\\": 0.0,\\n          \\"top10\\": 0.024,\\n          \\"recall_at_25\\": 0.048,\\n          \\"n\\": 250\\n        }\\n      }\\n    }\\n  },\\n  \\"candidate_coverage\\": {\\n    \\"C1\\": {\\n      \\"pool_truth_present\\": 246,\\n      \\"with_generation_truth_present\\": 246\\n    },\\n    \\"C2\\": {\\n      \\"pool_truth_present\\": 246,\\n      \\"with_generation_truth_present\\": 246\\n    },\\n    \\"C3\\": {\\n      \\"pool_truth_present\\": 0,\\n      \\"with_generation_truth_present\\": 26\\n    }\\n  },\\n  \\"primary\\": {\\n    \\"contrast\\": \\"Unperturbed test C2 MRR@25: fresh full model minus D1+D2 baseline\\",\\n    \\"delta_mrr\\": 0.6322495690343369,\\n    \\"ci_low\\": 0.5816074230207359,\\n    \\"ci_high\\": 0.682938028724887,\\n    \\"interpretation\\": \\"Single predeclared paired diagnostic; no automatic promotion\\"\\n  },\\n  \\"secondary_joint_intervals\\": {\\n    \\"method\\": \\"molecule-paired max-deviation bootstrap over all prespecified secondary contrasts\\",\\n    \\"n_molecules\\": 250,\\n    \\"n_bootstrap\\": 4000,\\n    \\"seed\\": 20261025,\\n    \\"family_size\\": 27,\\n    \\"joint_radius\\": 0.0663965811206416,\\n    \\"outcomes\\": {\\n      \\"no_d1_minus_full:C1\\": {\\n        \\"delta_mrr\\": -0.04705478982661646,\\n        \\"ci_low\\": -0.11345137094725807,\\n        \\"ci_high\\": 0.01934179129402514\\n      },\\n      \\"no_d1_minus_full:C2\\": {\\n        \\"delta_mrr\\": -0.07292318447544589,\\n        \\"ci_low\\": -0.1393197655960875,\\n        \\"ci_high\\": -0.006526603354804289\\n      },\\n      \\"no_d1_minus_full:C3\\": {\\n        \\"delta_mrr\\": -0.005858700890650009,\\n        \\"ci_low\\": -0.07225528201129161,\\n        \\"ci_high\\": 0.0605378802299916\\n      },\\n      \\"no_d2_minus_full:C1\\": {\\n        \\"delta_mrr\\": -0.056593578643578654,\\n        \\"ci_low\\": -0.12299015976422026,\\n        \\"ci_high\\": 0.00980300247706295\\n      },\\n      \\"no_d2_minus_full:C2\\": {\\n        \\"delta_mrr\\": -0.08761724024767505,\\n        \\"ci_low\\": -0.15401382136831665,\\n        \\"ci_high\\": -0.021220659127033442\\n      },\\n      \\"no_d2_minus_full:C3\\": {\\n        \\"delta_mrr\\": -0.002925467457416576,\\n        \\"ci_low\\": -0.06932204857805818,\\n        \\"ci_high\\": 0.06347111366322503\\n      },\\n      \\"no_d3_minus_full:C1\\": {\\n        \\"delta_mrr\\": -0.0048094199917729325,\\n        \\"ci_low\\": -0.07120600111241454,\\n        \\"ci_high\\": 0.061587161128868674\\n      },\\n      \\"no_d3_minus_full:C2\\": {\\n        \\"delta_mrr\\": -0.012641093873446817,\\n        \\"ci_low\\": -0.07903767499408842,\\n        \\"ci_high\\": 0.05375548724719478\\n      },\\n      \\"no_d3_minus_full:C3\\": {\\n        \\"delta_mrr\\": -0.00014627956584478325,\\n        \\"ci_low\\": -0.06654286068648639,\\n        \\"ci_high\\": 0.06625030155479682\\n      },\\n      \\"no_d4_minus_full:C1\\": {\\n        \\"delta_mrr\\": -0.07127755028507482,\\n        \\"ci_low\\": -0.13767413140571644,\\n        \\"ci_high\\": -0.004880969164433216\\n      },\\n      \\"no_d4_minus_full:C2\\": {\\n        \\"delta_mrr\\": -0.09894681435433891,\\n        \\"ci_low\\": -0.1653433954749805,\\n        \\"ci_high\\": -0.03255023323369731\\n      },\\n      \\"no_d4_minus_full:C3\\": {\\n        \\"delta_mrr\\": -0.006725084094649312,\\n        \\"ci_low\\": -0.07312166521529091,\\n        \\"ci_high\\": 0.059671497025992296\\n      },\\n      \\"no_gen_features_minus_full:C1\\": {\\n        \\"delta_mrr\\": -0.08674179861418252,\\n        \\"ci_low\\": -0.15313837973482414,\\n        \\"ci_high\\": -0.020345217493540915\\n      },\\n      \\"no_gen_features_minus_full:C2\\": {\\n        \\"delta_mrr\\": -0.13149350305353397,\\n        \\"ci_low\\": -0.19789008417417558,\\n        \\"ci_high\\": -0.06509692193289236\\n      },\\n      \\"no_gen_features_minus_full:C3\\": {\\n        \\"delta_mrr\\": 0.0191617251745129,\\n        \\"ci_low\\": -0.0472348559461287,\\n        \\"ci_high\\": 0.08555830629515451\\n      },\\n      \\"no_np_score_minus_full:C1\\": {\\n        \\"delta_mrr\\": 0.0047494125766248535,\\n        \\"ci_low\\": -0.06164716854401675,\\n        \\"ci_high\\": 0.07114599369726646\\n      },\\n      \\"no_np_score_minus_full:C2\\": {\\n        \\"delta_mrr\\": 0.005127869087434305,\\n        \\"ci_low\\": -0.0612687120332073,\\n        \\"ci_high\\": 0.07152445020807591\\n      },\\n      \\"no_np_score_minus_full:C3\\": {\\n        \\"delta_mrr\\": 0.0007863116415747993,\\n        \\"ci_low\\": -0.0656102694790668,\\n        \\"ci_high\\": 0.06718289276221641\\n      },\\n      \\"membership_only_minus_full:C1\\": {\\n        \\"delta_mrr\\": -0.44610111397628577,\\n        \\"ci_low\\": -0.5124976950969273,\\n        \\"ci_high\\": -0.37970453285564415\\n      },\\n      \\"membership_only_minus_full:C2\\": {\\n        \\"delta_mrr\\": -0.7327076906967042,\\n        \\"ci_low\\": -0.7991042718173458,\\n        \\"ci_high\\": -0.6663111095760625\\n      },\\n      \\"membership_only_minus_full:C3\\": {\\n        \\"delta_mrr\\": -0.031941910250949154,\\n        \\"ci_low\\": -0.09833849137159076,\\n        \\"ci_high\\": 0.03445467086969245\\n      },\\n      \\"full_blank_minus_none:C1\\": {\\n        \\"delta_mrr\\": -0.6056100955921777,\\n        \\"ci_low\\": -0.6720066767128193,\\n        \\"ci_high\\": -0.5392135144715361\\n      },\\n      \\"full_blank_minus_none:C2\\": {\\n        \\"delta_mrr\\": -0.5118579154924734,\\n        \\"ci_low\\": -0.578254496613115,\\n        \\"ci_high\\": -0.4454613343718318\\n      },\\n      \\"full_blank_minus_none:C3\\": {\\n        \\"delta_mrr\\": -0.03734975941932464,\\n        \\"ci_low\\": -0.10374634053996624,\\n        \\"ci_high\\": 0.029046821701316963\\n      },\\n      \\"full_intensity_shuffle_minus_none:C1\\": {\\n        \\"delta_mrr\\": -0.14139415584415582,\\n        \\"ci_low\\": -0.20779073696479744,\\n        \\"ci_high\\": -0.07499757472351422\\n      },\\n      \\"full_intensity_shuffle_minus_none:C2\\": {\\n        \\"delta_mrr\\": -0.09359914331193106,\\n        \\"ci_low\\": -0.15999572443257265,\\n        \\"ci_high\\": -0.027202562191289456\\n      },\\n      \\"full_intensity_shuffle_minus_none:C3\\": {\\n        \\"delta_mrr\\": -0.016535962105527324,\\n        \\"ci_low\\": -0.08293254322616893,\\n        \\"ci_high\\": 0.04986061901511428\\n      }\\n    },\\n    \\"limitations\\": \\"Approximate fixed-family uncertainty; no correction for domain shift or historical hypothesis selection\\"\\n  },\\n  \\"error_slices\\": {\\n    \\"library\\": {\\n      \\"gnps\\": {\\n        \\"n\\": 56,\\n        \\"status\\": \\"DESCRIPTIVE_ONLY\\",\\n        \\"classes\\": {\\n          \\"C1\\": {\\n            \\"baseline_mrr\\": 0.24911190233669225,\\n            \\"full_mrr\\": 0.7106827200577202,\\n            \\"delta_mrr\\": 0.4615708177210278,\\n            \\"ci_low\\": 0.3360824077737156,\\n            \\"ci_high\\": 0.5809605893493762\\n          },\\n          \\"C2\\": {\\n            \\"baseline_mrr\\": 0.19851666424145414,\\n            \\"full_mrr\\": 0.6857571248196248,\\n            \\"delta_mrr\\": 0.48724046057817066,\\n            \\"ci_low\\": 0.372144986687857,\\n            \\"ci_high\\": 0.5935427793276402\\n          },\\n          \\"C3\\": {\\n            \\"baseline_mrr\\": 0.0,\\n            \\"full_mrr\\": 0.09889346764346764,\\n            \\"delta_mrr\\": 0.09889346764346764,\\n            \\"ci_low\\": 0.037494276556776555,\\n            \\"ci_high\\": 0.17340029761904763\\n          }\\n        }\\n      },\\n      \\"massbank\\": {\\n        \\"n\\": 11,\\n        \\"status\\": \\"SUPPRESSED_SMALL_CELL\\"\\n      },\\n      \\"mona\\": {\\n        \\"n\\": 6,\\n        \\"status\\": \\"SUPPRESSED_SMALL_CELL\\"\\n      },\\n      \\"msdial\\": {\\n        \\"n\\": 2,\\n        \\"status\\": \\"SUPPRESSED_SMALL_CELL\\"\\n      },\\n      \\"pluskal_ms2\\": {\\n        \\"n\\": 141,\\n        \\"status\\": \\"DESCRIPTIVE_ONLY\\",\\n        \\"classes\\": {\\n          \\"C1\\": {\\n            \\"baseline_mrr\\": 0.46992672418204334,\\n            \\"full_mrr\\": 0.9517174245584759,\\n            \\"delta_mrr\\": 0.48179070037643257,\\n            \\"ci_low\\": 0.40367469693658686,\\n            \\"ci_high\\": 0.5607155645946321\\n          },\\n          \\"C2\\": {\\n            \\"baseline_mrr\\": 0.13634936989541427,\\n            \\"full_mrr\\": 0.9504171881518564,\\n            \\"delta_mrr\\": 0.8140678182564423,\\n            \\"ci_low\\": 0.7645289867398506,\\n            \\"ci_high\\": 0.859732782710127\\n          },\\n          \\"C3\\": {\\n            \\"baseline_mrr\\": 0.0,\\n            \\"full_mrr\\": 0.0,\\n            \\"delta_mrr\\": 0.0,\\n            \\"ci_low\\": 0.0,\\n            \\"ci_high\\": 0.0\\n          }\\n        }\\n      },\\n      \\"riken\\": {\\n        \\"n\\": 7,\\n        \\"status\\": \\"SUPPRESSED_SMALL_CELL\\"\\n      },\\n      \\"spectraverse\\": {\\n        \\"n\\": 27,\\n        \\"status\\": \\"DESCRIPTIVE_ONLY\\",\\n        \\"classes\\": {\\n          \\"C1\\": {\\n            \\"baseline_mrr\\": 0.6040123456790124,\\n            \\"full_mrr\\": 0.8551440329218107,\\n            \\"delta_mrr\\": 0.2511316872427984,\\n            \\"ci_low\\": 0.12510288065843622,\\n            \\"ci_high\\": 0.38343878600823056\\n          },\\n          \\"C2\\": {\\n            \\"baseline_mrr\\": 0.4975308641975309,\\n            \\"full_mrr\\": 0.7903292181069959,\\n            \\"delta_mrr\\": 0.292798353909465,\\n            \\"ci_low\\": 0.16080246913580248,\\n            \\"ci_high\\": 0.42943158436213996\\n          },\\n          \\"C3\\": {\\n            \\"baseline_mrr\\": 0.0,\\n            \\"full_mrr\\": 0.10957394725510666,\\n            \\"delta_mrr\\": 0.10957394725510666,\\n            \\"ci_low\\": 0.019244512867701277,\\n            \\"ci_high\\": 0.22475001917030904\\n          }\\n        }\\n      }\\n    },\\n    \\"adduct\\": {\\n      \\"[M+CH2O2-H]-\\": {\\n        \\"n\\": 3,\\n        \\"status\\": \\"SUPPRESSED_SMALL_CELL\\"\\n      },\\n      \\"[M+H]+\\": {\\n        \\"n\\": 129,\\n        \\"status\\": \\"DESCRIPTIVE_ONLY\\",\\n        \\"classes\\": {\\n          \\"C1\\": {\\n            \\"baseline_mrr\\": 0.3978795116825213,\\n            \\"full_mrr\\": 0.8562945591673362,\\n            \\"delta_mrr\\": 0.45841504748481493,\\n            \\"ci_low\\": 0.3750974159219713,\\n            \\"ci_high\\": 0.5423704594475799\\n          },\\n          \\"C2\\": {\\n            \\"baseline_mrr\\": 0.14377891408437163,\\n            \\"full_mrr\\": 0.8359780216996359,\\n            \\"delta_mrr\\": 0.6921991076152643,\\n            \\"ci_low\\": 0.6207810724735314,\\n            \\"ci_high\\": 0.7597687423470643\\n          },\\n          \\"C3\\": {\\n            \\"baseline_mrr\\": 0.0,\\n            \\"full_mrr\\": 0.035824554429205596,\\n            \\"delta_mrr\\": 0.035824554429205596,\\n            \\"ci_low\\": 0.011756774663751407,\\n            \\"ci_high\\": 0.06653746770025838\\n          }\\n        }\\n      },\\n      \\"[M+NH4]+\\": {\\n        \\"n\\": 3,\\n        \\"status\\": \\"SUPPRESSED_SMALL_CELL\\"\\n      },\\n      \\"[M+Na]+\\": {\\n        \\"n\\": 2,\\n        \\"status\\": \\"SUPPRESSED_SMALL_CELL\\"\\n      },\\n      \\"[M-H2O+H]+\\": {\\n        \\"n\\": 1,\\n        \\"status\\": \\"SUPPRESSED_SMALL_CELL\\"\\n      },\\n      \\"[M-H]-\\": {\\n        \\"n\\": 23,\\n        \\"status\\": \\"DESCRIPTIVE_ONLY\\",\\n        \\"classes\\": {\\n          \\"C1\\": {\\n            \\"baseline_mrr\\": 0.5991718426501036,\\n            \\"full_mrr\\": 0.8677536231884058,\\n            \\"delta_mrr\\": 0.2685817805383023,\\n            \\"ci_low\\": 0.09637034161490683,\\n            \\"ci_high\\": 0.438524844720497\\n          },\\n          \\"C2\\": {\\n            \\"baseline_mrr\\": 0.5437583729143832,\\n            \\"full_mrr\\": 0.8695652173913043,\\n            \\"delta_mrr\\": 0.3258068444769212,\\n            \\"ci_low\\": 0.18404244306418222,\\n            \\"ci_high\\": 0.4775057849226648\\n          },\\n          \\"C3\\": {\\n            \\"baseline_mrr\\": 0.0,\\n            \\"full_mrr\\": 0.0458592132505176,\\n            \\"delta_mrr\\": 0.0458592132505176,\\n            \\"ci_low\\": 0.007246376811594202,\\n            \\"ci_high\\": 0.1\\n          }\\n        }\\n      },\\n      \\"mixed\\": {\\n        \\"n\\": 89,\\n        \\"status\\": \\"DESCRIPTIVE_ONLY\\",\\n        \\"classes\\": {\\n          \\"C1\\": {\\n            \\"baseline_mrr\\": 0.5378783825413039,\\n            \\"full_mrr\\": 0.911933946203609,\\n            \\"delta_mrr\\": 0.37405556366230525,\\n            \\"ci_low\\": 0.28348234552588486,\\n            \\"ci_high\\": 0.46655352684144824\\n          },\\n          \\"C2\\": {\\n            \\"baseline_mrr\\": 0.20634549479385458,\\n            \\"full_mrr\\": 0.8631823856542958,\\n            \\"delta_mrr\\": 0.6568368908604412,\\n            \\"ci_low\\": 0.5803319460259749,\\n            \\"ci_high\\": 0.7325179729526704\\n          },\\n          \\"C3\\": {\\n            \\"baseline_mrr\\": 0.0,\\n            \\"full_mrr\\": 0.02780047667688117,\\n            \\"delta_mrr\\": 0.02780047667688117,\\n            \\"ci_low\\": 0.0015832482124616958,\\n            \\"ci_high\\": 0.06367041198501872\\n          }\\n        }\\n      }\\n    },\\n    \\"mode\\": {\\n      \\"mixed\\": {\\n        \\"n\\": 67,\\n        \\"status\\": \\"DESCRIPTIVE_ONLY\\",\\n        \\"classes\\": {\\n          \\"C1\\": {\\n            \\"baseline_mrr\\": 0.6205713854967586,\\n            \\"full_mrr\\": 0.9626865671641791,\\n            \\"delta_mrr\\": 0.3421151816674205,\\n            \\"ci_low\\": 0.23618969546208352,\\n            \\"ci_high\\": 0.45039179104477617\\n          },\\n          \\"C2\\": {\\n            \\"baseline_mrr\\": 0.20709592906464586,\\n            \\"full_mrr\\": 0.9024046434494195,\\n            \\"delta_mrr\\": 0.6953087143847737,\\n            \\"ci_low\\": 0.6118346625245277,\\n            \\"ci_high\\": 0.7768671790885414\\n          },\\n          \\"C3\\": {\\n            \\"baseline_mrr\\": 0.0,\\n            \\"full_mrr\\": 0.002103120759837178,\\n            \\"delta_mrr\\": 0.002103120759837178,\\n            \\"ci_low\\": 0.0,\\n            \\"ci_high\\": 0.005563093622795116\\n          }\\n        }\\n      },\\n      \\"negative\\": {\\n        \\"n\\": 29,\\n        \\"status\\": \\"DESCRIPTIVE_ONLY\\",\\n        \\"classes\\": {\\n          \\"C1\\": {\\n            \\"baseline_mrr\\": 0.5050903119868637,\\n            \\"full_mrr\\": 0.8462643678160918,\\n            \\"delta_mrr\\": 0.3411740558292282,\\n            \\"ci_low\\": 0.19051211001642038,\\n            \\"ci_high\\": 0.4942569786535304\\n          },\\n          \\"C2\\": {\\n            \\"baseline_mrr\\": 0.46114169805853383,\\n            \\"full_mrr\\": 0.8477011494252873,\\n            \\"delta_mrr\\": 0.3865594513667536,\\n            \\"ci_low\\": 0.2583964792813677,\\n            \\"ci_high\\": 0.5218902733507197\\n          },\\n          \\"C3\\": {\\n            \\"baseline_mrr\\": 0.0,\\n            \\"full_mrr\\": 0.07315270935960591,\\n            \\"delta_mrr\\": 0.07315270935960591,\\n            \\"ci_low\\": 0.014527914614121512,\\n            \\"ci_high\\": 0.1574753694581281\\n          }\\n        }\\n      },\\n      \\"positive\\": {\\n        \\"n\\": 154,\\n        \\"status\\": \\"DESCRIPTIVE_ONLY\\",\\n        \\"classes\\": {\\n          \\"C1\\": {\\n            \\"baseline_mrr\\": 0.3829900086756466,\\n            \\"full_mrr\\": 0.8284755224403811,\\n            \\"delta_mrr\\": 0.4454855137647345,\\n            \\"ci_low\\": 0.3723069277297696,\\n            \\"ci_high\\": 0.5207121240369427\\n          },\\n          \\"C2\\": {\\n            \\"baseline_mrr\\": 0.15842800530725223,\\n            \\"full_mrr\\": 0.8095090722239309,\\n            \\"delta_mrr\\": 0.6510810669166787,\\n            \\"ci_low\\": 0.5845700996271351,\\n            \\"ci_high\\": 0.7135218506708458\\n          },\\n          \\"C3\\": {\\n            \\"baseline_mrr\\": 0.0,\\n            \\"full_mrr\\": 0.04594222202917855,\\n            \\"delta_mrr\\": 0.04594222202917855,\\n            \\"ci_low\\": 0.019311120642642385,\\n            \\"ci_high\\": 0.07807735440887616\\n          }\\n        }\\n      }\\n    },\\n    \\"mass\\": {\\n      \\"150-300\\": {\\n        \\"n\\": 30,\\n        \\"status\\": \\"DESCRIPTIVE_ONLY\\",\\n        \\"classes\\": {\\n          \\"C1\\": {\\n            \\"baseline_mrr\\": 0.43494539229833346,\\n            \\"full_mrr\\": 0.7897354497354496,\\n            \\"delta_mrr\\": 0.3547900574371163,\\n            \\"ci_low\\": 0.20138623984947515,\\n            \\"ci_high\\": 0.5132889575021928\\n          },\\n          \\"C2\\": {\\n            \\"baseline_mrr\\": 0.16817601788190026,\\n            \\"full_mrr\\": 0.6673743386243385,\\n            \\"delta_mrr\\": 0.4991983207424385,\\n            \\"ci_low\\": 0.3566926972809326,\\n            \\"ci_high\\": 0.6479387166482754\\n          },\\n          \\"C3\\": {\\n            \\"baseline_mrr\\": 0.0,\\n            \\"full_mrr\\": 0.08888888888888888,\\n            \\"delta_mrr\\": 0.08888888888888888,\\n            \\"ci_low\\": 0.01111111111111111,\\n            \\"ci_high\\": 0.19999999999999998\\n          }\\n        }\\n      },\\n      \\"300-600\\": {\\n        \\"n\\": 194,\\n        \\"status\\": \\"DESCRIPTIVE_ONLY\\",\\n        \\"classes\\": {\\n          \\"C1\\": {\\n            \\"baseline_mrr\\": 0.45021306585739573,\\n            \\"full_mrr\\": 0.8982179282209604,\\n            \\"delta_mrr\\": 0.44800486236356457,\\n            \\"ci_low\\": 0.3819553291404034,\\n            \\"ci_high\\": 0.5139996312645647\\n          },\\n          \\"C2\\": {\\n            \\"baseline_mrr\\": 0.18368076909685058,\\n            \\"full_mrr\\": 0.8859040679689558,\\n            \\"delta_mrr\\": 0.7022232988721052,\\n            \\"ci_low\\": 0.64764079425244,\\n            \\"ci_high\\": 0.7554403104411662\\n          },\\n          \\"C3\\": {\\n            \\"baseline_mrr\\": 0.0,\\n            \\"full_mrr\\": 0.019274225202060253,\\n            \\"delta_mrr\\": 0.019274225202060253,\\n            \\"ci_low\\": 0.006095250682879549,\\n            \\"ci_high\\": 0.03649755384935798\\n          }\\n        }\\n      },\\n      \\"600-1170\\": {\\n        \\"n\\": 26,\\n        \\"status\\": \\"DESCRIPTIVE_ONLY\\",\\n        \\"classes\\": {\\n          \\"C1\\": {\\n            \\"baseline_mrr\\": 0.5698717948717948,\\n            \\"full_mrr\\": 0.7184829059829059,\\n            \\"delta_mrr\\": 0.14861111111111108,\\n            \\"ci_low\\": -0.0004300213675213736,\\n            \\"ci_high\\": 0.2980902777777778\\n          },\\n          \\"C2\\": {\\n            \\"baseline_mrr\\": 0.4218115218115219,\\n            \\"full_mrr\\": 0.6854700854700854,\\n            \\"delta_mrr\\": 0.26365856365856366,\\n            \\"ci_low\\": 0.1459252553002553,\\n            \\"ci_high\\": 0.3953688672438673\\n          },\\n          \\"C3\\": {\\n            \\"baseline_mrr\\": 0.0,\\n            \\"full_mrr\\": 0.11275282688326167,\\n            \\"delta_mrr\\": 0.11275282688326167,\\n            \\"ci_low\\": 0.018771798853320594,\\n            \\"ci_high\\": 0.227511845039019\\n          }\\n        }\\n      }\\n    },\\n    \\"spectra\\": {\\n      \\"1\\": {\\n        \\"n\\": 43,\\n        \\"status\\": \\"DESCRIPTIVE_ONLY\\",\\n        \\"classes\\": {\\n          \\"C1\\": {\\n            \\"baseline_mrr\\": 0.45717977113325947,\\n            \\"full_mrr\\": 0.7190660760428201,\\n            \\"delta_mrr\\": 0.2618863049095607,\\n            \\"ci_low\\": 0.13949796973052786,\\n            \\"ci_high\\": 0.3839244186046512\\n          },\\n          \\"C2\\": {\\n            \\"baseline_mrr\\": 0.3653805055273843,\\n            \\"full_mrr\\": 0.6740079365079364,\\n            \\"delta_mrr\\": 0.3086274309805522,\\n            \\"ci_low\\": 0.1826987284101727,\\n            \\"ci_high\\": 0.4339007098657497\\n          },\\n          \\"C3\\": {\\n            \\"baseline_mrr\\": 0.0,\\n            \\"full_mrr\\": 0.12037251031184297,\\n            \\"delta_mrr\\": 0.12037251031184297,\\n            \\"ci_low\\": 0.04065135137303995,\\n            \\"ci_high\\": 0.21458257097918376\\n          }\\n        }\\n      },\\n      \\"2-3\\": {\\n        \\"n\\": 28,\\n        \\"status\\": \\"DESCRIPTIVE_ONLY\\",\\n        \\"classes\\": {\\n          \\"C1\\": {\\n            \\"baseline_mrr\\": 0.3389579316575115,\\n            \\"full_mrr\\": 0.7506944444444443,\\n            \\"delta_mrr\\": 0.41173651278693296,\\n            \\"ci_low\\": 0.22901210673663408,\\n            \\"ci_high\\": 0.5763748567608863\\n          },\\n          \\"C2\\": {\\n            \\"baseline_mrr\\": 0.2449103126098924,\\n            \\"full_mrr\\": 0.7700396825396826,\\n            \\"delta_mrr\\": 0.5251293699297902,\\n            \\"ci_low\\": 0.38598212391017017,\\n            \\"ci_high\\": 0.6638883034274317\\n          },\\n          \\"C3\\": {\\n            \\"baseline_mrr\\": 0.0,\\n            \\"full_mrr\\": 0.10833333333333332,\\n            \\"delta_mrr\\": 0.10833333333333332,\\n            \\"ci_low\\": 0.019047619047619046,\\n            \\"ci_high\\": 0.22261904761904763\\n          }\\n        }\\n      },\\n      \\"4-5\\": {\\n        \\"n\\": 179,\\n        \\"status\\": \\"DESCRIPTIVE_ONLY\\",\\n        \\"classes\\": {\\n          \\"C1\\": {\\n            \\"baseline_mrr\\": 0.48076430707715617,\\n            \\"full_mrr\\": 0.9200425218335175,\\n            \\"delta_mrr\\": 0.4392782147563613,\\n            \\"ci_low\\": 0.3707760257734379,\\n            \\"ci_high\\": 0.5069947455874173\\n          },\\n          \\"C2\\": {\\n            \\"baseline_mrr\\": 0.16244468613179372,\\n            \\"full_mrr\\": 0.8891921183574156,\\n            \\"delta_mrr\\": 0.7267474322256221,\\n            \\"ci_low\\": 0.6717896854738171,\\n            \\"ci_high\\": 0.7776792046376771\\n          },\\n          \\"C3\\": {\\n            \\"baseline_mrr\\": 0.0,\\n            \\"full_mrr\\": 0.006302170827310492,\\n            \\"delta_mrr\\": 0.006302170827310492,\\n            \\"ci_low\\": 0.0009376098761573622,\\n            \\"ci_high\\": 0.013685848602049718\\n          }\\n        }\\n      }\\n    },\\n    \\"instrument\\": {\\n      \\"other_or_mixed\\": {\\n        \\"n\\": 250,\\n        \\"status\\": \\"DESCRIPTIVE_ONLY\\",\\n        \\"classes\\": {\\n          \\"C1\\": {\\n            \\"baseline_mrr\\": 0.46082545284780574,\\n            \\"full_mrr\\": 0.8665075884899415,\\n            \\"delta_mrr\\": 0.4056821356421357,\\n            \\"ci_low\\": 0.3474496532128003,\\n            \\"ci_high\\": 0.46274665558950856\\n          },\\n          \\"C2\\": {\\n            \\"baseline_mrr\\": 0.20658579723338238,\\n            \\"full_mrr\\": 0.8388353662677193,\\n            \\"delta_mrr\\": 0.6322495690343369,\\n            \\"ci_low\\": 0.5816074230207359,\\n            \\"ci_high\\": 0.682938028724887\\n          },\\n          \\"C3\\": {\\n            \\"baseline_mrr\\": 0.0,\\n            \\"full_mrr\\": 0.037349759419324634,\\n            \\"delta_mrr\\": 0.037349759419324634,\\n            \\"ci_low\\": 0.018833808751634835,\\n            \\"ci_high\\": 0.05876470780427302\\n          }\\n        }\\n      }\\n    }\\n  },\\n  \\"private_details_sha256\\": \\"b8f1a2b10f22fc90a50df4271882eacf2cfdc559551306a7d82e034182903b17\\",\\n  \\"environment\\": {\\n    \\"python\\": \\"3.13.15\\",\\n    \\"device\\": \\"cuda\\",\\n    \\"rdkit\\": \\"2026.3.3\\",\\n    \\"numpy\\": \\"2.1.3\\",\\n    \\"pandas\\": \\"2.3.3\\",\\n    \\"pyarrow\\": \\"23.0.1\\",\\n    \\"scipy\\": \\"1.16.3\\",\\n    \\"numba\\": \\"0.61.2\\",\\n    \\"torch\\": \\"2.11.0+cu128\\",\\n    \\"lightgbm\\": \\"4.6.0\\"\\n  },\\n  \\"limitations\\": [\\n    \\"Internal source data, not newly acquired external spectra or an independent laboratory replication.\\",\\n    \\"Family-disjoint supervised query fitting does not remove the task-defined retrieval reference corpus.\\",\\n    \\"Generic Murcko topology and acyclic connectivity are computational grouping rules, not a guarantee of distant chemistry.\\",\\n    \\"Fixed training sample and small fresh fingerprint architecture trade capacity for auditable zero-cost execution; no state-of-the-art claim.\\",\\n    \\"Feature ablations reuse a fixed generated candidate universe; no_gen_features does not remove candidate generation.\\",\\n    \\"Slices are descriptive; subgroup intervals do not establish multiplicity-adjusted subgroup effects.\\",\\n    \\"No raw spectra, structures, query identifiers, per-query ranks, model weights or submitted predictions are exported.\\"\\n  ],\\n  \\"raw_inputs\\": {\\n    \\"train.parquet\\": {\\n      \\"bytes\\": 3033286496,\\n      \\"sha256\\": \\"9423f90ec08b941a9646b535bf0606183e86ff0c5a297978f5a70693d41bcd98\\"\\n    },\\n    \\"coconut.csv\\": {\\n      \\"bytes\\": 576946638,\\n      \\"sha256\\": \\"6cfaa8685eb9bc44132239ba4544a03d4d3c36b634a0d67a78d55b7b6885bf7f\\"\\n    },\\n    \\"pubchemlite.csv\\": {\\n      \\"bytes\\": 224829130,\\n      \\"sha256\\": \\"b9bc8960d05561ea50cda2414eef82e19aecc793f49e3c4b79084aba636eecf4\\"\\n    }\\n  },\\n  \\"source_code_sha256\\": {\\n    \\"src/casmi/__init__.py\\": \\"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\",\\n    \\"src/casmi/channels/__init__.py\\": \\"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\",\\n    \\"src/casmi/channels/analog.py\\": \\"8592dcfc274257db6f4a803cb705ec6be4a5dbba2abddec626eddd01d56b4c19\\",\\n    \\"src/casmi/channels/bondmodel.py\\": \\"1ffb5ce34c7b1a1975277eb9eab76a11e7624ea8740f39e5c2cc19a1340680fb\\",\\n    \\"src/casmi/channels/fpinfer.py\\": \\"58315b7f7288a2988980b19a079ef0eb305897ffcd5094b1dcf5d3b370a71a7b\\",\\n    \\"src/casmi/channels/fpmodel.py\\": \\"1e8a63dcd407f7870918c34fe5dac4f8fbfcbd301d64806d077019e382ad70ea\\",\\n    \\"src/casmi/channels/fragments.py\\": \\"20656968ee6084ed948547a90ca79957a5b3cc6a65002ffb8601a411f0278bc0\\",\\n    \\"src/casmi/channels/library.py\\": \\"96dd852858d6e66319d174306091d05480c965037d8e6dc50a87985f2a4398c2\\",\\n    \\"src/casmi/channels/prefilter.py\\": \\"ac442621394b1d4ad1627f7776cf8f4d704b1a8e3d950f0480168e54d3b22a57\\",\\n    \\"src/casmi/channels/reference_selection.py\\": \\"c043a1ab42126d3b33c5de0870ee1a5922396956a69356a777149ebc9947d88b\\",\\n    \\"src/casmi/chem/__init__.py\\": \\"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\",\\n    \\"src/casmi/chem/canon.py\\": \\"6cdb71219a717cbcebefeb9678bec8aff30e8b3aeaa39ce67fd1d499e4f93fdb\\",\\n    \\"src/casmi/chem/keys.py\\": \\"2755a82fe24b854df0326c212df3696f0ac6197e46c3acdc5d33f4264e69aed2\\",\\n    \\"src/casmi/chem/masses.py\\": \\"1831039a9dd48ddfd92482b361c3c28353a0fc1fbff98aa72c5273adcb948cf5\\",\\n    \\"src/casmi/chem/npscore.py\\": \\"5fa24e1495bed25dfcd7001bb412e738d32148c5b3cab9dd3f25a28bb99a8945\\",\\n    \\"src/casmi/config.py\\": \\"5f8dc8985d2789bd93ae55b8bec11964f89472e3e710012c86c2c186e547b876\\",\\n    \\"src/casmi/engine.py\\": \\"1c690b735a39d077a369e9fa635c807578c790f520d82e1aa3c2796837f50d01\\",\\n    \\"src/casmi/generate/__init__.py\\": \\"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\",\\n    \\"src/casmi/generate/transforms.py\\": \\"20345fb5e22aa5e48da97c0d88ede2a9176be65c2b18935322437d20696f18ef\\",\\n    \\"src/casmi/pool/__init__.py\\": \\"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\",\\n    \\"src/casmi/pool/build_safety.py\\": \\"2421c9f41d26ab6e17ac20956ef1bfd5552ed28ae7255f13ca8b4fd585a79305\\",\\n    \\"src/casmi/pool/fingerprint_audit.py\\": \\"d0197f06cbe89d54494eed5bf8693f8ab0944668ce006012726b758c6b44c358\\",\\n    \\"src/casmi/pool/fingerprint_provenance.py\\": \\"ec64027e7120b8636408f13d2a617a012c5ddb1f97dae7c475d341f44b8e97fe\\",\\n    \\"src/casmi/pool/pool.py\\": \\"178d16b9ffb9b26052dd28adeff387794db9f5b405fb5a4e347faeefd14ae634\\",\\n    \\"src/casmi/pool/source_reconciliation.py\\": \\"268cf27522e22659fdb0560784ef787c7b25f66dc739a135e5d8d5b97e5fe4c2\\",\\n    \\"src/casmi/pool/structure_identity.py\\": \\"39519cf6128de5a8677e5675ac9f691b8943fb5386337320ff1d05bfdb2c146a\\",\\n    \\"src/casmi/pool/structure_quality.py\\": \\"654d3c57b836d857367c404c6eeb6f9d010ef710c36a3874dafbe2382f31d646\\",\\n    \\"src/casmi/predict.py\\": \\"d5416dc486fbab178da9073d48fa3c8adf6e4b224f11bc1af5c07c6e0cb2a2e1\\",\\n    \\"src/casmi/rank/__init__.py\\": \\"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\",\\n    \\"src/casmi/rank/features.py\\": \\"35479b0b218163f94b495ca1af66f7949bd1dc5ba0404e02fe3a9e63b26c81f8\\",\\n    \\"src/casmi/rank/ranker.py\\": \\"8b58478f310016485d8a877ff96f3fbac96278497dc0885ec267d294b11c3665\\",\\n    \\"src/casmi/spectra/__init__.py\\": \\"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\",\\n    \\"src/casmi/spectra/build_safety.py\\": \\"b70cab5a2af895061c6786811a2277e899475b7f9eaad29033f2b8990e8e04c0\\",\\n    \\"src/casmi/spectra/clean.py\\": \\"af67f80cbd5e5d6b489ed110be5dd943446a6e38d45a8976365a3c13c07a2bb6\\",\\n    \\"src/casmi/spectra/similarity.py\\": \\"4c63e21f048facdf4a1fbf2e07ec840a078c77b4f1b99f553d77db66cec70b4c\\",\\n    \\"src/casmi/spectra/store.py\\": \\"6b7cd04ac0350ce204a066a82bc250b8e60b552bdfbb95221152399bfdffe13c\\",\\n    \\"src/casmi/spectra/store_integrity.py\\": \\"3664d4c6278bea71c3fa403f017deda8af8dde7de9cb0afbf4c9424835d68243\\",\\n    \\"src/casmi/submit/__init__.py\\": \\"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\",\\n    \\"src/casmi/validate/__init__.py\\": \\"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\",\\n    \\"src/casmi/validate/analog_index_evidence.py\\": \\"2dac213fd7629937ea3198cd7cc4c36ee1daab92439e832bbabcbefec0952e3a\\",\\n    \\"src/casmi/validate/control_alignment.py\\": \\"4996027acd3d4c1b79c2d4e3351e325ad6278323c340af36ad0e63a2bd722cfb\\",\\n    \\"src/casmi/validate/error_slices.py\\": \\"ad0a7df1c02d2dc867e1a80e027ed8d665fad64b694a1ddb1ae04d0169e8baf1\\",\\n    \\"src/casmi/validate/fingerprint_evidence.py\\": \\"1d2d123d68a2f1a8005179c5b95ac5d3437e83a7c9d58df7276e77eb13b5b3da\\",\\n    \\"src/casmi/validate/knockout.py\\": \\"7c9e54a77d9d7d7cef98b0aadcdba5b918427c053af655c7ffe3e6e3ef01d238\\",\\n    \\"src/casmi/validate/metric.py\\": \\"b5c2ab1252a10d442399d140beea01f64d21df1c9ddbda5baf1803738dba3ed1\\",\\n    \\"src/casmi/validate/overlap_audit.py\\": \\"23ef2d7108c2a1f7ed93b3a469af0a6dd853c4d8850e270958e134d538f3df05\\",\\n    \\"src/casmi/validate/paired_bootstrap.py\\": \\"d40724ebf029d8eed57858cc6ec512b1d30672c50af0c9293504f0e2c9961695\\",\\n    \\"src/casmi/validate/ranked_slots.py\\": \\"3dec0248951c1d54ff4a91fafe6173b2e19854739be48041aca4c5ea681b53e0\\",\\n    \\"src/casmi/validate/run_alignment.py\\": \\"bcaed43318285185a8572c7ae2ce2fe53e341814821d9a2bfeaa92ba7f70860e\\",\\n    \\"src/casmi/validate/scaffold_study.py\\": \\"5005a72b5c4c8951dddc24e93f65d3d26683ce3edcf9dd3c3667cf7b95c46afc\\",\\n    \\"src/casmi/validate/simultaneous_ci.py\\": \\"990b45160da2287eb9debfa4c6e2b96bc5bd6e9fe3b27949877bb05c442159da\\",\\n    \\"src/casmi/validate/source_manifest.py\\": \\"d3a37d2b4879c0735e6998c0d40cc011cb4a6a9c5a1e8325be9443c536e262a0\\",\\n    \\"src/casmi/validate/spectral_controls.py\\": \\"12a49aeb2c72899993a178bda6e505ae382d5f0771df4a2af169343466d9828e\\",\\n    \\"src/casmi/validate/spectrum_overlap.py\\": \\"ed06679f55192f44b1f8db3596ec4be8ff279fce2e813544e5464e015b984aec\\",\\n    \\"scripts/build_structures.py\\": \\"e6a8e08b4146e0a8d77b11f820027afa5de48450d430b353dad961c25646fbe9\\",\\n    \\"scripts/build_store.py\\": \\"e3976223c78d0f926b0668608fff19668c1d75e11f220ed410764114736bd0a3\\",\\n    \\"scripts/build_pool.py\\": \\"49750d7f6c304bd93799f92a297abda6ec78c47ccaca1fee02505a9227ed6434\\",\\n    \\"scripts/build_fingerprints.py\\": \\"7cfd346ba76fcdbb65c8f0e3d726051b4eb09923026f00db0bac51b9c9f49c32\\",\\n    \\"scripts/run_advanced_study.py\\": \\"3ee2028ebe722f68291ed368eb5bbb25fd15605751f52bfc60af42113cb5ca16\\"\\n  },\\n  \\"execution\\": {\\n    \\"platform\\": \\"Kaggle\\",\\n    \\"competition_source\\": \\"enveda-CASMI26-molecule-id-mass-spectra\\",\\n    \\"dataset_sources\\": [\\n      \\"senanuretin/casmi-coconut-b\\",\\n      \\"senanuretin/casmi-pubchemlite-b\\",\\n      \\"metric/rdkit-2026-3-3-wheel\\"\\n    ],\\n    \\"pretrained_models_attached\\": false,\\n    \\"internet_enabled\\": false,\\n    \\"submitted_predictions\\": false,\\n    \\"protocol_preregistered_commit\\": \\"4f8ed74\\",\\n    \\"protocol_registered_before_any_new_test_results\\": true,\\n    \\"source_amendment_sha256\\": \\"4a0a4f4ad5fa84c0590def9d5d24e92eecc0fe756e71800aacfc4bda25aba505\\"\\n  },\\n  \\"installed_packages\\": {\\n    \\"Authlib\\": \\"1.8.0\\",\\n    \\"Boruta\\": \\"0.4.3\\",\\n    \\"Bottleneck\\": \\"1.4.2\\",\\n    \\"CacheControl\\": \\"0.14.4\\",\\n    \\"Cartopy\\": \\"0.26.0\\",\\n    \\"ConfigSpace\\": \\"1.2.2\\",\\n    \\"Cython\\": \\"3.0.12\\",\\n    \\"Deprecated\\": \\"3.0.0\\",\\n    \\"Farama-Notifications\\": \\"0.0.6\\",\\n    \\"Flask\\": \\"3.1.3\\",\\n    \\"GDAL\\": \\"3.8.4\\",\\n    \\"ImageHash\\": \\"4.3.2\\",\\n    \\"ImageIO\\": \\"2.37.4\\",\\n    \\"Janome\\": \\"0.5.0\\",\\n    \\"Jinja2\\": \\"3.1.6\\",\\n    \\"Mako\\": \\"1.4.3\\",\\n    \\"Markdown\\": \\"3.5.2\\",\\n    \\"MarkupSafe\\": \\"3.0.3\\",\\n    \\"POT\\": \\"0.9.7.post1\\",\\n    \\"PuLP\\": \\"3.3.2\\",\\n    \\"PyArabic\\": \\"0.6.15\\",\\n    \\"PyDrive2\\": \\"1.21.1\\",\\n    \\"PyGObject\\": \\"3.48.2\\",\\n    \\"PyJWT\\": \\"2.7.0\\",\\n    \\"PyOpenGL\\": \\"3.1.10\\",\\n    \\"PySocks\\": \\"1.7.1\\",\\n    \\"PyUpSet\\": \\"0.1.1.post7\\",\\n    \\"PyWavelets\\": \\"1.9.0\\",\\n    \\"PyYAML\\": \\"6.0.3\\",\\n    \\"Pygments\\": \\"2.21.0\\",\\n    \\"Pympler\\": \\"1.1\\",\\n    \\"QtPy\\": \\"2.4.3\\",\\n    \\"SQLAlchemy\\": \\"2.0.52\\",\\n    \\"SecretStorage\\": \\"3.5.0\\",\\n    \\"Send2Trash\\": \\"2.1.0\\",\\n    \\"Shimmy\\": \\"2.0.1\\",\\n    \\"Sphinx\\": \\"8.2.3\\",\\n    \\"TPOT\\": \\"1.1.0\\",\\n    \\"Wand\\": \\"0.7.2\\",\\n    \\"Werkzeug\\": \\"3.1.8\\",\\n    \\"a2a-sdk\\": \\"1.2.3\\",\\n    \\"absl-py\\": \\"1.4.0\\",\\n    \\"accelerate\\": \\"1.14.0\\",\\n    \\"access\\": \\"1.1.10.post3\\",\\n    \\"affine\\": \\"3.0.1\\",\\n    \\"aiobotocore\\": \\"2.26.0\\",\\n    \\"aiofiles\\": \\"25.1.0\\",\\n    \\"aiohappyeyeballs\\": \\"2.7.1\\",\\n    \\"aiohttp\\": \\"3.14.3\\",\\n    \\"aioitertools\\": \\"0.13.0\\",\\n    \\"aiosignal\\": \\"1.4.0\\",\\n    \\"aiosqlite\\": \\"0.22.1\\",\\n    \\"alabaster\\": \\"1.0.0\\",\\n    \\"albucore\\": \\"0.0.24\\",\\n    \\"albumentations\\": \\"2.0.8\\",\\n    \\"ale-py\\": \\"0.12.1\\",\\n    \\"alembic\\": \\"1.20.1\\",\\n    \\"altair\\": \\"5.5.0\\",\\n    \\"annotated-doc\\": \\"0.0.5\\",\\n    \\"annotated-types\\": \\"0.8.0\\",\\n    \\"antlr4-python3-runtime\\": \\"4.9.3\\",\\n    \\"anyio\\": \\"4.14.2\\",\\n    \\"anywidget\\": \\"0.9.21\\",\\n    \\"apsw\\": \\"3.53.4.0\\",\\n    \\"argon2-cffi\\": \\"25.1.0\\",\\n    \\"argon2-cffi-bindings\\": \\"26.1.0\\",\\n    \\"args\\": \\"0.1.0\\",\\n    \\"array_record\\": \\"0.8.3\\",\\n    \\"arrow\\": \\"1.4.0\\",\\n    \\"arviz\\": \\"0.22.0\\",\\n    \\"asn1crypto\\": \\"1.5.1\\",\\n    \\"astropy\\": \\"7.2.2\\",\\n    \\"astropy-iers-data\\": \\"0.2026.8.31.0.57.9\\",\\n    \\"asttokens\\": \\"3.0.2\\",\\n    \\"astunparse\\": \\"1.6.3\\",\\n    \\"async-lru\\": \\"2.4.0\\",\\n    \\"atpublic\\": \\"5.1\\",\\n    \\"attrs\\": \\"26.1.0\\",\\n    \\"audioop-lts\\": \\"0.2.2\\",\\n    \\"audioread\\": \\"3.1.0\\",\\n    \\"autograd\\": \\"1.9.1\\",\\n    \\"babel\\": \\"2.18.0\\",\\n    \\"backcall\\": \\"0.2.0\\",\\n    \\"bayesian-optimization\\": \\"3.4.0\\",\\n    \\"beartype\\": \\"0.22.9\\",\\n    \\"beautifulsoup4\\": \\"4.13.5\\",\\n    \\"betterproto\\": \\"2.0.0b7\\",\\n    \\"bigframes\\": \\"2.48.0\\",\\n    \\"bigquery-magics\\": \\"0.14.0\\",\\n    \\"black\\": \\"26.10.1\\",\\n    \\"blake3\\": \\"1.0.11\\",\\n    \\"bleach\\": \\"6.4.0\\",\\n    \\"blinker\\": \\"1.9.0\\",\\n    \\"blis\\": \\"1.3.3\\",\\n    \\"blobfile\\": \\"3.3.0\\",\\n    \\"blosc2\\": \\"4.12.0\\",\\n    \\"bokeh\\": \\"3.8.2\\",\\n    \\"boto3\\": \\"1.41.5\\",\\n    \\"botocore\\": \\"1.41.5\\",\\n    \\"bqplot\\": \\"0.12.47\\",\\n    \\"branca\\": \\"0.8.2\\",\\n    \\"brotli\\": \\"1.2.0\\",\\n    \\"build\\": \\"1.6.1\\",\\n    \\"cachebox\\": \\"5.2.3\\",\\n    \\"cachetools\\": \\"6.2.6\\",\\n    \\"catalogue\\": \\"2.0.10\\",\\n    \\"catboost\\": \\"1.2.10\\",\\n    \\"category_encoders\\": \\"2.11.1\\",\\n    \\"certifi\\": \\"2026.10.19\\",\\n    \\"cesium\\": \\"0.12.4\\",\\n    \\"cffi\\": \\"2.1.1\\",\\n    \\"chardet\\": \\"5.2.0\\",\\n    \\"charset-normalizer\\": \\"3.4.9\\",\\n    \\"chex\\": \\"0.1.90\\",\\n    \\"clarabel\\": \\"0.11.1\\",\\n    \\"click\\": \\"8.5.0\\",\\n    \\"click-plugins\\": \\"1.1.1.2\\",\\n    \\"cligj\\": \\"0.7.2\\",\\n    \\"clint\\": \\"0.5.1\\",\\n    \\"cloudpathlib\\": \\"0.25.0\\",\\n    \\"cloudpickle\\": \\"3.1.2\\",\\n    \\"cmake\\": \\"3.31.10\\",\\n    \\"cmdstanpy\\": \\"1.3.0\\",\\n    \\"colorama\\": \\"0.4.6\\",\\n    \\"colorcet\\": \\"3.2.1\\",\\n    \\"colorlog\\": \\"6.12.0\\",\\n    \\"colorlover\\": \\"0.3.0\\",\\n    \\"comm\\": \\"0.2.3\\",\\n    \\"community\\": \\"1.0.0b1\\",\\n    \\"confection\\": \\"1.3.3\\",\\n    \\"cons\\": \\"0.4.7\\",\\n    \\"contourpy\\": \\"1.3.3\\",\\n    \\"coverage\\": \\"7.16.2\\",\\n    \\"cramjam\\": \\"2.12.1\\",\\n    \\"cryptography\\": \\"50.0.1\\",\\n    \\"cucim-cu12\\": \\"26.2.1\\",\\n    \\"cuda-bindings\\": \\"12.9.7\\",\\n    \\"cuda-core\\": \\"0.3.2\\",\\n    \\"cuda-pathfinder\\": \\"1.8.0\\",\\n    \\"cuda-python\\": \\"12.9.7\\",\\n    \\"cuda-toolkit\\": \\"12.8.1\\",\\n    \\"cudf-cu12\\": \\"26.2.1\\",\\n    \\"cudf-polars-cu12\\": \\"26.2.1\\",\\n    \\"cufflinks\\": \\"0.17.3\\",\\n    \\"cuml-cu12\\": \\"26.2.0\\",\\n    \\"cupy-cuda12x\\": \\"14.0.1\\",\\n    \\"curl_cffi\\": \\"0.16.2\\",\\n    \\"cuvs-cu12\\": \\"26.2.0\\",\\n    \\"cvxopt\\": \\"1.3.2\\",\\n    \\"cvxpy\\": \\"1.6.7\\",\\n    \\"cycler\\": \\"0.12.1\\",\\n    \\"cyipopt\\": \\"1.5.0\\",\\n    \\"cymem\\": \\"2.0.13\\",\\n    \\"cytoolz\\": \\"1.2.0\\",\\n    \\"dacite\\": \\"1.9.2\\",\\n    \\"dask\\": \\"2026.1.1\\",\\n    \\"dask-cuda\\": \\"26.2.0\\",\\n    \\"dask-cudf-cu12\\": \\"26.2.1\\",\\n    \\"dask-expr\\": \\"2.0.0\\",\\n    \\"dask-jobqueue\\": \\"0.9.0\\",\\n    \\"dataclasses-json\\": \\"0.6.7\\",\\n    \\"dataproc-spark-connect\\": \\"1.1.0\\",\\n    \\"datasets\\": \\"4.8.5\\",\\n    \\"db-dtypes\\": \\"1.7.1\\",\\n    \\"dbus-python\\": \\"1.3.2\\",\\n    \\"deap\\": \\"1.4.4\\",\\n    \\"debugpy\\": \\"1.8.15\\",\\n    \\"decorator\\": \\"5.3.1\\",\\n    \\"deepdiff\\": \\"9.1.0\\",\\n    \\"defusedxml\\": \\"0.7.1\\",\\n    \\"deprecation\\": \\"2.1.0\\",\\n    \\"diffusers\\": \\"0.40.0\\",\\n    \\"dill\\": \\"0.4.1\\",\\n    \\"dipy\\": \\"1.12.1\\",\\n    \\"distributed\\": \\"2026.1.1\\",\\n    \\"distributed-ucxx-cu12\\": \\"0.48.0\\",\\n    \\"distro\\": \\"1.9.0\\",\\n    \\"dlib\\": \\"19.24.6\\",\\n    \\"dm-tree\\": \\"0.1.10\\",\\n    \\"dnspython\\": \\"2.9.0\\",\\n    \\"docker\\": \\"7.2.0\\",\\n    \\"docstring-to-markdown\\": \\"0.17\\",\\n    \\"docstring_parser\\": \\"0.18.0\\",\\n    \\"docutils\\": \\"0.21.2\\",\\n    \\"dopamine_rl\\": \\"4.1.2\\",\\n    \\"duckdb\\": \\"1.3.2\\",\\n    \\"earthengine-api\\": \\"1.7.39\\",\\n    \\"easydict\\": \\"1.13\\",\\n    \\"easyocr\\": \\"1.7.2\\",\\n    \\"editdistance\\": \\"0.8.1\\",\\n    \\"eerepr\\": \\"0.1.2\\",\\n    \\"einops\\": \\"0.8.2\\",\\n    \\"email-validator\\": \\"2.3.0\\",\\n    \\"emoji\\": \\"2.16.0\\",\\n    \\"en_core_web_sm\\": \\"3.8.0\\",\\n    \\"entrypoints\\": \\"0.4\\",\\n    \\"esda\\": \\"2.9.0\\",\\n    \\"et_xmlfile\\": \\"2.0.0\\",\\n    \\"etils\\": \\"1.14.0\\",\\n    \\"etuples\\": \\"0.3.10\\",\\n    \\"execnb\\": \\"0.3.3\\",\\n    \\"fastai\\": \\"2.8.8\\",\\n    \\"fastapi\\": \\"0.141.1\\",\\n    \\"fastcore\\": \\"2.2.33\\",\\n    \\"fastdownload\\": \\"0.0.7\\",\\n    \\"fastgit\\": \\"0.1.4\\",\\n    \\"fastjsonschema\\": \\"2.22.2\\",\\n    \\"fastprogress\\": \\"1.1.6\\",\\n    \\"fastspec\\": \\"0.2.5\\",\\n    \\"fasttext\\": \\"0.9.3\\",\\n    \\"fasttransform\\": \\"0.0.2\\",\\n    \\"fasttransport\\": \\"0.0.4\\",\\n    \\"fastuuid\\": \\"0.14.0\\",\\n    \\"featuretools\\": \\"1.31.0\\",\\n    \\"filelock\\": \\"3.32.5\\",\\n    \\"filetype\\": \\"1.2.0\\",\\n    \\"fiona\\": \\"1.10.1\\",\\n    \\"firebase-admin\\": \\"6.9.0\\",\\n    \\"flatbuffers\\": \\"25.12.19\\",\\n    \\"flax\\": \\"0.12.10\\",\\n    \\"folium\\": \\"0.20.0\\",\\n    \\"fonttools\\": \\"4.64.0\\",\\n    \\"formulaic\\": \\"1.2.2\\",\\n    \\"fqdn\\": \\"1.5.1\\",\\n    \\"frozendict\\": \\"2.4.7\\",\\n    \\"frozenlist\\": \\"1.8.0\\",\\n    \\"fsspec\\": \\"2025.12.0\\",\\n    \\"func_timeout\\": \\"4.3.5\\",\\n    \\"funcy\\": \\"2.1\\",\\n    \\"fury\\": \\"0.11.0\\",\\n    \\"future\\": \\"1.0.0\\",\\n    \\"fuzzywuzzy\\": \\"0.18.0\\",\\n    \\"gast\\": \\"0.7.0\\",\\n    \\"gatspy\\": \\"0.3\\",\\n    \\"gcsfs\\": \\"2025.12.0\\",\\n    \\"gdown\\": \\"5.2.2\\",\\n    \\"geemap\\": \\"0.38.4\\",\\n    \\"gensim\\": \\"4.4.0\\",\\n    \\"geocoder\\": \\"1.38.1\\",\\n    \\"geographiclib\\": \\"2.1\\",\\n    \\"geojson\\": \\"3.3.0\\",\\n    \\"geopandas\\": \\"1.1.4\\",\\n    \\"geopy\\": \\"2.4.1\\",\\n    \\"gepa\\": \\"0.1.4\\",\\n    \\"ghapi\\": \\"2.1.4\\",\\n    \\"giddy\\": \\"2.3.9\\",\\n    \\"gin-config\\": \\"0.5.0\\",\\n    \\"glob2\\": \\"0.7\\",\\n    \\"google\\": \\"3.0.0\\",\\n    \\"google-adk\\": \\"2.7.1\\",\\n    \\"google-ai-generativelanguage\\": \\"0.6.15\\",\\n    \\"google-api-core\\": \\"2.30.3\\",\\n    \\"google-api-python-client\\": \\"2.200.0\\",\\n    \\"google-auth\\": \\"2.49.0\\",\\n    \\"google-auth-httplib2\\": \\"0.4.2\\",\\n    \\"google-auth-oauthlib\\": \\"1.4.1\\",\\n    \\"google-cloud-aiplatform\\": \\"2.6.0\\",\\n    \\"google-cloud-bigquery\\": \\"3.44.0\\",\\n    \\"google-cloud-bigquery-connection\\": \\"1.22.0\\",\\n    \\"google-cloud-core\\": \\"2.7.0\\",\\n    \\"google-cloud-dataproc\\": \\"5.28.0\\",\\n    \\"google-cloud-datastore\\": \\"2.25.0\\",\\n    \\"google-cloud-firestore\\": \\"2.27.0\\",\\n    \\"google-cloud-functions\\": \\"1.24.0\\",\\n    \\"google-cloud-language\\": \\"2.21.0\\",\\n    \\"google-cloud-monitoring\\": \\"2.31.0\\",\\n    \\"google-cloud-resource-manager\\": \\"1.18.0\\",\\n    \\"google-cloud-spanner\\": \\"3.68.0\\",\\n    \\"google-cloud-storage\\": \\"3.13.1\\",\\n    \\"google-cloud-storage-control\\": \\"1.12.0\\",\\n    \\"google-cloud-texttospeech\\": \\"2.37.0\\",\\n    \\"google-cloud-translate\\": \\"3.27.0\\",\\n    \\"google-cloud-videointelligence\\": \\"2.20.0\\",\\n    \\"google-cloud-vision\\": \\"3.15.0\\",\\n    \\"google-colab\\": \\"1.0.0\\",\\n    \\"google-crc32c\\": \\"1.8.0\\",\\n    \\"google-genai\\": \\"2.12.1\\",\\n    \\"google-generativeai\\": \\"0.8.6\\",\\n    \\"google-pasta\\": \\"0.2.0\\",\\n    \\"google-resumable-media\\": \\"2.10.2\\",\\n    \\"googleapis-common-protos\\": \\"1.75.0\\",\\n    \\"googledrivedownloader\\": \\"1.1.0\\",\\n    \\"gpxpy\\": \\"1.6.2\\",\\n    \\"gradio\\": \\"6.26.0\\",\\n    \\"gradio_client\\": \\"2.6.1\\",\\n    \\"grain\\": \\"0.2.18\\",\\n    \\"graphviz\\": \\"0.21\\",\\n    \\"greenlet\\": \\"3.5.5\\",\\n    \\"groovy\\": \\"0.1.2\\",\\n    \\"grpc-google-iam-v1\\": \\"0.14.4\\",\\n    \\"grpc-interceptor\\": \\"0.15.4\\",\\n    \\"grpcio\\": \\"1.84.0\\",\\n    \\"grpcio-status\\": \\"1.71.2\\",\\n    \\"grpcio-tools\\": \\"1.71.2\\",\\n    \\"grpclib\\": \\"0.4.9\\",\\n    \\"gspread\\": \\"6.2.1\\",\\n    \\"gspread-dataframe\\": \\"4.0.0\\",\\n    \\"gym\\": \\"0.26.2\\",\\n    \\"gym-notices\\": \\"0.1.0\\",\\n    \\"gymnasium\\": \\"1.2.0\\",\\n    \\"gymnax\\": \\"0.0.8\\",\\n    \\"h11\\": \\"0.16.0\\",\\n    \\"h2\\": \\"4.4.1\\",\\n    \\"h2o\\": \\"3.46.0.12\\",\\n    \\"h5netcdf\\": \\"1.8.1\\",\\n    \\"h5py\\": \\"3.16.0\\",\\n    \\"haversine\\": \\"2.9.0\\",\\n    \\"hdbscan\\": \\"0.8.44\\",\\n    \\"hep_ml\\": \\"0.8.0\\",\\n    \\"hf-gradio\\": \\"0.4.1\\",\\n    \\"hf-xet\\": \\"1.6.0\\",\\n    \\"highspy\\": \\"1.15.1\\",\\n    \\"holidays\\": \\"0.103\\",\\n    \\"holoviews\\": \\"1.23.2\\",\\n    \\"hpack\\": \\"4.2.0\\",\\n    \\"html5lib\\": \\"1.1\\",\\n    \\"httpcore\\": \\"1.0.9\\",\\n    \\"httpcore2\\": \\"2.12.0\\",\\n    \\"httpimport\\": \\"1.4.1\\",\\n    \\"httplib2\\": \\"0.32.0\\",\\n    \\"httptools\\": \\"0.8.0\\",\\n    \\"httpx\\": \\"0.28.1\\",\\n    \\"httpx2\\": \\"2.12.0\\",\\n    \\"huggingface_hub\\": \\"1.29.0\\",\\n    \\"humanize\\": \\"4.16.0\\",\\n    \\"hyperframe\\": \\"6.1.0\\",\\n    \\"hyperopt\\": \\"0.3.0\\",\\n    \\"ibis-framework\\": \\"9.5.0\\",\\n    \\"id\\": \\"1.6.1\\",\\n    \\"idna\\": \\"3.19\\",\\n    \\"igraph\\": \\"1.0.0\\",\\n    \\"imageio-ffmpeg\\": \\"0.6.0\\",\\n    \\"imagesize\\": \\"2.0.1\\",\\n    \\"imbalanced-learn\\": \\"0.14.2\\",\\n    \\"immutabledict\\": \\"4.3.1\\",\\n    \\"importlib_metadata\\": \\"8.9.0\\",\\n    \\"importlib_resources\\": \\"7.1.0\\",\\n    \\"imutils\\": \\"0.5.4\\",\\n    \\"in-toto-attestation\\": \\"0.9.3\\",\\n    \\"inequality\\": \\"1.1.2\\",\\n    \\"inflect\\": \\"7.5.0\\",\\n    \\"iniconfig\\": \\"2.3.0\\",\\n    \\"intel-cmplr-lib-ur\\": \\"2025.3.3\\",\\n    \\"intel-openmp\\": \\"2025.3.3\\",\\n    \\"interface_meta\\": \\"2.0.1\\",\\n    \\"ipyevents\\": \\"2.0.4\\",\\n    \\"ipyfilechooser\\": \\"0.6.0\\",\\n    \\"ipykernel\\": \\"6.17.1\\",\\n    \\"ipyleaflet\\": \\"0.20.0\\",\\n    \\"ipympl\\": \\"0.10.0\\",\\n    \\"ipyparallel\\": \\"8.8.0\\",\\n    \\"ipython\\": \\"7.34.0\\",\\n    \\"ipython-genutils\\": \\"0.2.0\\",\\n    \\"ipython-sql\\": \\"0.5.0\\",\\n    \\"ipython_pygments_lexers\\": \\"1.1.1\\",\\n    \\"ipywidgets\\": \\"8.1.5\\",\\n    \\"isoduration\\": \\"20.11.0\\",\\n    \\"isoweek\\": \\"1.3.3\\",\\n    \\"itsdangerous\\": \\"2.2.0\\",\\n    \\"jaraco.classes\\": \\"3.4.0\\",\\n    \\"jaraco.context\\": \\"6.1.2\\",\\n    \\"jaraco.functools\\": \\"4.6.0\\",\\n    \\"jax\\": \\"0.11.1\\",\\n    \\"jax-cuda12-pjrt\\": \\"0.11.1\\",\\n    \\"jax-cuda12-plugin\\": \\"0.11.1\\",\\n    \\"jaxlib\\": \\"0.11.1\\",\\n    \\"jedi\\": \\"0.20.1\\",\\n    \\"jeepney\\": \\"0.9.0\\",\\n    \\"jieba\\": \\"0.42.1\\",\\n    \\"jiter\\": \\"0.10.0\\",\\n    \\"jmespath\\": \\"1.1.0\\",\\n    \\"joblib\\": \\"1.6.0\\",\\n    \\"joserfc\\": \\"1.7.5\\",\\n    \\"json-rpc\\": \\"1.15.0\\",\\n    \\"json5\\": \\"0.17.3\\",\\n    \\"jsonpatch\\": \\"1.33\\",\\n    \\"jsonpickle\\": \\"4.1.2\\",\\n    \\"jsonpointer\\": \\"3.1.1\\",\\n    \\"jsonschema\\": \\"4.26.0\\",\\n    \\"jsonschema-specifications\\": \\"2025.9.1\\",\\n    \\"jupyter-console\\": \\"6.6.3\\",\\n    \\"jupyter-events\\": \\"0.12.1\\",\\n    \\"jupyter-leaflet\\": \\"0.20.0\\",\\n    \\"jupyter-lsp\\": \\"2.3.1\\",\\n    \\"jupyter_client\\": \\"7.4.9\\",\\n    \\"jupyter_core\\": \\"5.9.1\\",\\n    \\"jupyter_kernel_gateway\\": \\"2.5.2\\",\\n    \\"jupyter_server\\": \\"2.12.5\\",\\n    \\"jupyter_server_proxy\\": \\"4.6.0\\",\\n    \\"jupyter_server_terminals\\": \\"0.5.4\\",\\n    \\"jupyterlab\\": \\"4.5.11\\",\\n    \\"jupyterlab-lsp\\": \\"5.3.0\\",\\n    \\"jupyterlab_pygments\\": \\"0.3.0\\",\\n    \\"jupyterlab_server\\": \\"2.28.1\\",\\n    \\"jupyterlab_widgets\\": \\"3.0.17\\",\\n    \\"jupytext\\": \\"1.19.5\\",\\n    \\"kaggle\\": \\"2.0.2\\",\\n    \\"kaggle-environments\\": \\"1.32.7\\",\\n    \\"kagglehub\\": \\"1.0.2\\",\\n    \\"kagglesdk\\": \\"0.1.37\\",\\n    \\"keras\\": \\"3.13.2\\",\\n    \\"keras-core\\": \\"0.1.7\\",\\n    \\"keras-cv\\": \\"0.9.0\\",\\n    \\"keras-hub\\": \\"0.26.0\\",\\n    \\"keras-nlp\\": \\"0.26.0\\",\\n    \\"keras-tuner\\": \\"1.4.8\\",\\n    \\"keyring\\": \\"25.7.0\\",\\n    \\"keyrings.google-artifactregistry-auth\\": \\"1.1.2\\",\\n    \\"kiwisolver\\": \\"1.5.1\\",\\n    \\"kornia\\": \\"0.8.3\\",\\n    \\"kornia_rs\\": \\"0.2.0\\",\\n    \\"kt-legacy\\": \\"1.0.5\\",\\n    \\"langchain\\": \\"1.3.18\\",\\n    \\"langchain-core\\": \\"1.6.1\\",\\n    \\"langchain-protocol\\": \\"0.0.19\\",\\n    \\"langgraph\\": \\"1.2.11\\",\\n    \\"langgraph-checkpoint\\": \\"4.2.0\\",\\n    \\"langgraph-prebuilt\\": \\"1.1.0\\",\\n    \\"langgraph-sdk\\": \\"0.4.4\\",\\n    \\"langid\\": \\"1.1.6\\",\\n    \\"langsmith\\": \\"0.12.1\\",\\n    \\"lark\\": \\"1.3.1\\",\\n    \\"launchpadlib\\": \\"1.11.0\\",\\n    \\"lazr.restfulclient\\": \\"0.14.6\\",\\n    \\"lazr.uri\\": \\"1.0.6\\",\\n    \\"lazy-loader\\": \\"0.5\\",\\n    \\"learntools\\": \\"0.3.5\\",\\n    \\"libclang\\": \\"18.1.1\\",\\n    \\"libcudf-cu12\\": \\"26.2.1\\",\\n    \\"libcugraph-cu12\\": \\"26.2.0\\",\\n    \\"libcuml-cu12\\": \\"26.2.0\\",\\n    \\"libcuvs-cu12\\": \\"26.2.0\\",\\n    \\"libkvikio-cu12\\": \\"26.2.0\\",\\n    \\"libpysal\\": \\"4.14.1\\",\\n    \\"libraft-cu12\\": \\"26.2.0\\",\\n    \\"librmm-cu12\\": \\"26.2.0\\",\\n    \\"librosa\\": \\"0.11.0\\",\\n    \\"libucx-cu12\\": \\"1.19.0\\",\\n    \\"libucxx-cu12\\": \\"0.48.0\\",\\n    \\"lightgbm\\": \\"4.6.0\\",\\n    \\"lightning-utilities\\": \\"0.15.3\\",\\n    \\"lime\\": \\"0.2.0.1\\",\\n    \\"line_profiler\\": \\"5.1.0\\",\\n    \\"linkify-it-py\\": \\"2.2.0\\",\\n    \\"litellm\\": \\"1.93.2\\",\\n    \\"llvmlite\\": \\"0.44.0\\",\\n    \\"lml\\": \\"0.2.0\\",\\n    \\"locket\\": \\"1.0.0\\",\\n    \\"logical-unification\\": \\"0.4.7\\",\\n    \\"lxml\\": \\"6.1.2\\",\\n    \\"mamba\\": \\"0.11.3\\",\\n    \\"mapclassify\\": \\"2.10.0\\",\\n    \\"markdown-it-py\\": \\"4.2.0\\",\\n    \\"marshmallow\\": \\"3.26.2\\",\\n    \\"matplotlib\\": \\"3.10.0\\",\\n    \\"matplotlib-inline\\": \\"0.2.2\\",\\n    \\"matplotlib-venn\\": \\"1.1.2\\",\\n    \\"mdit-py-plugins\\": \\"0.6.1\\",\\n    \\"mdurl\\": \\"0.1.2\\",\\n    \\"mgwr\\": \\"2.2.1\\",\\n    \\"miniKanren\\": \\"1.0.5\\",\\n    \\"minify_html\\": \\"0.18.1\\",\\n    \\"missingno\\": \\"0.5.2\\",\\n    \\"mistune\\": \\"0.8.4\\",\\n    \\"mizani\\": \\"0.13.5\\",\\n    \\"mkl\\": \\"2025.3.1\\",\\n    \\"ml_collections\\": \\"1.1.0\\",\\n    \\"ml_dtypes\\": \\"0.6.0\\",\\n    \\"mlcrate\\": \\"0.2.0\\",\\n    \\"mlxtend\\": \\"0.23.4\\",\\n    \\"mmh3\\": \\"5.3.0\\",\\n    \\"mne\\": \\"1.13.2\\",\\n    \\"model-signing\\": \\"1.1.1\\",\\n    \\"momepy\\": \\"0.11.0\\",\\n    \\"moocore\\": \\"0.3.2\\",\\n    \\"more-itertools\\": \\"10.8.0\\",\\n    \\"moviepy\\": \\"1.0.3\\",\\n    \\"mpld3\\": \\"0.5.12\\",\\n    \\"mpmath\\": \\"1.3.0\\",\\n    \\"msgpack\\": \\"1.2.2\\",\\n    \\"multidict\\": \\"6.7.1\\",\\n    \\"multimethod\\": \\"1.12\\",\\n    \\"multipledispatch\\": \\"1.0.0\\",\\n    \\"multiprocess\\": \\"0.70.19\\",\\n    \\"multitasking\\": \\"0.0.13\\",\\n    \\"murmurhash\\": \\"1.0.15\\",\\n    \\"music21\\": \\"9.9.2\\",\\n    \\"mypy_extensions\\": \\"1.1.0\\",\\n    \\"namex\\": \\"0.1.0\\",\\n    \\"narwhals\\": \\"2.25.0\\",\\n    \\"natsort\\": \\"8.4.0\\",\\n    \\"nbclassic\\": \\"1.3.3\\",\\n    \\"nbclient\\": \\"0.5.13\\",\\n    \\"nbconvert\\": \\"6.4.5\\",\\n    \\"nbdev\\": \\"3.3.25\\",\\n    \\"nbformat\\": \\"5.11.1\\",\\n    \\"ndindex\\": \\"1.10.1\\",\\n    \\"nest-asyncio\\": \\"1.6.0\\",\\n    \\"networkx\\": \\"3.6.1\\",\\n    \\"nh3\\": \\"0.3.7\\",\\n    \\"nibabel\\": \\"5.4.2\\",\\n    \\"nilearn\\": \\"0.14.1\\",\\n    \\"ninja\\": \\"1.13.2\\",\\n    \\"nltk\\": \\"3.9.1\\",\\n    \\"notebook\\": \\"6.5.7\\",\\n    \\"notebook_shim\\": \\"0.2.4\\",\\n    \\"numba\\": \\"0.61.2\\",\\n    \\"numba-cuda\\": \\"0.22.2\\",\\n    \\"numexpr\\": \\"2.14.2\\",\\n    \\"numpy\\": \\"2.1.3\\",\\n    \\"nvidia-cublas-cu12\\": \\"12.8.4.1\\",\\n    \\"nvidia-cuda-cccl-cu12\\": \\"12.9.27\\",\\n    \\"nvidia-cuda-cupti-cu12\\": \\"12.8.90\\",\\n    \\"nvidia-cuda-nvcc-cu12\\": \\"12.8.93\\",\\n    \\"nvidia-cuda-nvrtc-cu12\\": \\"12.8.93\\",\\n    \\"nvidia-cuda-runtime-cu12\\": \\"12.8.90\\",\\n    \\"nvidia-cudnn-cu12\\": \\"9.19.0.56\\",\\n    \\"nvidia-cufft-cu12\\": \\"11.3.3.83\\",\\n    \\"nvidia-cufile-cu12\\": \\"1.13.1.3\\",\\n    \\"nvidia-curand-cu12\\": \\"10.3.9.90\\",\\n    \\"nvidia-cusolver-cu12\\": \\"11.7.3.90\\",\\n    \\"nvidia-cusparse-cu12\\": \\"12.5.8.93\\",\\n    \\"nvidia-cusparselt-cu12\\": \\"0.7.1\\",\\n    \\"nvidia-libnvcomp-cu12\\": \\"5.1.0.21\\",\\n    \\"nvidia-ml-py\\": \\"13.610.43\\",\\n    \\"nvidia-nccl-cu12\\": \\"2.28.9\\",\\n    \\"nvidia-nccl-cu13\\": \\"2.31.2\\",\\n    \\"nvidia-nvimgcodec-cu12\\": \\"0.7.0.11\\",\\n    \\"nvidia-nvjitlink-cu12\\": \\"12.8.93\\",\\n    \\"nvidia-nvshmem-cu12\\": \\"3.4.5\\",\\n    \\"nvidia-nvtx-cu12\\": \\"12.8.90\\",\\n    \\"nvtx\\": \\"0.2.16\\",\\n    \\"nx-cugraph-cu12\\": \\"26.2.0\\",\\n    \\"oauth2client\\": \\"4.1.3\\",\\n    \\"oauthlib\\": \\"3.3.1\\",\\n    \\"odfpy\\": \\"1.4.1\\",\\n    \\"olefile\\": \\"0.47\\",\\n    \\"omegaconf\\": \\"2.3.1\\",\\n    \\"onemkl-license\\": \\"2025.3.1\\",\\n    \\"onnx\\": \\"1.22.0\\",\\n    \\"open-spiel\\": \\"2.0.1\\",\\n    \\"openai\\": \\"2.54.0\\",\\n    \\"opencv-contrib-python\\": \\"4.14.0.94\\",\\n    \\"opencv-python\\": \\"4.14.0.94\\",\\n    \\"opencv-python-headless\\": \\"4.14.0.94\\",\\n    \\"openpyxl\\": \\"3.1.5\\",\\n    \\"openslide-bin\\": \\"4.0.1.2\\",\\n    \\"openslide-python\\": \\"1.4.6\\",\\n    \\"opentelemetry-api\\": \\"1.42.1\\",\\n    \\"opentelemetry-resourcedetector-gcp\\": \\"1.14.0\\",\\n    \\"opentelemetry-sdk\\": \\"1.42.1\\",\\n    \\"opentelemetry-semantic-conventions\\": \\"0.63b1\\",\\n    \\"opt_einsum\\": \\"3.4.0\\",\\n    \\"optax\\": \\"0.2.8\\",\\n    \\"optree\\": \\"0.20.0\\",\\n    \\"optuna\\": \\"5.0.0\\",\\n    \\"orbax-checkpoint\\": \\"0.12.4\\",\\n    \\"orderly-set\\": \\"5.5.0\\",\\n    \\"orjson\\": \\"3.12.0\\",\\n    \\"ormsgpack\\": \\"1.12.2\\",\\n    \\"osqp\\": \\"1.1.3\\",\\n    \\"overrides\\": \\"7.7.0\\",\\n    \\"packaging\\": \\"26.3\\",\\n    \\"pandas\\": \\"2.3.3\\",\\n    \\"pandas-datareader\\": \\"0.11.1\\",\\n    \\"pandas-gbq\\": \\"0.30.0\\",\\n    \\"pandas-profiling\\": \\"3.6.6\\",\\n    \\"pandas-stubs\\": \\"2.2.3.250527\\",\\n    \\"pandasql\\": \\"0.7.3\\",\\n    \\"pandocfilters\\": \\"1.5.1\\",\\n    \\"panel\\": \\"1.9.4\\",\\n    \\"panel-material-ui\\": \\"0.14.2\\",\\n    \\"papermill\\": \\"2.7.0\\",\\n    \\"param\\": \\"2.4.1\\",\\n    \\"parso\\": \\"0.8.7\\",\\n    \\"parsy\\": \\"2.2\\",\\n    \\"partd\\": \\"1.4.2\\",\\n    \\"path\\": \\"17.1.1\\",\\n    \\"path.py\\": \\"12.5.0\\",\\n    \\"pathlib_abc\\": \\"0.5.2\\",\\n    \\"pathos\\": \\"0.3.5\\",\\n    \\"pathspec\\": \\"1.1.1\\",\\n    \\"patsy\\": \\"1.0.3\\",\\n    \\"pdf2image\\": \\"1.17.0\\",\\n    \\"peewee\\": \\"4.4.0\\",\\n    \\"peft\\": \\"0.20.0\\",\\n    \\"pettingzoo\\": \\"1.24.0\\",\\n    \\"pexpect\\": \\"4.9.0\\",\\n    \\"phik\\": \\"0.12.5\\",\\n    \\"pickleshare\\": \\"0.7.5\\",\\n    \\"pillow\\": \\"13.0.0\\",\\n    \\"pip\\": \\"26.2.2\\",\\n    \\"platformdirs\\": \\"4.11.7\\",\\n    \\"plotly\\": \\"5.24.1\\",\\n    \\"plotly-express\\": \\"0.4.1\\",\\n    \\"plotnine\\": \\"0.14.5\\",\\n    \\"pluggy\\": \\"1.6.0\\",\\n    \\"plum-dispatch\\": \\"2.9.0\\",\\n    \\"pointpats\\": \\"2.5.5\\",\\n    \\"pokerkit\\": \\"0.6.3\\",\\n    \\"polars\\": \\"1.35.2\\",\\n    \\"polars-runtime-32\\": \\"1.35.2\\",\\n    \\"pooch\\": \\"1.9.0\\",\\n    \\"portpicker\\": \\"1.5.2\\",\\n    \\"pox\\": \\"0.3.7\\",\\n    \\"ppft\\": \\"1.7.8\\",\\n    \\"preshed\\": \\"3.0.13\\",\\n    \\"prettytable\\": \\"3.18.0\\",\\n    \\"proglog\\": \\"0.1.12\\",\\n    \\"progressbar2\\": \\"4.5.0\\",\\n    \\"prometheus_client\\": \\"0.26.0\\",\\n    \\"promise\\": \\"2.3\\",\\n    \\"prompt_toolkit\\": \\"3.0.53\\",\\n    \\"propcache\\": \\"0.5.2\\",\\n    \\"prophet\\": \\"1.4.0\\",\\n    \\"proto-plus\\": \\"1.28.2\\",\\n    \\"protobuf\\": \\"5.29.5\\",\\n    \\"psutil\\": \\"5.9.5\\",\\n    \\"psycopg2\\": \\"2.9.12\\",\\n    \\"psygnal\\": \\"0.15.1\\",\\n    \\"ptyprocess\\": \\"0.7.0\\",\\n    \\"pudb\\": \\"2026.1\\",\\n    \\"puremagic\\": \\"2.2.0\\",\\n    \\"py-cpuinfo\\": \\"9.0.0\\",\\n    \\"py4j\\": \\"0.10.9.9\\",\\n    \\"pyLDAvis\\": \\"3.4.1\\",\\n    \\"pyOpenSSL\\": \\"26.4.0\\",\\n    \\"pyaml\\": \\"26.7.0\\",\\n    \\"pyarrow\\": \\"23.0.1\\",\\n    \\"pyasn1\\": \\"0.6.4\\",\\n    \\"pyasn1_modules\\": \\"0.4.2\\",\\n    \\"pybind11\\": \\"3.1.0\\",\\n    \\"pycairo\\": \\"1.29.1\\",\\n    \\"pyclipper\\": \\"1.4.0\\",\\n    \\"pycocotools\\": \\"2.0.11\\",\\n    \\"pycparser\\": \\"3.0\\",\\n    \\"pycryptodome\\": \\"4.1.0\\",\\n    \\"pycryptodomex\\": \\"3.23.0\\",\\n    \\"pycuda\\": \\"2026.1\\",\\n    \\"pydantic\\": \\"2.13.5\\",\\n    \\"pydantic_core\\": \\"2.46.5\\",\\n    \\"pydata-google-auth\\": \\"1.9.1\\",\\n    \\"pydicom\\": \\"3.0.2\\",\\n    \\"pydot\\": \\"4.0.1\\",\\n    \\"pydotplus\\": \\"2.0.2\\",\\n    \\"pydub\\": \\"0.25.1\\",\\n    \\"pyemd\\": \\"2.0.0\\",\\n    \\"pyerfa\\": \\"2.0.1.5\\",\\n    \\"pyexcel-io\\": \\"0.6.8\\",\\n    \\"pyexcel-ods\\": \\"0.6.0\\",\\n    \\"pygame\\": \\"2.6.1\\",\\n    \\"pygit2\\": \\"1.20.0\\",\\n    \\"pygltflib\\": \\"1.16.5\\",\\n    \\"pyiceberg\\": \\"0.12.0\\",\\n    \\"pyjson5\\": \\"2.0.1\\",\\n    \\"pylibcudf-cu12\\": \\"26.2.1\\",\\n    \\"pylibcugraph-cu12\\": \\"26.2.0\\",\\n    \\"pylibraft-cu12\\": \\"26.2.0\\",\\n    \\"pymc\\": \\"5.28.5\\",\\n    \\"pymongo\\": \\"4.19.0\\",\\n    \\"pynndescent\\": \\"0.6.0\\",\\n    \\"pyogrio\\": \\"0.13.0\\",\\n    \\"pyomo\\": \\"6.10.1\\",\\n    \\"pyparsing\\": \\"3.3.2\\",\\n    \\"pypdf\\": \\"6.21.2\\",\\n    \\"pyperclip\\": \\"1.11.0\\",\\n    \\"pyproj\\": \\"3.7.2\\",\\n    \\"pyproject_hooks\\": \\"1.3.3\\",\\n    \\"pyroaring\\": \\"1.1.0\\",\\n    \\"pysal\\": \\"25.7\\",\\n    \\"pyshp\\": \\"3.1.6\\",\\n    \\"pyspark\\": \\"4.0.4\\",\\n    \\"pytensor\\": \\"2.38.3\\",\\n    \\"pytesseract\\": \\"0.3.13\\",\\n    \\"pytest\\": \\"8.4.2\\",\\n    \\"python-apt\\": \\"2.7.7+ubuntu5.2\\",\\n    \\"python-bidi\\": \\"0.6.12\\",\\n    \\"python-box\\": \\"7.4.1\\",\\n    \\"python-dateutil\\": \\"2.9.0.post0\\",\\n    \\"python-dotenv\\": \\"1.2.3\\",\\n    \\"python-fasthtml\\": \\"0.14.12\\",\\n    \\"python-json-logger\\": \\"4.2.0\\",\\n    \\"python-louvain\\": \\"0.16\\",\\n    \\"python-lsp-jsonrpc\\": \\"1.1.2\\",\\n    \\"python-lsp-server\\": \\"1.15.0\\",\\n    \\"python-multipart\\": \\"0.0.32\\",\\n    \\"python-slugify\\": \\"8.0.4\\",\\n    \\"python-snappy\\": \\"0.7.3\\",\\n    \\"python-utils\\": \\"4.0.1\\",\\n    \\"pytokens\\": \\"0.4.1\\",\\n    \\"pytools\\": \\"2026.1.1\\",\\n    \\"pytorch-ignite\\": \\"0.5.5\\",\\n    \\"pytorch-lightning\\": \\"2.6.6\\",\\n    \\"pytz\\": \\"2025.2\\",\\n    \\"pyviz_comms\\": \\"3.0.6\\",\\n    \\"pyzmq\\": \\"26.2.1\\",\\n    \\"qgrid\\": \\"1.3.1\\",\\n    \\"qtconsole\\": \\"5.7.2\\",\\n    \\"quantecon\\": \\"0.11.4\\",\\n    \\"raft-dask-cu12\\": \\"26.2.0\\",\\n    \\"rapids-dask-dependency\\": \\"26.2.0\\",\\n    \\"rapids-logger\\": \\"0.2.3\\",\\n    \\"rasterio\\": \\"1.5.1\\",\\n    \\"rasterstats\\": \\"0.21.0\\",\\n    \\"ratelim\\": \\"0.1.6\\",\\n    \\"ray\\": \\"2.59.0\\",\\n    \\"rdkit\\": \\"2026.3.3\\",\\n    \\"referencing\\": \\"0.37.0\\",\\n    \\"regex\\": \\"2025.11.3\\",\\n    \\"requests\\": \\"2.34.2\\",\\n    \\"requests-oauthlib\\": \\"2.0.0\\",\\n    \\"requests-toolbelt\\": \\"1.0.0\\",\\n    \\"requirements-parser\\": \\"0.9.0\\",\\n    \\"rfc3161-client\\": \\"1.0.9\\",\\n    \\"rfc3339-validator\\": \\"0.1.4\\",\\n    \\"rfc3986-validator\\": \\"0.1.1\\",\\n    \\"rfc3987-syntax\\": \\"1.1.0\\",\\n    \\"rfc8785\\": \\"0.1.4\\",\\n    \\"rgf-python\\": \\"3.12.0\\",\\n    \\"rich\\": \\"13.9.4\\",\\n    \\"rmm-cu12\\": \\"26.2.0\\",\\n    \\"roman-numerals\\": \\"4.1.0\\",\\n    \\"roman-numerals-py\\": \\"4.1.0\\",\\n    \\"rouge_score\\": \\"0.1.2\\",\\n    \\"rpds-py\\": \\"2026.6.3\\",\\n    \\"rpy2\\": \\"3.5.17\\",\\n    \\"rsa\\": \\"4.9.1\\",\\n    \\"rtree\\": \\"1.4.1\\",\\n    \\"ruamel.yaml\\": \\"0.19.1\\",\\n    \\"s3fs\\": \\"2025.12.0\\",\\n    \\"s3transfer\\": \\"0.15.0\\",\\n    \\"safehttpx\\": \\"0.1.7\\",\\n    \\"safetensors\\": \\"0.8.0\\",\\n    \\"scikit-image\\": \\"0.25.2\\",\\n    \\"scikit-learn\\": \\"1.6.1\\",\\n    \\"scikit-multilearn\\": \\"0.2.0\\",\\n    \\"scikit-optimize\\": \\"0.10.2\\",\\n    \\"scikit-plot\\": \\"0.3.7\\",\\n    \\"scikit-surprise\\": \\"1.1.5\\",\\n    \\"scipy\\": \\"1.16.3\\",\\n    \\"scooby\\": \\"0.11.2\\",\\n    \\"scs\\": \\"3.2.11\\",\\n    \\"seaborn\\": \\"0.13.2\\",\\n    \\"securesystemslib\\": \\"1.6.0\\",\\n    \\"segment_anything\\": \\"1.0\\",\\n    \\"segregation\\": \\"2.5.4\\",\\n    \\"semantic-version\\": \\"2.10.0\\",\\n    \\"sentence-transformers\\": \\"5.7.0\\",\\n    \\"sentencepiece\\": \\"0.2.2\\",\\n    \\"sentry-sdk\\": \\"2.68.1\\",\\n    \\"setuptools\\": \\"81.0.0\\",\\n    \\"shap\\": \\"0.52.0\\",\\n    \\"shapely\\": \\"2.1.2\\",\\n    \\"shellingham\\": \\"1.5.4\\",\\n    \\"sigstore\\": \\"4.5.1\\",\\n    \\"sigstore-models\\": \\"0.0.6\\",\\n    \\"sigstore-rekor-types\\": \\"0.0.18\\",\\n    \\"simpervisor\\": \\"1.0.0\\",\\n    \\"simple-parsing\\": \\"0.1.9\\",\\n    \\"simpleitk\\": \\"2.5.6\\",\\n    \\"simplejson\\": \\"4.1.2\\",\\n    \\"simsimd\\": \\"6.5.16\\",\\n    \\"siphash24\\": \\"1.9\\",\\n    \\"six\\": \\"1.17.0\\",\\n    \\"sklearn-compat\\": \\"0.1.6\\",\\n    \\"sklearn-pandas\\": \\"2.2.0\\",\\n    \\"slicer\\": \\"0.0.8\\",\\n    \\"smart_open\\": \\"8.0.1\\",\\n    \\"sniffio\\": \\"1.3.1\\",\\n    \\"snowballstemmer\\": \\"3.1.1\\",\\n    \\"sortedcontainers\\": \\"2.4.0\\",\\n    \\"soundfile\\": \\"0.14.0\\",\\n    \\"soupsieve\\": \\"2.9.2\\",\\n    \\"soxr\\": \\"1.1.0\\",\\n    \\"spacy\\": \\"3.8.16\\",\\n    \\"spacy-legacy\\": \\"3.0.12\\",\\n    \\"spacy-loggers\\": \\"1.0.5\\",\\n    \\"spaghetti\\": \\"1.7.6\\",\\n    \\"spanner-graph-notebook\\": \\"1.1.10\\",\\n    \\"spglm\\": \\"1.1.0\\",\\n    \\"sphinxcontrib-applehelp\\": \\"2.0.0\\",\\n    \\"sphinxcontrib-devhelp\\": \\"2.0.0\\",\\n    \\"sphinxcontrib-htmlhelp\\": \\"2.1.0\\",\\n    \\"sphinxcontrib-jsmath\\": \\"1.0.1\\",\\n    \\"sphinxcontrib-qthelp\\": \\"2.0.0\\",\\n    \\"sphinxcontrib-serializinghtml\\": \\"2.0.0\\",\\n    \\"spint\\": \\"1.1.0\\",\\n    \\"splot\\": \\"1.1.7\\",\\n    \\"spopt\\": \\"0.7.0\\",\\n    \\"spreg\\": \\"1.9.0\\",\\n    \\"sqlglot\\": \\"25.20.2\\",\\n    \\"sqlparse\\": \\"0.6.0\\",\\n    \\"squarify\\": \\"0.4.5\\",\\n    \\"srsly\\": \\"2.5.3\\",\\n    \\"standard-aifc\\": \\"3.13.0\\",\\n    \\"standard-chunk\\": \\"3.13.0\\",\\n    \\"standard-sunau\\": \\"3.13.0\\",\\n    \\"stanio\\": \\"0.5.1\\",\\n    \\"starlette\\": \\"1.6.0\\",\\n    \\"statsmodels\\": \\"0.15.0\\",\\n    \\"stopit\\": \\"1.1.2\\",\\n    \\"strictyaml\\": \\"1.7.3\\",\\n    \\"stringzilla\\": \\"5.1.2\\",\\n    \\"stumpy\\": \\"1.14.1\\",\\n    \\"sympy\\": \\"1.14.0\\",\\n    \\"tables\\": \\"3.10.2\\",\\n    \\"tabulate\\": \\"0.9.0\\",\\n    \\"tbb\\": \\"2022.3.1\\",\\n    \\"tblib\\": \\"3.2.2\\",\\n    \\"tcmlib\\": \\"1.5.0\\",\\n    \\"tenacity\\": \\"9.1.4\\",\\n    \\"tensorboard\\": \\"2.20.0\\",\\n    \\"tensorboard-data-server\\": \\"0.7.2\\",\\n    \\"tensorflow\\": \\"2.20.0\\",\\n    \\"tensorflow-datasets\\": \\"4.9.10\\",\\n    \\"tensorflow-hub\\": \\"0.16.1\\",\\n    \\"tensorflow-metadata\\": \\"1.21.0\\",\\n    \\"tensorflow-probability\\": \\"0.25.0\\",\\n    \\"tensorflow-text\\": \\"2.20.1\\",\\n    \\"tensorstore\\": \\"0.1.85\\",\\n    \\"termcolor\\": \\"3.3.0\\",\\n    \\"terminado\\": \\"0.18.1\\",\\n    \\"testpath\\": \\"0.6.0\\",\\n    \\"text-unidecode\\": \\"1.3\\",\\n    \\"textblob\\": \\"0.19.0\\",\\n    \\"texttable\\": \\"1.7.2\\",\\n    \\"tf-slim\\": \\"1.1.0\\",\\n    \\"tf_keras\\": \\"2.20.1\\",\\n    \\"thinc\\": \\"8.3.13\\",\\n    \\"threadpoolctl\\": \\"3.6.0\\",\\n    \\"tifffile\\": \\"2026.8.23\\",\\n    \\"tiktoken\\": \\"0.14.0\\",\\n    \\"timm\\": \\"1.0.29\\",\\n    \\"tinycss2\\": \\"1.5.1\\",\\n    \\"tobler\\": \\"0.14.0\\",\\n    \\"tokenizers\\": \\"0.23.1\\",\\n    \\"toml\\": \\"0.10.2\\",\\n    \\"tomlkit\\": \\"0.14.0\\",\\n    \\"toolz\\": \\"0.12.1\\",\\n    \\"torch\\": \\"2.11.0+cu128\\",\\n    \\"torchao\\": \\"0.10.0\\",\\n    \\"torchaudio\\": \\"2.11.0+cu128\\",\\n    \\"torchcodec\\": \\"0.11.0+cu128\\",\\n    \\"torchdata\\": \\"0.11.0\\",\\n    \\"torchinfo\\": \\"1.8.0\\",\\n    \\"torchmetrics\\": \\"1.9.0\\",\\n    \\"torchsummary\\": \\"1.5.1\\",\\n    \\"torchtune\\": \\"0.6.1\\",\\n    \\"torchvision\\": \\"0.26.0+cu128\\",\\n    \\"tornado\\": \\"6.5.7\\",\\n    \\"tqdm\\": \\"4.67.3\\",\\n    \\"traitlets\\": \\"5.16.1\\",\\n    \\"traittypes\\": \\"0.2.3\\",\\n    \\"transformers\\": \\"5.16.1\\",\\n    \\"treelite\\": \\"4.7.1\\",\\n    \\"treescope\\": \\"0.1.10\\",\\n    \\"triton\\": \\"3.6.0\\",\\n    \\"truststore\\": \\"0.10.4\\",\\n    \\"trx-python\\": \\"0.6\\",\\n    \\"tsfresh\\": \\"0.21.2\\",\\n    \\"tuf\\": \\"7.0.1\\",\\n    \\"tweepy\\": \\"4.17.0\\",\\n    \\"typeguard\\": \\"4.6.0\\",\\n    \\"typer\\": \\"0.27.2\\",\\n    \\"types-pytz\\": \\"2026.3.1.20260727\\",\\n    \\"types-setuptools\\": \\"84.0.0.20260812\\",\\n    \\"typing-inspect\\": \\"0.9.0\\",\\n    \\"typing-inspection\\": \\"0.4.4\\",\\n    \\"typing_extensions\\": \\"4.16.0\\",\\n    \\"tzdata\\": \\"2026.3\\",\\n    \\"tzlocal\\": \\"5.4.4\\",\\n    \\"ucxx-cu12\\": \\"0.48.0\\",\\n    \\"ujson\\": \\"6.0.0\\",\\n    \\"umap-learn\\": \\"0.5.12\\",\\n    \\"umf\\": \\"1.0.3\\",\\n    \\"universal_pathlib\\": \\"0.3.10\\",\\n    \\"update_checker\\": \\"1.0.1\\",\\n    \\"uri-template\\": \\"1.3.0\\",\\n    \\"uritemplate\\": \\"4.2.0\\",\\n    \\"urllib3\\": \\"2.8.0\\",\\n    \\"urwid\\": \\"4.2.5\\",\\n    \\"urwid_readline\\": \\"0.15.1\\",\\n    \\"uuid_utils\\": \\"0.17.0\\",\\n    \\"uv\\": \\"0.13.5\\",\\n    \\"uvicorn\\": \\"0.52.4\\",\\n    \\"uvloop\\": \\"0.22.1\\",\\n    \\"vega-datasets\\": \\"0.9.0\\",\\n    \\"visions\\": \\"0.8.1\\",\\n    \\"vtk\\": \\"9.7.1\\",\\n    \\"wadllib\\": \\"1.3.6\\",\\n    \\"wandb\\": \\"0.28.1\\",\\n    \\"wasabi\\": \\"1.1.3\\",\\n    \\"watchdog\\": \\"6.0.0\\",\\n    \\"watchfiles\\": \\"1.2.0\\",\\n    \\"wavio\\": \\"0.0.9\\",\\n    \\"wcwidth\\": \\"0.8.3\\",\\n    \\"weasel\\": \\"1.0.0\\",\\n    \\"webcolors\\": \\"25.10.0\\",\\n    \\"webencodings\\": \\"0.6.1\\",\\n    \\"websocket-client\\": \\"1.9.2\\",\\n    \\"websockets\\": \\"15.0.1\\",\\n    \\"wheel\\": \\"0.48.0\\",\\n    \\"widgetsnbextension\\": \\"4.0.16\\",\\n    \\"woodwork\\": \\"0.31.0\\",\\n    \\"wordcloud\\": \\"1.9.6\\",\\n    \\"wrapt\\": \\"1.17.3\\",\\n    \\"xarray\\": \\"2025.12.0\\",\\n    \\"xarray-einstats\\": \\"0.11.0\\",\\n    \\"xgboost\\": \\"3.4.1\\",\\n    \\"xlrd\\": \\"2.0.2\\",\\n    \\"xvfbwrapper\\": \\"0.2.35\\",\\n    \\"xxhash\\": \\"4.0.1\\",\\n    \\"xyzservices\\": \\"2026.3.0\\",\\n    \\"yarl\\": \\"1.24.5\\",\\n    \\"ydata-profiling\\": \\"4.18.4\\",\\n    \\"ydf\\": \\"0.15.0\\",\\n    \\"ydf_tf\\": \\"2.20.0\\",\\n    \\"yellowbrick\\": \\"1.5\\",\\n    \\"yfinance\\": \\"0.2.66\\",\\n    \\"zict\\": \\"3.0.0\\",\\n    \\"zipp\\": \\"4.1.0\\",\\n    \\"zstandard\\": \\"0.25.0\\"\\n  }\\n}\\n", "reports/gnps_expanded_external_audit.json": "{\\n  \\"schema_version\\": 1,\\n  \\"status\\": \\"EXTERNAL_INDEPENDENCE_NOT_ESTABLISHED\\",\\n  \\"checked_at\\": \\"2026-10-28T14:28:12.998471+00:00\\",\\n  \\"protocol_sha256\\": \\"c7e06f57470e3df618d4b4d37a23729c1fc7fd0d3cde726f727aa31987f5f17a\\",\\n  \\"inputs\\": {\\n    \\"train.parquet\\": {\\n      \\"sha256\\": \\"9423f90ec08b941a9646b535bf0606183e86ff0c5a297978f5a70693d41bcd98\\",\\n      \\"bytes\\": 3033286496\\n    },\\n    \\"gnps_cc0.parquet\\": {\\n      \\"sha256\\": \\"4bb0db3c41b85fdeef67b4859d46c3d9afa894bdca08652f7a965fda29b8910f\\",\\n      \\"bytes\\": 13826158\\n    }\\n  },\\n  \\"external_source\\": {\\n    \\"schema_version\\": 1,\\n    \\"status\\": \\"PINNED_CC0_EXTERNAL_INPUT_PREPARED\\",\\n    \\"source\\": {\\n      \\"repository\\": \\"https://library.gnps2.org/libraries\\",\\n      \\"retrieved_on\\": \\"2026-10-27\\",\\n      \\"license_policy\\": \\"Directly contributed GNPS reference spectra default to CC0; exclude third-party imported and propagated/MassQL/suspect libraries.\\",\\n      \\"license_reference\\": \\"https://ccms-ucsd.github.io/GNPSDocumentation/browselibraries/#license\\",\\n      \\"allowed_record_licenses\\": [\\n        \\"CC0\\"\\n      ],\\n      \\"files\\": {\\n        \\"CEMBIO_OXYLIPINS.json\\": {\\n          \\"url\\": \\"https://library.gnps2.org/api/libraries/CEMBIO_OXYLIPINS/json\\",\\n          \\"sha256\\": \\"5bb461648eb111304b8e47765a177a01b443b9ad0589038ea5f261428814130e\\",\\n          \\"bytes\\": 192299\\n        },\\n        \\"USER-CONTRIBUTED-GNPS2.json\\": {\\n          \\"url\\": \\"https://library.gnps2.org/api/libraries/USER-CONTRIBUTED-GNPS2/json\\",\\n          \\"sha256\\": \\"3146f450ba71009edac2375041a05de3fb1dae282903119074701a84fbc15e7a\\",\\n          \\"bytes\\": 302638\\n        },\\n        \\"GNPS-LIBRARY.json\\": {\\n          \\"url\\": \\"https://library.gnps2.org/api/libraries/GNPS-LIBRARY/json\\",\\n          \\"sha256\\": \\"d463bc39362359329a520d8fdd1b352abc4807188fc6ef465501183b1a261264\\",\\n          \\"bytes\\": 334137083\\n        }\\n      },\\n      \\"eligibility\\": \\"Source-classified standard spectra; MS2, ESI evidenced in source/instrument annotation; supported adduct and matching mode; connected valid molecular structure, 150-1170 Da, precursor error <=20 ppm, at least five finite positive ordered peaks; reject explicit low-quality or inferred/synthetic acquisition annotations. Preserve raw record hash, original metadata and source URL. GNPS2 snapshot-relative record IDs are derived hashes, not claimed official accessions.\\"\\n    },\\n    \\"counts\\": {\\n      \\"source_records\\": 12842,\\n      \\"unverified_acquisition_or_quality\\": 8256,\\n      \\"unsupported_adduct\\": 611,\\n      \\"outside_mass_range\\": 790,\\n      \\"invalid_peaks\\": 25,\\n      \\"malformed_record\\": 82,\\n      \\"precursor_mass_conflict\\": 18,\\n      \\"invalid_structure\\": 9,\\n      \\"eligible_spectra\\": 3051,\\n      \\"eligible_metric_molecules\\": 936\\n    },\\n    \\"parquet_sha256\\": \\"4bb0db3c41b85fdeef67b4859d46c3d9afa894bdca08652f7a965fda29b8910f\\",\\n    \\"parquet_bytes\\": 13826158,\\n    \\"rdkit\\": \\"2026.03.3\\",\\n    \\"scope\\": \\"Source preparation only; CASMI overlap and external independence not yet established\\"\\n  },\\n  \\"identity_audit\\": {\\n    \\"training_rows_scanned\\": 2539608,\\n    \\"training_standard_identities\\": 275810,\\n    \\"distinct_training_smiles\\": 277566,\\n    \\"unparseable_training_labels\\": 0,\\n    \\"training_smiles_sharing_external_composition\\": 43310,\\n    \\"external_standard_overlap_molecules\\": 926,\\n    \\"external_metric_overlap_molecules\\": 925,\\n    \\"external_exact_content_overlap_molecules\\": 0,\\n    \\"excluded_external_molecules_union\\": 929,\\n    \\"novel_external_molecules\\": 7,\\n    \\"novel_external_spectra\\": 11,\\n    \\"novel_metric_identities_sha256\\": \\"b60cc9d8b7bb3fa2cbe5e66456f16ae84ca4ce49c9faf660b3ace203bb29fd2b\\",\\n    \\"novel_accessions_sha256\\": \\"e6a54c12deca0227ab6f8b04edb0758bb12634cbac0b2d5ac7763aa639dcdbf0\\"\\n  },\\n  \\"stored_content_audit\\": {\\n    \\"schema_version\\": 1,\\n    \\"status\\": \\"COMPLETE_CONTENT_AUDIT\\",\\n    \\"matching_method\\": \\"exact stored precursor and full ordered m/z/intensity numeric arrays; no rounding or tolerance\\",\\n    \\"metadata_fields\\": [\\n      \\"adduct\\",\\n      \\"ionization_mode\\"\\n    ],\\n    \\"train_rows_scanned\\": 2539608,\\n    \\"public_spectra\\": 3051,\\n    \\"public_molecules\\": 936,\\n    \\"exact_precursor_candidate_rows\\": 39945,\\n    \\"coarse_candidate_pairs_after_precursor_filter\\": 2861,\\n    \\"coarse_false_match_pairs\\": 2861,\\n    \\"exact_matching_pairs\\": 0,\\n    \\"content_matched_spectra\\": 0,\\n    \\"metadata_consistent_matched_spectra\\": 0,\\n    \\"metadata_disagreement_pairs\\": 0,\\n    \\"fully_content_matched_molecules\\": 0,\\n    \\"fully_metadata_consistent_molecules\\": 0,\\n    \\"content_matches_by_library\\": {},\\n    \\"limitations\\": [\\n      \\"Source-library match counts overlap and must not be summed as unique public spectra.\\",\\n      \\"Stored numerical equality does not independently prove common acquisition identity.\\",\\n      \\"This audit concerns mounted public examples, not the hidden competition evaluation set.\\",\\n      \\"No model evaluation, training or competition submission was performed.\\"\\n    ]\\n  },\\n  \\"model_performance_inspected\\": false,\\n  \\"limitations\\": [\\n    \\"External source acquisition is independently retrieved public reference data, not a new laboratory replication.\\",\\n    \\"Identity checks rely on source chemical annotations and the official computational canonicalizer.\\",\\n    \\"Exact stored-content comparison cannot detect arbitrary preprocessing/normalization variants of a common acquisition.\\",\\n    \\"This feasibility stage does not establish supervised-family separation, model accuracy or a C3 coverage improvement.\\"\\n  ],\\n  \\"source_code_sha256\\": {\\n    \\"src/casmi/__init__.py\\": \\"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\",\\n    \\"src/casmi/channels/__init__.py\\": \\"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\",\\n    \\"src/casmi/channels/analog.py\\": \\"8592dcfc274257db6f4a803cb705ec6be4a5dbba2abddec626eddd01d56b4c19\\",\\n    \\"src/casmi/channels/bondmodel.py\\": \\"1ffb5ce34c7b1a1975277eb9eab76a11e7624ea8740f39e5c2cc19a1340680fb\\",\\n    \\"src/casmi/channels/fpinfer.py\\": \\"58315b7f7288a2988980b19a079ef0eb305897ffcd5094b1dcf5d3b370a71a7b\\",\\n    \\"src/casmi/channels/fpmodel.py\\": \\"1e8a63dcd407f7870918c34fe5dac4f8fbfcbd301d64806d077019e382ad70ea\\",\\n    \\"src/casmi/channels/fragments.py\\": \\"20656968ee6084ed948547a90ca79957a5b3cc6a65002ffb8601a411f0278bc0\\",\\n    \\"src/casmi/channels/library.py\\": \\"96dd852858d6e66319d174306091d05480c965037d8e6dc50a87985f2a4398c2\\",\\n    \\"src/casmi/channels/prefilter.py\\": \\"ac442621394b1d4ad1627f7776cf8f4d704b1a8e3d950f0480168e54d3b22a57\\",\\n    \\"src/casmi/channels/reference_selection.py\\": \\"c043a1ab42126d3b33c5de0870ee1a5922396956a69356a777149ebc9947d88b\\",\\n    \\"src/casmi/chem/__init__.py\\": \\"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\",\\n    \\"src/casmi/chem/canon.py\\": \\"917d7002e84a047d8187187b6c2716da5adb013764bf6945e3748e2c63f5ee9e\\",\\n    \\"src/casmi/chem/keys.py\\": \\"2755a82fe24b854df0326c212df3696f0ac6197e46c3acdc5d33f4264e69aed2\\",\\n    \\"src/casmi/chem/masses.py\\": \\"1831039a9dd48ddfd92482b361c3c28353a0fc1fbff98aa72c5273adcb948cf5\\",\\n    \\"src/casmi/chem/npscore.py\\": \\"5fa24e1495bed25dfcd7001bb412e738d32148c5b3cab9dd3f25a28bb99a8945\\",\\n    \\"src/casmi/config.py\\": \\"5f8dc8985d2789bd93ae55b8bec11964f89472e3e710012c86c2c186e547b876\\",\\n    \\"src/casmi/engine.py\\": \\"3ec4a688c2e7b6a5542ecec159d85a2f71a4835174e9fa2e05239ba299f2e984\\",\\n    \\"src/casmi/generate/__init__.py\\": \\"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\",\\n    \\"src/casmi/generate/expanded.py\\": \\"e40c0abef2875f82b3f4343e7e51f13bafd6801bfe87543dc9f5e9fd6d00d090\\",\\n    \\"src/casmi/generate/transforms.py\\": \\"e43cca55be73612bdbf81631b888be0c0123650f751fd8465808747574f0ef49\\",\\n    \\"src/casmi/pool/__init__.py\\": \\"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\",\\n    \\"src/casmi/pool/build_safety.py\\": \\"2421c9f41d26ab6e17ac20956ef1bfd5552ed28ae7255f13ca8b4fd585a79305\\",\\n    \\"src/casmi/pool/fingerprint_audit.py\\": \\"d0197f06cbe89d54494eed5bf8693f8ab0944668ce006012726b758c6b44c358\\",\\n    \\"src/casmi/pool/fingerprint_provenance.py\\": \\"ec64027e7120b8636408f13d2a617a012c5ddb1f97dae7c475d341f44b8e97fe\\",\\n    \\"src/casmi/pool/pool.py\\": \\"178d16b9ffb9b26052dd28adeff387794db9f5b405fb5a4e347faeefd14ae634\\",\\n    \\"src/casmi/pool/source_reconciliation.py\\": \\"268cf27522e22659fdb0560784ef787c7b25f66dc739a135e5d8d5b97e5fe4c2\\",\\n    \\"src/casmi/pool/structure_identity.py\\": \\"39519cf6128de5a8677e5675ac9f691b8943fb5386337320ff1d05bfdb2c146a\\",\\n    \\"src/casmi/pool/structure_quality.py\\": \\"654d3c57b836d857367c404c6eeb6f9d010ef710c36a3874dafbe2382f31d646\\",\\n    \\"src/casmi/predict.py\\": \\"d5416dc486fbab178da9073d48fa3c8adf6e4b224f11bc1af5c07c6e0cb2a2e1\\",\\n    \\"src/casmi/rank/__init__.py\\": \\"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\",\\n    \\"src/casmi/rank/features.py\\": \\"578101099d2519aeb073244576e52368ad2915411bbe92a326496ad134b77dcc\\",\\n    \\"src/casmi/rank/ranker.py\\": \\"8b58478f310016485d8a877ff96f3fbac96278497dc0885ec267d294b11c3665\\",\\n    \\"src/casmi/spectra/__init__.py\\": \\"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\",\\n    \\"src/casmi/spectra/build_safety.py\\": \\"b70cab5a2af895061c6786811a2277e899475b7f9eaad29033f2b8990e8e04c0\\",\\n    \\"src/casmi/spectra/clean.py\\": \\"af67f80cbd5e5d6b489ed110be5dd943446a6e38d45a8976365a3c13c07a2bb6\\",\\n    \\"src/casmi/spectra/similarity.py\\": \\"4c63e21f048facdf4a1fbf2e07ec840a078c77b4f1b99f553d77db66cec70b4c\\",\\n    \\"src/casmi/spectra/store.py\\": \\"6b7cd04ac0350ce204a066a82bc250b8e60b552bdfbb95221152399bfdffe13c\\",\\n    \\"src/casmi/spectra/store_integrity.py\\": \\"3664d4c6278bea71c3fa403f017deda8af8dde7de9cb0afbf4c9424835d68243\\",\\n    \\"src/casmi/submit/__init__.py\\": \\"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\",\\n    \\"src/casmi/validate/__init__.py\\": \\"e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855\\",\\n    \\"src/casmi/validate/analog_index_evidence.py\\": \\"2dac213fd7629937ea3198cd7cc4c36ee1daab92439e832bbabcbefec0952e3a\\",\\n    \\"src/casmi/validate/budget_features.py\\": \\"8d2af4b739d4fc058a18b0c8c3f9c7619fd7e742396344f2b87bf893a926d788\\",\\n    \\"src/casmi/validate/cohort_bootstrap.py\\": \\"37ee133721adcc9b786ed35dbc8855082fe387159960ca41a4a96379a3c34bb1\\",\\n    \\"src/casmi/validate/control_alignment.py\\": \\"4996027acd3d4c1b79c2d4e3351e325ad6278323c340af36ad0e63a2bd722cfb\\",\\n    \\"src/casmi/validate/coverage_budget.py\\": \\"f3ae268d7cd4ecc2f5448ffde91b847c2cd08941e7b377a6c775bee47f3e7dd8\\",\\n    \\"src/casmi/validate/error_slices.py\\": \\"ad0a7df1c02d2dc867e1a80e027ed8d665fad64b694a1ddb1ae04d0169e8baf1\\",\\n    \\"src/casmi/validate/external_population.py\\": \\"38cebac369c8780ca794ab24e9b3f37e703c71047851bcd104e006d9a8458f65\\",\\n    \\"src/casmi/validate/fingerprint_evidence.py\\": \\"1d2d123d68a2f1a8005179c5b95ac5d3437e83a7c9d58df7276e77eb13b5b3da\\",\\n    \\"src/casmi/validate/knockout.py\\": \\"7c9e54a77d9d7d7cef98b0aadcdba5b918427c053af655c7ffe3e6e3ef01d238\\",\\n    \\"src/casmi/validate/massbank_records.py\\": \\"294c7061cd3c325a44e9e58e32cabfc1139dc543f1ad0182545a94ba3aa69a1a\\",\\n    \\"src/casmi/validate/metric.py\\": \\"da51b1ee98461facb6de18a2ad3ff90ca31d5e6d4a991c520ef894d817474ef1\\",\\n    \\"src/casmi/validate/overlap_audit.py\\": \\"23ef2d7108c2a1f7ed93b3a469af0a6dd853c4d8850e270958e134d538f3df05\\",\\n    \\"src/casmi/validate/paired_bootstrap.py\\": \\"42543ed9f11a684794c10aea17b3abe1c5e9cdbc5ec62816486e2be96809246a\\",\\n    \\"src/casmi/validate/rank_inputs.py\\": \\"c6faf51be68b80ba2f50bc5324ab802ffd1ebd9134eb71005d8dcb9123f8de62\\",\\n    \\"src/casmi/validate/ranked_slots.py\\": \\"3dec0248951c1d54ff4a91fafe6173b2e19854739be48041aca4c5ea681b53e0\\",\\n    \\"src/casmi/validate/run_alignment.py\\": \\"bcaed43318285185a8572c7ae2ce2fe53e341814821d9a2bfeaa92ba7f70860e\\",\\n    \\"src/casmi/validate/scaffold_study.py\\": \\"5005a72b5c4c8951dddc24e93f65d3d26683ce3edcf9dd3c3667cf7b95c46afc\\",\\n    \\"src/casmi/validate/simultaneous_ci.py\\": \\"990b45160da2287eb9debfa4c6e2b96bc5bd6e9fe3b27949877bb05c442159da\\",\\n    \\"src/casmi/validate/source_manifest.py\\": \\"d3a37d2b4879c0735e6998c0d40cc011cb4a6a9c5a1e8325be9443c536e262a0\\",\\n    \\"src/casmi/validate/spectral_controls.py\\": \\"12a49aeb2c72899993a178bda6e505ae382d5f0771df4a2af169343466d9828e\\",\\n    \\"src/casmi/validate/spectrum_overlap.py\\": \\"ed06679f55192f44b1f8db3596ec4be8ff279fce2e813544e5464e015b984aec\\",\\n    \\"scripts/run_external_audit.py\\": \\"2c1ead8e16915a3624f0170a8504b6501fb0ab46ad3347172930dda1b411ba3f\\"\\n  },\\n  \\"execution\\": {\\n    \\"platform\\": \\"Kaggle\\",\\n    \\"internet_enabled\\": false,\\n    \\"gpu_enabled\\": false,\\n    \\"model_training\\": false,\\n    \\"submitted_predictions\\": false,\\n    \\"protocol_registered_commit\\": \\"6be58a5\\"\\n  }\\n}\\n"}')
work=Path('/tmp/casmi_purged_external');code=work/'code';original_artifacts=work/'original-artifacts'
code.mkdir(parents=True,exist_ok=False);original_artifacts.mkdir()
for name,text in sources.items():
    path=code/name;path.parent.mkdir(parents=True,exist_ok=True);path.write_text(text,encoding='utf-8')
def unique(pattern):
    paths=sorted(Path('/kaggle/input').rglob(pattern))
    if len(paths)!=1:raise ValueError('Required declared input missing or ambiguous')
    return paths[0]
train=unique('train.parquet');coconut=unique('coconut_csv_lite-09-2026.csv');pcl=unique('PubChemLite_CCSbase_*.csv')
external=unique('gnps_cc0.parquet');external_manifest=unique('external_input_manifest.json')
wheel=unique(f'rdkit-2026.3.3-cp{sys.version_info.major}{sys.version_info.minor}-*.whl')
subprocess.run([sys.executable,'-m','pip','install','--no-deps','--force-reinstall',str(wheel)],check=True)
purged_data=work/'purged-data';artifacts=work/'filtered-artifacts'
env={**os.environ,'CASMI_DATA':str(train.parent),'CASMI_ARTIFACTS':str(original_artifacts),
    'CASMI_COCONUT':str(coconut),'CASMI_PUBCHEMLITE':str(pcl),'CASMI_POOL_TAG':'_purge_original',
    'CASMI_QUARANTINE_MASS_CONFLICTS':'1','CASMI_QUARANTINE_INVALID_STRUCTURES':'1',
    'CASMI_ORIGINAL_TRAIN':str(train),'CASMI_ORIGINAL_EXTERNAL':str(external),
    'CASMI_ORIGINAL_EXTERNAL_MANIFEST':str(external_manifest),'CASMI_PURGED_DATA':str(purged_data),
    'PYTHONPATH':str(code/'src')+os.pathsep+str(code),'PYTHONUNBUFFERED':'1','PYTHONHASHSEED':'20261028',
    'OMP_NUM_THREADS':'2','OPENBLAS_NUM_THREADS':'2','MKL_NUM_THREADS':'2','NUMBA_NUM_THREADS':'2'}
print('Original-snapshot novelty feasibility failed and remains preserved. Separate purged-training experiment begins.',flush=True)


In [ ]:
def run_stage(name,label=None):
    print('Stage '+(label or name),flush=True)
    log=work/((label or name)+'.log')
    with log.open('w',encoding='utf-8') as output:
        process=subprocess.Popen([sys.executable,str(code/'scripts'/name)],env=env,stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            output.write(line);output.flush()
            if re.match(r'^(Prepared |Trained fresh |Fitted prespecified |Sealed all |Completed sealed |External audit scanned |row group [0-9])',line):
                print(line.strip(),flush=True)
        status=process.wait()
    if status:
        text=log.read_text(encoding='utf-8',errors='replace')
        errors=re.findall(r'^([A-Za-z.]*Error|AssertionError):',text,flags=re.M)
        locations=re.findall(r'File "'+re.escape(str(code))+r'/([A-Za-z0-9_/]+\.py)", line ([0-9]+)',text)
        failure={'status':'FAILED_CLOSED','stage':label or name,'exception_type':errors[-1] if errors else 'Unknown',
                 'last_project_location':list(locations[-1]) if locations else None}
        Path('/kaggle/working/purged_external_failure.json').write_text(json.dumps(failure,indent=2))
        raise RuntimeError('Purged-training stage failed closed; sanitized failure evidence exported')
for name in ['build_structures.py','build_store.py']:
    run_stage(name,'original-'+name)
run_stage('prepare_purged_external.py')
artifacts.mkdir()
env.update(CASMI_DATA=str(purged_data),CASMI_ARTIFACTS=str(artifacts),CASMI_POOL_TAG='_purged_external',
    CASMI_EXTERNAL_INPUT=str(purged_data/'selected_external.parquet'),
    CASMI_EXTERNAL_MANIFEST=str(purged_data/'selected_external_manifest.json'),
    CASMI_PURGE_REPORT=str(purged_data/'purge_report.json'),
    CASMI_PROTECTED_POPULATION=str(purged_data/'protected_population.json'),
    CASMI_STUDY_PROTOCOL='protocols/purged-external-v1.json',
    CASMI_EXECUTION_PROTOCOL='protocols/purged-external-v1-execution.json',
    CASMI_EXTERNAL_PROTOCOL='protocols/external-source-v3.json',
    CASMI_EXTERNAL_STUDY_OUTPUT='/kaggle/working/purged_external_study.json')
for name in ['build_structures.py','build_store.py','build_pool.py','build_fingerprints.py','run_external_c3_study.py']:
    run_stage(name,'filtered-'+name)
output=Path('/kaggle/working/purged_external_study.json');report=json.loads(output.read_text())
def file_record(path):
    with path.open('rb') as stream:digest=hashlib.file_digest(stream,'sha256').hexdigest()
    return {'bytes':path.stat().st_size,'sha256':digest}
report['raw_inputs']={'original_train.parquet':file_record(train),'purged_train.parquet':file_record(purged_data/'train.parquet'),
    'coconut.csv':file_record(coconut),'pubchemlite.csv':file_record(pcl),'gnps_cc0.parquet':file_record(external)}
report['source_code_sha256']={n:hashlib.sha256(t.encode()).hexdigest() for n,t in sources.items()}
report['execution']={'platform':'Kaggle','competition_source':'enveda-CASMI26-molecule-id-mass-spectra',
    'dataset_sources':['senanuretin/casmi-coconut-b', 'senanuretin/casmi-pubchemlite-b', 'metric/rdkit-2026-3-3-wheel', 'senanuretin/casmi26-gnps-cc0-expanded'],'pretrained_models_attached':False,'gpu_enabled':False,'internet_enabled':False,
    'submitted_predictions':False,'protocol_registered_commit':'1b7a6cc','new_model_test_results_before_registration':False,
    'source_amendment_sha256':'73e0bcc63b837395e27a2433b3bef650ba8cb64f96d53ed4610bae3c83bdbdbe'}
report['installed_packages']=dict(sorted((d.metadata['Name'],d.version) for d in importlib.metadata.distributions()
    if d.metadata['Name'] and re.fullmatch(r'[A-Za-z0-9_.+-]+',d.version)))
output.write_text(json.dumps(report,indent=2,allow_nan=False)+'\n')
print('Completed separately registered purged-training external-family study; original novelty failures remain explicit.',flush=True)
